In [ ]:
#all imports here

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



#W1 D2 T1

'''
Build a daily DataFrame where each row splits the session in two:

**Morning window (observation): 03:00–09:55 ET**
- `am_open`, `am_high`, `am_low`, `am_close`, `am_bars`

**Afternoon window (target): 10:00–16:00 ET**
- `pm_high`, `pm_low`

Then:
- `pm_range` = `pm_high` − `pm_low`   ← **this is the target**

Keep only days where both windows have bars.
'''


In [ ]:
#W1 D2 T1
#FILTERING THE SESSION INTO 2 SESSIONS - EU (3-9:55 ET) + US (10:00 ET - 15:55 ET)

df = pd.read_csv('xauusd_m5_et.csv')
print(df.head())

df['et_time'] = pd.to_datetime(df['et_time'])
df['trade_date'] = pd.to_datetime(df['trade_date'])

eu_session = df[(df['et_time'].dt.hour >= 3) & (df['et_time'].dt.hour <= 9)]
eu_session['candle_range'] = eu_session['high'] - eu_session['low']
eu_session = eu_session.groupby(['trade_date']).agg(
    eu_open = ('open', 'first'),
    eu_high = ('high', 'max'),
    eu_low = ('low', 'min'),
    eu_close = ('close', 'last'),
    eu_bars = ('et_time', 'count'),
    eu_bar_rng_mean = ('candle_range', 'mean')
).reset_index()
eu_session.head()

us_session = df[(df['et_time'].dt.hour >= 10) & (df['et_time'].dt.hour <= 15)]
us_session['candle_range'] = us_session['high'] - us_session['low']
us_session = us_session.groupby(['trade_date']).agg(
    us_open = ('open', 'first'),
    us_high = ('high', 'max'),
    us_low = ('low', 'min'),
    us_close = ('close', 'last'),
    us_bars = ('et_time', 'count'),
    us_bar_rng_mean = ('candle_range', 'mean')
).reset_index()
us_session.head()

us_session['us_range'] = us_session['us_high'] - us_session['us_low']
eu_session['eu_range'] = eu_session['eu_high'] - eu_session['eu_low']

df = us_session.merge(eu_session, on = 'trade_date')
df = df[(df['us_bars'] >= 0) & (df['eu_bars'] >= 0)]
df.head()

#W1 D2 T2

## Task 2 — Features, all from before 10:00 (25 min)

Build these on the daily frame. Every one must be knowable at 10:00 ET.

**From this morning:**
- 2a. `am_range` = am_high − am_low #done already
- 2b. `am_return_pct` = open-to-close of the morning, in percent
- 2c. `am_close_loc` = where am_close sits inside the morning range, 0.0 to 1.0
- 2d. `am_bar_rng_mean` = mean of (high − low) across the morning's 5-min bars

**From previous days:**
- 2e. `pm_atr14` = mean `pm_range` over the previous 14 days
- 2f. `am_atr14` = mean `am_range` over the previous 14 days
- 2g. `prev_pm_range` = yesterday's `pm_range`

**Normalised:**
- 2h. `am_range_norm` = `am_range` / `am_atr14`

Drop rows with NaN. Print the shape and the correlation of each feature with `pm_range`.



In [ ]:
#W1 D2 T2

df['eu_return_pct'] = (df['eu_close'] - df['eu_open']) / df['eu_open'] * 100
df['eu_close_loc'] = (df['eu_close'] - df['eu_low']) / (df['eu_high'] - df['eu_low'])
#To achieve this step, I actualy had to modify the earlier step, as the current aggrgation is on daily level
#This is how I've done it 

# eu_session = df[(df['et_time'].dt.hour >= 3) & (df['et_time'].dt.hour <= 9)]
# eu_session['candle_range'] = eu_session['high'] - eu_session['low']
# eu_session = eu_session.groupby(['trade_date']).agg(
#     eu_open = ('open', 'first'),
#     eu_high = ('high', 'max'),
#     eu_low = ('low', 'min'),
#     eu_close = ('close', 'last'),
#     eu_bars = ('et_time', 'count'),
#     eu_bar_rng_mean = ('candle_range', 'mean')
# ).reset_index()
# eu_session.head()


df['us_atr14'] = df['us_range'].shift(1).rolling(14).mean()
df['eu_atr14'] = df['eu_range'].shift(1).rolling(14).mean()
df['prev_us_range'] = df['us_range'].shift(1)
df['eu_range_norm'] = df['eu_range'] / df['eu_atr14']

df = df.dropna()
df.head()
df.shape

test_df = df.copy().drop(columns = ['us_open', 'us_high', 'us_low', 'us_close', 'us_bars', 'us_bar_rng_mean'])

print('KENDALL')
kendall = test_df.corr('kendall')
display(kendall)

print('PEARSON')
pearson = test_df.corr('pearson')
display(pearson)

print('SPEARMAN')
spearman = test_df.corr('spearman')
display(spearman)



#W1 D2 T3

Task 3 — Look at the data before modelling (20 min)

Visualisation and statistics come first. A model trained on a relationship you have not
looked at is a guess with extra steps.

**3a. Distribution of the target.** Histogram of `pm_range`. Is it symmetric or skewed?
Add vertical lines for mean and median.

**3b. Does the morning actually relate to the afternoon?** Scatter of `am_range` (x) vs
`pm_range` (y). Then set both axes to log scale. In a comment: did that make the
relationship easier or harder to read, and why?

**3c. Correlation heatmap.** `sns.heatmap` of the correlation matrix for all your features
plus `pm_range`, with values annotated. Which feature relates most strongly to the target?

**3d. Volatility clustering, visually.** Line plot of `pm_range` across the full period.
In a comment: do quiet and violent stretches cluster together, or alternate randomly?
This is the entire premise of today's task — check whether it actually holds.

**3e. Quintile table.** Bucket `am_range` into 5 quantiles with `pd.qcut`. Per bucket print
mean and median `pm_range`, plus the count. Is the progression monotonic?

**In a comment:** based on 3a-3e alone, before fitting anything — do you expect a model to
work here? Yes or no, and name the evidence.

In [ ]:
#W1 D2 T3

mean_us_range = df['us_range'].mean()
median_us_range = df['us_range'].median()

histplot = sns.histplot(
    df['us_range']
)


histplot.axvline(
    x = mean_us_range,
    color = 'r'
)
histplot.axvline(
    x = median_us_range,
    color = 'yellow'
)
plt.show()


eu_vs_us_scatterplot = sns.scatterplot(
    df,
    x = 'eu_range',
    y = 'us_range'
)
plt.show()



eu_vs_us_scatterplot = sns.scatterplot(
    df,
    x = 'eu_range',
    y = 'us_range'
)
eu_vs_us_scatterplot.set_xscale('log'),
eu_vs_us_scatterplot.set_yscale('log')
plt.show()
#QUITE HONESTLY, IT MAKES the results look much more distributed in the center of the chart,
#but it's completely useless as human brain cannot understand it properly - perhaps algos do, not sure


corr_heatmap = sns.heatmap(
    test_df.select_dtypes(include = 'number')
)
plt.show()

#no visible correlations, I've used test_df with data leakage features cut out alreaydy

plt.figure(figsize = (20, 10)) #increased the size to see more
us_range_lineplot = sns.lineplot(
    x = df[:1000]['trade_date'], #I also limited the visualization scope as it was very difficult to observe anything while looking at the full history
    y = df[:1000]['us_range']
)
plt.show()
#volatility seems to cluster and have a wave structure, but high-volatility periods seem to be less popular as well


volatility_buckets = pd.qcut(df['eu_range'], 5)
print(volatility_buckets.head())

## Task 4 — Is the relationship statistically real? (15 min)

A visible pattern and a statistically supported one are different claims.

**4a.** Test whether `pm_range` is normally distributed (`scipy.stats.normaltest`).
State the null hypothesis in your own words and what the p-value means here.

**4b.** Split days into two groups: `am_range` above vs below its median. Test whether
`pm_range` differs between the groups. Pick the correct test based on 4a and justify it.

**4c.** Report the correlation between `am_range` and `pm_range` two ways: Pearson and
Spearman. Explain why the two numbers differ.

**In a comment:** if 4b came back with p = 0.30, would you still build the model? What would
that tell you that the scatter plot did not?

In [ ]:
from scipy.stats import normaltest, mannwhitneyu
import scikit_posthocs as sp

#H0: It's a normal distribution
#H1: It's NOT a normal distribution
us_range_distribution = normaltest(df['us_range'].sample(200))
print(us_range_distribution[1])
#close to 0, H0 rejected, not a normal distribution
#In that case, we'd use a nonparametric test - MannWhitney U, Kruskal Wallis

median_eu_range = df['eu_range'].median()
below_median_eu_range = df['us_range'][df['eu_range'] <= median_eu_range].sample(200)
above_median_eu_range = df['us_range'][df['eu_range'] >= median_eu_range].sample(200)

mw_test = mannwhitneyu(below_median_eu_range, 
                       above_median_eu_range
                       )

display(mw_test[1])
#picking samples instead of full DFs

#H0: There's no meaningful difference between the groups
#H1: There's a meaningful difference between the groups.
sp.posthoc_dunn(
    [below_median_eu_range, above_median_eu_range]
)


'''
3.380588784020326e-77
np.float64(5.2039612781732755e-21)
1	2
1	1.000000e+00	5.182604e-21
2	5.182604e-21	1.000000e+00
'''

#The p-values are below 0, there seems to be a statistically meaningfull difference between the two groups





You cannot judge a model without knowing what costs nothing.

Split the data chronologically, 80/20. On the **test** set only, compute MAE and R² for:

- 5a. Always predict the training-set **mean** of `pm_range`
- 5b. Always predict the training-set **median**
- 5c. Predict `prev_pm_range` (yesterday's value)
- 5d. Predict `pm_atr14` (the 14-day rolling mean)

Print all four in one table, sorted by MAE.

**In a comment:** which baseline is hardest to beat, and why is that one the honest
benchmark rather than the mean?


I don't know what do you mean here - how AM I SUPPOSED to compute mae and R2 without training a model DUDE?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score


test_df = test_df.select_dtypes(include = 'number')
X = test_df.drop(columns = ['us_range'])
y = test_df['us_range']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    shuffle = False
)

#I don't know what do you mean here - how AM I SUPPOSED to compute mae and R2 without training a model DUDE?



## Task 6 — First real model (20 min)

- 6a. Fit a `LinearRegression` on the training set, predict on test.
- 6b. Report MAE, RMSE, R² on test — and the same three on train.
- 6c. Compare against the best baseline from Task 5. Better or worse, and by how much?
- 6d. Plot predicted vs actual as a scatter, with a diagonal reference line.

**In comments:**
- Does linear regression need feature scaling here? Answer yes or no and justify it.
- What does the train-vs-test gap tell you?
- If the model beats the mean baseline but loses to `pm_atr14`, what is your verdict?

In [ ]:
#W1 D2 T6

from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

model = LinearRegression()
model.fit(X_train, y_train)

train_y_pred = model.predict(X_train)
train_mae = mean_absolute_error(y_train, train_y_pred)
train_r2 = r2_score(y_train, train_y_pred)
train_rmse = root_mean_squared_error(y_train, train_y_pred)

y_pred = model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)
test_rmse = root_mean_squared_error(y_test, y_pred)

print(f'Train values: MAE: {train_mae}, R2 {train_r2}, RMSE {train_rmse}')
print(f'Test values: MAE: {test_mae}, R2 {test_r2}, RMSE {test_rmse}')

## Task 7 — Where does it fail? (15 min)

Using the test-set predictions from Task 6:

- 7a. Split the actual `pm_range` into 5 quantile buckets.
- 7b. For each bucket: mean actual, mean predicted, MAE.
- 7c. State in one sentence where the model is weakest.

**In a comment:** is the error evenly spread, or concentrated? What would that mean for
someone using this to size a position?

In [ ]:
print(y_pred)



#W1 D3 T1 

BASELINE 

A baseline is not a model

A baseline is a rule so simple it needs no training. You write down what it would have
predicted, then measure how wrong it was - exactly like a model.

Why it matters: MAE 14.89 means nothing on its own. If guessing gives 18, your model
earned its place. If guessing gives 12, your model is worse than nothing.

**Worked example** - "always predict the training mean":

```python
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

# one number, repeated for every test row
pred = np.full(len(y_test), y_train.mean())

print(mean_absolute_error(y_test, pred), r2_score(y_test, pred))
```

That is the whole idea. No `.fit()`, no learning - just a constant.

When the rule is a column you already have, it is even shorter:

```python
pred = X_test['us_atr14']          # "tomorrow looks like the last 14 days"
print(mean_absolute_error(y_test, pred))


In [ ]:
'''## Task 1 - Four baselines (15 min)

`prev_day` and `atr14` are **columns you already have** - use them directly as predictions,
one value per test row. Only `train_mean` and `train_median` are single numbers repeated.

Build one table with these five rows, sorted by MAE:

| name | prediction |
|---|---|
| `train_mean` | `np.full(len(y_test), y_train.mean())` |
| `train_median` | same, with median |
| `prev_day` | `X_test['prev_us_range']` |
| `atr14` | `X_test['us_atr14']` |
| `my_model` | yesterday's `y_pred` |

Columns: MAE, RMSE, R2. Then add `vs_best_baseline` = MAE minus the best *baseline* MAE
(excluding your model).

**In a comment:**
- Which baseline is hardest to beat, and why is that one - not the mean - the honest benchmark?
- `train_mean` will have a negative R2. What does a negative R2 actually mean?

'''

In [ ]:
#W1 D3 T1

from sklearn.metrics import mean_absolute_percentage_error

train_mean = y_train.mean()
train_median = y_train.median()
prev_day = X_test['prev_us_range']
atr14 = X_test['us_atr14']
my_model = y_pred

pred_mean = np.full(len(y_test), train_mean)
pred_median = np.full(len(y_test), train_median)

rows = []
for name, prediction in [
    ('train_mean', pred_mean),
    ('train_median', pred_median),
    ('prev_day', prev_day),
    ('atr14', atr14),
    ('my_model', my_model)
]:
    rows.append({
        'name': name,
        'MAE': mean_absolute_error(y_test, prediction),
        'R2': r2_score(y_test, prediction),
        'RMSE': root_mean_squared_error(y_test, prediction),
        'MAPE': mean_absolute_percentage_error(y_test, prediction)
    }
    )

rows_df = pd.DataFrame(rows)
rows_df

#I've also added MAPE so it's a bit clearer for me.
#It's difficult to say, which will be the hardest baseline to beat
#negative r2 score signifies a VERY BAD performance of the model, not sure how to interpret it here in this context, but it feels like my model performs a lot well, 
#and it's able to explain 47% of the variance, if I'm not mistaken. It's not perfect, I'd rather see numbers like 70 here honestly, but it was a quickly trained model

#model wins in every single case now, it looks like it.
#I guess atr14 is the best to compare with, as it seems to be the closest to the model's current performance + it trims itself to the current market performance
#I'm thinking if we could somehow use rolling atr14 as the baseline comparator, but wondering if that's possible.


## Concept 2 - Average error hides where the model fails

MAE is one number for the whole test set. A model can be excellent on quiet days and
useless on violent ones and still show a decent average - and violent days are exactly
the ones that cost money.

The fix: bucket the **actual** values, then compute the error inside each bucket.

**Worked example:**

```python
res = pd.DataFrame({'actual': y_test, 'pred': y_pred})
res['bucket'] = pd.qcut(res['actual'], 5, labels=['Q1','Q2','Q3','Q4','Q5'])

res.groupby('bucket', observed=True).apply(
    lambda g: pd.Series({
        'mean_actual': g['actual'].mean(),
        'mean_pred':   g['pred'].mean(),
        'mae':         (g['actual'] - g['pred']).abs().mean(),
        'n':           len(g),
    })
)

In [ ]:
#W1 D3 T2

'''## Task 2 - Where does the model fail? (20 min)

- 2a. Build that table for your model's test predictions.
- 2b. Add a column `bias` = `mean_pred` - `mean_actual`. Positive means over-prediction.
- 2c. Do the same for the `atr14` baseline, so you can compare per bucket.

**In comments:**
- Is the error evenly spread across buckets, or concentrated? Where exactly?
- Look at `bias` in Q1 and Q5. Is there a pattern, and can you explain why it happens?
- Someone sizes a position using this model. Which bucket should worry them most?'''

In [ ]:
#W1 D3 T2

quantiles_df = pd.DataFrame({'actual': y_test, 'pred': y_pred})
quantiles_df['bucket'] = pd.qcut(quantiles_df['actual'], 5, labels = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

error_quantiles_df = quantiles_df.groupby('bucket').agg(
    mean_actual = ('actual', 'mean'),
    mean_pred = ('pred', 'mean'),
    n = ('bucket', 'count')
)

error_quantiles_df['mae'] = abs(error_quantiles_df['mean_actual'].abs() - error_quantiles_df['mean_pred'].abs())


#Seems that Q5 predictions are the absolute killer, which ruin our stats.
#If we'd be able to identify Q5 somehow earlier, we'd have a very good predictive power - at least that's what I think instinctively


atr_df = pd.DataFrame({'atr': atr14, 'actual': y_test, 'pred': y_pred})
atr_df['bucket'] = pd.qcut(atr_df['atr'], 5, labels = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

error_atr_df = atr_df.groupby('bucket').agg(
    mean_atr = ('atr', 'mean'),
    mean_actual = ('actual', 'mean'),
    mean_pred = ('pred', 'mean'),
    n = ('bucket', 'count')
)

error_atr_df['mae'] = abs(error_atr_df['mean_pred'] - error_atr_df['mean_atr'])

error_quantiles_df.head()

#Similar situation here, the high ATR periods seem to be very difficult to predict.
#I reckon it's because the data's mostly trained on lower ATR periods (80% of data is in the Q1-Q4 region, which is quite obvious), and it's hard to predict the outliers


## Concept 3 - Metrics can disagree, and that tells you something

Yesterday your numbers were:

```
train:  MAE  4.06   R2 0.323
test:   MAE 14.89   R2 0.490
```

MAE got 3.7x worse on test, but R2 got **better**. That looks impossible.

It is not. R2 measures error relative to the variance *of that particular set*. Between
train and test, gold went from ~2094 to ~4194 and the mean `us_range` went from 11.1 to
40.3. Everything got bigger - the errors and the variance both.

So: **MAE is in points and comparable across sets. R2 is relative and is not.**

In [ ]:
#W1 D3 T3

'''
## Task 3 - Diagnose the regime shift (15 min)

- 3a. Print mean and std of `us_range` separately for train and test.
- 3b. Same for `eu_close` (the price level).
- 3c. Line plot of `us_range` over the whole period, with a vertical line at the split point.
- 3d. Compute the model's MAE **as a percentage of the mean actual** in each set:
     `MAE / y.mean() * 100`.

**In a comment:** after 3d, is the model actually 3.7x worse on test, or was that an
artefact of scale? What does this suggest about predicting a raw price range across years?

'''


y_train.head()
y_test.head()

train_mean, train_std = (y_train.mean(), y_train.std())
print(f'Train values: {train_mean}, {train_std}')

test_mean, test_std = (y_test.mean(), y_test.std())
print(f'Test values: {test_mean}, {test_std}')


train_euc_mean, train_euc_std = (X_train['eu_close'].mean(), X_train['eu_close'].std())
print(f'Train EU close values: {train_euc_mean}, {train_euc_std}')

test_euc_mean, test_euc_std = (X_test['eu_close'].mean(), X_test['eu_close'].std())
print(f'Test EU close values: {test_euc_mean}, {test_euc_std}')


df.head()
split_point = df['trade_date'][int(round(0.8*len(df)))]
print(split_point)

plt.figure(figsize=(20, 10))
lineplot = sns.lineplot(
     df,
     x = 'trade_date',
     y = 'us_range'
)
plt.axvline(
     x = split_point,
     color = 'yellow'
)
plt.show()

#WAPE - ważona średnia, która dużo lepiej radzi sobie na prawoskośnych rozkładach.

train_pct = train_mae / y_train.mean() * 100
test_pct  = test_mae  / y_test.mean()  * 100
print(f'train WAPE: {train_pct:.1f}%   test WAPE: {test_pct:.1f}%')



In [ ]:
#W1 D3 T4

'''## Task 4 - One fix, measured (15 min)

If the problem is scale, model a scale-free target instead.

- 4a. Build `us_range_norm` = `us_range` / `us_atr14`. This is "today's range relative to
     recent days" - roughly 1.0 on a typical day.
- 4b. Retrain the same `LinearRegression` on this new target. Same features, same split.
- 4c. Report train and test MAE and R2.
- 4d. Compare train-vs-test on the normalised target against yesterday's raw-target run.

**In a comment:** did normalising the target reduce the train/test gap? If yes, what does
that confirm? If no, what else could explain it?'''


df.head()
df['us_range_norm'] = df['us_range'] / df['us_atr14']

y = df['us_range_norm']
X = df.drop(columns = ['trade_date', 'us_open', 'us_high', 'us_low', 'us_close', 'us_bars', 'us_bar_rng_mean', 'us_range', 'us_range_norm'])


X_train, X_test, y_train, y_test = train_test_split(
     X,
     y,
     test_size = 0.2,
     shuffle = False
)
model = LinearRegression()
model.fit(X_train, y_train)



train_pred = model.predict(X_train)
train_mae = mean_absolute_error(y_train, train_pred)
train_r2 = r2_score(y_train, train_pred)
train_rmse = root_mean_squared_error(y_train, train_pred)
train_mape = mean_absolute_percentage_error(y_train, train_pred)
print(f'Train metrics: MAE: {train_mae}, R2_SCORE: {train_r2}, RMSE: {train_rmse}, train_mape {train_mape}')


test_pred = model.predict(X_test)
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)
test_mape = mean_absolute_percentage_error(y_test, test_pred)
print(f'Test metrics: MAE: {test_mae}, R2_SCORE: {test_r2}, RMSE: {test_rmse}, train_mape {test_mape}')

'''Yesterday's metrics with absolute us_range

Train values: MAE: 4.026452759244498, R2 0.3344725256666109, RMSE 6.254345342145135
Test values: MAE: 14.993502938684792, R2 0.4709428259030948, RMSE 27.80561074625558
'''

#Today's results after using normalized us_range as a target:


# Train metrics: MAE: 0.38918616685539076, R2_SCORE: 0.0825434706588205, RMSE: 0.6432873326399401, train_mape 0.4271305702463293
# Test metrics: MAE: 0.6506124472685888, R2_SCORE: -1.039789597984146, RMSE: 0.9953337405214719, train_mape 0.8526300387331389

'''
It doesn't seem like we've achieved better results with this approach - 
perhaps normalizing that creates an effect of the outliers from the Q5 greatly increase the mean/median values, 
which maybe brings data closer to the Q5 data, but at the same time make it perform slightly worse on the most majority of data (Q1-Q4). 

That's my speculation. I've seen that this dataset has a very big right skew, which means some outliers are really extremely big.
'''



#W1 D4 T1 

## Reminder 1 - MAE is per row, then averaged

Yesterday you wrote:

```python
mae = abs(mean_actual - mean_pred)     # this is bias, not MAE
```

The difference matters: if a bucket has errors of +20 and -20, the difference of means is
0 while the true MAE is 20. Compute the error **per row first**, then group:

```python
res['abs_error'] = (res['actual'] - res['pred']).abs()   # per row
res['error']     = res['pred'] - res['actual']           # per row, keeps sign

res.groupby('bucket', observed=True).agg(
    mae  = ('abs_error', 'mean'),
    bias = ('error', 'mean'),
)
```

`bias` is a real term - mean error. It tells you the *direction* of the mistake.
`mae` tells you the *size*. You need both.




## Reminder 2 - WAPE, one more time

You have two error numbers from different periods and want to know which is worse.
You cannot compare them directly if the scales differ.

```
train MAE  4.06   with typical values around 11
test  MAE 14.89   with typical values around 40
```

Divide each by what it is predicting:

```python
4.06 / 11.06 * 100 = 36.7%
14.89 / 40.32 * 100 = 36.9%
```

Same accuracy. The MAE jump was scale, not failure.

**Rule:** whenever you compare error across sets with different scales, convert to percent
first. That is all WAPE is.

In [ ]:
'''

## Task 1 - Build the new target (15 min)

Predict `eu_range` using only data available **before today's morning session opens**.

Features - all lagged by one day:
- `prev_eu_range`, `prev_us_range`
- `eu_atr14`, `us_atr14`  (you already have these, they are already shift-based)
- `prev_eu_close_loc`, `prev_eu_return_pct`

Same 80/20 chronological split.

**In a comment:** which of your existing features would leak here, and why? Name at least
two and say exactly what they reveal.
'''


df['prev_eu_range'] = df['eu_range'].shift(1)
df['prev_eu_close_loc'] = (df['eu_close'].shift(1) - df['eu_low'].shift(1)) / (df['eu_high'].shift(1) - df['eu_low'].shift(1))
df['prev_eu_return_pct'] = (df['eu_close'].shift(1) - df['eu_open'].shift(1)) / df['eu_open'].shift(1) * 100
df.head()
filtered_df = df.drop(columns = ['trade_date', 'us_open', 'us_high', 'us_low', 'us_close', 'us_bars', 
                                 'us_bar_rng_mean', 'us_range', 'eu_open', 'eu_high', 
                                 'eu_low', 'eu_close', 'eu_bar_rng_mean', 'eu_bars',
                                 'eu_return_pct', 'eu_close_loc', 'eu_range_norm', 'us_range_norm'])

filtered_df = filtered_df.dropna()
filtered_df.head()

X = filtered_df.drop(columns = ['eu_range'])
y = filtered_df['eu_range']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    shuffle = False
)

X.head()
#Currently no features leak, I've dropped all the leakages in filtered_df, it's self-explanatory

In [ ]:
'''## Task 2 - Baselines, then the model (20 min)

- 2a. Four baselines on the test set: `train_mean`, `train_median`, `prev_eu_range`, `eu_atr14`.
     One table, MAE + R2, sorted by MAE.
- 2b. Fit `LinearRegression`. Add it to the table.
- 2c. WAPE for train and test.

**In comments:**
- Which baseline won this time? Same as yesterday or different?
- Does your model beat it? By how much, in percent?
- Compare the train/test WAPE gap here against yesterday's 36.7 / 36.9. Wider or narrower,
  and what would explain it?'''
  
rows = []
for name, prediction in [
     ('train_mean', np.full(len(y_test), y_train.mean())),
     ('train_median', np.full(len(y_test), y_train.median())),
     ('prev_eu_range', X_test['prev_eu_range']),
     ('eu_atr14', X_test['eu_atr14'])
]:
     rows.append(
          {
          'name': name,
          'MAE': mean_absolute_error(y_test, prediction),
          'R2': r2_score(y_test, prediction),
          }
     )

rows = pd.DataFrame(rows)
rows.head()

#It's similar, it seems like eu_atr14 is a pretty solid metric as a baseline, it self adjusts itself to the current market dynamics


model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
model_mae = mean_absolute_error(y_test, y_pred)
model_r2 = r2_score(y_test, y_pred)

rows.loc[len(rows)] = ({
     'name': 'model_predictions',
     'MAE': model_mae,
     'R2': model_r2
})

rows.head()

'''	name	MAE	R2
0	train_mean	39.893995	-0.691469
1	train_median	42.069544	-0.780908
2	prev_eu_range	27.668226	-0.080938
3	eu_atr14	23.976252	0.159418
4	model_predictions	21.242674	0.309893'''

#It seems like we're getting very close to the eu_ATR14 IN TERMS of MAE, but still perform about 2pp better + 0.15pp better in terms of predictions in the current state.
#Not bad I think.


train_wape = 39.89 / y_train.mean() * 100
test_wape = 21.24 / y_test.mean() * 100 #I manually took the MAE values here for convenience, since I didn't create variables for them - IMO it's not a big deal
print(train_wape, test_wape)

In [ ]:
''' Task 3 - Buckets and bias (20 min)

- 3a. Bucket the test set by actual `eu_range` into 5 quantiles.
- 3b. Per bucket: `mean_actual`, `mean_pred`, `mae`, `bias`, `n` - using the per-row method above.
- 3c. Plot `bias` per bucket as a bar chart, with a horizontal line at zero.

**In comments:**
- Yesterday the bias went from +6.6 in Q1 to -36.3 in Q5. Does the same pattern appear here?
- Why does a linear model do this? Answer in one sentence.
- If you were sizing a position off this model, which bucket is dangerous and why?'''


quantiles_df = pd.DataFrame({'actual': y_test, 'pred': y_pred})
quantiles_df['bucket'] = pd.qcut(quantiles_df['actual'], 5, ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

quantiles_df['abs_error'] = abs(quantiles_df['actual'] - quantiles_df['pred'])
quantiles_df['error'] = quantiles_df['actual'] - quantiles_df['pred']
quantiles_df.head()

quantiles_diff = quantiles_df.groupby('bucket').agg(
    mean_actual = ('actual', 'mean'),
    mean_pred = ('pred', 'mean'),
    mae = ('abs_error', 'mean'),
    bias = ('error', 'mean')
).reset_index()



plt.figure(figsize = (16, 9))
error_chart = sns.barplot(
    quantiles_diff,
    x = 'bucket',
    y = 'mae'
)
error_chart.set_title('MAE by EU range buckets for the model test results')
error_chart.axhline(color = 'r')
plt.show()


quantiles_diff.head()

In [ ]:
from sklearn.metrics import mean_absolute_error

mean_pred = np.full(len(y_test), y_train.median())
mean_actual = np.full(len(y_test), y_test.median())

train_mae = mean_absolute_error(y_test, mean_pred)
test_mae = mean_absolute_error(y_test, mean_actual)


#W3 D1 T1 


## Concept - your model cannot predict Q5, and that is not a bug

Every day this week the same pattern appeared:

```
Q1  bias +11.5    model over-predicts quiet days
Q5  bias -44.1    model under-predicts violent days
```

Linear regression minimises **squared error**, which means it fits a line through the
middle of the cloud. On a right-skewed target it cannot pass through both the bulk and the
tail, so it lands near the mean and misses both ends. This is called **regression toward
the mean** and it is a property of the method, not a mistake in your code.

You asked: *"if we could identify Q5 earlier, we'd have very good predictive power."*

That instinct has a name. The technique is **quantile regression**, and instead of
predicting the average outcome it predicts a chosen percentile.

**Ordinary regression** answers: *what is the expected range today?*
**Quantile regression** answers: *what range will not be exceeded on 90% of days like today?*

For stop placement, the second question is the useful one.

**Worked example:**

```python
from sklearn.linear_model import QuantileRegressor

# alpha=0 disables the regularisation; quantile=0.9 asks for the 90th percentile
q90 = QuantileRegressor(quantile=0.9, alpha=0)
q90.fit(X_train, y_train)
pred_q90 = q90.predict(X_test)

# how often did reality stay below the prediction?
coverage = (y_test <= pred_q90).mean()
print(f"{coverage:.1%} - should land near 90%")
```

`coverage` is the metric that matters here, not MAE. A q90 model that covers 89% of days is
working correctly even if its MAE looks worse than the ordinary model's.

In [ ]:
#W2 D1 T1

'''## Task 1 - Three quantiles (25 min)

Use the **`us_range` setup from Day 3** - predicting the afternoon from the morning.

- 1a. Fit three `QuantileRegressor` models: quantile 0.5, 0.8, 0.9. Same X, same split.
- 1b. For each, compute coverage on the test set: `(y_test <= pred).mean()`.
- 1c. Put the three coverages in a table next to their target quantiles.
- 1d. Also fit an ordinary `LinearRegression` and compute its coverage.

**In comments:**
- Are the coverages close to 0.5 / 0.8 / 0.9? If one is badly off, which and by how much?
- What is the ordinary model's coverage, and why is it near 50% rather than 90%?

Note: `QuantileRegressor` is slow on larger data. If it takes more than a minute, reduce
to `alpha=0` and fewer features, or say so and I will switch you to a faster approach.'''

from sklearn.linear_model import QuantileRegressor


eu_session.head()
eu_session.columns

df.head()
df = df.dropna()

y = df['us_range']
X = df.drop(columns = ['trade_date', 'us_open', 'us_high', 'us_low', 'us_close', 'us_bars',
       'us_bar_rng_mean', 'us_range', 'us_range_norm'])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    shuffle = False
)


q90_model = QuantileRegressor(quantile = 0.9)
q90_model.fit(X_train, y_train)
pred_q90 = q90_model.predict(X_test)

coverage_90 = (y_test <= pred_q90).mean()
print(f'{coverage_90} should land near 90%')


q80_model = QuantileRegressor(quantile = 0.8)
q80_model.fit(X_train, y_train)
pred_q80 = q80_model.predict(X_test)

coverage_80 = (y_test <= pred_q80).mean()
print(f'{coverage_80} should land near 80%')



q50_model = QuantileRegressor(quantile = 0.5)
q50_model.fit(X_train, y_train)
pred_q50 = q50_model.predict(X_test)

coverage_50 = (y_test <= pred_q50).mean()
print(f'{coverage_50} should land near 50%')




In [ ]:
#W2 D1 T2

'''## Task 2 - Does it fix Q5? (20 min)

- 2a. Build the bucket table again, this time with four prediction columns:
     ordinary, q50, q80, q90 - showing `mean_actual` and each model's mean prediction.
- 2b. For the Q5 bucket specifically: what does each model predict, against a mean actual
     of roughly 95?
- 2c. Compute MAE for each of the four, on the whole test set.

**In comments:**
- q90 will have the worst MAE. Explain why that is expected and not a failure.
- For someone placing a stop, which of the four numbers do they want, and why?
- Yesterday you said "it would be nice if we could mitigate Q5". Did this mitigate it?'''



quantiles_df = pd.DataFrame({'ordinary': y_pred, 'q50': pred_q50, 'q80': pred_q80, 'q90': pred_q90})
qs = ['q50', 'q80', 'q90']
for q in qs:
    q_name_mae = f'{q}_mae'
    q_name_bias = f'{q}_bias'
    quantiles_df[q_name_mae] = abs(quantiles_df['ordinary'] - quantiles_df[q])
    quantiles_df[q_name_bias] = quantiles_df['ordinary'] - quantiles_df[q]

quantiles_df['bucket'] = pd.qcut(quantiles_df['ordinary'], 5, ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
quantiles_df.head()

bucket_means = quantiles_df.groupby('bucket').agg(
    mean_actual = ('ordinary', 'mean'),
    q50_mean = ('q50', 'mean'),
    q50_mae = ('q50_mae', 'mean'),
    q50_bias = ('q50_bias', 'mean'),
    q80_mean = ('q80', 'mean'),
    q80_mae = ('q80_mae', 'mean'),
    q80_bias = ('q80_bias', 'mean'),
    q90_mean = ('q90', 'mean'),
    q90_mae = ('q90_mae', 'mean'),
    q90_bias = ('q90_bias', 'mean')
).reset_index()

bucket_means.head()

#As for the questions, I'll simply paste the results here


'''bucket	mean_actual	q50_mean	q50_mae	q50_bias	q80_mean	q80_mae	q80_bias	q90_mean	q90_mae	q90_bias
0	Q1	26.590864	18.601414	7.989450	7.989450	27.597410	2.194519	-1.006546	33.297246	6.706382	-6.706382
1	Q2	38.697974	22.848840	15.849134	15.849134	33.020842	5.758531	5.677132	39.210786	3.036578	-0.512812
2	Q3	51.301863	25.983851	25.318012	25.318012	36.932946	14.368916	14.368916	43.476422	7.899598	7.825441
3	Q4	61.933715	26.877338	35.056377	35.056377	37.494363	24.439352	24.439352	44.088573	17.845142	17.845142
4	Q5	98.404797	31.776297	66.628499	66.628499	41.824093	56.580703	56.580703	48.809575	49.595222	49.595222'''

#Q90 seems to work the best

In [ ]:
#W2 D1 T3

'''Print a small table for the last 10 test days:

```
date        actual   q50    q80    q90    within_q80?
```

**In a comment:** if you traded off `q80`, how many of those 10 days would have had the
range exceed your estimate? Is that roughly what you would expect?'''

small_df = df[['trade_date', 'us_range']].tail(10)
small_df['q50'] = pred_q50.mean()
small_df['q80'] = pred_q80.mean()
small_df['q90'] = pred_q90.mean()
small_df['within_q80'] = small_df['us_range'] <= small_df['q80']
small_df['within_q90'] = small_df['us_range'] <= small_df['q90']

exceeded_q80 = small_df['within_q80'].mean()
print(exceeded_q80)
exceeded_q90 = small_df['within_q90'].mean()
print(exceeded_q90)

#60% are within q80, 70% within q90


small_df.head(10)

In [ ]:
#W2 D2 WARMUP


quantiles_df = pd.DataFrame({'actual': y_test, 'pred': y_pred})
quantiles_df['bias'] = quantiles_df['actual'] - quantiles_df['pred']
quantiles_df['abs_diff'] = abs(quantiles_df['actual'] - quantiles_df['pred'])
quantiles_df['bucket'] = pd.qcut(quantiles_df['actual'], 5, ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

grouped_buckets = quantiles_df.groupby('bucket').agg(
  mean_actual = ('actual', 'mean'),
  mean_pred = ('pred', 'mean'),
  mae = ('abs_diff', 'mean'),
  bias = ('bias', 'mean'),
  n = ('bucket', 'count')
).reset_index()

grouped_buckets.head()

In [ ]:
#W2 D2 T2

'''## Task 1 - Rerun the quantiles on clean features (20 min)

**What we are predicting:** `us_range` - how far price travels during the US session
(10:00-16:00 ET), measured as high minus low.

**When we are predicting it:** at 10:00 ET, the moment the US session opens. So we may use
anything from this morning's EU session (03:00-09:55 ET) and anything from previous days,
but nothing from the US session itself.

**Use exactly these six features to avoid looking at raw prices which change during the years (from 1500s to 4000++):**

```python
X = ['eu_range',          # today's morning high - low, in points
     'eu_bar_rng_mean',   # mean high-low of the morning's 5-min bars
     'us_atr14',          # mean us_range over the previous 14 days
     'eu_atr14',          # mean eu_range over the previous 14 days
     'prev_us_range',     # yesterday's us_range
     'eu_range_norm']     # eu_range divided by eu_atr14

'''

from sklearn.model_selection import train_test_split
from sklearn.linear_model import QuantileRegressor, LinearRegression

X = df[['eu_range', 'eu_bar_rng_mean', 'us_atr14', 'eu_atr14', 'prev_us_range', 'eu_range_norm']]
y = df['us_range']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    shuffle = False
)

q50_model = QuantileRegressor(quantile = 0.5, alpha = 0, solver = 'highs')
q50_model.fit(X_train, y_train)
q50_pred = q50_model.predict(X_test)
q50_coverage = (y_test <= q50_pred).mean()

q80_model = QuantileRegressor(quantile = 0.8, alpha = 0, solver = 'highs')
q80_model.fit(X_train, y_train)
q80_pred = q80_model.predict(X_test)
q80_coverage = (y_test <= q80_pred).mean()

q90_model = QuantileRegressor(quantile = 0.9, alpha = 0, solver = 'highs')
q90_model.fit(X_train, y_train)
q90_pred = q90_model.predict(X_test)
q90_coverage = (y_test <= q90_pred).mean()


print(f'Q50 coverage: {q50_coverage}, Q80 coverage {q80_coverage}, Q90 coverage {q90_coverage}')

linear_reg_model = LinearRegression()
linear_reg_model.fit(X_train, y_train)
lin_pred = linear_reg_model.predict(X_test)

lin_coverage = (y_test <= lin_pred).mean()   # ← ile razy prawda zmieściła się pod predykcją
print(f'Lin coverage: {lin_coverage}')


In [ ]:
#W2 D2 T2

'''## Task 2 - Compare the four models properly (20 min)

Yesterday this table was built without `y_test` in it, so it measured how far the quantile
models sat from the ordinary model rather than from reality. Today reality goes in first.

Build a DataFrame with **five columns**: `actual`, `ordinary`, `q50`, `q80`, `q90`.
`actual` is `y_test`. The rest are the four prediction vectors (a vector here just means the
list of predictions, one per test row).

- 2a. Bucket by `actual` into 5 quantiles.
- 2b. Per bucket show: `mean_actual`, and the mean prediction of each of the four models.
- 2c. Across the whole test set, MAE of each of the four models against `actual`.

**In comments:**
- In the Q5 bucket (the most violent days), what does each model predict against the true
  mean? Which comes closest?
- q90 will have the worst overall MAE. Why is that expected rather than a failure?
- If you are setting a stop, which of the four numbers do you want and why?'''




quantiles_df = pd.DataFrame({'actual': y_test, 'ordinary': y_pred, 'q50': q50_pred, 'q80': q80_pred, 'q90': q90_pred})
quantiles_df['abs_diff_model'] = abs(quantiles_df['actual'] - quantiles_df['ordinary'])
quantiles_df['abs_diff_q50'] = abs(quantiles_df['actual'] - quantiles_df['q50'])
quantiles_df['abs_diff_q80'] = abs(quantiles_df['actual'] - quantiles_df['q80'])
quantiles_df['abs_diff_q90'] = abs(quantiles_df['actual'] - quantiles_df['q90'])
quantiles_df['bias_diff'] = quantiles_df['actual'] - quantiles_df['ordinary']
quantiles_df['bias_diff_q50'] = quantiles_df['actual'] - quantiles_df['q50']
quantiles_df['bias_diff_q80'] = quantiles_df['actual'] - quantiles_df['q80']
quantiles_df['bias_diff_q90'] = quantiles_df['actual'] - quantiles_df['q90']


quantiles_df['bucket'] = pd.qcut(quantiles_df['actual'], 5, ['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
results_df = quantiles_df.groupby('bucket').agg(
    mean_actual = ('actual', 'mean'),
    mean_pred = ('ordinary', 'mean'),
    model_mae = ('abs_diff_model', 'mean'),
    model_bias = ('bias_diff', 'mean'),
    q50_mean = ('q50', 'mean'),
    q50_mae = ('abs_diff_q50', 'mean'),
    q50_bias = ('bias_diff_q50', 'mean'),
    q80_mean = ('q80', 'mean'),
    q80_mae = ('abs_diff_q80', 'mean'),
    q80_bias = ('bias_diff_q80', 'mean'),
    q90_mean = ('q90', 'mean'),
    q90_mae = ('abs_diff_q90', 'mean'),
    q90_bias = ('bias_diff_q90', 'mean'),
).reset_index()

results_df.head()



In [ ]:
#W2 D2 T3

'''Print the last 15 test days as a table:

```
date        actual    q50     q80     q90    within_q80   within_q90
```

Careful here - yesterday this went wrong. `pred_q80` is a list with one prediction per test
row, so to get the last 15 you slice it: `pred_q80[-15:]`. Using `.mean()` would collapse
all of them into one number and every row would show the same value.

- 3a. Build the table.
- 3b. Count how many of the 15 days stayed within q80, and within q90.

**In a comment:** if you had used q80 as your stop distance on those 15 days, how many times
would the range have blown through it? Is that in line with what q80 promises?'''


test_df = pd.DataFrame(
    {'date': df['trade_date'][-15:],
     'actual': y_test[-15:],
     'q50': q50_pred[-15:],
     'q80': q80_pred[-15:],
     'q90': q90_pred[-15:]
    }
    )

test_df['within_q50'] = test_df['actual'] <= test_df['q50']
test_df['within_q80'] = test_df['actual'] <= test_df['q80']
test_df['within_q90'] = test_df['actual'] <= test_df['q90']

w50_count = test_df['within_q50'].mean()
w80_count = test_df['within_q80'].mean()
w90_count = test_df['within_q90'].mean()
print(F'Within q50: {w50_count}, Within q80: {w80_count}, within q90: {w90_count}')

test_df.head()

#Within q50: 0.6, Within q80: 0.8, within q90: 0.8666666666666667


WALK-FORWARD testing 

Here is what you have been doing all week:

```
[========== train 80% ==========][== test 20% ==]
 2021 ---------------------------> 2025 ------> 2026
```

You train once, test once, get one number. The problem: that number describes how the model
performed **on one specific stretch of 2025-2026**. If that stretch happened to be calm, or
happened to be violent, your result is coloured by it.

Walk-forward validation fixes this by repeating the exercise across the whole history:

```
[== train ==][test]
      [== train ==][test]
            [== train ==][test]
                  [== train ==][test]
```

The training window slides forward. Each time: fit on the window, predict the stretch
immediately after it, record the result, move on. Nothing ever trains on data that comes
after what it is tested on - the chronological rule still holds, just repeated.

Instead of one number you get several - one per fold (a fold is simply one train-then-test
cycle). And a spread of numbers tells you something a single number cannot:

- **q90 coverage = 88%, 89%, 87%, 90%** - the model is reliable, this is a real property
- **q90 coverage = 71%, 95%, 62%, 99%** - the average is still ~82%, but the model is
  unstable and the average is misleading

That second case is exactly what a single split would hide.

**Worked example - the loop structure:**

```python
train_size = 600      # how many days to train on
test_size  = 125      # how many days to predict, then move forward

results = []
start = train_size

while start + test_size <= len(df):
    train = df.iloc[start - train_size : start]      # the window before
    test  = df.iloc[start : start + test_size]       # the stretch right after

    model = QuantileRegressor(quantile=0.9, alpha=0, solver='highs')
    model.fit(train[features], train['us_range'])
    pred = model.predict(test[features])

    results.append({
        'test_start': test.index.min(),
        'coverage':   (test['us_range'] <= pred).mean(),
    })

    start += test_size        # slide the window forward by one test block

folds = pd.DataFrame(results)
```

Read the slicing carefully, because it is the part that trips people up:

- `df.iloc[start - train_size : start]` - the 600 rows **ending** at `start`
- `df.iloc[start : start + test_size]` - the 125 rows **beginning** at `start`

They touch but never overlap. `start += test_size` then moves everything forward by 125
days and the whole thing repeats.

In [ ]:
#W2 D3 T1 

'''## Task 1 - Walk-forward on q90 (25 min)

**What we are predicting:** `us_range` - the high minus low of the US session
(10:00-16:00 ET), in points.

**When:** at 10:00 ET. Features may use this morning's EU session and previous days,
nothing from the US session itself.

**Features - the same six as yesterday:**

```python
features = ['eu_range', 'eu_bar_rng_mean', 'us_atr14',
            'eu_atr14', 'prev_us_range', 'eu_range_norm']
```

Your dataframe needs to be sorted by date for any of this to be meaningful - check that
first with `df.index.is_monotonic_increasing` (or sort by `trade_date` if it is a column).

- 1a. Run the walk-forward loop with `train_size=600`, `test_size=125`, for q90.
- 1b. Print the resulting table: one row per fold, showing the test period start and the
     coverage achieved.
- 1c. Print the mean coverage across folds, and the min and max.'''


from sklearn.linear_model import QuantileRegressor

features = ['eu_range', 'eu_bar_rng_mean', 'us_atr14', 'eu_atr14', 'prev_us_range', 'eu_range_norm']

train_size = 300
test_size = 300

results = []
start = train_size
while start + test_size <= len(df):
     train = df.iloc[start - train_size : start]
     test = df.iloc[start : start + test_size]
     
     model = QuantileRegressor(quantile = 0.9, alpha = 0, solver = 'highs')
     model.fit(train[features], train['us_range'])
     pred = model.predict(test[features])
     
     results.append({
          'test_start': train.index.min(),
          'coverage': (test['us_range'] <= pred).mean()
     })
     
     start += test_size


folds = pd.DataFrame(results)
print(f'Mean coverage = {folds['coverage'].mean():.3f}, min coverage = {folds['coverage'].min():.3f}, max coverage = {folds['coverage'].max():.3f}')

folds.head()



In [ ]:
#W2 D3 T2

'''## Task 2 - All three quantiles, and the ordinary model (20 min)

- 2a. Extend the loop so each fold fits **four** models: q50, q80, q90, and an ordinary
     `LinearRegression`. Record coverage for each.
- 2b. Produce a summary table: one row per model, showing mean coverage, min, max, and the
     target it was aiming for (0.5 / 0.8 / 0.9 / and for the ordinary model, whatever it
     turns out to be).

**In comments:**
- Which of the four is most stable across folds? Which is least?
- Does any model systematically miss its target, rather than just varying around it?'''

from sklearn.linear_model import QuantileRegressor, LinearRegression

features = ['eu_range', 'eu_bar_rng_mean', 'us_atr14', 'eu_atr14', 'prev_us_range', 'eu_range_norm']
quantiles = [50, 80, 90]

train_size = 300
test_size = 300

results = []
start = train_size
while start + test_size <= len(df):
     train = df.iloc[start - train_size : start]
     test = df.iloc[start : start + test_size]
     
     lin_reg = LinearRegression()
     lin_reg.fit(train[features], train['us_range'])
     lin_reg_pred = lin_reg.predict(test[features])
     results.append({
     'model': 'linreg',
     'test_start': train.index.min(),
     'coverage': (test['us_range'] <= lin_reg_pred).mean()
     })
     
     for q in quantiles:
          name = f'q{q}'
          quantile_parameter = q/100
          model = QuantileRegressor(quantile = quantile_parameter, alpha = 0, solver = 'highs')
          model.fit(train[features], train['us_range'])
          pred = model.predict(test[features])
     
          results.append({
               'model': name,
               'test_start': test.index.min(),
               'coverage': (test['us_range'] <= pred).mean()
          })
     
     start += test_size


folds = pd.DataFrame(results)

model_summary = folds.groupby('model').agg(
     mean_coverage = ('coverage', 'mean'),
     min_coverage = ('coverage', 'min'),
     max_coverage = ('coverage', 'max'),
).reset_index()

model_summary.head()


In [ ]:
'''## Task 3 - Plot it (15 min)

- 3a. Line plot: x = fold test start date, y = coverage, one line per model (q50, q80, q90).
- 3b. Add three horizontal reference lines at 0.5, 0.8 and 0.9 so you can see at a glance
     whether each model tracks its own target.

**In a comment:** looking at the chart, is there a period where all three models degrade at
once? If yes, what might have been happening in the market then - and check it against your
data rather than guessing (`us_range` mean per year would tell you something).'''


import matplotlib.pyplot as plt
import seaborn as sns

folds.head()

plt.figure(figsize = (16, 10))
lp = sns.lineplot(
    folds,
    x = 'test_start',
    y = 'coverage',
    hue = 'model'
)
lp.axhline(y = 0.5, color = 'red')
lp.axhline(y = 0.8, color = 'red')
lp.axhline(y = 0.9, color = 'red')
plt.show()


#I wouldn't say all models degrade at once, the results are mixed + overall close to their expected coverage, while linear regression's coverage sits around 60-62% mark.

In [ ]:
#W2 D4 T1

'''## Task 1 - Refit on the full dataset (15 min)

Every model this week was trained on some window and tested on a later one - which means
none of them has ever seen the most recent data. Before building something usable, that
changes.

**What we are predicting:** `us_range` - the high minus low of the US session
(10:00-16:00 ET), in points, at the moment the US session opens (10:00 ET). Features come
from this morning's EU session (03:00-09:55 ET) and from previous days - nothing from the
US session itself, since it has not happened yet at prediction time.

**Features:**

```python
features = ['eu_range', 'eu_bar_rng_mean', 'us_atr14',
            'eu_atr14', 'prev_us_range', 'eu_range_norm']
```

- 1a. Fit three `QuantileRegressor` models (quantile 0.5, 0.8, 0.9, `alpha=0`,
     `solver='highs'`) on your **entire** dataframe - no train/test split this time.
- 1b. This is a different thing from what you have done all week, so name it in a comment:
     what can this final model tell you that a train/test split cannot, and what can it
     **not** tell you that a train/test split could? (You already have the answer to the
     second half from Monday to Wednesday.)
'''

from sklearn.linear_model import QuantileRegressor

features = ['eu_range', 'eu_bar_rng_mean', 'us_atr14',
            'eu_atr14', 'prev_us_range', 'eu_range_norm']
quantiles = [50, 80, 90]
filtered_df = df[features]


#first approach, I kinda wanted to use classes here as I thought it could be appropriate and I idn't use OOP for a long time, but it's just for testing purpsoes 
class QuantileModel():
    def __init__(self, model_name, quantile):
        self.name = model_name
        self.quantile_parameter = quantile
        self.model = None
        self.predictions = []

    def create_model(self):
        self.model = QuantileRegressor(quantile = self.quantile_parameter, alpha = 0, solver = 'highs')
    
    def train_model(self, train_x, train_target):
        self.model.fit(train_x, train_target)
    
    def model_predict(self, test_x):
        pred = self.model.predict(test_x)
        self.predictions.append(pred)
        return self.predictions
    

for q in quantiles:
    q_parameter = q / 100
    model_name = f'q{q}_model'
    model = QuantileModel(model_name, q_parameter)
    model.create_model()
    model.train_model(filtered_df, df['us_range'])
    pred = model.model_predict(filtered_df)
    
    print(pred)
    
#in this case training the models only takes 6 lines, so perhaps it's more practical :D
q50_model = QuantileRegressor(quantile = 0.5, alpha = 0, solver = 'highs')
q50_model.fit(filtered_df, df['us_range'])

q80_model = QuantileRegressor(quantile = 0.8, alpha = 0, solver = 'highs')
q80_model.fit(filtered_df, df['us_range'])

q90_model = QuantileRegressor(quantile = 0.9, alpha = 0, solver = 'highs')
q90_model.fit(filtered_df, df['us_range'])
    
    

    
    
    


In [ ]:
#W2 D4 T2
'''
Wrap the three fitted models into one function.

```python
def forecast_range(today_features, q50_model, q80_model, q90_model):
    """
    today_features: a single row of feature values (one day), same columns as
                     used for training - shape (1, 6), not a plain list.
    Returns a dict with the three quantile predictions.
    """
    return {
        'q50': q50_model.predict(today_features)[0],
        'q80': q80_model.predict(today_features)[0],
        'q90': q90_model.predict(today_features)[0],
    }
```

The `[0]` matters: `.predict()` always returns an array, even for one row, so you pull out
the single number inside it.

- 2a. Take the **last row** of your dataframe (today's actual morning data) and pass its
     features through the function. `X.iloc[[-1]]` gets you the last row **as a DataFrame**
     (not a Series) - the double brackets are what keep the two-dimensional shape the model
     expects.
- 2b. Print the three numbers next to the actual `us_range` that happened that day.
- 2c. Also run the function on 5 other random days from your data and build a small table:
     date, actual, q50, q80, q90.

**In a comment:** for those 5 days, how many times did the actual value exceed q80? Roughly
what would you expect, and does it match?'''



def forecast_range(today_features, q50_model, q80_model, q90_model):
    """
    today_features: a single row of feature values (one day), same columns as
                     used for training - shape (1, 6), not a plain list.
    Returns a dict with the three quantile predictions.
    """
    return {
        'q50': q50_model.predict(today_features)[0],
        'q80': q80_model.predict(today_features)[0],
        'q90': q90_model.predict(today_features)[0],
    }
    

features = ['eu_range', 'eu_bar_rng_mean', 'us_atr14',
            'eu_atr14', 'prev_us_range', 'eu_range_norm']

today_features = [[30, 5, 42, 24, 66, 1.3]]
xd = forecast_range(today_features, q50_model, q80_model, q90_model)


import random
results = []
random_list = random.sample(range(len(df)), 5)
filtered_df = df.iloc[random_list]

for i in range(len(filtered_df)):
    pred = forecast_range(filtered_df[features].iloc[[i]], q50_model, q80_model, q90_model)
    results.append(
    {
    'date': filtered_df['trade_date'].iloc[i],
    'actual': filtered_df['us_range'].iloc[i],
    'q50': pred['q50'],
    'q80': pred['q80'],
    'q90': pred['q90']
    }
    )

final_table = pd.DataFrame(results)
final_table.head()



In [ ]:
#latest_agg for T2 D4 T3
# df = pd.read_csv('xauusd_m5_latest.csv')
# print(df.head())

# df['et_time'] = pd.to_datetime(df['timestamp'], unit = 's', utc = True).map(lambda x: x.tz_convert('America/New_York'))
# df['trade_date'] = pd.to_datetime(df['et_time']).dt.date

# eu_session = df[(df['et_time'].dt.hour >= 3) & (df['et_time'].dt.hour <= 9)]
# eu_session['candle_range'] = eu_session['high'] - eu_session['low']
# eu_session = eu_session.groupby(['trade_date']).agg(
#     eu_open = ('open', 'first'),
#     eu_high = ('high', 'max'),
#     eu_low = ('low', 'min'),
#     eu_close = ('close', 'last'),
#     eu_bars = ('et_time', 'count'),
#     eu_bar_rng_mean = ('candle_range', 'mean')
# ).reset_index()
# eu_session.head()

# us_session = df[(df['et_time'].dt.hour >= 10) & (df['et_time'].dt.hour <= 15)]
# us_session['candle_range'] = us_session['high'] - us_session['low']
# us_session = us_session.groupby(['trade_date']).agg(
#     us_open = ('open', 'first'),
#     us_high = ('high', 'max'),
#     us_low = ('low', 'min'),
#     us_close = ('close', 'last'),
#     us_bars = ('et_time', 'count'),
#     us_bar_rng_mean = ('candle_range', 'mean')
# ).reset_index()
# us_session.head()

# us_session['us_range'] = us_session['us_high'] - us_session['us_low']
# eu_session['eu_range'] = eu_session['eu_high'] - eu_session['eu_low']

# df = us_session.merge(eu_session, on = 'trade_date')
# df = df[(df['us_bars'] >= 0) & (df['eu_bars'] >= 0)]


# df['eu_return_pct'] = (df['eu_close'] - df['eu_open']) / df['eu_open'] * 100
# df['eu_close_loc'] = (df['eu_close'] - df['eu_low']) / (df['eu_high'] - df['eu_low'])
# df['us_atr14'] = df['us_range'].shift(1).rolling(14).mean()
# df['eu_atr14'] = df['eu_range'].shift(1).rolling(14).mean()
# df['prev_us_range'] = df['us_range'].shift(1)
# df['eu_range_norm'] = df['eu_range'] / df['eu_atr14']

# df = df.dropna()
# df.head()
# df.shape

In [ ]:
#W2 D5 T1

'''## Task 1 - Walk-forward coverage for q80, model vs naive (25 min)

**What we are predicting:** `us_range` at 10:00 ET, using this morning's EU session and
previous days only - same as all week.

```python
features = ['eu_range', 'eu_bar_rng_mean', 'us_atr14',
            'eu_atr14', 'prev_us_range', 'eu_range_norm']
```

**The naive baseline for this task:** `prev_us_range` (yesterday's range) treated as an
80th-percentile guess. It's a real thing someone might actually do without a model - "today
will probably be like yesterday, plus some room."

- 1a. Run the walk-forward loop (`train_size=300`, `test_size=300` or whatever you prefer -
     your call, justify it in a comment like you did on D3) fitting `QuantileRegressor(
     quantile=0.8, alpha=0, solver='highs')` on each training window.
- 1b. In the same loop, for each fold also compute coverage using `prev_us_range` directly
     as the "prediction" (no model - just that column, scaled up by 1.3x to give it a fair
     shot at 80%: `naive_pred = test['prev_us_range'] * 1.3`).
- 1c. Build a table: one row per fold, two coverage columns (`model_coverage`,
     `naive_coverage`).

**In a comment:** does the model beat the naive baseline in every fold, most folds, or is it
close? What would it mean if the naive baseline won even one fold?'''


from sklearn.linear_model import QuantileRegressor

features = ['eu_range', 'eu_bar_rng_mean', 'us_atr14',
            'eu_atr14', 'prev_us_range', 'eu_range_norm']
filtered_df = df[features]

train_size = 300
test_size = 300


start = train_size
results = []

while start + test_size <= len(df):
    train = df.iloc[start - train_size : start]
    test = df.iloc[start : start + test_size]
    
    model = QuantileRegressor(quantile = 0.8, alpha = 0, solver = 'highs')
    model.fit(train[features], train['us_range'])
    pred = model.predict(test[features])
    
    results.append(
        {
            'window_start': train.index.min(),
            'model_prediction': (pred <= test['us_range']).mean(),
            'naive_coverage': ((test['prev_us_range'] * 1.3) <= test['us_range']).mean()
        }
    )
    
    start += test_size
    

folds = pd.DataFrame(results)
folds.head()

    


In [ ]:
#W2 D5 T2

'''Wednesday's function only predicted `us_range`. Today, predict `eu_range` for tomorrow
morning too - using only what's known at the US close (16:00 ET) today: today's full EU+US
data and history, nothing from tomorrow.

```python
features_eu = ['prev_eu_range', 'prev_us_range', 'eu_atr14', 'us_atr14']
```

(`prev_eu_range` and `prev_us_range` here mean *today's* completed sessions - they become
"previous" once tomorrow starts. You already built lagged versions like this in Week 1.)

- 2a. Fit q50/q80/q90 `QuantileRegressor` models for `eu_range` on the full dataset (no
     split - like Task 1 on D4).
- 2b. Write a function `forecast_both(row, us_models, eu_models)` that returns a dict with
     six keys: `us_q50`, `us_q80`, `us_q90`, `eu_q50`, `eu_q80`, `eu_q90`.
- 2c. Run it on the last available row and print the result next to what actually happened
     the next morning (if you have that data) or just print the six numbers with a note on
     what each represents.

**In a comment:** which of the six numbers would matter most to you if you were deciding
position size before the EU session opens, and why?'''

from sklearn.linear_model import QuantileRegressor

features_eu = ['prev_eu_range', 'prev_us_range', 'eu_atr14', 'us_atr14']
features_us = ['eu_range', 'eu_bar_rng_mean', 'us_atr14', 'eu_atr14', 'prev_us_range', 'eu_range_norm']


eu_q50_model = QuantileRegressor(quantile = 0.5, alpha = 0, solver = 'highs')
eu_q50_model.fit(df[features_eu], df['eu_range'])

eu_q80_model = QuantileRegressor(quantile = 0.8, alpha = 0, solver = 'highs')
eu_q80_model.fit(df[features_eu], df['eu_range'])

eu_q90_model = QuantileRegressor(quantile = 0.9, alpha = 0, solver = 'highs')
eu_q90_model.fit(df[features_eu], df['eu_range'])


from typing import Dict


def forecast_both(row,
                  us_models,
                  eu_models):
    
    features_us = ['eu_range', 'eu_bar_rng_mean', 'us_atr14',
            'eu_atr14', 'prev_us_range', 'eu_range_norm']
    features_eu = ['prev_eu_range', 'prev_us_range', 'eu_atr14', 'us_atr14']
    
    row_us = row[features_us]
    row_eu = row[features_eu]
    
    return {
        'us_q50': us_models[0].predict(row_us),
        'us_q80': us_models[1].predict(row_us),
        'us_q90': us_models[2].predict(row_us),
        'eu_q50': eu_models[0].predict(row_eu),
        'eu_q80': eu_models[1].predict(row_eu),
        'eu_q90': eu_models[2].predict(row_eu),
    }
    
    
xd = pd.DataFrame(forecast_both(row = df.iloc[[-1]],
                   us_models = [q50_model, q80_model, q90_model],
                   eu_models = [eu_q50_model, eu_q80_model, eu_q90_model]))
xd.head()



In [ ]:
#W2 D5 T3

'''## Task 3 - Reality check (10 min)

- 3a. Pick any 3 of the six predictions from Task 2 and sanity-check them against the
     historical `atr14` for that session (e.g. is `us_q90` in a plausible range compared to
     `us_atr14`, or wildly off?).

**In a comment:** if one of the three looked implausible, what would you check first -
feature order, a data leak, or something else? You don't need it to actually be wrong, just
say what your first move would be.
'''


historical_atr14 = df.iloc[-1]['us_atr14']
print(historical_atr14)
#35.10521428571422
'''
	us_q50	us_q80	us_q90	eu_q50	eu_q80	eu_q90
0	31.919022	45.169257	58.449321	55.244878	75.330539	91.297397
'''

'''
By looking at the us_qXX I can clearly see the percentages look very plausible - 
q50 would indeed be a dangerous stop to set and could get taken out
q80 seems a lot safer as it's about 30% above the atr14
q90 protects us even more, but it could be an overkill, as it only adds +10pp safety for 13 points more, not a deal I'd take here
'''


In [ ]:
#W3 D1 T1

#New data scheme added with LLM to speed things up, now I can move straight into adding features and ML

'''Task 1 - Up/down range asymmetry (15 min)'''

'''For the H1 dataframe, add a rolling feature comparing the size of recent up-candles vs
down-candles:

```python
h1["candle_range"] = h1["high"] - h1["low"]
h1["is_up"] = h1["close"] > h1["open"]

up_range_20 = h1["candle_range"].where(h1["is_up"]).rolling(20, min_periods=5).mean()
down_range_20 = h1["candle_range"].where(~h1["is_up"]).rolling(20, min_periods=5).mean()

h1["up_down_range_ratio"] = up_range_20 / down_range_20
```

`.where(condition)` keeps values where the condition is True and puts NaN elsewhere - so
`up_range_20` is "average range of up-candles among the last 20 bars, ignoring down-candles
in that window". A ratio above 1 means recent up-moves have been bigger than recent
down-moves.

- 1a. Add the column.
- 1b. Print `.describe()` on it - what's the typical range, is it centered near 1.0?
- 1c. Check the value at a handful of the `strong_bull` timestamps we looked at today
     (e.g. 2026-07-02, when price was clearly trending up) vs a `strong_bear` stretch
     (e.g. 2026-06-24). Does the ratio move in the direction you'd expect?

**In a comment:** does this look like it's adding information beyond what `regime` already
tells you, or does it mostly just restate "we're in an uptrend/downtrend"?'''



h1_df = pd.read_csv('project4_trend_regime/h1_with_regime.csv')
m15_df = pd.read_csv('project4_trend_regime/m15_with_regime.csv')


h1_df["candle_range"] = h1_df["high"] - h1_df["low"]
h1_df["is_up"] = h1_df["close"] > h1_df["open"]

up_range_20 = h1_df["candle_range"].where(h1_df["is_up"]).rolling(20, min_periods=5).mean()
down_range_20 = h1_df["candle_range"].where(~h1_df["is_up"]).rolling(20, min_periods=5).mean()

h1_df["up_down_range_ratio"] = up_range_20 / down_range_20


h1_df['trade_date'] = pd.to_datetime(h1_df['et_time']).dt.date
check_df = h1_df[h1_df['trade_date'].astype(str) == '2026-07-02']

h1_df.describe()

In [ ]:
#W3 D2 T1

'''## Task 1 - Load and orient yourself (10 min)

- 1a. Load `h1_with_regime.csv` with `et_time` as a datetime index.
- 1b. Print `.info()` and `.describe()` - anything look off (nulls, weird ranges)?
- 1c. Print `regime.value_counts(normalize=True)` yourself and compare against what the
     script printed yesterday. Do the numbers match your own read of the chart?

**In a comment:** how many total rows, and what date range does this cover?'''


h1_df.info()

start, end = h1_df['et_time'].min(), h1_df['et_time'].max()
print(start, end)

#30500 rows, no nulls
#it covers 5+ years of data, from 2021-01-04 00:00:00 to 2026-07-17 14:00:00


print(h1_df['regime'].value_counts())
'''
regime
strong_bull         12609
strong_bear          8607
range_recross        7442
retracement_bull      974
retracement_bear      889
Name: count, dtype: int64
'''

#This looks quite optimistic, as it seems like trends occupy the most time, but there are also range_recross periods
#that surely could wipe out any account if not taken seriously



In [ ]:
#W3 D2 T2

'''
## Task 2 - Regime duration (20 min)

You said "the important thing is red dominates in a downtrend" - let's check that with
numbers, not just eyeballing.

For each *contiguous block* of the same regime (e.g. 40 consecutive `strong_bear` bars in a
row counts as one block, not 40), compute how many bars it lasted.

Hint for finding contiguous blocks without a loop:
```python
block_id = (h1["regime"] != h1["regime"].shift(1)).cumsum()
block_lengths = h1.groupby(block_id)["regime"].agg(["first", "size"])
```
`block_id` increases by 1 every time the regime changes, so rows with the same `block_id`
are one uninterrupted run of the same label. `.groupby(block_id)` then collapses each run
into one row: `first` is the regime name, `size` is how many bars it lasted.

- 2a. Build `block_lengths` as above.
- 2b. Group by regime name (`first`) and get mean/median/max block length for each of the
     5 categories.
- 2c. What fraction of `strong_bear` blocks last more than 10 bars? More than 50?

**In a comment:** does this match the visual impression - are strong trend blocks
meaningfully longer-lived than range_recross blocks, or surprisingly similar?'''


h1_df['block_id'] = (h1_df['regime'] != h1_df['regime'].shift(1)).cumsum()
block_lengths = h1_df.groupby(['block_id'])['regime'].agg(['first', 'size'])
print(block_lengths.head(50))
mean_block_lengths = block_lengths.groupby('first').agg(
    avg_block_length = ('size', 'mean'),
    median_block_length = ('size', 'median'),
    min_block_length = ('size', 'min'),
    max_block_length = ('size', 'max'),
)

longer_than_10 = block_lengths[(block_lengths['first'] == 'strong_bear') & (block_lengths['size'] >= 10)].count() / block_lengths[block_lengths['first'] == 'strong_bear'].count() * 100
print(longer_than_10) #55% of the strong bear trends out of all bears are longer than 10 hours long

longer_than_10 = block_lengths[(block_lengths['first'] == 'strong_bull') & (block_lengths['size'] >= 10)].count() / block_lengths[block_lengths['first'] == 'strong_bull'].count() * 100
print(longer_than_10) #60% of the strong bull trends out of all bulls are longer than 10 hours long

longer_than_10 = block_lengths[(block_lengths['first'] == 'range_recross') & (block_lengths['size'] >= 10)].count() / block_lengths[block_lengths['first'] == 'range_recross'].count() * 100
print(longer_than_10) #84% of the range_RECROSS regime occurences out of all range recrosses are longer than 10 hours long


'''                  avg_block_length  median_block_length  min_block_length  \
first                                                                       
range_recross            30.500000                 23.0                 1   
retracement_bear          4.233333                  3.0                 1   
retracement_bull          3.959350                  3.0                 1   
strong_bear              36.625532                 14.0                 1   
strong_bull              47.224719                 18.0                 1   

                  max_block_length  
first                               
range_recross                  153  
retracement_bear                18  
retracement_bull                20  
strong_bear                    320  
strong_bull                    375  
'''
#Not sure what to think here, but there are some trends that definitely last, and the tendency is towards the bullish dominance (47 vs 36 bull vs bear mean).
#Range_recross periods are also long, the mean is lower than the  strong_bear and bull.

#However, we can also see that most of the trend are not that long with 14-18 median block lengths
#The mean is elevated by the extreme outliers that can last 200+, and the median for range_recross is 23 hours.

#This suggests that these periods definitively need patience to hold trades
#I'd also like to check how these numbers look after we filter out the night/Asian time range.


pct = (block_lengths[(block_lengths['first']=='strong_bear') & (block_lengths['size']>=10)].shape[0]
       / block_lengths[block_lengths['first']=='strong_bear'].shape[0] * 100)

print(pct)


In [ ]:
# check_df = h1_df[(h1_df['regime'] == 'strong_bear')].copy()
# linep = sns.lineplot(
#     check_df,
#     x = 'et_time',
#     y = 'close'
#     )
# plt.show()


# check_df = h1_df[(h1_df['regime'] == 'strong_bull')].copy()
# linep = sns.lineplot(
#     check_df,
#     x = 'et_time',
#     y = 'close'
#     )
# plt.show()


# check_df = h1_df[(h1_df['regime'] == 'range_recross')].copy()
# linep = sns.lineplot(
#     check_df,
#     x = 'et_time',
#     y = 'close'
#     )
# plt.show()

#There are subtle nuances between the three charts and you can clearly see the price movement is different and definitely bearish/bullish/range, which looks like we have filtered out at least a fraction of unvaforable trading conditions with this simple differentiation. And I must add that bullish is a way more upwards than the bearish is downwards - in other wards, the bearish chart overall looks a bit similar to the range_recross regime chart, which is interesting.

Overall I'd say we're unable to create perfect distinction, but it's probably as good as we can get with minimum effort for getting favorable trading conditions, and I'd stick with that without looking for more nuanced differentation for convenience. And most definitely there are more and less trending periods, but overall it looks promising from this point of view. And it could maybe give some actually useful edge in the logn term.


I don't see a reason to go deeply into specific trends, as every trend willl look a bit different, depending on the context. Building observations on the basis of single trends doesn't make sense IMHO.

In [ ]:
#W3 D3 T1

'''## Task 1 - Load, filter, orient (10 min)

**Where we are:** `project4_trend_regime/m15_pullback_events.csv` exists - 5,522 pullback
events, built on M15 candles with H1 as the regime context. Columns: `direction` (bull/bear),
`ref_time`, `ref_price`, `ref_atr`, `depth_price`, `depth_atr`, `duration_minutes`, `status`
(resumed/recrossed/unfinished), `same_candle_pullback`.


- 1a. Load the CSV, parse `ref_time` as datetime.
- 1b. Filter to `status == 'resumed'` only - this is the target we care about (recrossed
     and unfinished are different phenomena, not the pullback distribution itself).
- 1c. Print `.describe()` on `depth_atr` and `duration_minutes` for this filtered set.

**In a comment:** how many resumed events are you left with, and does the distribution
shape (mean vs median, skew) look like what you'd expect for something bounded at zero?'''

pullback_df = pd.read_csv('project4_trend_regime/m15_pullback_events.csv')

#converting ref time to datetime + filtering to resumed pullbacks only, also turning it into America/Ny Time to make sure it's ET time
pullback_df['ref_time'] = pd.to_datetime(pullback_df['ref_time']).dt.tz_localize('America/New_York')
# pullback_df['hour'] = pullback_df['ref_time'].dt.hour
# pullback_df['minute'] = pullback_df['ref_time'].dt.minute

#filtering to "resumed" pullbacks only 
pullback_df = pullback_df[pullback_df['status'] == 'resumed']
pullback_df.head(100)

print(pullback_df['depth_atr'].describe())
print(pullback_df['duration_minutes'].describe())

print(pullback_df.describe())

In [ ]:
#W3 D3 T2


'''## Task 2 - Is EU vs RTH statistically real, or noise? (25 min)

We found EU median 0.96 vs RTH median 0.75 - a ~30% difference. A difference in two medians
you compute from a sample is not proof of anything by itself - with enough noise, two
random subsets of the SAME data will almost always show slightly different medians. The
question a statistical test answers is: **how surprising is a gap this large if there were
actually no real difference between EU and RTH?** If it's very surprising (a small
p-value), you have evidence the gap is real. If it's not surprising at all (large p-value),
the gap you saw could easily be random noise and you should not build anything on it.

### Why normaltest first

Which test is "correct" depends on the shape of the data. A t-test compares means and
assumes each group is roughly normally distributed (bell-shaped, symmetric). We already
saw in Task 1 that `depth_atr` is bounded at zero and right-skewed (a long tail of deep
pullbacks, a hard floor at zero) - that is NOT what a t-test expects, and running one
anyway risks a wrong p-value.

`scipy.stats.normaltest` tests exactly this:
- **H0 (null hypothesis):** the data comes from a normal distribution.
- **H1 (alternative):** it does not.
- **p < 0.05** → reject H0, the data is not normal (this is what we expect here).

### Why Mann-Whitney U, specifically

If the data isn't normal, you don't fix it by fudging a t-test - you switch to a test that
doesn't require normality at all. Mann-Whitney U works by RANKING every value from both
groups together (smallest = rank 1, next = rank 2, ...) and checking whether one group's
ranks tend to be systematically higher or lower than the other's - not by comparing means.
That makes it robust to skew and outliers, at the cost of only telling you about the
overall *tendency* (are EU values typically larger?), not the exact size of the difference
in the original units.

- **H0:** EU and RTH `depth_atr` come from the same distribution (no systematic
  tendency for one to be larger).
- **H1:** they don't - one tends to be larger than the other.

- 2a. Build the two groups: EU (hour between 3 and 5), RTH (hour between 10 and 12), same
     as before.
- 2b. Run `scipy.stats.normaltest` on `depth_atr` for the full resumed set - confirm it's
     non-normal (state the p-value).
- 2c. Run `scipy.stats.mannwhitneyu` comparing EU vs RTH `depth_atr` (use
     `alternative='two-sided'` since you're asking "are they different", not "is EU
     specifically larger").
- 2d. Report the p-value.

**In a comment, write out before you run anything:**
- What exactly is H0 here, in terms of THIS data (not the generic template above)?
- What would a p-value of 0.30 mean for the EU-vs-RTH claim - would you still trust the
  30% gap you found earlier? Answer this before you see your actual result.

**Then after running:** state the conclusion in plain terms - what does your actual
p-value tell you about whether EU and RTH pullbacks are really different?'''

from scipy.stats import normaltest, mannwhitneyu

#2a - adding hr/min for convenience
pullback_df['hour'] = pullback_df['ref_time'].dt.hour
pullback_df['minute'] = pullback_df['ref_time'].dt.minute

eu_pullbacks = pullback_df[(pullback_df['hour'] >= 3) & (pullback_df['hour'] <= 4)] #3:00 et - 5:00 et (10:00 EU - 12:00 EU)
eu_pullbacks = pullback_df[(pullback_df['hour'] >= 3) & (pullback_df['hour'] <= 4)]
eu_pullbacks.head(15) 

rth_pullbacks = pullback_df[(pullback_df['hour'] >= 10) & (pullback_df['hour'] <= 11)] #10:00 ET - 12:00 ET (16:00 EU - 18:00 EU)
rth_pullbacks.head(15) 

#2B testing whether depth atr on the whole dataset has a normal distribution
normaltest_results = normaltest(pullback_df['depth_atr'])
print(normaltest_results) 
#NormaltestResult(statistic=np.float64(182.06953704951587), pvalue=np.float64(2.9114032894630856e-40))
#p_value = 1.7076775229562615e-46 # H0 rejected (Data does not come from a normal distribution)

#now it's time to compare rth vs eu sets, whether they're meaningfully statistically different from each other
#H0: There are no differences between the datasets
#H1: The datasets are meaningfully different
comparison_test_results = mannwhitneyu(rth_pullbacks['depth_atr'], eu_pullbacks['depth_atr'], alternative='two-sided')
print(comparison_test_results)
#MannwhitneyuResult(statistic=np.float64(16799.5), pvalue=np.float64(0.005643195590467532))
#Again, p_value is way below 0.05, H0 rejected, the two datasets have meaningful statistical differences


'''- What would a p-value of 0.30 mean for the EU-vs-RTH claim - would you still trust the
  30% gap you found earlier? Answer this before you see your actual result.'''
  
#It would mean the datasets are similar to each other, and I wouldn't trust that gap :)), but it's not the case.
#We can see clear differences between the RTH vs EU and it makes sense as well, as RTH is the EU/US overlapping period,
# which could be more volatile overall, but also more ordered in terms of following a given direction more decisively
#but this is just my pure speculation, that doesn't make any sens without proof, and the reason doesn't even matter as well



In [ ]:
#W3 D3 T3

'''## Task 3 - Baseline: how good is "always predict the median"? (20 min)

Before any model, establish what a trivial guess costs you - same principle as every
previous ML session.

- 3a. Split chronologically 80/20 by `ref_time` (sort first!).
- 3b. On the test set, compute MAE for: (i) always predict the training-set median
     `depth_atr`, (ii) always predict the training-set mean.
- 3c. Compute the same baselines separately for EU-hour and RTH-hour events in the test
     set - does a session-aware baseline (train-set EU median for EU events, train-set RTH
     median for RTH events) beat the single overall median?

**In a comment:** if a future model can't beat the session-aware baseline from 3c, is it
worth using?'''

import math
from sklearn.metrics import mean_absolute_error

pullback_df = pullback_df.sort_values(by = 'ref_time')
print(len(pullback_df))
train_set = pullback_df[:math.floor(0.8 * len(pullback_df))]
test_set = pullback_df[math.floor(0.8 * len(pullback_df)):]

median_baseline = np.full(len(test_set), train_set['depth_atr'].median())
mean_baseline = np.full(len(test_set), train_set['depth_atr'].mean())

test_baseline_mae_median = mean_absolute_error(test_set['depth_atr'], median_baseline)
test_baseline_mae_mean = mean_absolute_error(test_set['depth_atr'], mean_baseline)
print(test_baseline_mae_median, test_baseline_mae_mean) #1.004526664451128 1.2215080988640952



eu_train_set = pullback_df[:math.floor(0.8 * len(eu_pullbacks))]
eu_test_set = pullback_df[math.floor(0.8 * len(eu_pullbacks)):]

median_eu_baseline = np.full(len(eu_test_set), eu_train_set['depth_atr'].median())
mean_eu_baseline = np.full(len(eu_test_set), eu_train_set['depth_atr'].mean())

test_baseline_mae_median_eu = mean_absolute_error(eu_test_set['depth_atr'], median_eu_baseline)
test_baseline_mae_mean_eu = mean_absolute_error(eu_test_set['depth_atr'], mean_eu_baseline)

print(F'Eu baselines: MAE median: {test_baseline_mae_median_eu}, MAE mean: {test_baseline_mae_mean_eu}')

rth_train_set = pullback_df[:math.floor(0.8 * len(rth_pullbacks))]
rth_test_set = pullback_df[math.floor(0.8 * len(rth_pullbacks)):]

median_rth_baseline = np.full(len(rth_test_set), rth_train_set['depth_atr'].median())
mean_rth_baseline = np.full(len(rth_test_set), rth_train_set['depth_atr'].mean())

test_baseline_mae_median_rth = mean_absolute_error(rth_test_set['depth_atr'], median_rth_baseline)
test_baseline_mae_mean_rth = mean_absolute_error(rth_test_set['depth_atr'], mean_rth_baseline)

print(F'RTH baselines: MAE median: {test_baseline_mae_median_rth}, MAE mean: {test_baseline_mae_mean_rth}')


''
'''## Task 3 - Baseline: how good is "always predict the median"? (20 min)

Before any model, establish what a trivial guess costs you - same principle as every
previous ML session.

- 3a. Split chronologically 80/20 by `ref_time` (sort first!).
- 3b. On the test set, compute MAE for: (i) always predict the training-set median
     `depth_atr`, (ii) always predict the training-set mean.
- 3c. Compute the same baselines separately for EU-hour and RTH-hour events in the test
     set - does a session-aware baseline (train-set EU median for EU events, train-set RTH
     median for RTH events) beat the single overall median?

**In a comment:** if a future model can't beat the session-aware baseline from 3c, is it
worth using?'''

import math
from sklearn.metrics import mean_absolute_error

pullback_df = pullback_df.sort_values(by = 'ref_time')
print(len(pullback_df))
train_set = pullback_df[:math.floor(0.8 * len(pullback_df))]
test_set = pullback_df[math.floor(0.8 * len(pullback_df)):]

median_baseline = np.full(len(test_set), train_set['depth_atr'].median())
mean_baseline = np.full(len(test_set), train_set['depth_atr'].mean())

test_baseline_mae_median = mean_absolute_error(test_set['depth_atr'], median_baseline)
test_baseline_mae_mean = mean_absolute_error(test_set['depth_atr'], mean_baseline)
print(test_baseline_mae_median, test_baseline_mae_mean)#1.004526664451128 1.2215080988640952



eu_train_set = eu_pullbacks[:math.floor(0.8 * len(eu_pullbacks))]
eu_test_set = eu_pullbacks[math.floor(0.8 * len(eu_pullbacks)):]

median_eu_baseline = np.full(len(eu_test_set), eu_train_set['depth_atr'].median())
mean_eu_baseline = np.full(len(eu_test_set), eu_train_set['depth_atr'].mean())

test_baseline_mae_median_eu = mean_absolute_error(eu_test_set['depth_atr'], median_eu_baseline)
test_baseline_mae_mean_eu = mean_absolute_error(eu_test_set['depth_atr'], mean_eu_baseline)

print(F'Eu baselines: MAE median: {test_baseline_mae_median_eu}, MAE mean: {test_baseline_mae_mean_eu}')
#Eu baselines: MAE median: 0.9029923329570835, MAE mean: 1.0331881627738368


rth_train_set = rth_pullbacks[:math.floor(0.8 * len(rth_pullbacks))]
rth_test_set = rth_pullbacks[math.floor(0.8 * len(rth_pullbacks)):]

median_rth_baseline = np.full(len(rth_test_set), rth_train_set['depth_atr'].median())
mean_rth_baseline = np.full(len(rth_test_set), rth_train_set['depth_atr'].mean())

test_baseline_mae_median_rth = mean_absolute_error(rth_test_set['depth_atr'], median_rth_baseline)
test_baseline_mae_mean_rth = mean_absolute_error(rth_test_set['depth_atr'], mean_rth_baseline)

print(F'RTH baselines: MAE median: {test_baseline_mae_median_rth}, MAE mean: {test_baseline_mae_mean_rth}')
#RTH baselines: MAE median: 0.8467832584731276, MAE mean: 0.8873829123913888

#Very similar baselines for both EU/RTH.

In [ ]:
#W3 D3 T4


'''## Task 4 - Visualize the shape (15 min)

- 4a. Histogram of `depth_atr` (resumed only) - do the same skewed shape we've seen before.
- 4b. Same histogram, split by EU/RTH/other with `hue`, to see the shift visually alongside
     the statistical test.
- 4c. Box plot of `depth_atr` by hour of day (0-23) - is EU/RTH the whole story, or is there
     more structure across the full 24 hours?

**In a comment:** does the box plot suggest any other hour worth investigating beyond
EU/RTH, or does the pattern look like it's really just those two windows?'''

plt.figure(figsize = (20, 10))
histo_chart = sns.histplot(
     pullback_df['depth_atr']
)
plt.show()
#the left side looks a bit like a bell curve, but it has a huge tail to the right side, some extreme outrliers

#the issue with split by EU/RTH other will be a bit counterintuitive here, since I'm not naming EU/RTH as the whole EU/RTH session,
# but rather the windows that I usually like to trade in during these EU/RTH sessions
#For that reason I decided to do separate hisplots for EU/RTH pullbacks from the previously selected windows


plt.figure(figsize = (10, 8))
eu_pb_chart = sns.histplot(
     eu_pullbacks['depth_atr']
)
plt.show()

plt.figure(figsize = (10, 8))
rth_pb_chart = sns.histplot(
     rth_pullbacks['depth_atr']
)
plt.show()

#quite frankly all the charts look very similar, a bit bell-curve like structure on the left and a huge tail to the right, caused by the outliers


#and the boxplot you've asked for

plt.figure(figsize = (10, 8))
depth_atr_by_hour = sns.boxplot(
     pullback_df,
     x = 'hour',
     y = 'depth_atr'
     )
plt.show()

#by looking at the boxplot, I'm wondering whether I'm seeing the ET hours or hours in my local EU time,
#as the 6-7 window + 16-19 windows seem to be quite active


In [ ]:
#W3 D4 T1

'''
## Task 1 - Is the 6-7 / 16-19 pattern real? (25 min)

Yesterday's test compared exactly two groups (EU vs RTH). Today you're comparing across
all 24 hours at once, which changes what test is appropriate.

### Why not just run Mann-Whitney 24 times

You could compare hour 6 vs "everything else", then hour 7 vs "everything else", etc. But
running many tests on the same data inflates your false-positive rate: at a 5% significance
threshold, testing 24 independent hours means you'd expect ~1 to look "significant" by pure
chance even if none of them mean anything. This is the multiple comparisons problem.

### Kruskal-Wallis: the >2-group version of Mann-Whitney

`scipy.stats.kruskal` asks one combined question first: **"do these 24 groups all come from
the same distribution, or does at least one differ?"** It's the non-parametric equivalent of
a one-way ANOVA (which compares means across >2 groups and assumes normality - not
appropriate here, same reasoning as Mann-Whitney vs t-test).

- **H0:** all 24 hourly groups of `depth_atr` come from the same distribution (hour has no
  effect on pullback depth).
- **H1:** at least one hour's distribution differs from the others.

It does NOT tell you *which* hour differs, or by how much - only whether hour matters at
all, as a single yes/no gate before you go hunting for specific hours.

- 1a. Filter to `status == 'resumed'` (as before). Group `depth_atr` by hour (0-23).
- 1b. Run `scipy.stats.kruskal(*groups)` - unpack each hour's `depth_atr` values as a
     separate argument. Report the p-value.
- 1c. If (and only if) H0 is rejected, compute the median `depth_atr` for every hour and
     rank them - do hours 6, 7, 16, 17, 18, 19 actually stand out as a cluster, or was that
     an artifact of how the boxplot happened to look?

**In a comment:** based on the Kruskal-Wallis result and the per-hour medians, do you still
believe there's a real 6-7/16-19 effect, or does it look more like noise now that you're
looking at ranked medians instead of a visual impression?

'''

from scipy.stats import kruskal
'''
- **H0:** all 24 hourly groups of `depth_atr` come from the same distribution (hour has no
  effect on pullback depth).
- **H1:** at least one hour's distribution differs from the others.
'''
pullback_df.head()

grouped_by_hour = pullback_df.groupby('hour')['depth_atr'].agg(
     mean_atr = ('mean')
).reset_index()
grouped_by_hour.head(15)

groups = []
for i in (pullback_df['hour'].unique()):
     groups.append(pullback_df['depth_atr'][pullback_df['hour'] == i])
     
results = kruskal(*groups)
print(results) #p_value = 4.337509543465713e-28, very close to 0


grouped_by_hour_median = pullback_df.groupby('hour')['depth_atr'].agg(
     median_atr = ('mean')
).reset_index().sort_values(by = 'median_atr', ascending = False)
grouped_by_hour_median.head(15)

'''
hour	median_atr
16	16	2.223377
23	23	2.114033
18	18	2.071305
6	6	2.024532
17	17	1.897854
19	19	1.789194
7	7	1.745000
20	20	1.742685
2	2	1.731529
4	4	1.710913
0	0	1.708876
22	22	1.625740
5	5	1.620733
1	1	1.606385
12	12	1.580137'''


#16-20 + 6-7 are relatively high, with 16 being visibly high above the rest
#There could be a real effect, but I'm unsure whether I'd be interested in trading during these hours anyway.

#16-20 are aftermarket hours, which could be volatile due to a hollow orderbook
#6-7 is the same case, but the opposite, it's early pre-market, during EU session, yet before the US opens.

#The ATR can be higher, but it's always relative to the recent price action and it doesn't necessarily mean there's volume behind
#In other words, I'm not particularly interested in those periods, but maybe it's worth to check it... Idk.




In [ ]:
#W3 D4 T2

'''## Task 2 - First feature table for the depth model (30 min)

Time to build the X you'll eventually feed a model. Every feature here must be something
you'd actually know **at `ref_time`** - the moment the peak/trough was just established -
since that's when you'd want a depth prediction.

Build a new dataframe, one row per event (reuse the `resumed`-filtered set), with:

- `direction` (already have it) - encode as 0/1 (bear=0, bull=1) or keep as category, your
     choice, but note which you picked and why it matters for the model type you'd use later.
- `hour` (already have it, from `ref_time`).
- `ref_atr` (already have it) - this is the volatility level at the moment of reference.
- `is_same_candle_pullback` (already have it) - though think about whether this belongs as
     a *feature* (known before the fact) or is actually closer to a description of the
     outcome. Write down your reasoning either way.
- A new one: `prior_pullback_depth` - for events happening within the same underlying trend
     block, what was the depth of the *previous* pullback? (If you're not sure how to find
     "the previous one in the same trend", that's fine - describe the problem in a comment
     instead of guessing at code, and we'll solve it together next session.)

**In a comment:** which of these features do you expect to actually predict depth, and
which do you expect to be close to useless? You don't need to be right - the point is
committing to a guess before any model tells you the answer.'''

from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
pullback_df['direction_int'] = encoder.fit_transform(pullback_df[['direction']]).astype(int)
pullback_df['prior_pullback_depth'] = pullback_df['depth_atr'].shift(1)

#I frankly think predicting depth of a pullback will be veery difficult.
#IMO it should be easier to predict the chances a given pullback will result in a resume or recross on certain ATR levels


pullback_df.head()
#all features for your refernece
'''direction	ref_time	ref_price	ref_atr	depth_price	depth_atr	duration_minutes	status	same_candle_pullback	hour	minute	direction_int	prior_pullback_depth
0	bull	2021-01-04 14:00:00-05:00	1942.610	2.983302	1.790	0.600006	30.0	resumed	False	14	0	1	NaN
1	bull	2021-01-04 14:30:00-05:00	1942.740	2.605102	0.662	0.254117	15.0	resumed	True	14	30	1	0.600006
2	bull	2021-01-04 14:45:00-05:00	1944.135	2.532022	3.117	1.231032	150.0	resumed	False	14	45	1	0.254117
'''


In [ ]:
#W3 D4 T3

'''## Task 3 - Baseline once more, on the real feature set (15 min)

- 3a. Chronological 80/20 split on this new feature table.
- 3b. Report train-set median/mean `depth_atr` and the resulting MAE on test - same
     calculation as Day 3, just now living alongside the actual features you'll model with.

**In a comment:** has anything changed from yesterday's baseline numbers, and should it
have?'''


train = pullback_df[:math.floor(0.8 * len(pullback_df))]
test = pullback_df[math.floor(0.8 * len(pullback_df)):]

train_median = train['depth_atr'].median()
train_mean = train['depth_atr'].mean()

baseline_mae = mean_absolute_error(np.full(len(test), train_mean), test['depth_atr'])
baseline_mae_median = mean_absolute_error(np.full(len(test), train_median), test['depth_atr'])
print(baseline_mae, baseline_mae_median)

#1.2215080988640952 1.004526664451128

#I'm not exactly what does this data suggest, 1 ATR baseline doesn't look a very big error TBH
#The baseline is perhaps already very informative, or it trims dow and flattens bigger retracements into very small baseline


In [ ]:
#W3 D5 T11
#start 11:33


'''## Task 1 - The stop-survival table (35 min)

The question: **"if you set a stop at X ATR beyond the reference point, what fraction of
pullbacks stay within it (i.e. the trend resumes before price goes that deep)?"**

A pullback "surviving" a stop of X ATR means: its `depth_atr` never exceeded X before the
trend resumed. For `status == 'recrossed'` events, the trend never resumed at all - so by
definition, a stop couldn't have "survived" them in the same sense (there's a separate
question buried here, see the comment prompt below).

- 1a. Filter to `status == 'resumed'` only for now (recrossed handled separately below).
- 1b. For a range of stop levels - try `[0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 8.0]` (in ATR
     units, matching `depth_atr`) - compute what fraction of resumed events had
     `depth_atr <= X`. This is "the stop would have survived."
- 1c. Put this in a table: one row per stop level, one column for the survival fraction.
     Compare it to regimatic-ml's numbers for reference (not to match exactly - different
     instrument - but to see if the *shape* is similar): their 1.0 ATR stop survived 19%,
     2.0 ATR survived 53%, 5.0 ATR survived 82%.

**In a comment before building this:** `recrossed` events are excluded from the survival
fractions here, same as regimatic-ml did. Why does dropping them make the "survival" number
optimistic rather than neutral? (Hint: think about what a recrossed event actually
represents for someone who set a stop at, say, 3 ATR and the trend never came back.)'''


#I've reloaded the file to go back from getting the filtered pullbacks only
pullback_df = pd.read_csv('project4_trend_regime/m15_pullback_events.csv')

#converting ref time to datetime + filtering to resumed pullbacks only, also turning it into America/Ny Time to make sure it's ET time
pullback_df['ref_time'] = pd.to_datetime(pullback_df['ref_time']).dt.tz_localize('America/New_York')
pullback_df['hour'] = pullback_df['ref_time'].dt.hour
pullback_df['minute'] = pullback_df['ref_time'].dt.minute

pullback_df['status'].value_counts()
#5278 resumed
#243 recrossed (only about 5%)


pullback_df = pullback_df[pullback_df['status'] == 'resumed']

#how many % of the time the pullback depth is less than or equal toa  given ATR level
atr_range = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 8.0]
survival_rates = {
     'atr': atr_range, 
     'survival_rate': [
          (pullback_df['depth_atr'] <= i).sum() / len(pullback_df) for i in atr_range
          ]
     }

survival_df = pd.DataFrame(survival_rates)
survival_df.head(15)


'''	
     atr	survival_rate
0	0.5	0.213338
1	1.0	0.520652
2	1.5	0.692497
3	2.0	0.791967
4	3.0	0.878742
5	4.0	0.919667
6	5.0	0.941266
7	8.0	0.975748

'''

#Below there are values from regimatic-ml for reference from NQ - remember in this repo I work with gold
#How many of pullbacks WILL not reach a given ATR level and the trend will resume if we set a stop at that level?
'''

| Level | All | Bull | Bear |
|---|---|---|---|
| 1.0 ATR | 19 % | 21 % | 15 % |
| 1.5 ATR | 40 % | 41 % | 36 % |
| **2.0 ATR** | **53 %** | 54 % | 51 % |
| 2.5 ATR | 62 % | 63 % | 60 % |
| 3.0 ATR | 69 % | 70 % | 68 % |
| 4.0 ATR | 77 % | 77 % | 76 % |
| **5.0 ATR** | **82 %** | 82 % | 82 % |
| 6.0 ATR | 86 % | 86 % | 86 % |
| 8.0 ATR | 90 % | 90 % | 91 % |
| 10.0 ATR | 93 % | 92 % | 94 % |

'''




In [ ]:
#W3 D5 T2

'''## Task 2 - Does EU vs RTH change the safe stop level? (15 min)

You already proved statistically (Day 3) that EU pullbacks run deeper than RTH ones. That
should show up directly in this table.

- 2a. Rebuild the same survival table from Task 1, split into EU-hour (3-4) and RTH-hour
     (10-11) subsets.
- 2b. For a fixed target survival rate (pick 80%), what stop level (in ATR) does each
     session need to hit it?

**In a comment:** does this match what you'd expect given Wednesday's statistical result,
and would this change how you'd size a stop differently for a EU-session vs RTH-session
trade?'''


pullback_df.head()
eu_pullbacks = pullback_df[(pullback_df['hour'] >= 3) & (pullback_df['hour'] <= 4)] #3:00 et - 5:00 et (10:00 EU - 12:00 EU)
rth_pullbacks = pullback_df[(pullback_df['hour'] >= 10) & (pullback_df['hour'] <= 11)] #10:00 ET - 12:00 ET (16:00 EU - 18:00 EU)

atr_range = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 8.0]
survival_rates_eu = {
     'atr_range': atr_range,
     'survival_rate': [
          (eu_pullbacks['depth_atr'] <= i).sum() / len(eu_pullbacks) for i in atr_range
          ]
}

survival_df_eu = pd.DataFrame(survival_rates_eu)


survival_rates_rth = {
     'atr_range': atr_range,
     'survival_rate': [
          (rth_pullbacks['depth_atr'] <= i).sum() / len(rth_pullbacks) for i in atr_range
          ]
}

survival_df_rth = pd.DataFrame(survival_rates_rth)

display(survival_df_eu.head(15))
display(survival_df_rth.head(15))

'''
EU SURVIVAL RATES: 

atr_range	survival_rate
0	0.5	0.184579
1	1.0	0.535047
2	1.5	0.693925
3	2.0	0.801402
4	3.0	0.890187
5	4.0	0.925234
6	5.0	0.946262
7	8.0	0.974299
'''


'''
RTH SURVIVAL RATES

atr_range	survival_rate
0	0.5	0.314700
1	1.0	0.670807
2	1.5	0.809524
3	2.0	0.881988
4	3.0	0.939959
5	4.0	0.958592
6	5.0	0.973085
7	8.0	0.995859
'''


#To survive 80% of the time, EU needs a stop of 2.0 ATR, and RTH 1.5, so EU is more volatile (IN TERMS OF ATR!) and needs a bigger stop in terms of ATR.
#The findings match my expectations from Wednesday

#So sure, it could affect the sizing, but in the end, the RTH stop could still be bigger in points,
#as it will be certainly more volatile in terms of points, but it's interesting to see that in terms of ATR, the EU session is more volatile



In [ ]:
#W4 D1 Warmup

'''## Warm-up (15 min) - dict/DataFrame building practice

Goal: build a small summary table from scratch using a dict comprehension, same shape of
task as Friday's survival-rate bug, but simpler and lower-stakes.

Using `pullback_df` (reload `m15_pullback_events.csv`, filter to `status == 'resumed'`):

- For each `direction` value (`bull`, `bear`), compute the mean `depth_atr`.
- Build this as a dict comprehension: `{direction: value for direction in [...]}` - NOT a
  groupby this time, the point is to practice the comprehension syntax itself.
- Then do the same result via `.groupby('direction')['depth_atr'].mean()` and confirm both
  approaches agree.

**Concept - dict comprehension recap:** `{key_expr: value_expr for item in iterable}`. Each
loop iteration produces ONE key and ONE value - unlike Friday's bug, you cannot fit two
different `for` clauses building two different lists into one dict literal. If you need two
parallel lists (like `atr` and `survival_rate` were), build the dict as
`{'col_a': [...], 'col_b': [...]}` directly - two ordinary list comprehensions as the two
values, not a comprehension over the dict itself.'''


# pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
# filtered_df = pullback_df[pullback_df['status'] == 'resumed']

# mean_atrs = filtered_df.groupby('direction')['depth_atr'].mean()

# filtered_df.head()
# mean_atr_comprehension = {
#     'direction': [direction for direction in filtered_df['direction'].unique()],
#     'atr': [filtered_df['direction'].unique()].values()
# } 

# print(mean_atr_comprehension)

#Nie wiem, mam tu dziś olbrzymi problem - dawno już nie robiłem dict comprehensionów i uleciało to trochę

In [ ]:
#W4 D1 T1 


'''## Task 1 - Define the classification target (20 min)

**The goal in plain words:** given a pullback that has reached some depth so far, predict
whether it will ultimately resume (trend continues) or recross (trend reverses) - not the
exact depth_atr number, just the binary outcome.

- 1a. Load `m15_pullback_events.csv`. Keep only `status in ['resumed', 'recrossed']` (drop
     `unfinished` - those never resolved, so there's no label to predict).
- 1b. Create the target column: `target_resumed = 1 if status == 'resumed' else 0`.
- 1c. Check the class balance with `.value_counts(normalize=True)` - this matters, because
     if resumed is 95% of rows, a model that always predicts "resumed" looks falsely good.

**In a comment:** given Friday's finding (only ~5% recrossed), is this dataset balanced or
imbalanced? What would a "predict resumed always" baseline's accuracy be - and does that
mean accuracy is a good metric to judge this model by?'''


pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
filtered_df = pullback_df[pullback_df['status'] != 'unfinished']
filtered_df['target_resumed'] = filtered_df['status'].apply(lambda x : 1 if x == 'resumed' else 0)


filtered_df['target_resumed'].value_counts(normalize = True)
'''
target_resumed
1    0.955986
0    0.044014
Name: proportion, dtype: float64''' 
#resumed scenarios are WAAAY more common
#it means the dataset is imbalance, and perhaps accuracy is not the best metric here,
#as 95% of these scenarios will end up as resumed ANYWAY


#For such cases, one could consider using precision (out of all positives, how many were actually positive)
#or recall (out of all actual positives, how many did the model find), or F1 score, which is a harmonic mean of precision and recall







In [ ]:
#W4 D1 T2


'''## Task 2 - Feature table + time-aware split (25 min)

**Concept - why not a random train/test split here:** `train_test_split(..., shuffle=True)`
picks random rows for train vs test, mixing past and future events together. That's fine
for i.i.d. data (rows independent of each other), but this is a time series - events from
next month sitting in your training set while you "test" on events from three months ago
means the model could learn from the future to predict the past, which can't happen in
production. Instead: sort by `ref_time`, take the first ~80% as train, the last ~20% as
test - the test set is always strictly LATER in time than train.

- 2a. Build the feature table: `direction` (encode as 0/1, e.g. bull=1/bear=0), `hour`
     (extract from `ref_time`), `ref_atr`, `same_candle_pullback` (already 0/1),
     `prior_pullback_depth` (will have NaNs for the first event in each block - decide how
     to handle: drop those rows, or fill with something sensible, and say why you picked
     that option).
- 2b. Sort by `ref_time`, split 80/20 in time order (no shuffling) into train/test.
- 2c. Print the date range covered by train and by test, to confirm there's no overlap.

**In a comment:** why would `prior_pullback_depth` filled with the column mean be a worse
choice here than just dropping those rows, given it's roughly a per-block "first event"
marker rather than random missingness?'''

from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()

filtered_df = filtered_df.dropna()

filtered_df['direction_int'] = encoder.fit_transform(filtered_df[['direction']])
filtered_df['ref_time'] = pd.to_datetime(filtered_df['ref_time']).dt.tz_localize('America/New_York')
# df['trade_date'] = pd.to_datetime(df['et_time']).dt.date
filtered_df['hour'] = filtered_df['ref_time'].dt.hour


feature_df = filtered_df[['ref_time', 'direction_int', 'ref_atr', 'same_candle_pullback', 'prior_pullback_depth', 'hour', 'target_resumed']].sort_values(by = 'ref_time')

train = feature_df[:int(0.8 * len(feature_df))]
test = feature_df[int(0.8 * len(feature_df)):]


print(f'Train time range = {train['ref_time'].min()} - {train['ref_time'].max()}')
train.tail()
test.head()
print(f'TEST time range = {test['ref_time'].min()} - {test['ref_time'].max()}')

# Train time range = 2021-01-04 14:00:00-05:00 - 2025-08-07 16:00:00-04:00
# TEST time range = 2025-08-14 17:00:00-04:00 - 2026-07-17 06:45:00-04:00


#Honestly I wouldn't make any assessments before actually checking it.
#But generally the prior pullback could be long time ago and the ATR could change until then.

In [ ]:
#W4 D2 Warmup

# pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
# resumed_df = pullback_df

# directions = resumed_df['direction'].unique()
# statuses = resumed_df['status'].unique()

# mean_atr_by_direction = {
#     d: resumed_df.loc[resumed_df['direction'] == d, 'depth_atr'].mean()
#     for d in directions
# }

# median_duration_time_by_status = {
#     status: resumed_df.loc[resumed_df['status'] == status, 'duration_minutes'].median()
#     for status in statuses
# }

# print(median_duration_time_by_status)


# median_dur_groupby = resumed_df.groupby('status')['duration_minutes'].median()
# print(median_dur_groupby)


# #W4 D2 Warmup

# pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
# resumed_df = pullback_df

# directions = resumed_df['direction'].unique()
# statuses = resumed_df['status'].unique()

# mean_atr_by_direction = {
#     d: resumed_df.loc[resumed_df['direction'] == d, 'depth_atr'].mean()
#     for d in directions
# }

# median_duration_time_by_status = {
#     status: resumed_df.loc[resumed_df['status'] == status, 'duration_minutes'].median()
#     for status in statuses
# }

# print(median_duration_time_by_status)


# median_dur_groupby = resumed_df.groupby('status')['duration_minutes'].median()
# print(median_dur_groupby)

# {'resumed': np.float64(15.0), 'recrossed': np.float64(2175.0), 'unfinished': np.float64(465.0)}
# status
# recrossed     2175.0
# resumed         15.0
# unfinished     465.0
# Name: duration_minutes, dtype: float64 

#everything correct




In [ ]:
#W4 D2 T1

'''## Task 1 - Close yesterday's open items (10 min)

- 1a. Decide: drop rows where `prior_pullback_depth` is NaN, or fill them? (Yesterday's
     hint: these NaNs mark the FIRST event in each `block_id` - not random missingness, but
     a structural "there is no prior pullback yet" case.)
- 1b. Answer in a comment: why would filling with the column mean be worse than dropping
     here? (Think about what a mean-fill actually claims about a brand-new trend block vs.
     what's true.)
- 1c. Apply your decision to `feature_df` (both train and test) and confirm no NaNs remain
     in the feature columns you'll actually feed the model.'''
    

# filtered_df = filtered_df.dropna()
#I've dropped all NaNs in the cell above to avoid running code again

# Filling the columns carries an obvious risk that we would skew data in any direction,
# which would make our later assumptions possible false or different


train.isna().sum()
# ref_time                0
# direction_int           0
# ref_atr                 0
# same_candle_pullback    0
# prior_pullback_depth    0
# hour                    0
# dtype: int64


test.isna().sum()
# ref_time                0
# direction_int           0
# ref_atr                 0
# same_candle_pullback    0
# prior_pullback_depth    0
# hour                    0
# dtype: int64

     
#No nulls confirmed

In [ ]:
#W4 D2 T2

'''## Task 2 - Fit the first classifier: logistic regression (30-40 min)

**Concept - logistic regression in one paragraph:** linear regression predicts a number;
logistic regression predicts a PROBABILITY between 0 and 1 (here: probability the pullback
resumes) by squashing a linear combination of features through a sigmoid function. You don't
need the sigmoid formula - what matters is: `model.predict_proba(X)` gives you a probability
per row, and `model.predict(X)` applies a 0.5 cutoff to that probability to give you a hard
0/1 guess. For imbalanced classes (95.6% resumed, like ours), that default 0.5 cutoff is
usually a bad idea - worth inspecting the probabilities directly rather than trusting
`.predict()` blindly.

- 2a. `from sklearn.linear_model import LogisticRegression`. Fit on your TRAIN split only,
     features = [direction_int, hour, ref_atr, same_candle_pullback, prior_pullback_depth],
     target = target_resumed.
- 2b. Predict probabilities on the TEST split with `.predict_proba(X_test)` - note this
     returns TWO columns (probability of class 0, probability of class 1); you want column
     index 1 (probability of "resumed").
- 2c. Compare: what's the model's predicted probability distribution (`.describe()`) versus
     the actual test-set class balance? Is the model just predicting ~95.6% for everyone
     (a sign it learned nothing useful beyond the base rate), or is there real spread?

**In a comment:** given yesterday's point about accuracy being a bad metric here, what would
you actually look at to judge whether this model is useful - not yet computing it, just
naming a metric or comparison that would tell you something `.predict().mean() == accuracy`
wouldn't?'''


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score


features = ['direction_int', 'hour', 'ref_atr', 'same_candle_pullback', 'prior_pullback_depth']

model = LogisticRegression()
model.fit(train[features], train['target_resumed'])

probabilities = model.predict_proba(test[features])[:, 1]
probabilities = pd.DataFrame({'probability_of_resumed': probabilities})
display(probabilities.describe())

# probability
# count	1056.000000
# mean	0.909131
# std	0.132960
# min	0.011932
# 25%	0.861607
# 50%	0.990356
# 75%	0.997344
# max	0.998849

#the mean is 0.90 here, which is not super amazing tbh, but I'm wondering how precise these predictions would be


display(test['target_resumed'].mean())
# target_resumed
# 1    1025
# 0      31
# Name: count, dtype: int64

#mean = np.float64(0.9706439393939394) 

#That signifiesa the basic mean is a better indicator than the current probability, if I'm not mistaken



predictions = model.predict(test[features])
print(predictions)

pred_precision = precision_score(test['target_resumed'], predictions)
#checks how many true predictions were really true  (how much we can trust if the models says 'True')
print(pred_precision) #0.9749276759884281

pred_recall = recall_score(test['target_resumed'], predictions)
print(pred_recall)
#How much of the actual TRUE cases were caught (and how much were missed)
#0.9863414634146341

#Not bad

In [ ]:
pred_precision_0 = precision_score(test['target_resumed'], predictions, pos_label=0)
pred_recall_0 = recall_score(test['target_resumed'], predictions, pos_label=0)
print(pred_precision_0, pred_recall_0)

In [ ]:
#W4 D3 warmup
'''**Your turn:** build a list of `ref_atr` values, but only for events where
`status == 'recrossed'` AND `direction == 'bear'`. Then check `len()` of your result against
`((pullback_df['status']=='recrossed') & (pullback_df['direction']=='bear')).sum()` to
confirm they match.'''

#start 14:50

bearish_recrosses_atrs = [
    depth for depth, direction, status in zip(pullback_df['depth_atr'], pullback_df['direction'], pullback_df['status'])
    if status == 'recrossed' and direction == 'bear'
]
print(bearish_recrosses_atrs)
print(len(bearish_recrosses_atrs))


print(((pullback_df['status']=='recrossed') & (pullback_df['direction']=='bear')).sum())

In [ ]:
#W4 D3 T1
'''- 1a. Refit: `LogisticRegression(class_weight='balanced')` on the same train features/target
     as yesterday.
- 1b. Predict on test, compute precision/recall for `pos_label=0` (recrossed) again.
- 1c. Compare directly to yesterday's numbers (0.26 precision / 0.16 recall) - table or just
     print both side by side.

**In a comment:** precision and recall usually move in OPPOSITE directions when you make a
model "more aggressive" about calling something positive (here: recrossed). Did that happen?
Which one improved more, and does that trade feel worth it for this specific
problem (missing a real recrossed event vs. false-alarming on a real resumed event)?'''

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

pullback_df = pd.read_csv('project4_trend_regime/m15_pullback_events.csv')
pullback_df.dtypes

filtered_df = pullback_df.copy().dropna()
features = ['direction_int', 'hour', 'ref_atr', 'same_candle_pullback', 'prior_pullback_depth']

filtered_df['target_resumed'] = filtered_df['status'].map(lambda x: 1 if x == 'resumed' else 0)
filtered_df['direction_int'] = filtered_df['direction'].map(lambda x: 1 if x == 'bull' else 0)
filtered_df['ref_time'] = pd.to_datetime(filtered_df['ref_time'])
filtered_df['hour'] = filtered_df['ref_time'].dt.hour
filtered_df = filtered_df.dropna()


train = filtered_df[:int(0.8 * len(filtered_df))]
test = filtered_df[int(0.8 * len(filtered_df)):]

model = LogisticRegression(class_weight = 'balanced') 
# with class weight 'balanced' each class's mistakes
# get weighted inversely to how common that class is, so a wrong guess on a rare recrossed row
# costs the model much more than a wrong guess on a common resumed row

model.fit(train[features], train['target_resumed'])
predictions = model.predict(test[features])

precision_pred = precision_score(test['target_resumed'], predictions, pos_label = 0)
recall_pred = recall_score(test['target_resumed'], predictions, pos_label = 0)

print(precision_pred, recall_pred) #0.06794055201698514 1.0 - recall is 1.0, which could be an overfitting issue, it suggests a lot of false positives, with a lack of false negatives
#I could've done something wrong here, but the yesterday numbers were precision 0.26 and recall 0.16
#So it's a weird transition, from 0.26 to 0.06 and from 0.16 to 1 (prolly an overfit or some kind of issue, not sure)







In [ ]:
#W4 D3 T2
'''- 2a. Using the ORIGINAL (non-balanced) model from yesterday, get `predict_proba` on test
     again (column index 1 = P(resumed)).
- 2b. Try a few manual thresholds instead of 0.5 - e.g. treat P(resumed) < 0.7 as a
     "recrossed" prediction, then < 0.8, then < 0.9. For each, compute precision/recall for
     the recrossed class.
- 2c. Build a small table: threshold | precision (recrossed) | recall (recrossed). Does
     recall keep climbing as the threshold rises? What happens to precision?

**In a comment:** given the shape of that table, is there a threshold that looks like a
genuinely good practical tradeoff, or does the model just not have enough separating power
between the two classes no matter where you draw the line? (Hint: think back to Task 1's
comparison - does moving the threshold get you anywhere close to what class_weight achieved
in one line?)'''

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

features = ['direction_int', 'hour', 'ref_atr', 'same_candle_pullback', 'prior_pullback_depth']

model = LogisticRegression()
model.fit(train[features], train['target_resumed'])

#predict probabilites for the 'recrossed' class
base_predictions = (model.predict_proba(test[features])[:, 1] >= 0.5).astype(int)
predictions_07 = (model.predict_proba(test[features])[:, 1] >= 0.7).astype(int) #predictions where 1 is only when there's a probability of 70% for 'recrossed' class
predictions_08 = (model.predict_proba(test[features])[:, 1] >= 0.8).astype(int)
predictions_09 = (model.predict_proba(test[features])[:, 1] >= 0.9).astype(int)

base_prec, base_rec = precision_score(test['target_resumed'], base_predictions), recall_score(test['target_resumed'], base_predictions)
prec_07, recall_07 = precision_score(test['target_resumed'], predictions_07), recall_score(test['target_resumed'], predictions_07)
prec_08, recall_08 = precision_score(test['target_resumed'], predictions_08), recall_score(test['target_resumed'], predictions_08)
prec_09, recall_09 = precision_score(test['target_resumed'], predictions_09), recall_score(test['target_resumed'], predictions_09)

results = pd.DataFrame({'threshold': [70, 80, 90], 'precision': [prec_07, prec_08, prec_09], 'recall': [recall_07, recall_08, recall_09]})
results.head()

# threshold	precision(recrossed)	recall(recrossed)
# 0	70	          0.979778	          0.946289
# 1	80	          0.985459            0.860352
# 2	90	          0.997033	          0.656250

#It's evident that precision goes up with the threshold, but recall goes down at the same time.
#That means we can tune the model to get the precision higher at the cost of recall

#precision - total number of our observations classified as positive vs. false positives, 
#recall - total number of our observations classified as positive vs total actual positives

#Which means we basically delete false positives from our positive classifications at the cost of number of detected instances of a given class
#So there's less 1s that are false alarms, but we also detect less of them from the total population

#I need to write it until I remember the difference every single time




In [ ]:
#W4 D3 T3
'''## Task 3 - Which fix actually worked, and why (15 min)

Pull Task 1 and Task 2's recrossed-class numbers into one place (a small DataFrame or just
printed side by side): baseline, class_weight='balanced', and your best threshold from Task 2.

**In a comment, answer plainly:** which approach gave the best precision/recall balance on
the class that actually matters here? Is threshold tuning fundamentally a different kind of
fix than class_weight (hint: one changes what the model LEARNS, the other changes how you
READ an already-trained model's output) - and does that difference explain why one worked
better than the other today?'''





balanced_c = pd.DataFrame({
    'threshold': ['class_weight = balanced', 'baseline'],
    'precision': [
        precision_pred,
        base_prec,
    ],
    'recall': [
        recall_pred,
        base_rec
    ]
})

results = pd.concat([results, balanced_c])

display(results.head())


'''
threshold	precision	recall
0	70	0.979778	0.946289
1	80	0.985459	0.860352
2	90	0.997033	0.656250
0	class_weight = balanced	0.067941	1.000000
1	baseline	0.973963	0.986328
'''

#Model baseline tak słabo wypada pod kątem precyzji, bo przy tak niezbalansowanych klasach (5-95)
#kara za pomyłkę przy klasie recrossed kosztuje dużo więcej w treningu, więc przesuwa swoją gotowość do obstawinia recrossed znacznie niżej
#nie ptorzebuje juz dużej pewności, żeby obstawić 'recrossed'

#We're able to get above the baseline's precision (+2.4pp, not bad at 97,3%)
#This comes at a cost of -33pp of total % of detected population of 'recrossed' class

#It's difficult to say whether this is a good trade-off, it would need a context.
#But I'd say it's not the best.

In [ ]:
#W4 D4 Warmup

'''Same shape as Monday's dict comprehension (`{key: value for item in iterable}`),
except this time the value expression is `.tolist()` instead of `.mean()` - a whole list of
numbers instead of one number. `depths_by_direction['bull']` is now a full list you could
loop over, average, or plot. The second line is a second, separate dict comprehension - it
loops over `.items()` (key-value pairs of the FIRST dict) to build a summary of it. Nesting
comprehensions like this (one consuming the output of another) is common in real code.

**Build `atrs_by_status` - a dict mapping each `status` value to a list of its
`ref_atr` values (use the full `pullback_df`, not filtered). Then print just the count and
mean of each list (you can do this with a second small comprehension like the example, or a
plain loop - your choice this time).'''



atrs_by_status = {status : [value for value in pullback_df[pullback_df['status'] == status]['depth_atr']]
                  for status in pullback_df['status'].unique()
                  }

print(atrs_by_status)
#I think it works :))




In [ ]:
#W4 D4 T1

'''## Task 1 - Does the model actually separate the classes at all? (25 min)

**Concept - ROC AUC in plain words:** precision/recall at ONE threshold only tells you about
that one cutoff point. ROC AUC asks a broader question: if you picked one random `resumed`
event and one random `recrossed` event, what's the probability the model gives the
`recrossed` one a LOWER predicted P(resumed) than the resumed one - i.e., ranks them
correctly relative to each other? 0.5 = no better than a coin flip (random ranking), 1.0 =
perfect separation. It's threshold-independent, which is exactly what yesterday's closing
question was really asking for.

- 1a. `from sklearn.metrics import roc_auc_score`. Using the baseline (non-balanced) model's
     `predict_proba` on test, compute `roc_auc_score(test['target_resumed'], probabilities)`.
- 1b. Sanity check: what would a model with NO real signal (random guessing) score here?
     What would a perfect model score?

**In a comment:** given this number, was yesterday's "the model doesn't have much
separating power" suspicion correct, partially correct, or wrong? Does this change how you'd
value the class_weight/threshold experiments from yesterday - are they squeezing real signal
out of a weak model, or is there just not much signal to squeeze?'''


from sklearn.metrics import roc_auc_score

model = LogisticRegression()
model.fit(train[features], train['target_resumed'])

#predict probabilites for the 'recrossed' class
base_predictions = model.predict_proba(test[features])[:, 1]

base_roc_auc = roc_auc_score(test['target_resumed'], base_predictions)
print(base_roc_auc) #0.5712890625 #+7.1pp vs a coinflip; -42.8pp vs perfect score

#
#0.87744140625 - a lot better


In [ ]:
#W4 D4 T2

'''## Task 2 - Do the features even carry the signal we assumed? (25 min)

Before reaching for a fancier model, check the cheaper explanation: maybe the features
themselves are weak, not the algorithm.

- 2a. `model.coef_` gives you one number per feature (in the order you passed them into
     `LogisticRegression`) - the direction and rough size of each feature's pull on the
     prediction. Print feature names next to their coefficients.
- 2b. Separately (not from the model): compute, for `resumed` vs `recrossed` groups, the
     median of each raw feature (`ref_atr`, `hour`, `prior_pullback_depth`) - does any single
     feature show an actual visible difference between the two groups, on its own, before
     any model is involved?

**In a comment:** if none of the raw features show much difference between resumed and
recrossed groups by themselves, what does that suggest is the real bottleneck - the model
algorithm choice (logistic regression specifically), or the feature set itself?'''

coefs = [feature for feature in model.coef_[0]]
print(features)
#array([[ 0.37561072,  0.02021929, -0.10986281,  4.04105333, -0.03115294]])

features_strength = pd.DataFrame({'features': features, 'coef': coefs})
print(features_strength)


'''               features      coef
0         direction_int  0.375611
1                  hour  0.020219
2               ref_atr -0.109863
3  same_candle_pullback  4.041053
4  prior_pullback_depth -0.031153'''

#it seems that ref_atr + prio_pullback_depth don't carry any signal
#The most is in same_candle_pullback and direction :))

#Not sure whether 4.04 and 0.37 are STRON signals, butthe rest are expendable and perhaps even a bit misleading.

#It's already established some features are uesless, but I'll check the numbers aynway for practice
#using filtred_df here for convenience
filtered_df = filtered_df[filtered_df['status'] != 'unfinished']
grouped_by_status = filtered_df.groupby('status').agg(
     median_atr = ('ref_atr', 'median'),
     median_hour = ('hour', 'median'),
     median_prior_pb_depth = ('prior_pullback_depth', 'median')
).reset_index()

grouped_by_status.head()
'''	status	median_atr	median_hour	median_prior_pb_depth
0	recrossed	 3.543084	          8.0	          1.175144
1	resumed	 3.454020	          9.0	          0.948750
'''


#and the obvious answer here is that the feature set is the bottleneck here


In [ ]:

#W4 D4 T3

'''## Task 3 - Decide the actual next step (10-15 min)

Pull Task 1 (AUC) and Task 2 (feature separation) together.

**In a comment, make an actual call:** given what you found, which of these is the better
use of tomorrow's time - (a) trying a different model (e.g. a tree-based model that can
capture non-linear patterns logistic regression can't), (b) engineering new/different
features (e.g. something capturing recent volatility trend, not just a single ATR snapshot),
or (c) accepting this feature set may have a low ceiling for this specific target and
reconsidering the target itself? There's no single correct answer here - the point is
reasoning from today's evidence, not guessing.'''

#There's no point in trying a different model
# when there's barely any influence of the current features set.
#Engineering new/different features is the best path or accepting c),
# but it's difficult to say before we check different scenarios/paths before.


#Claiming acceptance over failure without PROPERLY checking the results first would be overly pessimistic and unprofessional.

In [ ]:
#W4 D5 warmup

'''Same family as this week's warm-ups, one more layer: the value itself is a small calculation,
not just `.mean()` on an existing column.

**Worked example:**

```python
pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
resumed_df = pullback_df[pullback_df['status'] == 'resumed']

# for each direction, what fraction of its events were same_candle_pullback?
same_candle_rate_by_direction = {
    d: resumed_df.loc[resumed_df['direction'] == d, 'same_candle_pullback'].mean()
    for d in resumed_df['direction'].unique()
}
```

`.mean()` on a boolean column (`same_candle_pullback` is True/False, or 0/1) gives you the
FRACTION that are True - not a typo, this is a real, common pandas pattern: True counts as 1,
False as 0, so averaging a boolean column is exactly "what % of rows are True."

**Your turn:** build a dict mapping each `status` value to the fraction of its rows where
`ref_atr > 3.0` (a rough "was this a high-volatility reference point" flag) - don't create the
flag as a separate column first, compute it inline inside the comprehension's value
expression.'''


filtered_df.head()

atrs_above_3 = {status: (filtered_df.loc[filtered_df['status'] == status, 'ref_atr'] > 3.0).mean()
                for status in filtered_df['status'].unique()
                }

print(atrs_above_3)
#resumed 0.584, recrossed 0.613




In [ ]:
#W4 D5 T1

'''**Concept - what AUC does NOT tell you:** AUC asks "if I rank every test row by predicted
probability, do resumed rows tend to rank above recrossed rows overall?" A model can be very
good at that ranking (high AUC) while still being unusable at one SPECIFIC cutoff, if the
probabilities for both classes are bunched up close together near the top of the range - the
ranking is right, but the absolute numbers don't spread out enough around 0.5 for that
particular cutoff to separate them cleanly.

- 1a. Using yesterday's baseline model, get `predict_proba` on test again. Split it into two
     groups: probabilities for rows where `target_resumed == 1` (actual resumed) and rows
     where `target_resumed == 0` (actual recrossed).
- 1b. Compare the two groups with `.describe()` - specifically look at the median and the
     25th percentile of EACH group. Is there a lot of overlap between the two distributions,
     or are they cleanly separated?
- 1c. Recall Task 2 from yesterday: `same_candle_pullback` had by far the largest
     coefficient (4.04). Check: for `target_resumed == 0` (recrossed) rows specifically, what
     fraction have `same_candle_pullback == 1`? Does this feature single-handedly explain why
     most recrossed probabilities still land above 0.5 despite the model "knowing" something?

**In a comment - answer the open question plainly:** given what you just found, explain in
your own words how a model can rank correctly overall (high AUC) but still fail hard at one
specific threshold. Is this a flaw in AUC as a metric, or is it AUC correctly reporting
something that precision/recall (at 0.5) reports differently, both being right about
different things?'''



from sklearn.metrics import roc_auc_score

model = LogisticRegression()
model.fit(train[features], train['target_resumed'])

#predict probabilites for the 'recrossed' class
probabilities = model.predict_proba(test[features])[:, 1] #(P) resumed DLA KAŻDEGO WIERSZA


probas_for_actual_resumed = probabilities[test['target_resumed'] == 1]
probas_for_actual_recrossed = probabilities[test['target_resumed'] == 0]

# display(pd.Series(probas_for_actual_resumed).describe())
# display(pd.Series(probas_for_actual_recrossed).describe())


# count    1024.000000
# mean        0.916328
# std         0.121121
# min         0.029302
# 25%         0.869240
# 50%         0.991362 #even 50th percentile is already +99% probability, interresting
# 75%         0.997371 
# max         0.998850 #more conviction in extreme cases
# dtype: float64


# count    32.000000
# mean      0.678495 #significantly lower mean than probas for actual resumed
# std       0.242201
# min       0.012117
# 25%       0.646822
# 50%       0.760669
# 75%       0.843841
# max       0.920851 #way less conviction in extreme cases, -7.7pp
# dtype: float64


# coefs = [feature for feature in model.coef_[0]]
# print(features)
# #array([[ 0.37561072,  0.02021929, -0.10986281,  4.04105333, -0.03115294]])
# features_strength = pd.DataFrame({'features': features, 'coef': coefs})
# print(features_strength)

'''               features      coef
0         direction_int  0.375611
1                  hour  0.020219
2               ref_atr -0.109863
3  same_candle_pullback  4.041053
4  prior_pullback_depth -0.031153'''



same_candle_pullback_fraction = (filtered_df.loc[filtered_df['target_resumed'] == 0, 'same_candle_pullback'] == 1).mean()
print(same_candle_pullback_fraction) #0.0

#I'm not sure whether that single-handedly explains why most recrossed probabilities still land above 0.5, but this feature is definitely very strong
#but also there CANNOT be a 'same candle pullback' within the 'recrossed' group - it just DOESN'T MAKE SENSE.
#Same candle pullback is a variation of a pullback, it's a GUARANTEED resumed basically, it already has an information, that this is a resumed class
#Still, you wouldn't have that data available normally to you unless that candle closes, AND IF THAT'S THE CASE, you couldn't use that information.
#It shows us it's impossible to have a one candle recross, which is quite obvious.

#As for correctly ranking overall - with such a high group imbalance, the model will easily learn to rank most of the cases as 'resumed'
#High accuracy/high AUC comes with that by default, which is obvious
#However, in such cases, high accuracy/AUC doesn't mean shit, as we have 95%/5% rates by default, so unless its 99+%, it's nothign impressive.
#That's how I see it.

#The featuers are poor, same_candle_pullback one is very strong, yet it's obviously an exmaple of data leakage in a sense
#You wouldn't be able to use same_candle_pullback - it's already useless as it's ranked as one
#And perhaps it's also useless to rank non same_candle_pullbacks.
#The rest of the features have close to zero predictive power.

In [ ]:
#W4 D5 t2

'''
Quick self-assessment, no new code - just review the week.

- What was the single most useful concept this week (time-aware split, class_weight,
  threshold tuning, ROC AUC, feature-signal checking)?
- What's still shaky and needs to come back next week?
- Rate the week's overall difficulty (1-10, 5 = ideal) and pace (too slow / about right /
  too fast).
'''



In [ ]:
#W5 D1 WARMUP

'''
**Worked example:**

```python
pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
resumed_df = pullback_df[pullback_df['status'] == 'resumed']

stats_by_direction = {
    d: {
        'mean_depth': resumed_df.loc[resumed_df['direction'] == d, 'depth_atr'].mean(),
        'median_depth': resumed_df.loc[resumed_df['direction'] == d, 'depth_atr'].median(),
    }
    for d in resumed_df['direction'].unique()
}
# stats_by_direction['bull']['mean_depth'] now works - a dict living inside a dict



Reading it: the OUTER comprehension is exactly last week's pattern (`{d: ... for d in ...}`).
The INNER part - what becomes the value for each `d` - is just a plain dict literal (not
even a comprehension itself here), with two key:value pairs computed the normal way. Nesting
just means "the value expression can be arbitrarily complex, including another dict."

**Your turn:** build `stats_by_status` - for each `status` value, a dict with `count` (number
of rows) and `mean_duration` (mean `duration_minutes`). Access and print
`stats_by_status['recrossed']['count']` to confirm it works.

'''

stats_by_status = {
    s: {
    'count': len(pullback_df.loc[pullback_df['status'] == s]),
    'mean_duration': pullback_df.loc[pullback_df['status'] == s, 'duration_minutes'].mean()
    } 
    for s in pullback_df['status'].unique()          
}
print(stats_by_status)

In [ ]:
# #W5 D1 T1

# '''
# ## Task 1 - Close the quiz loose end: coefficient magnitude vs. leaking feature (15 min)

# This directly follows up Saturday's quiz C1, where you said "I'm not sure."

# **Concept - what a coefficient tells you, and what it can't:** a logistic regression
# coefficient's sign and magnitude tell you the DIRECTION and RELATIVE STRENGTH of a feature's
# pull on the prediction, holding other features fixed. That's it. It cannot tell you WHY that
# pull exists - whether the feature captures a genuine, generalizable pattern, or whether it's
# a structural/definitional shortcut (like `same_candle_pullback`, which isn't "strongly
# predictive" so much as "definitionally impossible for one class"). The only way to tell them
# apart is to go check the raw data directly, the way you did Friday.

# - 1a. Reload the pullback data, refit last week's baseline `LogisticRegression` (same
#      features as before). Print `model.coef_` again, sorted by absolute value, largest first.
# - 1b. For the SECOND-largest coefficient (not `same_candle_pullback`, which you already know
#      is leaking) - go check the raw data the way you did Friday: does this feature's value
#      differ meaningfully between resumed and recrossed groups, or could it also be some kind
#      of structural artifact?

# **In a comment, answer plainly (this is the quiz question, properly this time):** if you saw
# a coefficient of -3.1 for some feature with no other information, what would that number
# alone tell you, and what would it NOT tell you? Reference what you found in 1b as a concrete
# example either way.'''

filtered_df.head()
model = LogisticRegression()
model.fit(train[features], train['target_resumed'])

coefs = [feature for feature in model.coef_[0]]
print(features)
#array([[ 0.37561072,  0.02021929, -0.10986281,  4.04105333, -0.03115294]])
features_strength = pd.DataFrame({'features': features, 'coef': coefs})
print(features_strength)

#hour is the second largest value here

pullbacks_by_hour = filtered_df[['hour', 'target_resumed']]
xd_plot = sns.barplot(
    pullbacks_by_hour,
    x = 'hour',
    y = 'target_resumed'
)

plt.show()

#I wouldnt' say there are many meaningful cases,but remembering that 0.95 is the baseline for our df,
#seeing around 0.82-0.84 at hour 16 is rather interesting, showing perhaps some potential, but it's the ET market close time
#Nothing too important I'd say, perhaps worth looking to check for specific gold strategy.

#As for seeing a plain number, we'd know it has a strong negative effect on the predicted target,
#we wouldn't know why though. (e.g. the target goes the opposite was as the feature basically)



In [ ]:
#W5 D1 T2
'''## Task 2 - Drop the leaking feature and re-measure honestly (25 min)

Now that `same_candle_pullback` is known to be a structural leak, see what the model looks
like without it - this is the real, honest baseline going forward.

- 2a. Refit `LogisticRegression` on the SAME features as before MINUS `same_candle_pullback`.
- 2b. Recompute ROC AUC and precision/recall (`pos_label=0`, recrossed) on test. Compare
     directly to last week's numbers (AUC 0.877 with the leak included).
- 2c. Is the drop in AUC large or small? What does that tell you about how much of last
     week's apparent performance was really coming from the leak vs. the other features?

**In a comment:** given this corrected, honest baseline - does the original Week 4 Day 4
conclusion ("feature set is the bottleneck") hold up even more strongly now, or does removing
the leak change the picture?
'''

from sklearn.metrics import roc_auc_score, precision_score, recall_score

train = train.drop(columns = ['same_candle_pullback'])
test = test.drop(columns = ['same_candle_pullback'])

features = ['direction_int', 'hour', 'ref_atr', 'prior_pullback_depth']

print(train.head())
print(test.head())


filtered_df.head()
model = LogisticRegression()
model.fit(train[features], train['target_resumed'])

coefs = [feature for feature in model.coef_[0]]
print(features)
#array([[ 0.37561072,  0.02021929, -0.10986281,  4.04105333, -0.03115294]])
features_strength = pd.DataFrame({'features': features, 'coef': coefs})
print(features_strength)

#                features      coef
# 0         direction_int  0.428552
# 1                  hour  0.018116
# 2               ref_atr -0.083314
# 3  prior_pullback_depth -0.029020

#direction seems to be the strongest value now and it got a bit stronger as we've deleted the same_candle_pullback
#It's interesting as it shows a clear +0.05 raise after deleting the leaking feature


base_predictions = model.predict(test[features])
probabilities = model.predict_proba(test[features])[:, 1]


rauc_score = roc_auc_score(test['target_resumed'], probabilities)
prec_score = precision_score(test['target_resumed'], base_predictions)
rec_score = recall_score(test['target_resumed'], base_predictions, pos_label=0)
print(f'Precision: {prec_score}')
print(f'Recall: {rec_score}')

# Precision: 0.9712643678160919
# recall_score: 0.990234375


#for recrossed
prec_score = precision_score(test['target_resumed'], base_predictions, pos_label=0)
rec_score = recall_score(test['target_resumed'], base_predictions, pos_label=0)
print(f'Precision: {prec_score}')
print(f'Recall score: {rec_score}')
print(f'ROC_AUC: {rauc_score}')

# Precision: 0.16666666666666666
# Recall score: 0.0625

#ROC_AUC: 0.720550537109375
#The drop coming from the leak was definitely noticeable. it's not a coinflip, but definitely worse by -15pp


#That definitely makes the features bottleneck thesis hold! The current features are really bad.

In [ ]:
#W5 D1 T3

'''
This follows up Saturday's quiz B2, where you flagged genuine uncertainty.

**Concept - leakage from statistics computed before the split:** even with a perfectly
time-ordered train/test split (train strictly before test), you can still leak information
if you compute some statistic - a mean, a std, a min/max used for scaling - on the FULL
dataset (train+test combined) BEFORE splitting, then apply it to both halves. The model's
training data ends up "informed" by numbers that were only knowable once future (test-period)
data existed - a subtler leak than shuffling rows, but a real one.

- 3a. Concretely reproduce this: compute the mean and std of `ref_atr` over the ENTIRE
     dataset (train+test combined), then imagine normalizing every row (train and test) using
     THAT mean/std. Why would this be a leak, even though the row order itself is untouched
     and nothing was shuffled?
- 3b. Contrast: what would the correct version look like instead (which subset should the
     mean/std be computed from)?

**In a comment:** name one other place in this project (Week 3 or 4) where a similar mistake
could easily have happened if you weren't careful about what "before/after the split" means -
even if you got it right there, explain why it was a real risk.'''


entire_dataset = pd.concat([train, test])
entire_dataset['mean_atr'] = entire_dataset['depth_atr'].mean()
entire_dataset['std_atr'] = entire_dataset['depth_atr'].std()

entire_dataset.head()

#It makes no sense, as we're using data from the future to calculate the atr, which is a direct data leak
#Perhaps it's flattened, but still a book example of a data leak

#it should computed from the past values, that were available before

entire_dataset['mean_atr'] = entire_dataset['depth_atr'].shift(1).expanding().mean()
entire_dataset['std_atr'] = entire_dataset['depth_atr'].shift(1).expanding().std()
#the expanding() window is definitely SUPER USEFUL!


entire_dataset.head()


#I don't remember bud, but in general I always try to use shuffle = False etc.


In [ ]:
#W5 D1 T4
'''
Short, no new concepts - just look at what Task 2 actually gave you.

- 4a. With `same_candle_pullback` removed, what's the class balance-aware precision/recall
     picture now telling you? Is this model closer to "no better than noise" or "modestly
     useful" for the recrossed class?
- 4b. Given everything from Weeks 3-4, if you had to pick ONE next move for this project -
     new features (e.g. something about recent volatility trend, not a single ATR snapshot),
     a different model type, or reconsidering the target - which would you pick now, with the
     leak removed, and does that match or differ from your answer on Day 4 last week?
'''
     
     
#4a - It basically useless, no better than noise would be a fair assessment
#4b - I'd probably still work on new features a bit starting with a proper research or completely abandon this direction.
#I don't remember my asnswer from D4 last week, but the current feature set does not give ANY predictive power.




In [ ]:
#W5 D2 warmup

#8:13

## Warm-up (10 min) - repetition, not a new level
'''
Per your Friday feedback: no escalation today, straight repeat of a pattern with
a fresh column, to build fluency rather than chase difficulty.

```python
pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
```

**Your turn:** build a plain dict comprehension mapping each `direction` value to the
fraction of its rows where `status == 'unfinished'`. Same shape as last week's "fraction of
rows above a threshold" pattern - no new mechanics, just muscle memory.'''


pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')

unfinished_by_direction = {
    direction : (pullback_df.loc[pullback_df['direction'] == direction, 'status'] == 'unfinished').mean()
    for direction in pullback_df['direction'].unique()
    
}

print(unfinished_by_direction)


#done, but I had to check the previous tasks, as this .loc pattern with extraction and comparison is still not hat obvious
#good we're practicing it


In [ ]:
#W5 D2 T1

'''## Task 1 - Build all three candidate features (30 min)

- 1a. **`atr_trend`:** `.groupby('block_id')['ref_atr'].shift(1)` gives the previous row's
     ATR within the same block; `atr_trend = ref_atr - previous_ref_atr` (positive = rising
     volatility).
- 1b. **`pullback_count_in_block`:** within each `block_id`, this is just "how many rows have
     come before this one in the block" - `.groupby('block_id').cumcount()` gives exactly
     that (0 for the first event in a block, 1 for the second, etc. - check what `cumcount()`
     actually returns before assuming, print a few rows of one block to confirm).
- 1c. **`block_duration_so_far`:** for each row, time elapsed (in minutes) between the FIRST
     `ref_time` in its `block_id` and this row's own `ref_time`. One way:
     `.groupby('block_id')['ref_time'].transform('min')` gives every row in a block the
     block's start time; subtract that from `ref_time` and convert to minutes.
- 1d. Handle NaNs from `atr_trend` (first event per block) the same way as
     `prior_pullback_depth` before - drop those rows.

**In a comment, before any modeling:** for each of the three features, one sentence on
whether you expect it to matter, and why - a real prediction, not a shrug.'''


#Not sure - as for atr and trends perhaps it could matter as I think volume and volatility is autocorrelated, 
# but does it necessarily mean directional autocorrelation or not, 
# I wouldn't bet my house for that. 
# 
# Not sure whether there's any monotonic relationship between the pullback time/count vs the odds of resume/continuation,
# but I wouldn't say so. Quite frankly, these features do not seem to be very strong, but we will see.
pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')

pullback_df['prev_atr'] = pullback_df.groupby('block_id')['ref_atr'].shift(1)
pullback_df['atr_trend'] = np.where(
    pullback_df['ref_atr'] > pullback_df['prev_atr'], 'positive', 'negative'
)
pullback_df['atr_trend_positive'] = pullback_df['atr_trend'].apply(lambda x: 1 if x == 'positive' else 0)
pullback_df['direction_int'] = pullback_df['direction'].apply(lambda x: 1 if x == 'bull' else 0)


pullback_df['target_resumed'] = pullback_df['status'].apply(lambda x: 1 if x == 'resumed' else 0)
pullback_df['pullback_count_in_block'] = pullback_df.groupby('block_id').cumcount()
pullback_df['block_starting_time'] = pullback_df.groupby('block_id')['ref_time'].transform('min')
pullback_df['block_duration_so_far'] = (pd.to_datetime(pullback_df['ref_time']) - pd.to_datetime(pullback_df['block_starting_time'])).dt.seconds / 60

pullback_df['ref_time'] = pd.to_datetime(pullback_df['ref_time'])
pullback_df['hour'] = pullback_df['ref_time'].dt.hour                                      
          


pullback_df.head(15)
print(pullback_df['atr_trend'].value_counts())
print(pullback_df['target_resumed'].value_counts())

print(pullback_df.shape)
pullback_df = pullback_df.dropna()
print(pullback_df.shape)

In [ ]:
#W5 D2 T2

'''## Task 2 - Check each feature BEFORE fitting anything (25 min)

This is the "verify, don't assume" step - checking with plots/stats whether each feature
actually separates resumed from recrossed, independent of any model.

- 2a. For each of the three new features, plot its distribution split by `target_resumed`
     (e.g. `sns.boxplot(data=df, x='target_resumed', y='atr_trend')`, repeat for the other
     two). Visually: does either group look shifted relative to the other?
- 2b. Pick the one that looks most promising from the plots. Run
     `scipy.stats.mannwhitneyu` comparing that feature's values between the resumed and
     recrossed groups (same test you used in Week 3 for EU vs RTH depth) - does the
     difference look statistically real, or could it plausibly be noise?

**In a comment:** which of the three looked most promising visually, and did the
Mann-Whitney result agree with that impression?'''
from scipy.stats import mannwhitneyu

plt.Figure(figsize = (16, 10))
resume_by_atr_trend_chart = sns.heatmap(
     pullback_df[['target_resumed', 'atr_trend_positive']]
)
plt.show() #doesn't look there's any visible difference


plt.Figure(figsize = (16, 10))
resume_by_pullback_count = sns.boxplot(data = pullback_df, x='target_resumed', y='pullback_count_in_block')
plt.show()
#interestingly there could be something here, resumed target has many more outliers in terms of pullback_count_in_block
#and a bit lifted IQR/median. It could also be the clock effect, but it's worth exploring I GUESS

plt.Figure(figsize = (16, 10))
resume_by_block_duration = sns.boxplot(data = pullback_df, x='target_resumed', y='block_duration_so_far')
plt.show() #no visible differences again

#I presume 2 of these features will not have any effect on our target
#pullback_count_in_block could show something - could be a clock effect though

#pora na MW testy
#H0: Efektu nie ma/Nic się nie dzieje.
#H1: Efekt jest.


group_resumed = pullback_df.loc[pullback_df['target_resumed'] == 1, 'atr_trend_positive']
group_recrossed = pullback_df.loc[pullback_df['target_resumed'] == 0, 'atr_trend_positive']

mw_test_atr_trend = mannwhitneyu(group_resumed, group_recrossed)
print(mw_test_atr_trend.pvalue) #2.755114916970596e-07 - statystycznie istotny efekt


group_resumed = pullback_df.loc[pullback_df['target_resumed'] == 1, 'pullback_count_in_block']
group_recrossed = pullback_df.loc[pullback_df['target_resumed'] == 0, 'pullback_count_in_block']

mw_test_pullback_count = mannwhitneyu(group_resumed, group_recrossed)
print(mw_test_pullback_count.pvalue) #0.0005346832222164845 - statystycvznie istoyn efekt



group_resumed = pullback_df.loc[pullback_df['target_resumed'] == 1, 'block_duration_so_far']
group_recrossed = pullback_df.loc[pullback_df['target_resumed'] == 0, 'block_duration_so_far']

mw_test_block_duration = mannwhitneyu(group_resumed, group_recrossed)
print(mw_test_block_duration.pvalue) #0.8486550482707517 - brak efektu, hipoteza 0 potwierdzona




In [ ]:
#W5 D2 T3

'''## Task 3 - Fit with all three added, compare to baseline (20 min)

- 3a. Rebuild the feature table: yesterday's honest baseline features (direction, hour,
     ref_atr, prior_pullback_depth) PLUS all three new ones. Same time-aware 80/20 split.
- 3b. Fit `LogisticRegression`, compute ROC AUC on test. Compare to yesterday's 0.720.
- 3c. Check `model.coef_` - which of the three new features got the largest coefficient?
     Does that match what Task 2's plots/test suggested was most promising?

**In a comment:** did AUC move meaningfully? If yes, which feature(s) likely drove it - if
barely at all, does that contradict or confirm Task 2's statistical check?'''
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

features = ['direction_int', 'hour', 'ref_atr', 'prior_pullback_depth', 'atr_trend_positive', 'block_duration_so_far', 'pullback_count_in_block']

train = pullback_df[:int(0.8*len(pullback_df))]
test = pullback_df[int(0.8*len(pullback_df)):]


train.head()

model = LogisticRegression()
model.fit(train[features], train['target_resumed'])
predictions = model.predict(test[features])
probabilities = model.predict_proba(test[features])[:, 1]


'''
Weird unexpected warning here, taking note, will verify/cehck with LLM


:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
       
'''


score_rocauc = roc_auc_score(test['target_resumed'], probabilities)
print(score_rocauc) #0.73486328125 #It's +1.2pp better than yesterday, not a big deal

coefs = [i for i in model.coef_[0]]
coefs_features = pd.DataFrame({
     'feature': features,
     'coef': coefs 
})

print(coefs_features)

'''   feature      coef
0            direction_int  0.415618
1                     hour  0.011022
2                  ref_atr -0.072211
3     prior_pullback_depth -0.003800
4       atr_trend_positive -0.663685
5    block_duration_so_far -0.000124
6  pullback_count_in_block  0.006135'''


#The ROC/AUC score barely moved at all.
#While we have two statiscially important features, their effect on target is low/negligible.
#In other words, statistical importance does not correspond with the strength of the effect


In [ ]:
#W5 D3 Warmup
#Start 12:16

'''## Warm-up (10 min) - repetition: group-vs-group comparison, correct setup

Direct repeat of yesterday's Task 2 setup mistake, on a brand new pair of columns, no
escalation - purely to make "which two things go into a 2-sample test" automatic.

**Your turn (no worked example this time - you've now done this correctly at least twice,
Week 3 and yesterday's corrected version):** using `pullback_df`, split `ref_atr` into two
groups by `direction` (bull vs bear), and run `scipy.stats.mannwhitneyu` comparing THOSE TWO
GROUPS of `ref_atr` values. Before running it, write one line stating explicitly which two
things you are comparing (e.g. "ref_atr values where direction==bull vs ref_atr values where
direction==bear") - a forcing function to catch the feature-vs-label mixup before it happens.'''

from scipy.stats import mannwhitneyu

#First I have to distinguish ref_atr series for both directions to be able to compare them later
group_bullish = pullback_df[pullback_df['direction'] == 'bull']['ref_atr']
group_bearish = pullback_df[pullback_df['direction'] == 'bear']['ref_atr']

print(group_bullish.shape, group_bearish.shape) #(3377,) (1901,)

mw_results = mannwhitneyu(group_bullish, group_bearish)
print(mw_results) #MannwhitneyuResult(statistic=np.float64(3323999.0), pvalue=np.float64(0.03169887186656704))

#It shows there's a statistically meaningful difference, or an effect between direction and ref_atr
#However, IT DOESN'T MEAN that the effect is strong - we don't know that, we just know an effect is there.



In [ ]:
#W5 D3 T1

'''## Task 1 - Cohen's d, the actual standard tool (10 min)

You already know from yesterday's coefficients and today's AUC comparison that these two
features have a weak practical effect - this task isn't about reaching a new conclusion,
it's about learning the ACTUAL standard tool for this question (not a made-up substitute),
so you have it ready next time you're looking at raw data with no model built yet.

**Concept - Cohen's d:** `(mean of group A - mean of group B) / pooled standard deviation`.
The "pooled std" combines both groups' spread into one number (not just one group's std) -
`np.sqrt((std_a**2 + std_b**2) / 2)` is a simple version. Rough interpretation: d≈0.2 is a
small effect, d≈0.5 medium, d≈0.8+ large. It's the standard, named tool for "how big is the
gap between two groups' averages" - unlike a p-value, it doesn't get smaller just because you
have more rows.

- 1a. Compute Cohen's d for `atr_trend`, comparing target_resumed==1 vs ==0 groups.
- 1b. One line: does the resulting d value match the "weak effect" conclusion you already
     reached from the coefficients/AUC yesterday?'''

atr_trend_resumed = pullback_df[pullback_df['target_resumed'] == 1]['atr_trend_positive']
atr_trend_recrossed = pullback_df[pullback_df['target_resumed'] == 0]['atr_trend_positive']

print(atr_trend_resumed)
print(atr_trend_recrossed)

pooled_std = np.sqrt((atr_trend_resumed.std()**2 + atr_trend_recrossed.std()**2) / 2)
cohens_d = (atr_trend_resumed.mean() - atr_trend_recrossed.mean()) / pooled_std

print(cohens_d) #-0.3633052250077581

#Following the interpretaions it shows medium negative effect, which does not neccesarily agree with yesterday's conclusion


In [ ]:
#W5 D3 T2

'''## Task 2 - Try a tree-based model: RandomForestClassifier (35 min)

**Concept - why trees might do better here:** logistic regression assumes each feature pulls
the prediction in a straight-line way, independent of the others (a LINEAR combination). A
decision tree instead asks a sequence of yes/no questions ("is ref_atr > 2.3? if yes, is hour
> 14?..."), which lets it capture INTERACTIONS - e.g. "atr_trend only matters when
pullback_count_in_block is high" - something logistic regression cannot represent no matter
how the coefficients are tuned. A `RandomForestClassifier` fits many such trees on random
subsets of data/features and averages them, which tends to generalize better than one tree
alone.

- 2a. `from sklearn.ensemble import RandomForestClassifier`. Same features and time-aware
     train/test split as yesterday's Task 3. Fit with default settings first.
- 2b. Compute ROC AUC on test, compare to yesterday's 0.735.
- 2c. `model.feature_importances_` gives one number per feature (all non-negative, sum to 1)
     - a different way of ranking feature usefulness than logistic regression's coefficients.
     Print features sorted by importance, largest first.

**In a comment:** did the random forest beat logistic regression on AUC? Does the feature
importance ranking roughly agree with yesterday's coefficient ranking, or does it surface
something different (a sign the tree is picking up on an interaction a linear model missed)?'''


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
model = RandomForestClassifier()
features = ['direction_int', 'hour', 'ref_atr', 'prior_pullback_depth', 'atr_trend_positive', 'block_duration_so_far', 'pullback_count_in_block']

model.fit(train[features], train['target_resumed'])
predictions = model.predict(test[features])
probabilities = model.predict_proba(test[features])[:, 1]

tree_base_roc_auc = roc_auc_score(test['target_resumed'], probabilities) #0.633056640625


model.feature_importances_
feature_importances = pd.DataFrame(
     {
          'feature': [feature for feature in features],
          'importance': [importance for importance in model.feature_importances_]
     }
).sort_values(by = 'importance', ascending = False)
display(feature_importances)

'''feature	importance
2	ref_atr	0.255827
3	prior_pullback_depth	0.246144
5	block_duration_so_far	0.175128
6	pullback_count_in_block	0.166840
1	hour	0.116970
0	direction_int	0.023070
4	atr_trend_positive	0.016020
'''

#I'd say the ROC-AUC is a bit worse, as it's 0.633, unless I did something wrong
#As for the features, they're way more balanced and it seems like the tree creates some relationships with features
#that didn't seem to matter in logistic regression, which is interesting.

#However, feature importances seem to look way more positive here, and yet that doesn't create meaningful increase in ROC-AUC score
#It's quite the opposite, unless I did any errors on the way.

In [ ]:
#W5 D3 T3

'''
## Task 3 - Same overfitting check, applied to the new model (15 min)

Trees are more prone to overfitting than logistic regression (they can carve arbitrarily
complex boundaries if left unchecked) - a quick, concrete check for that.

- 3a. Compute ROC AUC on the TRAINING set (not test) for the random forest. Compare it to the
     test AUC from Task 2.
- 3b. A large gap (train AUC much higher than test AUC) is a classic overfitting signature -
     is there a large gap here?

**In a comment:** if there is a gap, what's one concrete lever you could pull to reduce it
(hint: `RandomForestClassifier` has a `max_depth` parameter - what would limiting it likely
do to both train and test AUC)?'''

predictions_train = model.predict(train[features])
probabilities_train = model.predict_proba(train[features])[:, 1]

roc_auc_train = roc_auc_score(train['target_resumed'], probabilities_train) #1.0

#Yes, we could say it's definitely overfitted
#Limiting max_depth as perhaps changing other parameters could help that, but I'd rather use
#a randomizedsearch or some other variant of that to pick the proper set of parameters rather than do it on my own

#unless there's a methodology that helps with doing that manually, I'm unsure whether it should come with testing and experience
#or simply using the searches for that.


In [ ]:
#W4 D4 WARMUP
'''
No escalation - straight repeat to keep the pattern automatic.

```python
pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
```

**Your turn:** build a dict comprehension mapping each `status` value to the fraction of its
rows where `direction == 'bull'`.'''


pullback_df.head()

bullish_direction_by_status = {
    status: (pullback_df.loc[pullback_df['status'] == status, 'direction'] == 'bull').mean()
    for status in pullback_df['status'].unique()
}

print(bullish_direction_by_status)

# {'resumed': np.float64(0.6456848772763262), 'recrossed': np.float64(0.5111111111111111), 'unfinished': np.float64(0.0)}

In [ ]:
#W4 D4 t1

'''## Task 1 - Why cross-validation before we tune anything (15 min)

**Concept - why not just use the test set to pick hyperparameters:** if you try 20 different
`max_depth` values and pick whichever scores best ON THE TEST SET, you've quietly turned the
test set into a second training set - you're now fitting hyperparameters to its specific
quirks, and your final "test AUC" is optimistic, not honest. Cross-validation solves this by
splitting the TRAINING data itself into several folds, training on some, validating on the
rest, rotating through - so hyperparameter choices are judged without ever touching the real
test set. The test set stays untouched until the very end, for one final honest check.

- 1a. In your own words (one or two sentences): why would picking `max_depth` based on test
     AUC directly be a form of leakage, even though no future data is involved?
- 1b. `TimeSeriesSplit` (from `sklearn.model_selection`) is cross-validation adapted for
     time-ordered data - instead of random folds, each fold's validation portion is always
     LATER in time than its training portion, preserving the same "no future leakage" rule
     you've applied all along. Import it and print what `list(TimeSeriesSplit(n_splits=5).split(train))`
     actually gives you (a list of (train_indices, validation_indices) pairs) - just to see
     its shape before using it for real.'''

from sklearn.model_selection import TimeSeriesSplit
     
#1a - we'd be unable to honestly test the results in that way - we should do a randomized search on train data and then test the parameters on test data

data_splits = list(TimeSeriesSplit(n_splits = 5).split(train))

print(data_splits)

'''[(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181,
       182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194,
       195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207,
       208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220,
       221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233,
       234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246,
       247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259,'''
       
       #We get 5 lists of indices
       

In [ ]:
#W4 D4 t2

'''## Task 2 - Tune RandomForest with RandomizedSearchCV (35 min)

- 2a. `from sklearn.model_selection import RandomizedSearchCV`. Define a small parameter grid
     to search over: `max_depth` (try [3, 5, 8, 12, None]), `min_samples_leaf` (try
     [1, 5, 10, 20, 50]), `n_estimators` (try [100, 200, 300]).
- 2b. `RandomizedSearchCV(RandomForestClassifier(), param_distributions=..., n_iter=15,
     cv=TimeSeriesSplit(n_splits=5), scoring='roc_auc')` - fit this on your TRAIN split only
     (features/target from yesterday, honest set, no `same_candle_pullback`).
- 2c. `.best_params_` and `.best_score_` give you the winning combination and its
     cross-validated AUC. Print both.
- 2d. NOW evaluate that best model on the actual, untouched TEST set - compute ROC AUC there.
     This is the number that actually matters, not `.best_score_` (which is a cross-validated
     TRAINING estimate).

**In a comment:** how does the tuned RF's test AUC compare to (a) yesterday's overfit default
RF (0.633) and (b) yesterday's logistic regression (0.735)? Did tuning fix the problem, or
does logistic regression still win here?'''


from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer


roc_auc_scorer = make_scorer(roc_auc_score, response_method = 'predict_proba') #need_proba IS CRUCIAL HERE, otherwise it's a binary check and basically a coinflip foir auc_roc
parameter_grid = {
     'max_depth': [2, 3, 4, 5],
     'min_samples_leaf': [1, 5, 10, 20, 50],
     'n_estimators': [100, 200, 300, 500]
}

searcher = RandomizedSearchCV(
     estimator =  model,
     param_distributions = parameter_grid,
     n_iter = 15,
     cv = data_splits,
     scoring = roc_auc_scorer,
     error_score = 'raise'
     )

searcher.fit(train[features], train['target_resumed']) #- commented out so I will not rerun it with every run of all cells, as it took 30 sec
print(searcher.best_params_)
print(searcher.best_score_) 

#Results after fixing searcher, as there were issues before

# {'n_estimators': 200, 'min_samples_leaf': 10, 'max_depth': 3} 
#0.6063952812511696


model = RandomForestClassifier(
     n_estimators = 200,
     min_samples_leaf = 10,
     max_depth = 3
     )

model.fit(train[features], train['target_resumed'])
probabilities = model.predict_proba(test[features])[:, 1]

scored_after_search = roc_auc_score(test['target_resumed'], probabilities)
print(scored_after_search) #0.7297515869140625





In [ ]:
#W4 D4 t3

'''## Task 3 - Check the overfitting gap again, post-tuning (15 min)

- 3a. Compute train AUC for the TUNED model (from `.best_params_`, refit if needed) and
     compare to its test AUC.
- 3b. Is the train/test gap much smaller than yesterday's (train=1.0, test=0.633)? If some
     gap remains, is that necessarily bad, or is a small gap actually expected/healthy?

**In a comment:** given everything from today, would you now recommend RandomForest or
logistic regression for this specific dataset going forward - and is your answer about the
ALGORITHM, or more about the FEATURES available (tie back to Week 5 Day 1's finding that the
feature set itself has a fairly low ceiling)?'''

train_probas = model.predict_proba(train[features])[:, 1]
train_roc_auc = roc_auc_score(train['target_resumed'], train_probas)
print(train_roc_auc) #0.9999795247699096 at first, before changes- overfitting was there XD

#0.7358349798830864 AFTER dealing with the searcher's issue - AWESOME, handled overfitting


In [ ]:
#W4 D5 WARMUP

'''## Warm-up (10 min) - repetition: list comprehension with condition

Straight repeat of the Week 4 Day 3 pattern, fresh columns, no escalation.

```python
pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
```

**Your turn:** build a list of `depth_atr` values, but only for events where
`direction == 'bear'` AND `duration_minutes > 60`. Confirm the count matches a boolean-mask
`.sum()` check, same as before.'''


pullback_df.head()

long_bearish_events = pullback_df.loc[(pullback_df['direction'] == 'bear') & (pullback_df['duration_minutes'] > 60), 'depth_atr'].tolist()
print(len(long_bearish_events)) #581


In [ ]:
#W4 D5 T1
'''## Task 1 - Build a bigger, honest feature set (30 min)

Mechanical features only, from columns already in the dataset - no new domain research
needed. Build ALL of these (reuse `atr_trend`, `pullback_count_in_block`,
`block_duration_so_far` from Week 5 Day 2 if you still have that code, otherwise rebuild
quickly):

- `direction_int`, `hour`, `ref_atr`, `prior_pullback_depth` (original honest baseline)
- `atr_trend`, `pullback_count_in_block`, `block_duration_so_far` (Week 5 Day 2)
- **New today:** `minute` (extract from `ref_time` - does position within the hour matter?),
  `day_of_week` (`.dt.dayofweek` on `ref_time` - Monday vs Friday effects are a real, common
  thing to check in market data), `atr_zscore_in_block` (how unusual is this event's
  `ref_atr` relative to the OTHER events in the same `block_id` - compute per-block mean/std
  of `ref_atr` via `.groupby('block_id')['ref_atr'].transform(['mean','std'])`, then
  `(ref_atr - block_mean) / block_std`).

That's 10 features total. Handle NaNs (from shifts/z-scores with insufficient block history)
by dropping - same reasoning as before, restate briefly why.
'''

pullback_df['minute'] = pullback_df['ref_time'].dt.minute
pullback_df['day_of_week'] = pullback_df['ref_time'].dt.dayofweek
pullback_df['mean_ref_atr_block'] = pullback_df.groupby('block_id')['ref_atr'].transform('mean')
pullback_df['std_ref_atr_block'] = pullback_df.groupby('block_id')['ref_atr'].transform('std')
pullback_df['atr_zscore_in_block'] = (pullback_df['ref_atr'] - pullback_df['mean_ref_atr_block']) / pullback_df['std_ref_atr_block']

print(f'Before deleting Nans: {pullback_df.shape}')
pullback_df = pullback_df.dropna()
print(f'After deleting Nans: {pullback_df.shape}')
# Before deleting Nans: (5278, 25)
# After deleting Nans: (5269, 25) #git

In [ ]:
#W4 D5 T2
'''## Task 2 - Fit both models on the full 10-feature set (25 min)

- 2a. Same time-aware 80/20 split. Fit logistic regression AND your tuned RandomForest
     (`max_depth=3, min_samples_leaf=10, n_estimators=200` from yesterday - refit on the new,
     larger feature set, don't just reuse yesterday's fitted model object).
- 2b. Compute ROC AUC for both on test. Compare to yesterday's 7-feature numbers (0.735 LR,
     0.730 RF).
- 2c. For the RandomForest: print `feature_importances_` sorted, largest first. Do any of
     today's 3 new features crack the top 5?

**In a comment:** did adding 3 more features move AUC meaningfully for either model? If not,
does this add more weight to the "the fundamental target may just be hard to predict from
price-derived features alone" theory you raised at the start of today?'''

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, make_scorer

features = ['direction_int', 'hour', 'minute', 'ref_atr', 'prior_pullback_depth', 'atr_trend_positive', 'pullback_count_in_block', 'block_duration_so_far', 'day_of_week', 'atr_zscore_in_block']

pullback_df.head()

train = pullback_df[:int(0.8 * len(pullback_df))]
test = pullback_df[int(0.8 * len(pullback_df)):]

data_splits = list(TimeSeriesSplit(n_splits = 5).split(train))
model = RandomForestClassifier()
scorer = make_scorer(roc_auc_score, response_method = 'predict_proba')

parameter_grid = {
     'max_depth': [2, 3, 4, 5],
     'min_samples_split': [2, 3, 10, 15],
     'min_samples_leaf': [1, 2, 3, 5, 7],
     'max_features': [2, 3, 5, 7, 10, 12],
     'max_leaf_nodes': [2, 3, 5, 10, 30]
}

searcher = RandomizedSearchCV(
     estimator = model,
     param_distributions = parameter_grid,
     n_iter = 30,
     cv = data_splits,
     scoring = scorer,
     error_score = 'raise'
)


searcher.fit(train[features], train['target_resumed'])
print(f'Searcher best params: {searcher.best_params_}')
print(f'Searcher best score: {searcher.best_score_}')

# Searcher best params: {'min_samples_split': 10, 'min_samples_leaf': 5, 'max_leaf_nodes': 100, 'max_features': 3, 'max_depth': 10}
# Searcher best score: 0.688191723708769

#Retraining the model on the best set of featres

model = RandomForestClassifier(max_depth = 10, max_features = 3, max_leaf_nodes = 100, min_samples_leaf = 5, min_samples_split = 10)
model.fit(train[features], train['target_resumed'])
probabilities = model.predict_proba(test[features])[:, 1]
score_rocauc = roc_auc_score(test['target_resumed'], probabilities)

print(score_rocauc) #0.7600353167470753
#It's also quite weird, as I get slightly differenc roc_auc score with every other rerun of this cell (from 0.74 to 0.78)

overfit_test_probas = model.predict_proba(train[features])[:, 1]
train_rocauc = roc_auc_score(train['target_resumed'], overfit_test_probas)
print(f'Train rocauc: {train_rocauc}') #Train rocauc: 0.9884479993167812
#Still massive freaking overfitting! Why and HOW? I used randomized search and data split ://

feature_matrix = pd.DataFrame({
  'feature': [feature for feature in features],
  'importance': [importance for importance in model.feature_importances_]
}).sort_values(by = 'importance', ascending = False)

display(feature_matrix)
#Interestingly atr_zscore_in_block turns out to be the strongest feature in our feature matrix
#minute and day of week are quite weak in this context

'''feature	importance
9	atr_zscore_in_block	0.271053
3	ref_atr	0.164064
4	prior_pullback_depth	0.126660
6	pullback_count_in_block	0.125582
1	hour	0.105537
7	block_duration_so_far	0.101158
8	day_of_week	0.034284
2	minute	0.032862
0	direction_int	0.021678
5	atr_trend_positive	0.017122
'''


In [ ]:
#W4 D5 T3

'''## Task 3 - RandomForest regularization toolbox (25 min)

Since you want real RF competence, not a one-off tuning run: three more levers beyond
`max_depth`, tried individually so you can see what each one actually does.

**Concept - three more overfitting levers:**
- `min_samples_split` - minimum rows required to even consider splitting a node further
  (higher = simpler trees, similar effect to `min_samples_leaf` but at the split decision
  itself, not the leaf).
- `max_features` - how many features each individual tree is allowed to consider at each
  split (e.g. `'sqrt'` = only look at sqrt(n_features) candidates per split) - this is what
  makes a RANDOM forest random; lower values make trees more different from each other,
  which usually helps generalization.
- `max_leaf_nodes` - a hard cap on total leaves in a tree, a different way of limiting
  complexity than depth (a very asymmetric tree could have few levels but many leaves).

- 3a. Starting from yesterday's tuned params, try changing ONLY `max_features` to `'sqrt'`
     (if not already default) and refit - does test AUC change?
- 3b. Try adding `max_leaf_nodes=20` on top - does it change train/test gap at all?

**In a comment:** of the 4 levers now covered (max_depth, min_samples_leaf, max_features,
max_leaf_nodes), which do you now understand as controlling roughly the SAME thing
(tree complexity/size) versus which one is doing something conceptually different
(hint: max_features affects randomness/diversity BETWEEN trees, not one tree's own size)?'''

#I've already used all of these in my randomized search, will try max_leaf_nopdes = 20 and the sqrt and test things out

model = RandomForestClassifier(max_depth = 5, max_features = 'sqrt', max_leaf_nodes = 5, min_samples_leaf = 5, min_samples_split = 3)
model.fit(train[features], train['target_resumed'])
probabilities = model.predict_proba(test[features])[:, 1]
score_rocauc = roc_auc_score(test['target_resumed'], probabilities)

print(score_rocauc)  #0.75


overfit_test_probas = model.predict_proba(train[features])[:, 1]
train_rocauc = roc_auc_score(train['target_resumed'], overfit_test_probas)
print(f'Train rocauc: {train_rocauc}') #Train rocauc: 0.897611937111853 - a bit better, not perfect yet

feature_matrix = pd.DataFrame({
  'feature': [feature for feature in features],
  'importance': [importance for importance in model.feature_importances_]
}).sort_values(by = 'importance', ascending = False)

display(feature_matrix)
#Feature imps with additions:

'''feature	importance
9	atr_zscore_in_block	0.406822
3	ref_atr	0.145506
4	prior_pullback_depth	0.106164
6	pullback_count_in_block	0.084396
1	hour	0.078482
5	atr_trend_positive	0.077166
7	block_duration_so_far	0.056171
0	direction_int	0.028146
8	day_of_week	0.010577
2	minute	0.006569'''




In [ ]:
#W4 D5 T4 - qs

'''## Task 4 - Week 5 summary + weekend quiz plan (10 min)

Quick review, no new code.

- What's the single most useful thing from this week (RF tuning, effect size, leakage
  patterns, feature engineering discipline)?
- What's still shaky?
- Rate the week's difficulty (1-10) and pace.

Weekend: this week's quiz will keep the transfer-question format from last week, and given
today's discussion, will lean toward "judge whether an approach generalizes to a new
scenario" rather than pure recall - consistent with what's worked well recently.'''

# 1. All of these concepts are very useful and I'd like to continue working on all of them as it feels like all are important.
# 2. Randomized search is definitely a bit shaky, getting all the parametres right, getting the time split correctly + making sure evrything works to handle overfitting.
# I wouldn't trust my models done on my own yet without proper guidance - getting there and I get the ideas in a general sense, but still being 100% confident from end to end is not there yet.
# 3. 4-5, the pace is alright - I like today's session as an example of good task volume.



In [ ]:
#W6 D1 WARMUP
'''
Straight repeat of the Week 4 Day 5 pattern (boolean condition computed inline), fresh
columns, no escalation.

```python
pullback_df = pd.read_csv('./project4_trend_regime/m15_pullback_events.csv')
```

**Your turn:** build a dict comprehension mapping each `direction` value to the fraction of
its rows where `ref_atr` is above that direction's OWN median (not a fixed threshold - the
median has to be computed per-direction, inline, inside the comprehension's value
expression).
'''

above_own_median = {direction: (pullback_df.loc[pullback_df['direction'] == direction, 'ref_atr'] > pullback_df.loc[pullback_df['direction'] == direction, 'ref_atr'].median()).mean()
                    for direction in pullback_df['direction'].unique()
                    }

print(above_own_median)

#{'bull': np.float64(0.4998517640083012), 'bear': np.float64(0.5)} #Makes sense XD



In [ ]:
#W6 D1 t1
'''
One-line mental model to replace "bigger sample = more range to differ": a statistical
test compares the observed difference to the STANDARD ERROR of that difference, and standard
error shrinks proportionally to `1/sqrt(n)`. At n=200, noise is large, so only big true
differences look significant. At n=2,000,000, noise is tiny, so even a real but practically
trivial difference clears the significance bar. The sample isn't "gaining more range" - the
measurement is just getting more precise, precise enough to detect small stuff. Say this
back in one sentence in your own words, then move on - not a full task, just locking it in.

'''

#So with bigger sample size there's more room to find meaningful differences between the sets, as it measures the difference to the standard error.


In [ ]:
#W6 D1 t2

'''##One clean, final feature/model pipeline (30 min)

Bring together the best of Week 5 into ONE script section, rather than leaving it scattered
across days - useful practice for how you'd actually hand this off or revisit it later.

- 1a. Rebuild the full 10-feature table from Week 5 Day 5 (direction_int, hour, minute,
     ref_atr, prior_pullback_depth, atr_trend, pullback_count_in_block,
     block_duration_so_far, day_of_week, atr_zscore_in_block), dropping NaN rows, time-aware
     80/20 split.
- 1b. Fit the BEST bounded RandomForest config from Friday (`max_depth=5, max_features='sqrt',
     max_leaf_nodes=5, min_samples_leaf=5, min_samples_split=10`) - no search this time, you
     already know these work well and generalize (test 0.750, train 0.738).
- 1c. Print test AUC, train AUC, and the train/test gap in one clean block, plus
     `feature_importances_` sorted - this is your reference "known good" result to compare
     anything new against today.
'''


model = RandomForestClassifier(max_depth = 5, max_features = 'sqrt', max_leaf_nodes = 5, min_samples_leaf = 5, min_samples_split = 3)
model.fit(train[features], train['target_resumed'])
probabilities = model.predict_proba(test[features])[:, 1]
score_rocauc = roc_auc_score(test['target_resumed'], probabilities)

print(f'Test AUC: {score_rocauc}')  #0.75


overfit_test_probas = model.predict_proba(train[features])[:, 1]
train_rocauc = roc_auc_score(train['target_resumed'], overfit_test_probas)
print(f'Train rocauc: {train_rocauc}') #Train rocauc: 0.897611937111853 - a bit better, not perfect yet

feature_matrix = pd.DataFrame({
  'feature': [feature for feature in features],
  'importance': [importance for importance in model.feature_importances_]
}).sort_values(by = 'importance', ascending = False)


display(feature_matrix)
#Feature imps with additions:

'''
feature	importance
9	atr_zscore_in_block	0.364974
3	ref_atr	0.161094
4	prior_pullback_depth	0.130114
6	pullback_count_in_block	0.088848
5	atr_trend_positive	0.076539
1	hour	0.075123
7	block_duration_so_far	0.043001
0	direction_int	0.038316
8	day_of_week	0.011066
2	minute	0.010924
'''



In [ ]:
#W6 D1 t3


'''**Concept - what's actually different about XGBoost vs RandomForest:** RandomForest builds
many trees INDEPENDENTLY (in parallel) and averages them - each tree doesn't know what the
others got wrong. XGBoost (gradient boosting) builds trees SEQUENTIALLY - each new tree is
trained specifically to correct the mistakes of the trees before it. This often gives
stronger performance, but also makes it easier to overfit if left unchecked (each new tree
is actively hunting for remaining error, including noise) - so it needs its own regularization
levers, not the same ones as RandomForest.

- 2a. `from xgboost import XGBClassifier` (if not installed: `pip install xgboost` - check
     first with a quick import, don't assume). Fit with conservative defaults to start:
     `n_estimators=100, max_depth=3, learning_rate=0.1`. Same features/split as Task 1.
- 2b. Compute test AUC and train AUC. Compare directly to Task 1's RandomForest numbers.
- 2c. XGBoost has `learning_rate` - a new lever RandomForest doesn't have. In plain terms:
     it controls how much each new tree is allowed to correct - lower means slower, more
     cautious learning (needs more trees, usually more robust); higher means faster fitting
     but higher overfitting risk. Try `learning_rate=0.05` with `n_estimators=300` (lower
     rate, more trees to compensate) - does the train/test gap change?

**In a comment:** does XGBoost beat the tuned RandomForest here? Given everything you now
know about this feature set's apparent ceiling (Week 5 Day 1/2 findings), would you expect a
fundamentally more powerful algorithm to break through that ceiling, or is the ceiling more
about the FEATURES than any single algorithm's capability?'''



from xgboost import XGBClassifier
xgb_model = XGBClassifier(n_estimators = 100, max_depth = 3, learning_rate = 0.1)

xgb_model.fit(train[features], train['target_resumed'])
probabilities = xgb_model.predict_proba(test[features])[:, 1]
score_rocauc = roc_auc_score(test['target_resumed'], probabilities)

print(f'Test AUC: {score_rocauc}')  #0.75


overfit_test_probas = xgb_model.predict_proba(train[features])[:, 1]
train_rocauc = roc_auc_score(train['target_resumed'], overfit_test_probas)
print(f'Train rocauc: {train_rocauc}') #Train rocauc: 0.897611937111853 - a bit better, not perfect yet

feature_matrix = pd.DataFrame({
  'feature': [feature for feature in features],
  'importance': [importance for importance in xgb_model.feature_importances_]
}).sort_values(by = 'importance', ascending = False)


display(feature_matrix)


#Test AUC: 0.7098350834042821
#Train rocauc: 0.9063663440059567

'''Featuer importances:



feature	importance
5	atr_trend_positive	0.220599
2	minute	0.128110
9	atr_zscore_in_block	0.120543
1	hour	0.115691
0	direction_int	0.088307
6	pullback_count_in_block	0.078240
3	ref_atr	0.070371
4	prior_pullback_depth	0.065492
7	block_duration_so_far	0.059469
8	day_of_week	0.053177
'''


#Run 2: lower learning rate means slower, more cautious learning - more treess, usually more robust;
# higher means faster fitting but higher overfitting irsk; higher = faster fitting but higher overfitting risk

#lower estimators = more trees to compensate

xgb_model = XGBClassifier(n_estimators = 300, max_depth = 3, learning_rate = 0.05)

xgb_model.fit(train[features], train['target_resumed'])
probabilities = xgb_model.predict_proba(test[features])[:, 1]
score_rocauc = roc_auc_score(test['target_resumed'], probabilities)

print(f'Test AUC: {score_rocauc}')  #0.75


overfit_test_probas = xgb_model.predict_proba(train[features])[:, 1]
train_rocauc = roc_auc_score(train['target_resumed'], overfit_test_probas)
print(f'Train rocauc: {train_rocauc}') #Train rocauc: 0.897611937111853 - a bit better, not perfect yet

feature_matrix = pd.DataFrame({
  'feature': [feature for feature in features],
  'importance': [importance for importance in xgb_model.feature_importances_]
}).sort_values(by = 'importance', ascending = False)


display(feature_matrix)

'''
Run2 :


Test AUC: 0.7009112982057831
Train rocauc: 0.9388499240719835
feature	importance
9	atr_zscore_in_block	0.140947
5	atr_trend_positive	0.139864
1	hour	0.137370
0	direction_int	0.101421
2	minute	0.090568
8	day_of_week	0.089280
6	pullback_count_in_block	0.084685
3	ref_atr	0.078827
4	prior_pullback_depth	0.071242
7	block_duration_so_far	0.065796


#The overfitting issue is +3pp more dangerous here, not a big deal at all, as both are overfitted, but not like 0.99 or 1.00
Also, the atr_zscore_in_block's importance is greatly reduced in the second run, as if it averages down
'''



In [ ]:
#W6 D2 WARMUP

'''```python
build a list of `ref_atr` values for events where `status == 'resumed'` AND
`hour` is between 8 and 16 inclusive (a rough "daytime" filter). Confirm the count against a
boolean-mask `.sum()`.'''



resumed_rth_atrs = pullback_df.loc[(pullback_df['status'] == 'resumed') & (pullback_df['hour'] >= 8) & (pullback_df['hour'] <= 16), 'ref_atr']
print(len(resumed_rth_atrs)) 

mask = (pullback_df['status'] == 'resumed') & (pullback_df['hour'] >= 8) & (pullback_df['hour'] <= 16)
print(mask.sum()) #1831 - ale ten ogólnie krok jest mega dziwny

In [ ]:
#W6 D2 T1

'''
Last attempt still framed it as "more room to differ." Let's ground it in numbers you
already have: yesterday's `above_own_median` warm-up gave `bull: 0.4998...` - almost exactly
0.5, as expected for a median split.

Imagine you ran that SAME calculation on just 50 random bull rows instead of all ~3300.
Purely from small-sample noise, you might get 0.46 or 0.55 instead of ~0.50, by chance alone
- not because the true relationship changed, but because 50 rows barely pins down the true
proportion. With all ~3300 rows, that same calculation lands much closer to the true 0.50,
because more data means your ESTIMATE has less random wobble around the true value.

**In one sentence, using this example:** why does a huge sample make a real-but-tiny
difference look "significant," while a tiny sample can make an even LARGER-looking
difference not look significant at all? (The word "estimate" or "wobble/noise around the
true value" should appea'''

# Because in a sample the difference could get randomly covered by the noise, or some noise could accidentally cover the real difference,
# just by the sheer low number of examples.

# And on huge sample, it's easier to pick really imporatnt differences, as there's enough room to pick evidence, that makes statistical sense.
# Some phenomena require more room to observe, as they occur rarely, and yet the differ meaningfully.

In [ ]:
#W6 D2 T2
'''Same tool, same discipline as last week's RF tuning - but with a DELIBERATELY BOUNDED grid
from the start, applying last week's lesson directly instead of repeating the "too-wide grid"
mistake.

- 1a. `from xgboost import XGBClassifier`. Build a parameter grid with intentionally modest
     ranges (not wide-open): `max_depth`: [2, 3, 4], `learning_rate`: [0.01, 0.05, 0.1],
     `n_estimators`: [50, 100, 200], `subsample`: [0.6, 0.8, 1.0] (subsample = fraction of
     rows each tree sees - another randomness/regularization lever, similar spirit to RF's
     max_features but for ROWS instead of columns).
- 1b. `RandomizedSearchCV(XGBClassifier(), param_distributions=..., n_iter=20,
     cv=TimeSeriesSplit(n_splits=5), scoring=make_scorer(roc_auc_score,
     response_method='predict_proba'), error_score='raise')` - fit on train, same 10-feature
     set as yesterday.
- 1c. Print `.best_params_` and `.best_score_`, then evaluate the best model on the REAL test
     set (not `.best_score_`, which is a CV estimate) - test AUC, train AUC, gap.

**In a comment:** now that XGBoost has had a fair, bounded-grid tuning pass - does it beat,
match, or still lose to yesterday's RF result (test AUC 0.758, train 0.733)? Was yesterday's
conclusion ("RF handles this feature set better") still fair, or was it premature given
XGBoost hadn't been tuned yet?'''


from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer, roc_auc_score

scorer = make_scorer(roc_auc_score, response_method = 'predict_proba')
parameter_grid = {
     'max_depth': [2, 3, 4],
     'learning_rate': [0.01, 0.05, 0.1],
     'n_estimators': [50, 100, 200],
     'subsample': [0.6, 0.8, 1.0] #fraction of rows each tree sees
}

searcher = RandomizedSearchCV(
     estimator = XGBClassifier(),
     param_distributions = parameter_grid,
     scoring = scorer,
     n_iter = 20,
     cv = TimeSeriesSplit(n_splits = 5),
     error_score='raise'
)

searcher.fit(train[features], train['target_resumed'])
print(searcher.best_params_)
print(searcher.best_score_)

'''
{'subsample': 0.6, 'n_estimators': 50, 'max_depth': 4, 'learning_rate': 0.05}
0.6878859506967048
'''

model = XGBClassifier(
     subsample = 0.6,
     n_estimators = 50,
     max_depth = 4,
     learning_rate = 0.05
)
model.fit(train[features], train['target_resumed'])
probas = model.predict_proba(test[features])[:, 1]
test_rocauc = roc_auc_score(test['target_resumed'], probas)

train_probas = model.predict_proba(train[features])[:, 1]
train_rocauc = roc_auc_score(train['target_resumed'], train_probas)

print(f'Test rocauc: {test_rocauc}, train rocauc: {train_rocauc}')

#Test rocauc: 0.7539179516286697, train rocauc: 0.8885379386544328
#Still an instance of overfitting XD - 0.88 vs 0.75 - not terribly bad, yet still bad


In [ ]:
#W6 D2 T3

'''- 2a. Print the tuned XGBoost's `feature_importances_`, sorted. Compare directly to
     yesterday's tuned-RF importances (`atr_zscore_in_block` at 0.365, dominant).
- 2b. Is `atr_zscore_in_block` still the top feature for XGBoost now that it's properly
     tuned (lower learning_rate, bounded depth), or does tuning change which feature XGBoost
     leans on most?

**In a comment:** does today's result change your hypothesis from yesterday (RF vs XGBoost
and how each handles one dominant feature among weak ones)? If the gap between the two
models shrank after fair tuning, what does that tell you about how much of yesterday's
difference was really about the ALGORITHM versus just about tuning effort?'''


model.feature_importances_
feature_df = pd.DataFrame({
     'feature': [feature for feature in features],
     'importance': [importance for importance in model.feature_importances_]
}
).sort_values(by = 'importance', ascending = False)

display(feature_df)

'''
feature	importance
5	atr_trend_positive	0.152349
9	atr_zscore_in_block	0.142851
1	hour	0.138997
3	ref_atr	0.102270
4	prior_pullback_depth	0.096191
2	minute	0.089893
6	pullback_count_in_block	0.083155
7	block_duration_so_far	0.069765
0	direction_int	0.067994
8	day_of_week	0.056536
'''

#atr_zscore_in_block still gets massively diluted by other features, it's not even the first feature
#Today's tuning effort didn't give much tbh :))

In [ ]:
#W6 D3 WARMUP
'''
**Your turn:** build a dict comprehension mapping each `regime` value to the fraction of its
rows where `close > ema144_high` (a rough "trading above the slow channel" flag) - same
inline-boolean-then-mean pattern as before.
'''

m15_df = pd.read_csv('./project4_trend_regime/m15_with_regime.csv', parse_dates=['et_time'])
m15_df.head()


regime_above_slow_channel = {
    
    regime: (m15_df.loc[m15_df['regime'] == regime, 'close'] > m15_df.loc[m15_df['regime'] == regime, 'ema144_low']).mean()
    for regime in m15_df['regime'].unique()
    
}

print(regime_above_slow_channel)
'''
'range_recross': np.float64(0.6224893440863513), 
 'strong_bull': np.float64(1.0), 
 'retracement_bull': np.float64(0.6225739026575097), 
 'strong_bear': np.float64(0.0), 
 'retracement_bear': np.float64(1.0)
'''

In [ ]:
#W6 D3 T1
'''## Task 1 - Stationarity via ADF test (25 min)

**Concept - what ADF actually tests:** you don't run ADF on raw price (which obviously
drifts over years) - you run it on a DEVIATION series, like `close - rolling_mean(close, N)`.
If that deviation series is stationary (keeps returning to zero, doesn't wander off
permanently), ADF's null hypothesis ("this series has a unit root / is a random walk, NOT
stationary") gets rejected at a low p-value - meaning: evidence FOR mean-reversion. This is
the same H0/H1 logic as Kruskal-Wallis back in Week 3 - a low p-value rejects "no structure
here," it doesn't itself measure how strong or tradeable the effect is.

- 1a. `from statsmodels.tsa.stattools import adfuller`. Load `m15_with_regime.csv`. Build
     `deviation = close - close.rolling(200).mean()` (200 M15 bars ≈ 2 days - a reasonable
     starting window, arbitrary but round). Drop the resulting NaNs (first 199 rows).
- 1b. Run `adfuller(deviation)` on the FULL deviation series - print the test statistic and
     p-value. `adfuller` returns a tuple; the p-value is index 1.
- 1c. In a comment: given the p-value, does XAUUSD's price show evidence of mean-reversion
     around its own 200-bar rolling mean, or does the deviation behave more like a random
     walk (fails to reject H0)?'''
     
from statsmodels.tsa.stattools import adfuller
m15_df.head()   
m15_df['deviation'] = m15_df['close'] - m15_df['close'].rolling(200).mean()
m15_df['deviation_50'] = m15_df['close'] - m15_df['close'].rolling(50).mean()
m15_df = m15_df.dropna()

adf_scores = adfuller(m15_df['deviation'])
print(adf_scores)
print(f'Adfuller p-value = {adf_scores[1]}') #0.0 ("this series has a unit root / is a random walk, NOT stationary" - rejceted)

'''(np.float64(-26.035859761171643), 
0.0, 
71, 
121348, 
{'1%': np.float64(-3.4304038899556217),
'5%': np.float64(-2.861563818562268),
'10%': np.float64(-2.566782677779183)},
np.float64(692412.2842732007))
'''


#It seems like it shows evidence of mean reversion, unless I'm interpreting it wrong - it's clear that H0 is rejected.

In [ ]:
#W6 D3 T2
'''## Task 2 - Hurst Exponent, done at a reliable window size (30 min)

**Concept - a simple, honest Hurst estimate:** the rigorous R/S method has real small-sample
bias issues (research found upward bias vs. DFA, especially under ~256 points). For today, a
simpler, well-known approximation via the variance of the differenced series at different
lags is good enough to get a directional read, AS LONG AS you use a window that's actually
large enough (256+ points, not 50).

- 2a. Write a small `hurst_exponent(series)` function using the variance-scaling method: for
     a range of lags (e.g. `[2, 4, 8, 16, 32, 64]`), compute
     `np.std(np.subtract(series[lag:], series[:-lag]))` for each lag, then fit a line to
     `log(lag)` vs `log(that std)` - the SLOPE of that line is your Hurst estimate. (This is
     a standard, simple implementation - if you want, look up "Hurst exponent Python variance
     method" to see the exact small function, then adapt it rather than reinventing it from
     scratch blind.)
- 2b. Apply it on a ROLLING basis: for every 300-bar window (stepping every 50 bars, not
     every single bar, to keep runtime sane), compute the Hurst exponent of `close` in that
     window. Build a Series/DataFrame of (window end time, Hurst value).
- 2c. Plot the rolling Hurst values over time (a simple line plot). What fraction of windows
     fall below 0.45 (mean-reverting), between 0.45-0.55 (random-walk-like), above 0.55
     (trending)?

**In a comment:** does this match or contradict Task 1's ADF read? Remember the research
finding - a single Hurst reading near 0.5 with real estimation noise doesn't mean much on its
own; the DISTRIBUTION across many windows over time is more informative than any one value.'''

from hurst import compute_Hc



H, c, data = compute_Hc(m15_df['close'], kind='price', simplified=True)
print(H, c, data)


# Plot
f, ax = plt.subplots()
ax.plot(data[0], c*data[0]**H, color="deepskyblue")
ax.scatter(data[0], data[1], color="purple")
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Time interval')
ax.set_ylabel('R/S ratio')
ax.grid(True)
plt.show()

print("H={:.4f}, c={:.4f}".format(H,c))

#H=0.6277, c=0.7637


#If I'm not mistaken, it shows a persistent behavior, but I could've done something wrong here.
#I've found a relevant library on GitHub to calculate this.
#I could've done something wrong or oversimplified things.

In [ ]:
#W6 D3 T3
'''## Task 3 - Half-life, and an honest verdict (25 min)

**Concept - half-life via AR(1):** fit `deviation[t] = a + b * deviation[t-1] + error` (a
simple linear regression of the deviation on its own previous value - `LinearRegression` on
one feature). If `b` is between 0 and 1, the deviation shrinks back toward zero over time;
half-life (in bars) is `-log(2) / log(b)`. A very long half-life means mean-reversion is
technically present but too slow to trade profitably after costs; research suggests 5-60
BARS is the tradeable range for a strategy at this kind of frequency (not days, since we're
on M15 not daily bars - so translate that intuition to bars, not days).

- 3a. Using the same `deviation` series from Task 1, build `deviation_lagged = deviation.shift(1)`,
     drop NaNs, fit `LinearRegression` predicting `deviation` from `deviation_lagged`.
- 3b. Extract the slope (`b`), compute half-life via the formula above. Is it a sane, finite,
     positive number, or does something look wrong (e.g. b >= 1, meaning no reversion at
     all)?

**In a comment - the actual verdict for today:** pulling ADF (Task 1), Hurst distribution
(Task 2), and half-life (Task 3) together - is there real, measurable, plausibly-tradeable
mean-reversion in XAUUSD M15 around a 200-bar rolling mean? Say this plainly, one way or the
other - a clear "no" backed by these three tools is just as valuable a result as a "yes."'''

import numpy as np
import statsmodels.api as sm

def calculate_half_life(time_series):
    # Przesunięcie szeregu o 1 okres (lag)
    y = np.asarray(time_series)
    dy = np.diff(y)              # Δy_t = y_t - y_{t-1}
    y_lag = y[:-1]               # y_{t-1}
    
    # Dodanie stałej do regresji OLS
    X = sm.add_constant(y_lag)
    model = sm.OLS(dy, X)
    res = model.fit()
    
    # Współczynnik lambda przy y_{t-1}
    lam = res.params[1]
    
    # Obliczenie half-life
    half_life = -np.log(2) / lam
    return half_life

hl_value = calculate_half_life(m15_df['deviation'])

m15_df['half_life'] = hl_value
m15_df.tail()  #93.419468

#Frankly, I have no idea, I'm not a data scientist yet.

In [ ]:
#W6 D4 WARMUP
'''```python
m15 = pd.read_csv('./project4_trend_regime/m15_with_regime.csv', parse_dates=['et_time'])
```

**Your turn:** build a dict comprehension mapping each `regime` value to the fraction of its
rows where `close < ema144_low` (mirror of yesterday's warm-up, opposite side).'''

regime_below_ema144_low = {
    regime : (m15_df.loc[m15_df['regime'] == regime, 'close'] < m15_df.loc[m15_df['regime'] == regime, 'ema144_low']).mean()
    for regime in m15_df['regime'].unique()
}
print(regime_below_ema144_low)

'''
{'strong_bull': np.float64(0.0), 
'retracement_bull': np.float64(0.3782569631626235), 
'range_recross': np.float64(0.37778868874090893), 
'strong_bear': np.float64(1.0), 
'retracement_bear': np.float64(0.0)}
'''

In [ ]:
#W6 D4 T1

m15_df.head()
#deviation is already there
H, c, data = compute_Hc(series = m15_df['deviation'], kind = 'change', simplified = True)
print(H) #0.522782653194648
#It's definitely closer than yesterday, not sure whether 0.52 is a signifier of a strong mean reverting data, but it's definitely closer.


In [ ]:
#W6 D4 T2

'''## Task 2 - Try a shorter window: does speed improve? (25 min)

Yesterday's 93-bar half-life was measured on a 200-bar rolling mean. A shorter mean might
track price more closely, producing smaller, faster-reverting deviations - worth testing
directly rather than assuming.

**Reminder - the half-life function, self-contained (same as yesterday's Task 3):**

```python
import numpy as np
import statsmodels.api as sm

def calculate_half_life(time_series):
    y = np.asarray(time_series)
    dy = np.diff(y)              # change from one bar to the next
    y_lag = y[:-1]                # the series one step behind dy

    X = sm.add_constant(y_lag)    # adds an intercept column so OLS can fit a + b*y_lag
    model = sm.OLS(dy, X)
    res = model.fit()

    lam = res.params[1]           # the slope on y_lag - how strongly today's level predicts tomorrow's change
    half_life = -np.log(2) / lam  # bars needed for the deviation to shrink halfway back to zero
    return half_life
```

- 2a. Rebuild `deviation_50 = close - close.rolling(50).mean()` (quarter of yesterday's
     window). Run `calculate_half_life(deviation_50)` (drop any NaNs first).
- 2b. Re-run ADF (`adfuller(deviation_50.dropna())`) too - still p≈0, or does a shorter
     window change that picture at all?

**In a comment - updated verdict:** compare the 50-bar window's half-life to yesterday's
200-bar result (93.4 bars). Is reversion meaningfully faster at this shorter window - closer
to the 5-60 bar tradeable range - or does it stay similarly slow regardless of window choice?'''


#m15_df['deviation_50'] = m15_df['close'] - m15_df['close'].rolling(50).mean() 
# - executed above in the same cell where I calculate deviation, so I don't delete another 49 rows

half_life_value = calculate_half_life(m15_df['deviation_50'])
adfuller_value = adfuller(m15_df['deviation_50'])

print(half_life_value) #22.17001591847692
print(adfuller_value) #(np.float64(-34.09721333869626), 0.0, 71, 121746, {'1%': np.float64(-3.430403713780186), '5%': np.float64(-2.8615637406960395), '10%': np.float64(-2.5667826363336177)}, np.float64(690650.5356657901))

In [ ]:
#W6 D5 WARMUP
'''**Your turn:** build a dict comprehension mapping each `regime` value to the mean `atr14`
for that regime (plain mean this time, no boolean condition needed - just to vary the shape
slightly).'''



regimes_mean_atr14 = {
    regime: m15_df.loc[m15_df['regime'] == regime, 'atr14'].mean()
    for regime in m15_df['regime'].unique()
}

print(regimes_mean_atr14)

'''{'strong_bull': np.float64(3.752236649663511), 
'retracement_bull': np.float64(4.543574201774487), 
'range_recross': np.float64(3.9195455391136904), 
'strong_bear': np.float64(4.338025358183098), 
'retracement_bear': np.float64(4.150508765228352)}'''

#Quite frankly, there are no striking differences between these regimes
#ATR is relative as well, so it doesnt surprise me

In [ ]:
#W6 D5 T1
'''## 
Task 1 - Pandas practice: half-life across several window sizes (25 min)

Pure pandas/aggregation work - no new statistical concepts, applying what you already have
across more window sizes and organizing results into one clean table.

**Reminder - the half-life function, self-contained:**

```python
import numpy as np
import statsmodels.api as sm

def calculate_half_life(time_series):
    y = np.asarray(time_series)
    dy = np.diff(y)
    y_lag = y[:-1]
    X = sm.add_constant(y_lag)
    model = sm.OLS(dy, X)
    res = model.fit()
    lam = res.params[1]
    half_life = -np.log(2) / lam
    return half_life
```

- 1a. For window sizes `[20, 30, 50, 75, 100, 150, 200]`: for each one, build
     `deviation_N = close - close.rolling(N).mean()`, drop NaNs, compute
     `calculate_half_life(deviation_N)` and the ADF p-value
     (`from statsmodels.tsa.stattools import adfuller`; `adfuller(deviation_N)[1]`). Do this
     as a loop building up a list of dicts (one dict per window with keys `window`,
     `half_life`, `adf_pvalue`), then `pd.DataFrame(that_list)`.
- 1b. Sort the table by `half_life` ascending. Does half-life shrink smoothly and predictably
     as the window gets shorter, or does it jump around unevenly?

**In a comment:** which window size looks like the best candidate - fastest reversion while
still having a sane, stable ADF p-value (not one weird window that might be a fluke)?'''


import numpy as np
import statsmodels.api as sm

def calculate_half_life(time_series):
    y = np.asarray(time_series)
    dy = np.diff(y)
    y_lag = y[:-1]
    X = sm.add_constant(y_lag)
    model = sm.OLS(dy, X)
    res = model.fit()
    lam = res.params[1]
    half_life = -np.log(2) / lam
    return half_life

windows = [20, 30, 50, 75, 100, 150, 200]
results_dict = {}
for window in windows:
    m15_df[f'deviation_{window}'] = m15_df['close'] - m15_df['close'].rolling(window).mean()
#I decided to use double loop, which may look a bit weird,
#BUT I would delete the N-1 number of rows with every other iteration otherwise when using dropna()
m15_df = m15_df.dropna() #one collective dropna() to deal with all windows at once
for window in windows:
    hl_calc = calculate_half_life(m15_df[f'deviation_{window}'])
    adfuller_calc = adfuller(m15_df[f'deviation_{window}'])[1]
    results_dict[window] = {
        'half_life': hl_calc,
        'adfuller': adfuller_calc
    }

results = pd.DataFrame(
    results_dict
)
display(results)
'''	20	30	50	75	100	150	200
half_life	8.726207	13.162958	22.165758	33.897877	46.079748	70.562235	93.381607
adfuller	0.000000	0.000000	0.000000	0.000000	0.000000	0.000000	0.000000
'''
    
    
    

In [ ]:
#W6 D5 T2
'''## Task 2 - Build the actual feature: a z-score, not just a raw deviation (25 min)

**Concept - why a z-score instead of raw deviation:** `deviation` (price minus rolling mean)
is in RAW PRICE UNITS (dollars for gold) - a deviation of $5 means something very different
when ATR is $3 (huge, unusual) versus when ATR is $20 (small, normal). A z-score fixes this:
`(deviation) / deviation.rolling(N).std()` expresses the deviation in STANDARD DEVIATIONS
from the mean, which is comparable across different volatility regimes - this is the
mean-reversion equivalent of what `atr_zscore_in_block` did for the pullback project.

- 2a. Using your best window size from Task 1, build
     `z = deviation_BEST / deviation_BEST.rolling(BEST).std()` (same window for both the
     mean and the std, for consistency). Drop NaNs.
- 2b. `.describe()` the z-score column - does it look roughly centered on 0, with most values
     in a sane range (say -3 to 3), or are there extreme outliers suggesting something's off?
- 2c. Plot a histogram of `z` (seaborn `histplot` or similar) - does it look roughly
     bell-shaped (as you'd expect if deviations are "random noise around a reverting mean"),
     or heavily skewed?

**In a comment:** if you picked a threshold like `abs(z) > 2` (roughly top/bottom ~5% of
deviations under a normal assumption) as "price is unusually far from its mean, a reversion
candidate" - what fraction of all rows would that flag? Does that fraction seem like a
reasonable number of potential trade signals, or wildly too many/too few?'''

#assuming 30 is the best window, but THAT COULD BE ENTIRELY WRONG - we need to consult this :))
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import normaltest

m15_df['z_score'] = m15_df['deviation_30'] / m15_df['deviation_30'].rolling(30).std()
m15_df = m15_df.dropna()
display(m15_df['z_score'].describe())
'''
count    121503.000000
mean          0.100424
std           1.578609
min          -8.437553
25%          -0.944998
50%           0.088523
75%           1.137666
max           8.510436
Name: z_score, dtype: float64'''
#I'd say it looks pretty centered, maybe not perfectly and percentiles seem to work in a reversible way, eg.
#25% is similar to 75% but as negative value, min similar to max etc.


plt.Figure(figsize = (16, 10))
histogram_zscore = sns.histplot(
    m15_df['z_score']
)
plt.show()
#Quite frankly the hist plot looks like a normal distirbution with a clean bell curve
#I'd even consider doing a normaltest

n_test = normaltest(m15_df['z_score']) #H0 - this is a normal distribution
print(n_test) #pvalue=np.float64(1.333000796203678e-111) - well, not a normal distribution XD


#I assume choosing extreme threshold of deviations would result in a very low number of signals, and I'd like to explore that


display(m15_df['z_score'].describe(percentiles = [0.01, 0.03, 0.05, 0.1, 0.9, 0.95, 0.97, 0.99]))
#That looks more reasonable, as it feels like we'd be able to capture 99% of the highest ATR entries 
#within 4 STD threshold
'''1%           -3.664082
3%           -2.852683
5%           -2.450508
10%          -1.878286
90%           2.085188
95%           2.693124
97%           3.101494
99%           3.912681
max           8.510436
Name: z_score, dtype: float64'''

#IMO it's not always about the quantity of setups, but rather their quality and precision,
#Not saying it makes sense as it would have to be tested properly, but it's interesting


In [ ]:
#W6 D5 T3
'''
## Task 3 - Does the z-score actually predict reversion? A first real check (25 min)

Not a model yet - just the first honest look at whether extreme z-score rows actually behave
differently afterward, the same "verify before modeling" discipline from the pullback
project.

- 3a. For each row, compute what happens N bars later: `future_return_N = close.shift(-N) - close`
     (try N=20, roughly matching your Task 1 window scale). This looks FORWARD - a reminder
     that this column could never be used as a model FEATURE (it needs future data), only as
     a target/label to check against, same logic as your resumed/recrossed labels before.
- 3b. Split rows into three groups: `z > 2` (unusually high), `z < -2` (unusually low), and
     `-2 <= z <= 2` (normal range). Compare `future_return_N`'s mean/median across the three
     groups.
- 3c. If mean-reversion is real: rows with `z > 2` (price too high) should show NEGATIVE
     future returns on average (price pulling back down), and `z < -2` rows should show
     POSITIVE future returns (pulling back up), relative to the normal-range group.

**In a comment - the actual verdict:** does the pattern in 3c show up in your numbers? If
yes, how large is the difference between groups (is it a few cents, or a meaningfully-sized
move relative to typical ATR)? If no, does that contradict yesterday's ADF/half-life
findings, or could "exists statistically" and "shows up cleanly in a simple 3-group split"
be different bars to clear?
'''

m15_df['future_return_20'] = m15_df['close'].shift(-20) - m15_df['close']

unusually_low_z = m15_df.loc[m15_df['z_score'] < -2, 'future_return_20'] #z < -2
high_z = m15_df.loc[m15_df['z_score'] > 2, 'future_return_20']
normal_z = m15_df.loc[(m15_df['z_score'] <= 2) & (m15_df['z_score'] >= -2), 'future_return_20']

plt.Figure(figsize = (24, 12))
sns.histplot(
    unusually_low_z
)
plt.title('Returns after 20 candles from rows with z_score below -2')
plt.show()


plt.Figure(figsize = (24, 12))
sns.histplot(
    normal_z
)
plt.title('Returns after 20 candles from rows with z_score above -2 and below 2 (normal)')
plt.show()


plt.Figure(figsize = (24, 12))
sns.histplot(
    high_z
)
plt.title('Returns after 20 candles from rows with z_score above 2')
plt.show()



display(unusually_low_z.describe())
display(high_z.describe())
display(normal_z.describe())

'''count    10548.000000
mean        -0.226044
std         22.734432
min       -440.470000
25%         -5.457000
50%          0.514500
75%          6.222500
max        321.030000
Name: future_return_20, dtype: float64count    13326.000000
mean         1.007914
std         16.202162
min       -135.330000
25%         -5.233000
50%          0.235000
75%          6.249250
max        229.000000
Name: future_return_20, dtype: float64count    97580.000000
mean         0.329577
std         18.352904
min       -433.057000
25%         -4.360000
50%          0.320500
75%          5.340000
max        253.990000
Name: future_return_20, dtype: float64'''

#It's not entirely clear which option gives what result but:
#z_score below -2 results are more skewed to the left
# and a bit skewed to the right compared with the "normal_z"
#z_score above 2 are actually a bit more skewed to the right than to the left,
# and also the tail on the left stops on much lower vlaues

In [ ]:

#W7 D1 WARMUP
'''
## Warm-up (10 min) - repetition: dict comprehension, fresh column

Straight repeat, no escalation.

```python
m15 = pd.read_csv('./project4_trend_regime/m15_with_regime.csv', parse_dates=['et_time'])
```

**Your turn:** build a dict comprehension mapping each `regime` value to the median
`atr14` for that regime (median this time, not mean - to see if the Week 6 Day 5 "no
striking difference" finding holds up with a different, more outlier-resistant statistic).'''


median_atr_by_regime = {
    regime: m15_df.loc[m15_df['regime'] == regime, 'atr14'].median()
    for regime in m15_df['regime'].unique()
}

print(median_atr_by_regime)
'''
{'strong_bear': np.float64(2.6036040966142777), 
 'retracement_bear': np.float64(2.785985671824304), 
 'range_recross': np.float64(2.673737652445282), 
 'strong_bull': np.float64(2.585759253717905), 
 'retracement_bull': np.float64(3.037218947902601)}
 '''



In [ ]:
#W7 D1 T1

'''## Task 1 - Remove the drift: measure return relative to the rolling mean, not raw price (25 min)

**Concept - why this fixes the confound:** `close.shift(-N) - close` measures the ABSOLUTE
price change N bars later - if the whole market drifted up during your sample period, EVERY
group (regardless of z-score) inherits some of that positive drift, making the comparison
unfair. Instead, measure where price ends up RELATIVE TO THE SAME ROLLING MEAN used to build
the z-score: `future_deviation_N = close.shift(-N) - close.rolling(WINDOW).mean()` (the mean
itself isn't shifted - you're asking "N bars from now, how far is price from TODAY's rolling
mean," which is a drift-neutral way to ask whether price reverted toward where it recently
was centered).

- 1a. Rebuild your best window's `deviation` and `z` columns from Friday (pick whichever
     window you landed on - 30 or 50). Build
     `future_deviation_20 = close.shift(-20) - close.rolling(WINDOW).mean()`.
- 1b. Split into the same three groups as Friday (`z < -2`, `z > 2`, `-2 <= z <= 2`). Compare
     `future_deviation_20` mean/median across groups - same direction as before, or does it
     flip once drift is removed?

**In a comment:** does removing the drift change the sign of the pattern? If yes, that
confirms the drift was the real culprit, not an actual momentum effect in the data.'''\
     
     
m15_df = m15_df.drop(columns = ['deviation', 'deviation_50', 'deviation_20',	'deviation_75', 'deviation_100', 'deviation_150', 'deviation_200'])
m15_df['future_deviation_20'] = m15_df['close'].shift(-20) - m15_df['close'].rolling(30).mean()
m15_df = m15_df.dropna()




unusually_low_z = m15_df.loc[m15_df['z_score'] < -2, 'future_deviation_20'] #z < -2
high_z = m15_df.loc[m15_df['z_score'] > 2, 'future_deviation_20']
normal_z = m15_df.loc[(m15_df['z_score'] <= 2) & (m15_df['z_score'] >= -2), 'future_deviation_20']

plt.Figure(figsize = (24, 12))
sns.histplot(
    unusually_low_z
)
plt.title('Deviation after 20 candles from rows with z_score below -2')
plt.show()


plt.Figure(figsize = (24, 12))
sns.histplot(
    normal_z
)
plt.title('Deviation after 20 candles from rows with z_score above -2 and below 2 (normal)')
plt.show()


plt.Figure(figsize = (24, 12))
sns.histplot(
    high_z
)
plt.title('Deviation after 20 candles from rows with z_score above 2')
plt.show()



display(unusually_low_z.describe(percentiles = [0.01, 0.03, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.97, 0.99]))
display(high_z.describe(percentiles = [0.01, 0.03, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.97, 0.99]))
display(normal_z.describe(percentiles = [0.01, 0.03, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.97, 0.99]))




#zscore -2 - very long left tail
#normal - more left tail  than right tail,
#above 2 - medium right tail, left tail short

In [ ]:
#W7 D1 T2
'''## Task 2 - Test more extreme thresholds (25 min)

Build the same three-way comparison, but sweep through a range of thresholds instead of
assuming ±2 is where any effect lives.

- 2a. For thresholds `[1.5, 2.0, 2.5, 3.0, 3.5]`: for each, compute the mean
     `future_deviation_20` for rows with `z > threshold` and separately for `z < -threshold`
     (two numbers per threshold). Build this into a small DataFrame (loop -> list of dicts ->
     `pd.DataFrame`, same pandas pattern as Friday's Task 1 table).
- 2b. Does the pattern (sign and size of the effect) get cleaner/stronger as the threshold
     gets more extreme, or does it stay noisy/flat regardless of threshold? Also check: how
     many rows does each threshold leave you with (`.shape[0]`) - a threshold with a "strong"
     effect but only 50 rows is a different kind of finding than one with 5,000 rows.

**In a comment:** based on this table, is there a threshold where a real, usable-looking
reversion signal shows up? Or does extending further into the tails just produce noisier
results from shrinking sample sizes, with no clear signal at any point?'''

thresholds = [1.5, 2.0, 2.5, 3.0, 3.5]
results = []

for threshold in thresholds:
     below_threshold_mean = m15_df.loc[m15_df['z_score'] < (threshold * -1), 'future_deviation_20'].mean()
     below_threshold_median = m15_df.loc[m15_df['z_score'] < (threshold * -1), 'future_deviation_20'].median()
     below_threshold_q10 = m15_df.loc[m15_df['z_score'] < (threshold * -1), 'future_deviation_20'].quantile(0.1)
     below_threshold_q25 = m15_df.loc[m15_df['z_score'] < (threshold * -1), 'future_deviation_20'].quantile(0.25)
     below_threshold_q75 = m15_df.loc[m15_df['z_score'] < (threshold * -1), 'future_deviation_20'].quantile(0.75)
     below_threshold_q90 = m15_df.loc[m15_df['z_score'] < (threshold * -1), 'future_deviation_20'].quantile(0.90)
      
      
     above_threshold_mean = m15_df.loc[m15_df['z_score'] > threshold, 'future_deviation_20'].mean()
     above_threshold_median = m15_df.loc[m15_df['z_score'] > threshold, 'future_deviation_20'].median()
     above_threshold_q10 = m15_df.loc[m15_df['z_score'] > threshold, 'future_deviation_20'].quantile(0.1)
     above_threshold_q25 = m15_df.loc[m15_df['z_score'] > threshold, 'future_deviation_20'].quantile(0.25)
     above_threshold_q75 = m15_df.loc[m15_df['z_score'] > threshold, 'future_deviation_20'].quantile(0.75)
     above_threshold_q90 = m15_df.loc[m15_df['z_score'] > threshold, 'future_deviation_20'].quantile(0.90)
     
     results.append(
          {'threshold': threshold,
           'below_threshold_mean': below_threshold_mean,
           'below_threshold_median': below_threshold_median,
           'below_threshold_q10': below_threshold_q10,
           'below_threshold_q25': below_threshold_q25,
           'below_threshold_q75': below_threshold_q75,
           'below_threshold_q90': below_threshold_q90,
           'above_threshold_mean': above_threshold_mean,
           'above_threshold_median': above_threshold_median,
           'above_threshold_q10': above_threshold_q10,
           'above_threshold_q25': above_threshold_q25,
           'above_threshold_q75': above_threshold_q75,
           'above_threshold_q90': above_threshold_q90
           }
     )

results_df = pd.DataFrame(results)
results_df.head()


     


In [ ]:
#W7 D1 T3
'''## Task 3 - Make it formal: Mann-Whitney on the best-looking threshold (20 min)

Pick whichever threshold from Task 2 looked most promising (biggest, most consistent effect,
reasonable sample size) and test it properly instead of eyeballing the table.

- 3a. `from scipy.stats import mannwhitneyu`. Compare `future_deviation_20` for
     `z > best_threshold` against the `-2 <= z <= 2` "normal" group (write out explicitly
     which two groups you're comparing before running it - same forcing-function habit from
     Week 5). Repeat for `z < -best_threshold` vs normal.
- 3b. Print the p-values. Are the differences statistically real, or could they plausibly be
     noise given the sample sizes involved?

**In a comment - the actual verdict:** pulling Tasks 1-3 together, do you now believe there's
a real, drift-adjusted mean-reversion signal in XAUUSD at extreme z-score deviations? State
it plainly either way - "yes, roughly at threshold X" or "no, the apparent effect doesn't
survive a fair, formal test" are both acceptable, useful conclusions.'''


from scipy.stats import mannwhitneyu

above_25 = m15_df.loc[m15_df['z_score'] > 2.5, 'future_deviation_20']
below_25 = m15_df.loc[m15_df['z_score'] < -2.5, 'future_deviation_20']

above_3 = m15_df.loc[m15_df['z_score'] > 3, 'future_deviation_20']
below_3 = m15_df.loc[m15_df['z_score'] < -3, 'future_deviation_20']

above_35 = m15_df.loc[m15_df['z_score'] > 3.5, 'future_deviation_20']
below_35 = m15_df.loc[m15_df['z_score'] < -3.5, 'future_deviation_20']

normal_z = m15_df.loc[(m15_df['z_score'] <= 2) & (m15_df['z_score'] >= -2), 'future_deviation_20']


p_vals = []
groups = [('above_25', above_25), ('below_25', below_25), ('above_3', above_3), ('below_3', below_3), ('above_35', above_35), ('below_35', below_35)]


### below cohens_d function was fetched from the internet!
import numpy as np

def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    n1, n2 = len(x), len(y)
    s1, s2 = x.std(ddof=1), y.std(ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    return (x.mean() - y.mean()) / pooled_sd
###########

for group in groups:
     test_mw = mannwhitneyu(group[1], normal_z)
     cohen = cohens_d(group[1], normal_z)
     p_vals.append(
          {
          'group': group[0],
          'p_val': test_mw[1],
          'cohens_d': cohen
          }
     )

result_df = pd.DataFrame(p_vals)
result_df.head()


'''
group	p_val	cohens_d
0	above_25	0.0	0.713224
1	below_25	0.0	-0.804704
2	above_3	0.0	0.750902
3	below_3	0.0	-0.915584
4	above_35	0.0	0.750297'''

In [ ]:
#W7 D2 WARMUP
'''## Warm-up (10 min) - repetition: dict comprehension with per-group computed value

Straight repeat, no escalation.

```python
m15 = pd.read_csv('./project4_trend_regime/m15_with_regime.csv', parse_dates=['et_time'])
```

**Your turn:** build a dict comprehension mapping each `regime` value to the fraction of its
rows where `atr14` is above the OVERALL (not per-regime) median `atr14` - same
inline-boolean-then-mean pattern as several previous warm-ups, just comparing against a
single global threshold instead of a per-group one this time.'''


regimes_fraction_above_atr14 = {
    regime : (m15_df.loc[m15_df['regime'] == regime, 'atr14'] > m15_df['atr14'].median()).mean()
    for regime in m15_df['regime'].unique()
}
print(regimes_fraction_above_atr14)
'''
{
'strong_bear': np.float64(0.4932324180984476), 
'retracement_bear': np.float64(0.5411061285500748),
'range_recross': np.float64(0.5098298212201265), 'strong_bull': np.float64(0.48897082745906084),
'retracement_bull': np.float64(0.5886056971514243)}
'''



In [ ]:

##W7 D2 T1
'''## Task 1 - Define the actual target (30 min)

**The goal in plain words:** right now you've only measured AVERAGE future deviation across
three z-score groups. A real target asks, PER ROW: "once this row crosses an extreme z-score
threshold, does price actually revert by a meaningful amount within N bars, or not?" - a
binary yes/no outcome per event, not a population-level average.

- 1a. Rebuild `deviation`, `z_score` (window=30, matching yesterday) and reload
     `m15_with_regime.csv` fresh - keep this self-contained rather than assuming leftover
     notebook state.
- 1b. Define "extreme event" rows: `|z_score| > 2.5` (yesterday's threshold with a strong,
     confirmed effect). For EACH such row, compute whether price reverts: for an extreme-HIGH
     row (z>2.5), reversion means price drops back toward the mean - check if
     `close.rolling(30).mean()` (today's mean) is reached or crossed by the close price at
     ANY point in the next 20 bars (use `close.shift(-k)` for k in 1..20, or a rolling-min/
     max over the forward window - whichever feels more natural, just explain your choice).
     Mirror the opposite logic for extreme-LOW rows (z<-2.5, reversion means price rises back
     up to the mean).
- 1c. Build `target_reverted` (1 if it reverted within 20 bars, 0 if not) for extreme rows
     only (same pattern as the pullback project dropping non-extreme rows before building
     the target - you're not trying to predict anything for "normal" rows here).

**In a comment:** what fraction of extreme events reverted within 20 bars? Does this
roughly match what you'd expect given yesterday's half-life numbers (half-life ~13-22 bars
means HALF reverted by then, not necessarily fully "touched the mean")?'''


m15_df = pd.read_csv('project4_trend_regime/m15_with_regime.csv')
m15_df['deviation_30'] = m15_df['close'] - m15_df['close'].rolling(30).mean()
m15_df['future_return_20'] = m15_df['close'].shift(-20) - m15_df['close']
m15_df['future_deviation_20'] = m15_df['close'].shift(-20) - m15_df['close'].rolling(30).mean()
m15_df['z_score_30'] = m15_df['deviation_30'] / m15_df['deviation_30'].rolling(30).std()
m15_df['extreme_high_event'] = m15_df['z_score_30'].apply(lambda x: 1 if x > 2.5 else 0)
m15_df['extreme_low_event'] = m15_df['z_score_30'].apply(lambda x: 1 if x < -2.5 else 0)
m15_df = m15_df.dropna()
m15_df['new_high_episode'] = m15_df['extreme_high_event'] & ~m15_df['extreme_high_event'].shift(1, fill_value=False).astype(bool)
m15_df['new_low_episode'] = m15_df['extreme_low_event'] & ~m15_df['extreme_low_event'].shift(1, fill_value = False).astype(bool)




extreme_rows = m15_df.loc[(m15_df['new_high_episode'] == 1) | (m15_df['new_low_episode'] == 1)]

extreme_high_events_returns = m15_df.loc[m15_df['new_high_episode'] == 1, 'future_return_20'].mean()
extreme_high_events_future_deviation = m15_df.loc[m15_df['new_high_episode'] == 1, 'future_deviation_20'].mean()
extreme_low_events_returns = m15_df.loc[m15_df['new_low_episode'] == 1, 'future_return_20'].mean()
extreme_low_events_future_deviation = m15_df.loc[m15_df['new_low_episode'] == 1, 'future_deviation_20'].mean()


extreme_high_mean_deviation = m15_df.loc[m15_df['new_high_episode'] == 1, 'deviation_30'].mean()
extreme_low_mean_deviation = m15_df.loc[m15_df['new_low_episode'] == 1, 'deviation_30'].mean()
print(f'Extreme high returns: {extreme_high_events_returns}, deviation: {extreme_high_events_future_deviation}, deviation at the event {extreme_high_mean_deviation}')
print(f'Extreme low returns: {extreme_low_events_returns}, deviation: {extreme_low_events_future_deviation}, deviation at the event {extreme_low_mean_deviation}')
# Extreme high returns: 0.8944549590536808, deviation: 12.540217546254171, deviation at the event 11.645762587200487
# Extreme low returns: -1.1753205325140832, deviation: -14.600458713090967, deviation at the event -13.425138180576882


#I'm not sure exactly hwo to interpret this, but if 0 suggests that the price returns to the mean, it doesn't work. 
#It suggests it's quite the opposite, but my interpretation could be entirely wrong XD

In [ ]:
#W7 D3 WARMUP

regimes_high_closes = {
     regime: (abs(m15_df.loc[m15_df['regime'] == regime, 'close'] - m15_df.loc[m15_df['regime'] == regime, 'ema33_high']) / m15_df.loc[m15_df['regime'] == regime, 'close'] < 0.005).mean()
     for regime in m15_df['regime'].unique()
}
print(regimes_high_closes)
# {'strong_bull': np.float64(0.9308316210662918), 
#  'retracement_bull': np.float64(0.7873992236488504), 
#  'range_recross': np.float64(0.9243614931237721), 
#  'strong_bear': np.float64(0.8453916671241148), 
#  'retracement_bear': np.float64(0.9258594917787742)}

#not sure - these avalues look a bit suspisicous


In [ ]:
#W7 D3 T1
'''
## Task 1 - Rebuild the episode events cleanly (15 min)

Quick, self-contained rebuild of yesterday's fix - no new concepts, just getting back to a
clean starting point before testing multiple horizons.

```python
m15_df = pd.read_csv('project4_trend_regime/m15_with_regime.csv')
m15_df['deviation_30'] = m15_df['close'] - m15_df['close'].rolling(30).mean()
m15_df['z_score_30'] = m15_df['deviation_30'] / m15_df['deviation_30'].rolling(30).std()
m15_df = m15_df.dropna()

is_extreme_high = m15_df['z_score_30'] > 2.5
is_extreme_low = m15_df['z_score_30'] < -2.5
m15_df['new_high_episode'] = is_extreme_high & ~is_extreme_high.shift(1, fill_value=False)
m15_df['new_low_episode'] = is_extreme_low & ~is_extreme_low.shift(1, fill_value=False)
```

- 1a. Run this, confirm `m15_df['new_high_episode'].sum()` and `['new_low_episode'].sum()`
     give you roughly the same episode counts as yesterday (sanity check that nothing broke).
'''


m15_df = pd.read_csv('project4_trend_regime/m15_with_regime.csv')
m15_df['deviation_30'] = m15_df['close'] - m15_df['close'].rolling(30).mean()
m15_df['z_score_30'] = m15_df['deviation_30'] / m15_df['deviation_30'].rolling(30).std()
m15_df = m15_df.dropna()

is_extreme_high = m15_df['z_score_30'] > 2.5
is_extreme_low = m15_df['z_score_30'] < -2.5
m15_df['new_high_episode'] = is_extreme_high & ~is_extreme_high.shift(1, fill_value=False)
m15_df['new_low_episode'] = is_extreme_low & ~is_extreme_low.shift(1, fill_value=False)

print(m15_df['new_high_episode'].sum()) #2198
print(m15_df['new_low_episode'].sum()) #1953



In [ ]:
#W7 D3 T2
'''
## Task 2 - Multi-horizon table: does deviation shrink at ANY tested horizon? (35 min)

- 2a. For horizons `[5, 10, 20, 50, 100]`: for each N, compute
     `future_deviation_N = close.shift(-N) - close.rolling(30).mean()` (same drift-neutral
     formula as Week 7 Day 1). For episode-start rows only (`new_high_episode` or
     `new_low_episode` True), get the mean `future_deviation_N` for high-episodes and
     separately for low-episodes.
- 2b. Also grab the mean `deviation_30` AT the episode start (same for every horizon - it's
     the starting point, doesn't depend on N) for comparison.
- 2c. Build this into one table: one row per horizon, columns for
     `high_deviation_at_start`, `high_deviation_at_N`, `low_deviation_at_start`,
     `low_deviation_at_N` (loop -> list of dicts -> `pd.DataFrame`, same pattern as before).
- 2d. For each horizon, compute whether `abs(deviation_at_N) < abs(deviation_at_start)` -
     i.e. did the deviation actually SHRINK (move toward zero/the mean) by that horizon, for
     both high and low episodes.

**In a comment:** across all 5 horizons, is there ANY point where deviation clearly shrinks
compared to the episode start? If yes, which horizon, and does the shrinking get
stronger or weaker as N increases past that point? If no shrinkage shows up at any horizon,
that's a stronger, more conclusive negative finding than yesterday's single-horizon result.'''

high_deviation_at_start = m15_df.loc[m15_df['new_high_episode'] == 1, 'deviation_30'].mean()
low_deviation_at_start = m15_df.loc[m15_df['new_low_episode'] == 1, 'deviation_30'].mean()


intervals = [5, 10, 20, 50, 100]
results = {}

results[0] = {
          'N': 'start',
          'high_episodes_deviation': high_deviation_at_start,
          'low_episodes_deviation': low_deviation_at_start}


for i in intervals:
     name = f'deviation_at_{i}'
     m15_df[name] = m15_df['close'].shift(-i) - m15_df['close'].rolling(30).mean()
     high_deviation_at_N = m15_df.loc[m15_df['new_high_episode'] == 1, f'deviation_at_{i}'].mean()
     low_deviation_at_N = m15_df.loc[m15_df['new_low_episode'] == 1, f'deviation_at_{i}'].mean()
     
     results[i] = {
          'N': i,
          'high_episodes_deviation': high_deviation_at_N,
          'low_episodes_deviation': low_deviation_at_N}
     
xd = pd.DataFrame(results)
xd.head()

'''	

0	5	10	20	50	100
N	start	5.000000	10.000000	20.000000	50.000000	100.000000
high_episodes_deviation	11.645763	11.942501	12.274196	12.540218	13.704311	15.271340
low_episodes_deviation	-13.425138	-13.630402	-13.951115	-14.600459	-13.849258	-12.754887


The findings back up the yesterday's observations, which simply means that the deviation DOES NOT shrink at all in any of these episodes.
In other words, this could perhaps be an evidence of a phenomenon that supports playing extended trend movements beyond a given deviation threshold.
That's perhaps worth checking instead of the MR approach, whichb is quite surprising for me. 

Perhaps there are other MR techniques or instruments, or we shoudl just follow the evidence and look for trend approach instead.

'''



In [ ]:
#W7 D3 T3
'''
## Task 3 - The actual decision (15 min)

No new code - synthesis, based on everything from Week 6-7.

- 3a. Summarize in your own words: episode-based testing (correcting for the row-vs-episode
     counting issue) at multiple horizons shows [whatever Task 2 found]. How does this square
     with Week 6's descriptive findings (large Cohen's d, clean threshold-sweep pattern) -
     were those real but specific to "row-level, not episode-first-crossing" framing, or do
     they still hold up at the episode level too?
- 3b. Given everything now known: continue developing this mean-reversion thread (if some
     horizon shows real reversion), or is it time to treat this as a well-documented,
     thoroughly-tested NEGATIVE result and move to a different ML exercise? Either answer is
     fine - the goal is a reasoned call backed by this week's actual evidence, not
     attachment to the original idea.
'''

'''3a + 3b at once:

Testing different horizons vs the established 30 interval clearly shows that
the deviation from the mean does not return to normal with time. 
There ARE no differences between deviation at start vs deviation at N, both in low/high scenarios, which was surprising.
I'm not entirely sure why that's the case, because there's a chance I might be doing somethign wrong,
but in the light of all my current knowledge, it seems like the mean-reversion should be rejected in this case.

It seems like the gold market has more of a trending chaaracter, and the characteristics seem to hold.
Extreme deviations from the mean are rather a signal for continuation rather than Mean reversion plays.
Perhaps that's exactly what we're looking for - instead of trying to play trends every time, we should wait for extreme deviation from the mean,
and then continue playing in that direction, instead of hoping for a return to the mean.

Or I misinterpreted the results, I'm not entirely sure.
'''

In [ ]:
#W7 D4 WARMUP
'''
Build a dict comprehension mapping each `regime` value to the fraction of its
rows where `atr14` is above the overall 75th percentile of `atr14` (same inline-boolean-then
-mean shape, different threshold - a percentile instead of median/mean this time).'''


#I could do it in one go, BUT IMO IT MAKES 100% SENSE to create a separate variable for that, makes things clearer
#I might also consider using .describe() to verify it :))
perc_75 = m15_df['atr14'].quantile(0.75)
regimes_above_75perc = {
    regime: (m15_df.loc[m15_df['regime'] == regime, 'atr14'] > perc_75).mean()
    for regime in m15_df['regime'].unique()
}
print(regimes_above_75perc)

# {'strong_bull': np.float64(0.2430842435384449), 
#  'retracement_bull': np.float64(0.29979098238280083), 
#  'range_recross': np.float64(0.2557982574549024), 
#  'strong_bear': np.float64(0.24822967557775705), 
#  'retracement_bear': np.float64(0.2585949177877429)}



In [ ]:
#W7 D4 T1
'''## Task 1 - Define the continuation target (30 min)

**The goal in plain words:** for each episode (first bar of a continuous extreme run, same
definition as yesterday), does price make a NEW, MORE EXTREME high/low within N bars - i.e.
does the move continue further - or does it fail to extend and instead pull back? This
mirrors `resumed` (new extreme reached) vs `recrossed` (reverted without extending) from the
pullback project exactly, just applied to z-score episodes instead of EMA-channel pullbacks.

- 1a. Rebuild `deviation_30`, `z_score_30`, and the episode flags
     (`new_high_episode`/`new_low_episode`) fresh, self-contained - same code as the last two
     days, don't assume leftover notebook state.
- 1b. For each HIGH episode: record `ref_close` (the close price at the episode's first
     bar). Over the next N bars (try N=20 first, matching earlier tests), check if ANY later
     close exceeds `ref_close` (a new, more extreme high - continuation) BEFORE any close
     drops back to `close.rolling(30).mean()` taken at the episode bar (a reversion/failure
     to continue). Mirror this with lows for LOW episodes (new lower low = continuation,
     touching the rolling mean first = failure).
- 1c. Build `target_continued` (1 if continuation happened first, 0 if reversion/mean-touch
     happened first, within the N-bar window). Rows where NEITHER happens within N bars are
     ambiguous - drop them for now (same "can't resolve within the window" logic as
     `unfinished` events in the pullback project).

**In a comment:** what fraction of episodes show continuation vs. reversion-first vs.
ambiguous/dropped? Does the high/low asymmetry found yesterday show up here too (e.g. high
episodes continuing more often than low episodes)?'''


m15_df = pd.read_csv('project4_trend_regime/m15_with_regime.csv')
m15_df['deviation_30'] = m15_df['close'] - m15_df['close'].rolling(30).mean()
m15_df['z_score_30'] = m15_df['deviation_30'] / m15_df['deviation_30'].rolling(30).std()

is_extreme_high = m15_df['z_score_30'] > 2.5
is_extreme_low = m15_df['z_score_30'] < -2.5
m15_df['new_high_episode'] = is_extreme_high & ~is_extreme_high.shift(1, fill_value=False)
m15_df['new_low_episode'] = is_extreme_low & ~is_extreme_low.shift(1, fill_value=False)

m15_df['ref_close'] = m15_df.apply(lambda x: x['close'] if x['new_high_episode'] == 1 or x['new_low_episode'] == 1 else np.nan, axis = 1)
m15_df['ref_close'] = m15_df['ref_close'].ffill()

####Added 13.10 
m15_df['atr_trend'] = m15_df['atr14'] - m15_df['atr14'].shift(5)
m15_df['atr_trend'] = m15_df['atr_trend'].apply(lambda x: 1 if x > 0 else 0)
m15_df['regime_id'] = (m15_df['regime'] != m15_df['regime'].shift(1)).cumsum()
m15_df['regime_start'] = m15_df.groupby('regime_id')['et_time'].transform('first')
m15_df['regime_duration_so_far_mins'] = (pd.to_datetime(m15_df['et_time']) - pd.to_datetime(m15_df['regime_start'])).dt.seconds // 60
m15_df['day_of_week'] = pd.to_datetime(m15_df['et_time']).dt.day_of_week

m15_df['volatility_ratio'] = m15_df['close'].pct_change().rolling(10).std() / m15_df['close'].pct_change().rolling(50).std()
m15_df['realized_power'] = m15_df['close'].pct_change().abs().rolling(10).sum() #atr proxy - sum of absolute moves
m15_df['variance_ratio_10']  = m15_df['close'].pct_change(10).rolling(200).var() / (10 * m15_df['close'].pct_change().rolling(200).var())


#7 NEW MATH-HEAVY FEATURES ADDED AFTER RESEARCH, ON 14.10

#F1 - Efficiency Ratio ER
m15_df['net_displacement_10'] = m15_df['close'].pct_change().rolling(10).sum().abs()
m15_df['efficiency_ratio_10'] = m15_df['net_displacement_10'] / m15_df['realized_power']

#F2 - Concentration (CONC)
m15_df['l1_sum_10'] = m15_df['close'].pct_change().abs().rolling(10).sum()
m15_df['l2_sum_10'] = (m15_df['close'].pct_change()**2).rolling(10).sum()
m15_df['concentration_10'] = (m15_df['l1_sum_10'] ** 2) / (10 * m15_df['l2_sum_10'])

#F3 - signed semivariance assymmetry (SJV)
m15_df['ret_1bar'] = m15_df['close'].pct_change()
m15_df['squared_returns_10_positive_bars'] = (m15_df['ret_1bar'].where(m15_df['ret_1bar'] > 0, 0) ** 2).rolling(10).sum()
m15_df['squared_returns_10_negative_bars'] = (m15_df['ret_1bar'].where(m15_df['ret_1bar'] < 0, 0) ** 2).rolling(10).sum()
m15_df['sjv_10'] = (m15_df['squared_returns_10_positive_bars'] -  m15_df['squared_returns_10_negative_bars']) / (m15_df['squared_returns_10_positive_bars'] +  m15_df['squared_returns_10_negative_bars'])

#F4 - relative RP
m15_df['realized_power_50'] = m15_df['close'].pct_change().abs().rolling(50).sum()
m15_df['relative_rp'] = (m15_df['realized_power'] / 10) / (m15_df['realized_power_50'] / 50)

#F5 - Center of Mass (COM) - LLM assisted here, as it was a bit too difficult
def center_of_mass(abs_returns_window):
    n = len(abs_returns_window)
    positions = np.arange(1, n + 1)
    weighted_sum = np.sum(positions * abs_returns_window)
    total_weight = np.sum(abs_returns_window)
    if total_weight == 0:
        return 0.0 
    raw_com = weighted_sum / total_weight
    center = (n + 1) / 2
    radius = (n - 1) / 2
    return (raw_com - center) / radius

m15_df['abs_returns_1bar'] = m15_df['close'].pct_change().abs()
m15_df['center_of_mass'] = m15_df['abs_returns_1bar'].rolling(10).apply(center_of_mass, raw=True)

#F6+F7 Wick to close ratio (WTC) + Wick rejection assymetry (WRA)
m15_df['wtc_10'] = np.log(m15_df['high'] / m15_df['low']).rolling(10).sum() / m15_df['realized_power']
m15_df['upper_wick'] = m15_df['high'] - np.maximum(m15_df['open'], m15_df['close'])
m15_df['lower_wick'] = np.minimum(m15_df['open'], m15_df['close']) - m15_df['low']
m15_df['wra_10'] = (m15_df['upper_wick'].rolling(10).sum() - m15_df['lower_wick'].rolling(10).sum()) / (m15_df['high'] - m15_df['low']).rolling(10).sum()

#######

#Volatility target pivot - 16.10.26
m15_df['rp_at_event'] = np.where((m15_df['new_low_episode'] == True) | (m15_df['new_high_episode'] == True), m15_df['realized_power'], None)
m15_df['rp_at_event'] = m15_df['rp_at_event'].ffill()
m15_df = m15_df.dropna()
m15_df['rp_future_20'] = m15_df['realized_power'].shift(-20)
m15_df['target_volatility_rose'] = m15_df.apply(lambda x: 1 if x['rp_future_20'] > x['rp_at_event'] else 0, axis = 1)


#otf_streaksy --- 

from typing import List

def otf_streak(highs: List, lows: List) -> List:
     otf_streaks = [0] * len(highs)
     try:
          for i in range(1, len(otf_streaks)):
               if (lows[i] > lows[i - 1]) and (highs[i] < highs[i - 1]): #inside bar
                    otf_streaks[i] = otf_streaks[i - 1]
                    
               elif (lows[i] > lows[i - 1]):
                    if otf_streaks[i - 1] < 0:
                         otf_streaks[i] = 1
                    else:
                         otf_streaks[i] = otf_streaks[i - 1] + 1
                         
               elif (highs[i] < highs[i - 1]):
                    if otf_streaks[i - 1] > 0:
                         otf_streaks[i] = -1
                    else:
                         otf_streaks[i] = otf_streaks[i - 1] - 1
     except Exception as e: 
          print(f'Unhandled exception: {str(e)}')
     return otf_streaks 
          
     #low > low-1 #otfup
     #high < high -1 #otfdown
     
m15_df['otf_streak'] = otf_streak(m15_df['high'].tolist(), m15_df['low'].tolist())


#abs z-score + hour

m15_df['abs_z_score'] = m15_df['z_score_30'].abs()
m15_df['hour'] = pd.to_datetime(m15_df['et_time']).dt.hour

m15_df = m15_df.reset_index(drop=True)


def check_continuation_high(df, event_idx, n_bars=20):
    ref_close = df['close'].iloc[event_idx]
    ref_mean = df['close'].rolling(30).mean().iloc[event_idx]
    
    for k in range(1, n_bars + 1):
        future_idx = event_idx + k
        if future_idx >= len(df):
            return None  
        future_close = df['close'].iloc[future_idx]
        if future_close > ref_close:
            return 1  
        if future_close <= ref_mean:
            return 0 
    return None


high_event_positions = m15_df.index[m15_df['new_high_episode']].tolist()

results = []
for pos in high_event_positions:
    outcome = check_continuation_high(m15_df, pos, n_bars=20)
    results.append({'event_idx': pos, 'target_continued': outcome})

high_targets_df = pd.DataFrame(results)
print(len(high_targets_df))
high_targets_df = high_targets_df.dropna()
print(len(high_targets_df))
print(high_targets_df['target_continued'].value_counts())

# 2198
# 2113
# target_continued
# 1.0    1860
# 0.0     253
# Name: count, dtype: int64




def check_continuation_low(df, event_idx, n_bars=20):
    ref_close = df['close'].iloc[event_idx]
    ref_mean = df['close'].rolling(30).mean().iloc[event_idx]
    
    for k in range(1, n_bars + 1):
        future_idx = event_idx + k
        if future_idx >= len(df):
            return None  
        future_close = df['close'].iloc[future_idx]
        if future_close < ref_close:
            return 1  
        if future_close >= ref_mean:
            return 0 
    return None


low_event_positions = m15_df.index[m15_df['new_low_episode']].tolist()

results = []
for pos in low_event_positions:
    outcome = check_continuation_low(m15_df, pos, n_bars=20)
    results.append({'event_idx': pos, 'target_continued': outcome})

low_targets_df = pd.DataFrame(results)
print(len(low_targets_df))
low_targets_df = low_targets_df.dropna()
print(len(low_targets_df))
print(low_targets_df['target_continued'].value_counts())

# 1953
# 1867
# target_continued
# 1.0    1586
# 0.0     281
# Name: count, dtype: int64




In [ ]:
#W7 D4 T2
'''## Task 2 - First feature table (25 min)

Same mechanical-feature approach as the pullback project.

- 2a. For each resolved episode (continuation or reversion-first, not ambiguous), build:
     `direction` (1 for high episodes, 0 for low - the asymmetry from yesterday suggests this
     matters a lot here), `hour` (from `et_time`), `atr14` (volatility context),
     `abs_z_score` (how extreme the triggering deviation was).
- 2b. Quick sanity check: group by `direction` and compute `target_continued.mean()` - does
     this match yesterday's finding (high episodes continuing more, given they showed
     monotonic growth vs. low episodes' partial reversal)?

**In a comment:** does this numeric check from 2b match your expectation from yesterday's
multi-horizon table, or is there a surprise?'''

# m15_df.head()
# low_targets_df.head()
# high_targets_df.head()

high_targets_df['direction'] = 1
low_targets_df['direction'] = 0

all_targets_df = pd.concat([high_targets_df, low_targets_df], ignore_index=True)


m15_df['abs_z_score'] = m15_df['z_score_30'].abs()
m15_df['hour'] = pd.to_datetime(m15_df['et_time']).dt.hour

#feature column for all_targets df
feature_cols = [
               'hour', 'atr14','abs_z_score', 'et_time', 'atr_trend', 'regime_duration_so_far_mins', 
                'day_of_week', 'volatility_ratio', 'realized_power', 'variance_ratio_10', 'wra_10', 'wtc_10', 
                'relative_rp', 'center_of_mass', 'sjv_10', 'concentration_10', 'efficiency_ratio_10',
                'target_volatility_rose'
                ]


all_targets_df = all_targets_df.merge(
    m15_df[feature_cols],
    left_on='event_idx',
    right_index=True
)

all_targets_df.head()
print(all_targets_df['target_continued'].mean())


In [ ]:
#W7 D4 T3
'''
## Task 3 - Time-aware split + baseline (20 min)

- 3a. Sort by the episode's `et_time`, time-aware 80/20 split (no shuffling).
- 3b. Compute the "always predict the majority class" baseline accuracy on test - is this
     target closer to balanced (50/50-ish, given continuation and reversion are both common
     outcomes) or imbalanced like the pullback project's resumed/recrossed (95/5)?

**In a comment:** given the class balance here, does this feel like a genuinely different
(and in some ways easier/more interesting) ML problem than the pullback classifier's extreme
imbalance - why or why not?'''

all_targets_df = all_targets_df.sort_values(by = 'et_time')

train = all_targets_df[:int(0.8*(len(all_targets_df)))]
test = all_targets_df[int(0.8*(len(all_targets_df))):]


all_targets_df['target_continued'].value_counts()
print(534 / (534 + 3446)) #13.4% reversed / 83.6% continued

#It feels similar, but 83.6 - 13.4 is not as drastic difference as in returned/recrossed


In [ ]:
#W7 D5 WARMUP
'''# build a dict comprehension mapping each `regime` value to the fraction of its
# rows where `atr14` is below the overall 25th percentile of `atr14` (mirror of Thursday's
# warm-up, opposite tail).'''


perc_25 = m15_df['atr14'].quantile(0.25)
regime_fraction_below_25percentile = {
    regime: (m15_df.loc[m15_df['regime'] == regime, 'atr14'] < perc_25).mean()
    for regime in m15_df['regime'].unique()
}
print(regime_fraction_below_25percentile)

# {'strong_bull': np.float64(0.2602321451773572), 
#  'retracement_bull': np.float64(0.16183935503135263), 
#  'range_recross': np.float64(0.2389557000859001), 
#  'strong_bear': np.float64(0.2597024757095021), 
#  'retracement_bear': np.float64(0.19910313901345292)}


In [ ]:
#W7 D5 T1
'''## Task 1 - Fit a baseline logistic regression (30 min)

Same approach as Week 4 Day 2 for the pullback project - a first honest baseline before
anything fancier.

- 1a. Using yesterday's `all_targets_df` (or rebuild it fresh if you cleared your notebook -
     it's `event_idx`, `target_continued`, `direction`, `hour`, `atr14`, `abs_z_score`, plus
     `et_time` sorted for the split), fit `LogisticRegression()` on the TRAIN split with
     features `[direction, hour, atr14, abs_z_score]`, target `target_continued`.
- 1b. Predict probabilities on TEST (`predict_proba(...)[:, 1]`, not `.predict()` -
     remember why, from earlier this month). Compute `roc_auc_score`.
- 1c. Compare to the "always predict continued" baseline accuracy from yesterday (83.6%) -
     does the model's ROC AUC suggest it's learning something real, or is it close to what
     you'd expect from a model that just learned the base rate?

**In a comment:** print `model.coef_` next to feature names, sorted by absolute value. Does
`direction` show up as a strong coefficient (consistent with yesterday's ~88%/85% split by
side), or is it weaker than expected once combined with the other features?
'''
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, precision_score
features = ['direction', 'hour', 'atr14', 'abs_z_score']

model = LogisticRegression()
model.fit(train[features], train['target_continued'])
probas = model.predict_proba(test[features])[:, 1]
train_probas = model.predict_proba(train[features])[:, 1]

train_rocauc = roc_auc_score(train['target_continued'], train_probas)
test_rocauc = roc_auc_score(test['target_continued'], probas)
print(f'Test ROCAUC: {test_rocauc}')
print(f'Train rocauc: {train_rocauc}')

'''Test ROCAUC: 0.5614709298353487
Train rocauc: 0.5599774550071912'''

#absolutely useless, almost -30pp vs baseline :D
#Unless we're 56% sure whether it's NOT CONTINUED, that COULD MAKE SENSE, as then its 0.56 vs 0.13, which IS NICE

print(model.coef_)
coefs = [coef for coef in model.coef_[0]]
print(coefs)
print(features)

results = {
     'coef': [coef for coef in coefs],
     'features': [feature for feature in features]
}
results = pd.DataFrame(results)
results.sort_values(by = 'coef', ascending = False)
results.head()
'''	coef	features
0	0.302758	direction
1	0.005298	hour
2	0.104008	atr14
3	0.027954	abs_z_score'''

#interestingly direction is a very strong feature, atr14 could be useful as well, hour and abs_z_score seem to be useless in LogReg :))



In [ ]:
#W7 D5 T2
'''Since `target_continued` is imbalanced (83.6/13.4, milder than the pullback project's 95/5
but still real) - check whether the model can actually find the minority class
(`reversed`/`target_continued == 0`), not just the majority.

- 2a. Compute precision and recall for `pos_label=0` (reversed) on TEST, using `.predict()`
     at the default 0.5 threshold.
- 2b. Compare this to what you'd expect "for free" just from the 83.6/13.4 base rate (same
     reasoning as the pullback project's Week 4 imbalance work) - is the model doing better
     than a naive "always predict continued" guess at catching reversals specifically?

**In a comment:** given this result, would `class_weight='balanced'` or threshold tuning (the
two levers from Week 4/5) be worth trying here, or does the milder imbalance (13.4% vs the
pullback project's 4.4%) make this less urgent than it was there?'''


predictions = model.predict(test[features])
test_precision = precision_score(test['target_continued'], predictions, pos_label = 0)
test_recall = recall_score(test['target_continued'], predictions, pos_label = 0)

print(predictions.sum(), len(predictions))  # ile razy model przewidział 1 (continued), z ilu wszystkich
print(test['target_continued'].value_counts())  # jak wygląda prawdziwy rozkład w teście


print(f'Test precision: {test_precision:.4f}, test_recall: {test_recall:.4f}')
#Test precision: 0.0000, test_recall: 0.0000

#Model is unavailable to predict ANY reversed instance XDD!



In [ ]:
#W8 D1 WARMUP
'''build a dict comprehension mapping each `regime` value to the fraction of its
rows where `low` is above the row's OWN `ema33_low` (same inline-boolean-then-mean pattern,
comparing a column to another column instead of to a fixed threshold this time).'''


regime_low_below_ema33_low = {
    regime: (m15_df.loc[m15_df['regime'] == regime, 'low'] < m15_df.loc[m15_df['regime'] == regime, 'ema33_low']).mean()
    for regime in m15_df['regime'].unique()
}

print(regime_low_below_ema33_low)

# {'strong_bull': np.float64(0.2309108233357876), 
#  'retracement_bull': np.float64(1.0), 
#  'range_recross': np.float64(0.4903055589642901), 
#  'strong_bear': np.float64(0.722731514519405), 
#  'retracement_bear': np.float64(0.02242152466367713)}

In [ ]:
#W8 D1 T1
'''## Task 1 - Worked example: a simple streak-counting function (20 min)

**Before building OTF, a simpler, fully worked warm-up for the stateful-loop pattern itself**
- counting how many consecutive bars in a row closed higher than the previous bar (the
"consecutive close streak" idea, simpler than OTF's low/high logic, to build the mechanic
before adding OTF's extra condition).

```python
def consecutive_up_streak(closes):
    """Returns a list, same length as closes: for each position, how many bars in a row
    (ending at this one) have closed higher than the previous bar. Resets to 0 on any
    bar that does NOT close higher."""
    streaks = [0] * len(closes)
    for i in range(1, len(closes)):
        if closes[i] > closes[i - 1]:
            streaks[i] = streaks[i - 1] + 1   # extend yesterday's streak
        else:
            streaks[i] = 0                     # broke the streak, reset
    return streaks
```

Read through this line by line before running it: `streaks[i-1]` is WHY this needs a loop
and can't be a simple vectorized one-liner - each value depends on the PREVIOUS computed
value in the same list, not just on the raw input data at that position. This is the core
shape of every stateful function you'll write this week.

- 1a. Run this on a small test list you make up by hand first (e.g.
     `[10, 11, 12, 10, 11, 12, 13, 14]` - work out what the output SHOULD be on paper, then
     check the function matches). This step matters - don't skip straight to real data.
- 1b. Run it on `m15_df['close'].tolist()` (a sample, like `.head(500)`, to keep it fast
     while testing) and sanity check a few values against the raw `close` column by eye.'''
     

test_list = [10, 11, 12, 10, 11, 12, 13, 14]

def consecutive_up_streak(closes):
    """Returns a list, same length as closes: for each position, how many bars in a row
    (ending at this one) have closed higher than the previous bar. Resets to 0 on any
    bar that does NOT close higher."""
    streaks = [0] * len(closes)
    for i in range(1, len(closes)):
        if closes[i] > closes[i - 1]:
            streaks[i] = streaks[i - 1] + 1   # extend yesterday's streak
        else:
            streaks[i] = 0                     # broke the streak, reset
    return streaks


streaks = consecutive_up_streak(test_list)
print(streaks)
m15_streaks = consecutive_up_streak(m15_df['close'].tail(500).tolist())
print(m15_streaks)
print(m15_df['close'].tail(500).head(15))

'''
Works just fine, nice to work on such funcs from time to time

[0, 0, 0, 0, 1, 0]
121402    4126.255
121403    4124.975
121404    4121.215
121405    4119.915
121406    4120.515
121407    4117.195'''

In [ ]:
#W8 D1 T2
'''## Task 2 - Build the OTF streak feature (35 min)

Same loop shape as Task 1, slightly more complex condition (tracking both highs and lows,
and two directions).

- 2a. Write `otf_streak(highs, lows)` that returns a list, same length as the input: a
     POSITIVE number for an ongoing OTF-up streak (bar's low > previous bar's low), a
     NEGATIVE number for an ongoing OTF-down streak (bar's high < previous bar's high), and
     0 when neither condition holds (a bar that breaks both, or can't be classified) -
     magnitude counts how many consecutive bars the current streak has lasted. Structure it
     like Task 1's function: a loop, checking the condition against the PREVIOUS bar, and a
     separate variable tracking the running streak length that resets when the condition
     fails.
- 2b. Run it on `m15_df`, store the result as `m15_df['otf_streak']`. Print
     `m15_df['otf_streak'].describe()` and the longest positive/negative streaks found - do
     these numbers look plausible (not absurdly long, not all zero)?

**In a comment:** before checking any relationship to a target, what's your own prediction -
do you expect LONGER otf streaks to predict MORE continuation (confidence building) or LESS
(exhaustion, due for a break)? Write this down before testing it in Task 3.'''
from typing import List

def otf_streak(highs: List, lows: List) -> List:
     otf_streaks = [0] * len(highs)
     try:
          for i in range(1, len(otf_streaks)):
               if (lows[i] > lows[i - 1]) and (highs[i] < highs[i - 1]): #inside bar
                    otf_streaks[i] = otf_streaks[i - 1]
                    
               elif (lows[i] > lows[i - 1]):
                    if otf_streaks[i - 1] < 0:
                         otf_streaks[i] = 1
                    else:
                         otf_streaks[i] = otf_streaks[i - 1] + 1
                         
               elif (highs[i] < highs[i - 1]):
                    if otf_streaks[i - 1] > 0:
                         otf_streaks[i] = -1
                    else:
                         otf_streaks[i] = otf_streaks[i - 1] - 1
     except Exception as e: 
          print(f'Unhandled exception: {str(e)}')
     return otf_streaks 
          
     #low > low-1 #otfup
     #high < high -1 #otfdown
     


m15_df['otf_streak'] = otf_streak(m15_df['high'].tolist(), m15_df['low'].tolist())
m15_df.head()

In [ ]:
#W8 D1 T3
'''## Task 3 - Quick check against the continuation target (15 min)

- 3a. Using Friday's episode/target data (`all_targets_df`, rebuilt if needed), merge
     `otf_streak` (value at the episode's `event_idx`) as a new feature.
- 3b. Quick groupby or boxplot: does `target_continued` differ by `otf_streak` value
     (binned into a few buckets - e.g. short/medium/long) at all, before any model is
     involved?

**In a comment:** did reality match your Task 2 prediction (confidence vs exhaustion), or
was it a surprise? This is purely descriptive today - full feature-table integration and
RF/XGBoost comparison come Tuesday/Wednesday.'''

import matplotlib.pyplot as plt
import seaborn as sns

all_targets_df = all_targets_df.merge(m15_df[['otf_streak', 'et_time']], how = 'left', on = 'et_time')

plt.Figure(figsize = (20, 10))
boxplot = sns.boxplot(
     all_targets_df,
     x = 'target_continued',
     y = 'otf_streak'
)
plt.show()

#quite frankly it doesn't show much difference, there are some outliers with very high otf streaks in both groups
#Anyway we will test that, and I assume we'll add more features.
#I guess OTF might be useful in certain context, not sure it will help here


In [ ]:
#W8 D2 WARMUP

'''build a dict comprehension mapping each `regime` value to the fraction of its
rows where `high` is above the row's OWN `ema144_high` (same shape as yesterday's warm-up,
different EMA).'''

regime_high_above_ema144h = {
    regime: (m15_df.loc[m15_df['regime'] == regime, 'high'] > m15_df.loc[m15_df['regime'] == regime, 'ema144_high']).mean()
    for regime in m15_df['regime'].unique()
}
print(regime_high_above_ema144h)

'''{'strong_bull': np.float64(1.0), 
'retracement_bull': np.float64(0.44490892803822035), 
'range_recross': np.float64(0.4890170573076451), 
'strong_bear': np.float64(0.002168304331119284), 
'retracement_bear': np.float64(0.5121076233183857)}'''




In [ ]:
#W8 D2 T1
'''
Same mechanical-feature-engineering approach as the pullback project (no domain guessing,
built from columns/concepts you already have) - this time applied to the continuation
target's episode structure.

- 1a. **`otf_streak_at_event`** (if not already merged): the OTF streak value AT each
     episode's `event_idx` - does the extreme z-score event happen during an already-long
     OTF run, or a short/fresh one? (Yesterday checked this against the target broadly;
     today it becomes a real feature in the table.)
- 1b. **`atr_trend`**: whether volatility (`atr14`) was rising or falling into the event -
     `atr14_at_event - atr14.shift(5)_at_event` (5-bar lookback, simple difference) -
     mirrors the `atr_trend` feature from the pullback project (Week 5 Day 2).
- 1c. **`block_duration_so_far`**: minutes since the current `regime` last changed, at the
     event bar - mirrors the pullback project's `block_duration_so_far`
     (`(regime != regime.shift(1)).cumsum()` for the block id, then time since that block's
     first row).
- 1d. **`day_of_week`**: from `et_time`, `.dt.dayofweek` - session/weekday effects, already
     explored in the pullback project.

**In a comment:** before merging these into `all_targets_df`, one-sentence prediction per
feature - do you expect `atr_trend`, `block_duration_so_far`, or `day_of_week` to matter
most for predicting continuation, and why (gut instinct, to compare against Task 2's actual
check)?

'''

#otf_streak's done, I merged it yesterday, pasting the current all_targets_df's scheme 

all_targets_df.head()
'''event_idx	target_continued	direction	hour	atr14	abs_z_score	et_time	otf_streak
0	0	1.0	1	5	2.476584	2.689785	2021-01-05 05:45:00	0
1	7	1.0	1	7	3.128382	2.566619	2021-01-05 07:30:00	1
2	9	1.0	1	8	3.344518	3.064920	2021-01-05 08:00:00	-1'''

#pasted above, in the sections of the code BEFORE WE'VE ACTUALLY CREATED all_targets_df, included it there
# m15_df['atr_trend'] = m15_df['atr14'] - m15_df['atr14'].shift(5)
# m15_df['atr_trend'] = m15_df['atr_trend'].apply(lambda x: 1 if x > 0 else 0)
# m15_df['regime_id'] = (m15_df['regime'] != m15_df['regime'].shift(1)).cumsum()
# m15_df['regime_start'] = m15_df.groupby('regime_id')['et_time'].transform('first')
# m15_df['regime_duration_so_far_mins'] = (pd.to_datetime(m15_df['et_time']) - pd.to_datetime(m15_df['regime_start'])).dt.seconds // 60
# m15_df['day_of_week'] = pd.to_datetime(m15_df['et_time']).dt.day_of_week

all_targets_df.head()
features = ['hour', 'atr14', 'abs_z_score', 'et_time', 'atr_trend', 'regime_duration_so_far_mins', 'day_of_week']

#current shape:
'''event_idx	target_continued	direction	hour	atr14	abs_z_score	et_time	atr_trend	regime_duration_so_far_mins	day_of_week	otf_streak
0	0	1.0	1	5	2.476584	2.689785	2021-01-05 05:45:00	0	120	1	0
1	7	1.0	1	7	3.128382	2.566619	2021-01-05 07:30:00	1	225	1	1
2	9	1.0	1	8	3.344518	3.064920	2021-01-05 08:00:00	1	255	1	-1'''


#As for predictions, I doubt 'day of week', 'regime duration', 'hour' will be any useful, but whatever.
#If they weren't useful before, I doubt they will be here, but then why do we even consider them?
#Shouldn't we move forward and stop using features that tend to fail and look for actually working features?
#Maybe we could derive more features from atr/otf, add more angles to our analysis by using data we already have
#We can have different atr intervals, different deviations, some transformed price action indicators etc. (and I don't mean like typical trading indicators, but who knows)
#I'm just wondering whether the current number of features is enough and actually gives the model what it needs...






In [ ]:
#W8 D2 T2
'''## Task 2 - Verify before modeling: check each new feature against the target (25 min)

Same "verify, don't assume" discipline as the pullback project (Week 5 Day 2's Mann-Whitney
checks) - quick looks before anything goes into a model tomorrow.

- 2a. For each of the 4 new features (plus the corrected `otf_streak`), split
     `target_continued` into its two groups and compare the feature's distribution (boxplot
     or `.describe()` per group) - does any show a visible separation?
- 2b. Pick whichever ONE feature looked most promising and run a proper two-group
     comparison on it (Mann-Whitney, same tool as before) - state explicitly which two
     groups you're comparing before running it.

**In a comment:** which of today's new features (plus corrected OTF) looks most worth
including tomorrow, and which looks like dead weight already, before any model gets a
chance to use it?'''

from scipy.stats import mannwhitneyu

all_targets_df.head()

results = []
features_to_check = ['atr_trend', 'otf_streak', 'day_of_week', 'regime_duration_so_far_mins']
for feature in features_to_check:
    t_continued = all_targets_df.loc[all_targets_df['target_continued'] == 1.0, feature]
    t_reversed = all_targets_df.loc[all_targets_df['target_continued'] == 0.0, feature]
    
    mw_test = mannwhitneyu(t_continued, t_reversed)
    
    results.append(
        {
            'feature': feature,
            'Mann Whitney p_value': mw_test[1],
        }
    )
    
xd = pd.DataFrame(results)
xd.head()
'''	feature	Mann Whitney p_value
0	atr_trend	0.207119
1	otf_streak	0.011280
2	day_of_week	0.191714
3	regime_duration_so_far_mins	0.867570
'''

#It seems like only otf_streak has a statistically viable effect here \
#however, while they don't have effect ON THEIR OWN, they might have an effect if combined with any other feature
#and I won't be able to observe it this way
#But we definitely need to look for more of such features;;

#I also decided to check the effect's strength here

#cohen's func imported from the internet quickly :P
def cohens_d(x, y):
  nx = len(x)
  ny = len(y)
  dof = nx + ny - 2
  pooled_std = np.sqrt(
      ((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / dof
  )
  return (np.mean(x) - np.mean(y)) / pooled_std


t_continued = all_targets_df.loc[all_targets_df['target_continued'] == 1.0, 'otf_streak']
t_reversed = all_targets_df.loc[all_targets_df['target_continued'] == 0.0, 'otf_streak']

print(cohens_d(t_continued, t_reversed)) #0.10 - small effect


In [ ]:
#W8 D2 T3 - extra task, o który poprosiłem

## Task 3 - Research-grounded features, round 2 (30 min)
'''
Quick research check (done before writing this) into what the finance/stats literature
actually uses for volatility/trend-persistence features, since today's univariate screening
flagged that the current feature set is thin. Key findings, each pointing to a concrete,
buildable feature:

- **Volatility clustering/persistence** - today's volatility correlates with its OWN recent
  past (foundational property behind GARCH-style models) - suggests comparing CURRENT
  volatility to its own recent history, not just its absolute level.
- **"Realized power" beats plain realized volatility** in some forecasting literature - sum
  of ABSOLUTE short-interval returns over a window, rather than their standard deviation -
  worth trying as an alternative volatility measure to `atr14`.
- **Variance-ratio tests (Lo-MacKinlay)** - a classic econometric tool for testing whether
  returns behave more like a trend or mean-reversion at different horizons, by comparing the
  variance of N-bar returns to N times the variance of 1-bar returns (if markets were pure
  random walk, these would be equal - a ratio far from 1 signals structure, a ratio > 1
  leans trending/persistent, < 1 leans mean-reverting). Different tool than ADF/Hurst, same
  spirit.

Build these three as features, same mechanical approach as always (no need to understand
the deep math, just what each number tells you):

- 3a. **`vol_ratio`**: current 10-bar realized volatility (`close.pct_change().rolling(10).std()`)
     divided by a longer 50-bar realized volatility (`close.pct_change().rolling(50).std()`)
     - a ratio notably above 1 means recent volatility is elevated vs. its own longer-run
     level (not just "high" in absolute terms).
- 3b. **`realized_power`**: sum of absolute 1-bar returns over the last 10 bars
     (`close.pct_change().abs().rolling(10).sum()`) - an alternative volatility proxy to
     `atr14`, computed differently (sum of absolute moves vs. true-range-based).
- 3c. **`variance_ratio_10`**: variance of 10-bar returns divided by (10 times the variance
     of 1-bar returns), computed on a trailing window (e.g. last 200 bars of 1-bar returns
     to get a stable variance estimate) - at the event bar. Values meaningfully above 1
     suggest trend-like persistence in the recent window; below 1 suggests mean-reversion.
     (If this feels like too much to build cleanly in the time left, `vol_ratio` and
     `realized_power` alone are enough for today - `variance_ratio_10` can carry to
     tomorrow if needed.)

**In a comment:** run the same Mann-Whitney + Cohen's d screen from Task 2 on these 3 new
features. Does anything here beat `otf_streak`'s p=0.011 / d=0.106, or is it more of the
same modest-to-nothing effect?'''

#changed name to volatilitiry ratio, as vol wrongly suggests VOLUME :))
#I've also put this in the part of the code above, where we delete nans and before we merge the all targets df
# m15_df['volatility_ratio'] = m15_df['close'].pct_change().rolling(10).std() / m15_df['close'].pct_change().rolling(50).std()
# m15_df['realized_power'] = m15_df['close'].pct_change().abs().rolling(10).sum() #atr proxy - sum of absolute moves
# m15_df['variance_ratio_10']  = m15_df['close'].pct_change(10).rolling(200).var() / (10 * m15_df['close'].pct_change().rolling(200).var())
#commented this part

new_features = ['volatility_ratio', 'realized_power', 'variance_ratio_10']
all_targets_df.head()

def cohens_d(x, y):
  nx = len(x)
  ny = len(y)
  dof = nx + ny - 2
  pooled_std = np.sqrt(
      ((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / dof
  )
  return (np.mean(x) - np.mean(y)) / pooled_std


results = []
for feature in new_features:
    t_continued = all_targets_df.loc[all_targets_df['target_continued'] == 1.0, feature]
    t_reversed = all_targets_df.loc[all_targets_df['target_continued'] == 0.0, feature]
    
    mw_test = mannwhitneyu(t_continued, t_reversed)
    cohen_test = cohens_d(t_continued, t_reversed)
    
    results.append(
        {
            'feature': feature,
            'Mann Whitney p_value': mw_test[1],
            'Cohen D effect value': cohen_test
        }
    )
    
xd = pd.DataFrame(results)
xd.head()

'''feature	Mann Whitney p_value	Cohen D effect value
0	volatility_ratio	0.484161	0.043544
1	realized_power	0.000002	0.250477
2	variance_ratio_10	0.331604	-0.079133
'''

#Realized power seems to have a very strong correlation, and interesting effect
#still weak, but it's the strongest effect so far as a single feature
#perhaps it's worth to move into that direciton and invent more features that are proxies or explore realized power
#from a different angle or using different information.



In [ ]:
#W8 D3 WARMUP

#abs(close - open) / open < 0.003 --- CLOSE IS WITHIN


regimes_close_to_open = {
    regime: ((m15_df.loc[m15_df['regime'] == regime, 'close'] - m15_df.loc[m15_df['regime'] == regime, 'open']) / m15_df.loc[m15_df['regime'] == regime, 'open'] < 0.003).mean()
    for regime in m15_df['regime'].unique()
    }

print(regimes_close_to_open)

In [ ]:
#W8 D3 T1-T3 combined
#All of the features added today are quite complex mathemathical functions that are a result of my yesterday research and could potentially be interesting
#I might be using LLMs to day to aid me a bit if anything's too complex in mathematical sense (it's not exactly my area of expertise)

#I will be pasting the code from these above anyway, durign the part where we build features and then delete NaNs, to avoid unnecessary data waste
#and match the pipeline's order properly


'''## Task 1 - Efficiency Ratio and Concentration (25 min)

**Concept - Efficiency Ratio (ER):** `realized_power` (call it RP) is
`Σ|r_t|` - total distance traveled, ignoring direction. The NET displacement over the same
window is `|Σr_t|` - how far price actually ended up from where it started (direction
matters here, since positive and negative returns can cancel out). ER = net displacement /
total distance traveled = `|Σr_t| / Σ|r_t|`. Picture someone walking back and forth on a
street: if they end up far from where they started, ER is close to 1 (efficient, trend-like);
if they end up near their starting point despite a lot of walking, ER is close to 0
(inefficient, choppy).

- 1a. Build `net_displacement_10 = close.pct_change().rolling(10).sum().abs()` (note: this
     sums the signed returns first, THEN takes abs - different from `realized_power`, which
     takes abs of each return BEFORE summing).
- 1b. Build `efficiency_ratio_10 = net_displacement_10 / realized_power` (reuse Tuesday's
     `realized_power` column - same 10-bar window). Handle division-by-zero (when RP is 0,
     which shouldn't happen often but check) by setting those rows to NaN.

**Concept - Concentration (CONC):** compares the SAME quantity measured two ways -
`Σ|r_t|` (L1, each move counted once) vs `Σr_t²` (L2, each move counted squared, so big
moves count disproportionately more). `CONC = (Σ|r_t|)² / (N · Σr_t²)`. If one single bar
did almost all the moving, L2 is dominated by that one big squared term, making CONC LOW.
If effort was spread evenly across all N bars, CONC is HIGHER (closer to its max of 1).

- 1c. Build `l1_sum_10 = close.pct_change().abs().rolling(10).sum()` (this is just
     `realized_power` again, confirm they match) and `l2_sum_10 = (close.pct_change()**2).rolling(10).sum()`.
- 1d. Build `concentration_10 = (l1_sum_10 ** 2) / (10 * l2_sum_10)`.
'''

##############
##############
##################

#F1 - Efficiency Ratio ER
# m15_df['net_displacement_10'] = m15_df['close'].pct_change().rolling(10).sum().abs()
# m15_df['efficiency_ratio_10'] = m15_df['net_displacement_10'] / m15_df['realized_power']

# #F2 - Concentracion (CONC)
# m15_df['l1_sum_10'] = m15_df['close'].pct_change().abs().rolling(10).sum()
# m15_df['l2_sum_10'] = (m15_df['close'].pct_change()**2).rolling(10).sum()
# m15_df['concentration_10'] = (m15_df['l1_sum_10'] ** 2) / (10 * m15_df['l2_sum_10'])


'''
## Task 2 - Signed jump asymmetry and Relative RP (25 min)

**Concept - signed semivariance asymmetry (SJV):** split squared returns into "up" and
"down" buckets based on sign, then compare: `RS+ = Σr_t² for r_t>0`,
`RS- = Σr_t² for r_t<0`, `SJV = (RS+ - RS-) / (RS+ + RS-)`. Positive SJV means the big
(squared-weighted) moves lately were skewed upward; negative means skewed downward. This is
NOT the same as just checking the sign of the net move - it's asking specifically where the
big, squared-weighted moves were concentrated.

- 2a. Build `returns_1bar = close.pct_change()`. Build `rs_pos_10` (sum of squared returns
     where return > 0, over a 10-bar rolling window - you'll need a helper column like
     `returns_1bar.where(returns_1bar > 0, 0) ** 2` then `.rolling(10).sum()`) and `rs_neg_10`
     (mirror, for return < 0).
- 2b. Build `sjv_10 = (rs_pos_10 - rs_neg_10) / (rs_pos_10 + rs_neg_10)`.

**Concept - Relative RP:** is today's `realized_power` high in absolute terms, or high
relative to ITS OWN recent history? A fixed RP value might mean something totally different
for a usually-calm market vs. a usually-wild one.

- 2c. Build `rp_long_50 = close.pct_change().abs().rolling(50).sum()` (same idea, longer
     window). Build `relative_rp = (realized_power / 10) / (rp_long_50 / 50)` (normalizing
     each by its own window length before comparing, so you're comparing average per-bar
     effort at two different scales, not raw sums of different sizes).

'''
##############
##############
##################

#F3 - signed semivariance assymmetry (SJV)
# m15_df['ret_1bar'] = m15_df['close'].pct_change()
# m15_df['squared_returns_10_positive_bars'] = (m15_df['ret_1bar'].where(m15_df['ret_1bar'] > 0, 0) ** 2).rolling(10).sum()
# m15_df['squared_returns_10_negative_bars'] = (m15_df['ret_1bar'].where(m15_df['ret_1bar'] < 0, 0) ** 2).rolling(10).sum()
# m15_df['sjv_10'] = (m15_df['squared_returns_10_positive_bars'] -  m15_df['squared_returns_10_negative_bars']) / (m15_df['squared_returns_10_positive_bars'] +  m15_df['squared_returns_10_negative_bars'])

# #F4 - relative RP
# m15_df['realized_power_50'] = m15_df['close'].pct_change().abs().rolling(50).sum()
# m15_df['relative_rp'] = (m15_df['realized_power'] / 10) / (m15_df['realized_power_50'] / 50)


'''
## Task 3 - Center of mass and wick-based features (25 min)

**Concept - Center of Mass (COM) of effort in time:** same total `realized_power` could come
from heavy movement early in the window (fading out) or late in the window (accelerating) -
RP itself can't distinguish these, since summation doesn't care about order. COM computes a
weighted average POSITION (in time) of where the |returns| were concentrated, then rescales
to [-1, 1] (-1 = all effort at the start of the window, +1 = all at the end).

- 3a. This one genuinely needs a small loop (not easily vectorized) - for a 10-bar window
     ending at each row, you need `Σ(i * |r_i|) / Σ|r_i|` where `i` runs 1 to 10 for the
     bars in that window, then rescale. Write a function `center_of_mass(abs_returns_window)`
     taking a list of 10 absolute returns, returning the rescaled COM value - same
     stateful-function practice as this week's OTF work, applied to a fixed-size window
     instead of an open-ended streak. `.rolling(10).apply(center_of_mass, raw=True)` can
     then apply it across the whole series once the function works on one window.

**Concept - Wick-to-Close ratio (WTC) and Wick Rejection Asymmetry (WRA):** these use
`high`/`low` directly (information `realized_power` never touches, since RP only looks at
closes). WTC = `Σln(High/Low) / Σ|r_t|` over a window - compares the full high-low range
each bar offered to the actual close-to-close movement; a value near 2 is "normal" for
random walk-like price action, much higher means lots of rejected wicks (indecision), lower
means clean, directional bars. WRA = `(Σ(upper wick) - Σ(lower wick)) / Σ(High-Low)`,
positive meaning upside moves got rejected more (selling pressure at highs), negative the
mirror.

- 3b. Build `wtc_10 = np.log(high/low).rolling(10).sum() / realized_power`.
- 3c. Build upper wick (`high - np.maximum(open, close)`) and lower wick
     (`np.minimum(open, close) - low`) as their own columns first, then
     `wra_10 = (upper_wick.rolling(10).sum() - lower_wick.rolling(10).sum()) / (high - low).rolling(10).sum()`.
'''

# #F1 - Efficiency Ratio ER
# m15_df['net_displacement_10'] = m15_df['close'].pct_change().rolling(10).sum().abs()
# m15_df['efficiency_ratio_10'] = m15_df['net_displacement_10'] / m15_df['realized_power']

# #F2 - Concentration (CONC)
# m15_df['l1_sum_10'] = m15_df['close'].pct_change().abs().rolling(10).sum()
# m15_df['l2_sum_10'] = (m15_df['close'].pct_change()**2).rolling(10).sum()
# m15_df['concentration_10'] = (m15_df['l1_sum_10'] ** 2) / (10 * m15_df['l2_sum_10'])

# #F3 - signed semivariance assymmetry (SJV)
# m15_df['ret_1bar'] = m15_df['close'].pct_change()
# m15_df['squared_returns_10_positive_bars'] = (m15_df['ret_1bar'].where(m15_df['ret_1bar'] > 0, 0) ** 2).rolling(10).sum()
# m15_df['squared_returns_10_negative_bars'] = (m15_df['ret_1bar'].where(m15_df['ret_1bar'] < 0, 0) ** 2).rolling(10).sum()
# m15_df['sjv_10'] = (m15_df['squared_returns_10_positive_bars'] -  m15_df['squared_returns_10_negative_bars']) / (m15_df['squared_returns_10_positive_bars'] +  m15_df['squared_returns_10_negative_bars'])

# #F4 - relative RP
# m15_df['realized_power_50'] = m15_df['close'].pct_change().abs().rolling(50).sum()
# m15_df['relative_rp'] = (m15_df['realized_power'] / 10) / (m15_df['realized_power_50'] / 50)

# #F5 - Center of Mass (COM) - LLM assisted here, as it was a bit too difficult
# def center_of_mass(abs_returns_window):
#     n = len(abs_returns_window)
#     positions = np.arange(1, n + 1)
#     weighted_sum = np.sum(positions * abs_returns_window)
#     total_weight = np.sum(abs_returns_window)
#     if total_weight == 0:
#         return 0.0 
#     raw_com = weighted_sum / total_weight
#     center = (n + 1) / 2
#     radius = (n - 1) / 2
#     return (raw_com - center) / radius

# m15_df['center_of_mass'] = m15_df['close'].rolling(10).apply(center_of_mass, raw=True)

# #F6+F7 Wick to close ratio (WTC) + Wick rejection assymetry (WRA)
# m15_df['wtc_10'] = np.log(m15_df['high'] / m15_df['low']).rolling(10).sum() / m15_df['realized_power']
# m15_df['upper_wick'] = m15_df['high'] - np.maximum(m15_df['open'], m15_df['close'])
# m15_df['lower_wick'] = np.minimum(m15_df['open'], m15_df['close']) - m15_df['low']
# m15_df['wra_10'] = (m15_df['upper_wick'].rolling(10).sum() - m15_df['lower_wick'].rolling(10).sum()) / (m15_df['high'] - m15_df['low']).rolling(10).sum()

# ##############
# ##############
# ##################
# m15_df.head()

In [ ]:
#W8 D3 T4

'''## Task 4 - Screen everything against the target (15 min)

- 4a. Merge all 7 new features (`efficiency_ratio_10`, `concentration_10`, `sjv_10`,
     `relative_rp`, `center_of_mass_10`, `wtc_10`, `wra_10`) into `all_targets_df` at each
     episode's `event_idx`, same pattern as Tuesday.
- 4b. Run the Mann-Whitney + Cohen's d screen (Tuesday's function) across all 7 at once -
     build one table, sorted by Cohen's d magnitude.

**In a comment:** which of today's 7 features comes closest to `realized_power`'s own
effect size (d=0.250)? Does any of them look like it's capturing genuinely NEW information
(worth keeping alongside RP), or do some look redundant with RP itself (very correlated,
not adding much)?'''

from scipy.stats import mannwhitneyu


new_features = ['wra_10', 'wtc_10', 'relative_rp', 'center_of_mass', 'sjv_10', 'concentration_10', 'efficiency_ratio_10']
all_targets_df.head()

def cohens_d(x, y):
  nx = len(x)
  ny = len(y)
  dof = nx + ny - 2
  pooled_std = np.sqrt(
      ((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / dof
  )
  return (np.mean(x) - np.mean(y)) / pooled_std


results = []
for feature in new_features:
    t_continued = all_targets_df.loc[all_targets_df['target_continued'] == 1.0, feature]
    t_reversed = all_targets_df.loc[all_targets_df['target_continued'] == 0.0, feature]
    
    mw_test = mannwhitneyu(t_continued, t_reversed)
    cohen_test = cohens_d(t_continued, t_reversed)
    
    results.append(
        {
            'feature': feature,
            'Mann Whitney p_value': mw_test[1],
            'Cohen D effect value': cohen_test
        }
    )
    
xd = pd.DataFrame(results)
xd.head(7)

# '''	feature	Mann Whitney p_value	Cohen D effect value
# 0	wra_10	0.120444	0.102800
# 1	wtc_10	0.002258	-0.144598
# 2	relative_rp	0.110710	0.101594
# 3	center_of_mass	0.017944	-0.088932
# 4	sjv_10	0.000175	0.159831
# 5	concentration_10	0.966114	-0.015305
# 6	efficiency_ratio_10	0.000176	0.158616'''

#Interesting, as 4/7 features seem to have an effect, it's low as 0.15 is the best we have, and wtc_10 is -0.14,
#and if these features are not just a reprint of realized_power, but they measure something a bit different,
#these features could actually be useful and make a difference together
#and we're actually having features that matter, which is nice


In [ ]:
#W8 D4 WARMUP
''''m15 = pd.read_csv('./project4_trend_regime/m15_with_regime.csv', parse_dates=['et_time'])
```

**Your turn:** build a dict comprehension mapping each `regime` value to the fraction of its
rows where `realized_power` (if already merged onto `m15_df`) is above the overall median -
same inline-boolean-then-mean pattern as all week, now using one of this week's own new
columns instead of an original dataset column.'''

regimes_rp_above_medianrp = {
    regime: (m15_df.loc[m15_df['regime'] == regime, 'realized_power'] > m15_df['realized_power'].median()).mean()
    for regime in m15_df['regime'].unique()
}
print(regimes_rp_above_medianrp)

In [ ]:
#W8 D4 T1  - Feature table check + time-aware split
'''## Task 1 - Assemble the full feature table + time-aware split (15 min)

- 1a. Confirm `all_targets_df` has all ~18 features merged (list them explicitly, drop any
     rows with NaN from the newer features' rolling-window warmup period).
- 1b. Sort by `et_time`, time-aware 80/20 split into train/test (same as before - no
     shuffling).'''

all_targets_df.head()
print(all_targets_df.columns) #
'''Index(['event_idx', 'target_continued', 'direction', 'hour', 'atr14',
       'abs_z_score', 'et_time', 'atr_trend', 'regime_duration_so_far_mins',
       'day_of_week', 'volatility_ratio', 'realized_power',
       'variance_ratio_10', 'wra_10', 'wtc_10', 'relative_rp',
       'center_of_mass', 'sjv_10', 'concentration_10', 'efficiency_ratio_10',
       'otf_streak'],
      dtype='str')'''
print(all_targets_df.isna().sum()) #no nans confirmed
all_targets_df = all_targets_df.sort_values(by = 'et_time')

train = all_targets_df[:int(0.8*len(all_targets_df))]
test = all_targets_df[int(0.8*len(all_targets_df)):]




In [ ]:
#W8 D4 T2  - hyperparameter tuning #1 + RF fit + check
'''## Task 2 - Tune and fit RandomForest with the full overfitting checklist (30 min)

- 2a. Define a BOUNDED parameter grid: `max_depth`: [3, 4, 5, 6], `min_samples_leaf`:
     [5, 10, 20, 30], `max_features`: ['sqrt', 0.5, 0.7], `n_estimators`: [100, 200, 300].
     (Deliberately avoiding `max_depth` above 6 or `min_samples_leaf` below 5, per Week 5's
     lesson about what a too-wide grid allows.)
- 2b. `RandomizedSearchCV(RandomForestClassifier(), param_distributions=..., n_iter=25,
     cv=TimeSeriesSplit(n_splits=5), scoring=make_scorer(roc_auc_score,
     response_method='predict_proba'), error_score='raise')` - fit on train.
- 2c. Evaluate the BEST model on train AND test: print both ROC AUC values side by side,
     plus the gap. Is the gap small (healthy) or large (overfitting, even after tuning)?

**In a comment:** compare today's test AUC to Friday's weak 4-feature baseline (0.561) -
did the expanded, research-grounded feature set actually move the needle?
'''
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, roc_auc_score

parameter_grid = {
    'max_depth': [3, 4, 5, 6],
    'min_samples_leaf': [5, 10, 20, 30],
    'max_features': ['sqrt', 0.5, 0.7],
    'n_estimators': [100, 200, 300]
}

all_features = ['direction', 'hour', 'atr14',
       'abs_z_score', 'atr_trend', 'regime_duration_so_far_mins',
       'day_of_week', 'volatility_ratio', 'realized_power',
       'variance_ratio_10', 'wra_10', 'wtc_10', 'relative_rp',
       'center_of_mass', 'sjv_10', 'concentration_10', 'efficiency_ratio_10',
       'otf_streak', 'target_volatility_rose']
print(all_features)

scorer = make_scorer(roc_auc_score, response_method = 'predict_proba')
model = RandomForestClassifier()
param_search = RandomizedSearchCV(
    estimator = model,
    param_distributions = parameter_grid,
    n_iter = 25,
    cv = TimeSeriesSplit(n_splits = 5),
    scoring = scorer
)
param_search.fit(train[all_features], train['target_continued'])
print(param_search.best_params_) #{'n_estimators': 200, 'min_samples_leaf': 30, 'max_features': 'sqrt', 'max_depth': 6}
print(param_search.best_index_) #13


rf_model = RandomForestClassifier(n_estimators = 200, min_samples_leaf = 30, max_features = 'sqrt', max_depth = 6)
rf_model.fit(train[all_features], train['target_continued'])
test_probas = rf_model.predict_proba(test[all_features])[:, 1]
train_probas = rf_model.predict_proba(train[all_features])[:, 1]

train_rocuac = roc_auc_score(train['target_continued'], train_probas) #
test_rocauc = roc_auc_score(test['target_continued'], test_probas)

print(f'Train ROCAUC: {train_rocauc}, Test ROCAUC: {test_rocauc}')
#Train ROCAUC: 0.559811557625931 #Test ROCAUC: 0.5414019266243083

#Nothing too impressive, but at least there's no overfitting XD
#Perhaps I'm doing something wrong, but I doubt it
#It DOESN'T GO ABOVE THE baseline, which is quite surprising

In [ ]:
#W8 D4 T3  - hyperparameter tuning #1 + XGB fit + check
'''## Task 3 - Tune and fit XGBoost, same discipline (30 min)

- 3a. Same bounded-grid approach: `max_depth`: [2, 3, 4], `learning_rate`: [0.01, 0.05, 0.1],
     `n_estimators`: [50, 100, 200], `subsample`: [0.6, 0.8, 1.0] (same grid shape as Week 6
     Day 2's fair XGBoost tuning pass, which worked well).
- 3b. Same `RandomizedSearchCV` + `TimeSeriesSplit` + `error_score='raise'` setup.
- 3c. Same train vs test AUC comparison, same gap check.

**In a comment:** which model wins on test AUC, and does either show a concerning
train/test gap? Given this week's much richer feature set (vs. Week 6's comparison on a
thin set), does the RF-vs-XGBoost pattern from Week 6 (RF better exploits one dominant
feature; XGBoost spreads importance more) still hold, or does more features change that?'''


from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer, roc_auc_score


parameter_grid = {
    'max_depth': [2, 3, 4], 
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 200], 
    'subsample': [0.6, 0.8, 1.0]
    }


all_features = ['direction', 'hour', 'atr14',
       'abs_z_score', 'atr_trend', 'regime_duration_so_far_mins',
       'day_of_week', 'volatility_ratio', 'realized_power',
       'variance_ratio_10', 'wra_10', 'wtc_10', 'relative_rp',
       'center_of_mass', 'sjv_10', 'concentration_10', 'efficiency_ratio_10',
       'otf_streak', 'target_volatility_rose']

scorer = make_scorer(roc_auc_score, response_method = 'predict_proba')
model = XGBClassifier()
param_search = RandomizedSearchCV(
    estimator = model,
    param_distributions = parameter_grid,
    n_iter = 25,
    cv = TimeSeriesSplit(n_splits = 5),
    scoring = scorer,
    error_score = 'raise'
)
param_search.fit(train[all_features], train['target_continued'])
print(param_search.best_params_)  #{'subsample': 0.6, 'n_estimators': 50, 'max_depth': 3, 'learning_rate': 0.01}
print(param_search.best_index_)  #12


xgb_model = XGBClassifier(subsample = 0.6, n_estimators = 50, max_depth = 3, learning_rate = 0.01)
xgb_model.fit(train[all_features], train['target_continued'])
test_probas = xgb_model.predict_proba(test[all_features])[:, 1]
train_probas = xgb_model.predict_proba(train[all_features])[:, 1]

train_rocuac = roc_auc_score(train['target_continued'], train_probas) #
test_rocauc = roc_auc_score(test['target_continued'], test_probas)

print(f'Train ROCAUC: {train_rocauc}, Test ROCAUC: {test_rocauc}')
#Train ROCAUC: 0.559811557625931, Test ROCAUC: 0.5502835280453644
#Managed to avoid overfitting again, yet the scores are rather miserable, similar as the baseline XD
#We've earned similar score as the RF, nothing too great.
#The baseline is 87-13.
#I'm not sure - maybe that's a good signal that if we're looking for reversals and not continuations,
#we are getting a 55% chance instead of a 13%, but I might be wrong with that assumption...

#The explanation could be that direction seems to be very difficult/impossible to predict overall, this scheme repeats itself in another research
#Perhaps I should focus on predicting different factors like volatility or something else

In [ ]:
#W8 D4 T4 - final feature check for both RF vs XGB

rf_imps = [importance for importance in rf_model.feature_importances_]
xgb_imps = [importance for importance in xgb_model.feature_importances_]

feature_importance_comparison = pd.DataFrame(
    {
        'feature': [feature for feature in all_features],
        'rf_importance': [importance for importance in rf_imps],
        'xgb_importance': [importance for importance in xgb_imps]
        
    }
).sort_values(by = ['rf_importance', 'xgb_importance'], ascending = False)

feature_importance_comparison.head(15)

'''feature	rf_importance	xgb_importance
16	efficiency_ratio_10	0.103655	0.075756
13	center_of_mass	0.101323	0.080783
14	sjv_10	0.097478	0.084778
8	realized_power	0.079460	0.077380
10	wra_10	0.076376	0.074292
9	variance_ratio_10	0.076340	0.056963
2	atr14	0.072220	0.064165
3	abs_z_score	0.061396	0.058481
11	wtc_10	0.052662	0.057403
12	relative_rp	0.052555	0.055991
15	concentration_10	0.051468	0.051398
7	volatility_ratio	0.043420	0.057313
5	regime_duration_so_far_mins	0.040305	0.047401
1	hour	0.040114	0.081125
17	otf_streak	0.023331	0.048331'''

'''Quite frankly, and interestingly:
- there is no dominating feature, all of the features are low
- I'm not sure what threshold would make sense to actually trim any feature, 
as all of them are in the same order of magnitude (e.g. 0.02 - 0.1), XGB seems more "balanced".

Difficult to make a decision, impossible to make a clear feature trim decision.
Perhaps it's the choice of the target here that doesn't make sense, 
as directionality has proven itself to be nearly impossible to predict at this time horizon. 
Maybe it's predictable in the next 10-30 seconds, if one has features from full market data like MBO including whole limit book + market orders etc., 
but in minutes/hours it seems very random, as I try to explore/predict this in concurrent researches'''

In [ ]:
#W8 D5 WARMUP

regime_atr_fraction_above_75ownperc = {
    regime : (m15_df.loc[m15_df['regime'] == regime, 'atr14'] > m15_df.loc[m15_df['regime'] == regime, 'atr14'].quantile(0.75)).mean()
    for regime in m15_df['regime'].unique()
}

print(regime_atr_fraction_above_75ownperc)
'''{'retracement_bull': np.float64(0.2499250824093497), 
'range_recross': np.float64(0.2500076717709516), 
'strong_bear': np.float64(0.2500137234451337), 
'retracement_bear': np.float64(0.2499252615844544), 
'strong_bull': np.float64(0.249994577594621)}
'''
#makes sense XD

In [ ]:
#W8 D5 T1
'''## Task 1 - Define a volatility target (30 min)

**The goal in plain words:** instead of "does price continue in the same direction," ask
"does REALIZED VOLATILITY rise or fall in the N bars after an extreme z-score event,
compared to its level going into the event?" This sidesteps direction entirely - you're
only asking about the MAGNITUDE of future movement, not which way it goes.

- 1a. Using the same episode definitions (`new_high_episode`/`new_low_episode`) and the same
     `realized_power` column (10-bar sum of absolute returns) as this week's feature work -
     for each episode, record `rp_at_event` (realized_power value AT the event bar).
- 1b. Compute `rp_future_20 = realized_power.shift(-20)` at the event bar's position (a
     peek 20 bars ahead at what realized_power looks like then - note this is a FORWARD-
     LOOKING value, usable as a target, never as a feature).
- 1c. Build the target: `target_vol_rose = 1 if rp_future_20 > rp_at_event else 0` - did
     volatility (by this measure) end up higher 20 bars later than it was at the event?
     Drop any rows where either value is NaN (shouldn't be many, given this week's earlier
     NaN handling, but confirm).

**In a comment:** check the class balance of `target_vol_rose` - is it closer to 50/50 than
`target_continued` was (83.6/13.4)? A more balanced target would itself be a meaningful
difference from this week's direction-based attempts.


#below part added at the top in the part where we build other features in the code for maximum efficiency/clarity'''


#below part added at the top in the part where we build other features

# m15_df['new_high_episode'].value_counts()
# m15_df['rp_at_event'] = np.where((m15_df['new_low_episode'] == True) | (m15_df['new_high_episode'] == True), m15_df['realized_power'], None)
# m15_df['rp_at_event'] = m15_df['rp_at_event'].ffill()
# m15_df['rp_future_20'] = m15_df['realized_power'].shift(-20)
# m15_df['target_volatility_rose'] = m15_df.apply(lambda x: 1 if x['rp_future_20'] >= x['rp_at_event'] else 0, axis = 1)


all_targets_df['target_volatility_rose'].mean()
#np.float64(0.44466800804828976)

#pullback_df['target_resumed'] = pullback_df['status'].apply(lambda x: 1 if x == 'resumed' else 0)


In [ ]:
#W8 D5 T2
'''
## Task 2 - Quick univariate check before modeling (20 min)

Same "verify before modeling" discipline as always - does anything already separate the
two volatility-target groups, before any model gets involved?

- 2a. Run the Mann-Whitney + Cohen's d screen (reuse this week's function) across the
     SAME ~18 features from yesterday, against `target_vol_rose` instead of
     `target_continued`.
- 2b. Sort by Cohen's d magnitude. Does anything here look meaningfully stronger than
     yesterday's best direction-predicting feature (`sjv_10`/`efficiency_ratio_10` at
     d≈0.16)?

**In a comment:** based on this screen alone (before fitting anything), does the
volatility target look more promising than the direction target did at this same stage of
Week 8 Day 2/3?'''


print(all_features)
results = []
for feature in all_features:
    vol_rose = all_targets_df.loc[all_targets_df['target_volatility_rose'] == 1.0, feature]
    vol_fell = all_targets_df.loc[all_targets_df['target_volatility_rose'] == 0.0, feature]
    
    mw_test = mannwhitneyu(vol_rose, vol_fell)
    cohen_test = cohens_d(vol_rose, vol_fell)
    
    results.append(
        {
            'feature': feature,
            'Mann Whitney p_value': mw_test[1],
            'Cohen D effect value': cohen_test
        }
    )
    
xd = pd.DataFrame(results)
xd.head(17)

In [ ]:
#W8 D5 T3
'''## Task 3 - Fit RF/XGBoost on the volatility target (25 min)

Same disciplined approach as yesterday - no repeating old mistakes.

- 3a. Time-aware split (same as before). Fit ONE of RF or XGBoost (pick whichever you want,
     no need to redo both today given time) with a bounded grid + `TimeSeriesSplit`, same
     checklist as yesterday.
- 3b. Compare train vs test AUC - healthy gap or not? Compare test AUC to yesterday's
     direction-target results (~0.54-0.55).

**In a comment - this week's actual verdict:** does predicting volatility show a real,
usable improvement over predicting direction, or is this ALSO weak? Either answer is a
valid, useful finding to close the week on.
'''

#RF FIT and check
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, roc_auc_score

parameter_grid = {
    'max_depth': [3, 4, 5, 6],
    'min_samples_leaf': [5, 10, 20, 30],
    'max_features': ['sqrt', 0.5, 0.7],
    'n_estimators': [100, 200, 300]
}

all_features = ['direction', 'hour', 'atr14',
       'abs_z_score', 'atr_trend', 'regime_duration_so_far_mins',
       'day_of_week', 'volatility_ratio', 'realized_power',
       'variance_ratio_10', 'wra_10', 'wtc_10', 'relative_rp',
       'center_of_mass', 'sjv_10', 'concentration_10', 'efficiency_ratio_10',
       'otf_streak']
print(all_features)
#'target_vol_rose'

scorer = make_scorer(roc_auc_score, response_method = 'predict_proba')
model = RandomForestClassifier()
param_search = RandomizedSearchCV(
    estimator = model,
    param_distributions = parameter_grid,
    n_iter = 25,
    cv = TimeSeriesSplit(n_splits = 5),
    scoring = scorer
)
param_search.fit(train[all_features], train['target_volatility_rose'])
print(param_search.best_params_) 
print(param_search.best_index_)


rf_model = RandomForestClassifier(n_estimators = 200, min_samples_leaf = 30, max_features = 'sqrt', max_depth = 6)
rf_model.fit(train[all_features], train['target_volatility_rose'])
test_probas = rf_model.predict_proba(test[all_features])[:, 1]
train_probas = rf_model.predict_proba(train[all_features])[:, 1]

train_rocuac = roc_auc_score(train['target_volatility_rose'], train_probas) #
test_rocauc = roc_auc_score(test['target_volatility_rose'], test_probas)

print(f'RF Train ROCAUC: {train_rocauc}, Test ROCAUC: {test_rocauc}')





#XGB TEST


parameter_grid = {
    'max_depth': [2, 3, 4], 
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 200], 
    'subsample': [0.6, 0.8, 1.0]
    }


all_features = ['direction', 'hour', 'atr14',
       'abs_z_score', 'atr_trend', 'regime_duration_so_far_mins',
       'day_of_week', 'volatility_ratio', 'realized_power',
       'variance_ratio_10', 'wra_10', 'wtc_10', 'relative_rp',
       'center_of_mass', 'sjv_10', 'concentration_10', 'efficiency_ratio_10',
       'otf_streak']

scorer = make_scorer(roc_auc_score, response_method = 'predict_proba')
model = XGBClassifier()
param_search = RandomizedSearchCV(
    estimator = model,
    param_distributions = parameter_grid,
    n_iter = 25,
    cv = TimeSeriesSplit(n_splits = 5),
    scoring = scorer,
    error_score = 'raise'
)
param_search.fit(train[all_features], train['target_volatility_rose'])
print(param_search.best_params_)  
print(param_search.best_index_)


xgb_model = XGBClassifier(subsample = 0.6, n_estimators = 50, max_depth = 3, learning_rate = 0.01)
xgb_model.fit(train[all_features], train['target_volatility_rose'])
test_probas = xgb_model.predict_proba(test[all_features])[:, 1]
train_probas = xgb_model.predict_proba(train[all_features])[:, 1]

train_rocuac = roc_auc_score(train['target_volatility_rose'], train_probas) #
test_rocauc = roc_auc_score(test['target_volatility_rose'], test_probas)

print(f'XGB Train ROCAUC: {train_rocauc}, Test ROCAUC: {test_rocauc}')

In [ ]:
#W9 D1 WARMUP

'''**Your turn:** build a dict comprehension mapping each `regime` value to the fraction of its
rows where `otf_streak` (if merged onto `m15_df`) is positive - same inline-boolean-then-mean
shape as all month, using another of this month's own feature columns this time.'''


regime_positive_otf_streak = {
    regime: (m15_df.loc[m15_df['regime'] == regime, 'otf_streak'] > 0).mean()
    for regime in m15_df['regime'].unique()
}

print(regime_positive_otf_streak)
'''{'retracement_bull': np.float64(0.20437518729397663), 'range_recross': np.float64(0.4389787338509221), 'strong_bear': np.float64(0.40857440851951476), 'retracement_bear': np.float64(0.694170403587444), 'strong_bull': np.float64(0.46554603622166796)}'''

In [ ]:
#W9 D1 T1
'''- 0a. Compute the correlation between `realized_power` (at the event bar) and
     `target_vol_rose` directly: `all_targets_df[['realized_power', 'target_vol_rose']].corr()`
     (or a quick Mann-Whitney between the two `realized_power` groups split by
     `target_vol_rose`, same pattern you've used all month). Is there a visible, strong
     relationship - confirming `realized_power` "knows" something about the target just by
     construction, not by real predictive power?
- 0b. One sentence: in your own words, why does `volatility_ratio` and `relative_rp` being
     BUILT FROM `realized_power` mean they inherit this same problem, even though neither one
     is literally equal to `realized_power`?'''
     
     
print(all_targets_df[['realized_power', 'target_volatility_rose']].corr())
     
# #                        realized_power  target_volatility_rose
# realized_power                1.000000               -0.375671
# target_volatility_rose       -0.375671                1.000000

#There is a negative correlation, but it's 0.37, so not AS BIG, as perhaps expected, I guess.
#I will not draw any conclusions from that, I'm moving straight to the next task.

In [ ]:
#W9 D1 T2
'''- 1a. Build a new feature list EXCLUDING `realized_power`, `volatility_ratio`, and
     `relative_rp` (keep everything else - `direction`, `hour`, `atr14`, `abs_z_score`,
     `atr_trend`, `regime_duration_so_far_mins`, `day_of_week`, `variance_ratio_10`,
     `wra_10`, `wtc_10`, `center_of_mass`, `sjv_10`, `concentration_10`,
     `efficiency_ratio_10`, `otf_streak` - 15 features).
- 1b. Re-run the Mann-Whitney + Cohen's d screen (reuse the function) across these 15,
     against `target_vol_rose`. Sort by Cohen's d magnitude.

**In a comment:** compare this new table to Friday's leaked one. Do the remaining features'
effect sizes look like realistic, modest effects now (in the same ballpark as this month's
direction-target screens, roughly d=0.05-0.25), or does anything still look suspiciously
strong (which would mean there's a SECOND leak path to track down)?'''



all_targets_df.head()
trimmed_targets_df = all_targets_df.drop(columns = ['realized_power', 'volatility_ratio', 'relative_rp'])
trimmed_features = ['direction', 'hour', 'atr14', 'abs_z_score', 'atr_trend', 'regime_duration_so_far_mins',
                    'day_of_week', 'variance_ratio_10', 'wra_10', 'wtc_10', 'center_of_mass', 'sjv_10',
                    'concentration_10', 'efficiency_ratio_10', 'otf_streak']


results = []
for feature in trimmed_features:
     volatility_rose = trimmed_targets_df.loc[trimmed_targets_df['target_volatility_rose'] == 1.0, feature]
     volatility_fell = trimmed_targets_df.loc[trimmed_targets_df['target_volatility_rose'] == 0.0, feature]
     
     cohen_test = cohens_d(volatility_rose, volatility_fell)
     mw_test = mannwhitneyu(volatility_fell, volatility_rose)
     
     results.append({
          'feature': feature,
          'mw_result': mw_test[1],
          'cohens_d': cohen_test
     })
     
test_results = pd.DataFrame(results)
test_results.head(20)

'''
feature	mw_result	cohens_d
0	direction	9.171385e-03	0.083216
1	hour	4.336078e-13	-0.021670
2	atr14	6.510102e-110	-0.388329
3	abs_z_score	1.113643e-10	-0.256054
4	atr_trend	1.043019e-10	-0.207249
5	regime_duration_so_far_mins	3.704397e-06	0.130924
6	day_of_week	7.402686e-04	0.109728
7	variance_ratio_10	5.591952e-07	0.174087
8	wra_10	3.703089e-02	0.057979
9	wtc_10	1.173205e-70	0.570061
10	center_of_mass	2.002357e-08	-0.202831
11	sjv_10	6.551712e-02	0.071583
12	concentration_10	1.694942e-03	0.155516
13	efficiency_ratio_10	7.177730e-01	-0.012234
14	otf_streak	3.007901e-01	0.024405'''

In [ ]:
#W9 D1 T3
'''Same disciplined approach as last week - bounded grid, `TimeSeriesSplit`,
`error_score='raise'`, mandatory train/test gap check.

- 2a. Time-aware split (same as before). Fit RandomForest with a bounded grid (reuse last
     week's grid shape: `max_depth` [3,4,5,6], `min_samples_leaf` [5,10,20,30], etc.) via
     `RandomizedSearchCV` + `TimeSeriesSplit`, on the 15 cleaned features.
- 2b. Compare train AUC vs test AUC - is the gap small and in the EXPECTED direction now
     (train >= test, not inverted)?

**In a comment - the actual verdict:** with the leak removed, is there a genuine, usable
signal for predicting volatility direction, or does it collapse back toward the ~0.54-0.56
range seen for the direction/continuation target all last week? Either answer closes this
question properly - state it plainly.
'''






#RF FIT and check
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, roc_auc_score

parameter_grid = {
    'max_depth': [3, 4, 5, 6],
    'min_samples_leaf': [5, 10, 20, 30],
    'max_features': ['sqrt', 0.5, 0.7],
    'n_estimators': [100, 200, 300]
}

trimmed_features = ['direction', 'hour', 'atr14', 'abs_z_score', 'atr_trend', 'regime_duration_so_far_mins',
                    'day_of_week', 'variance_ratio_10', 'wra_10', 'wtc_10', 'center_of_mass', 'sjv_10',
                    'concentration_10', 'efficiency_ratio_10', 'otf_streak']


scorer = make_scorer(roc_auc_score, response_method = 'predict_proba')
model = RandomForestClassifier()
param_search = RandomizedSearchCV(
    estimator = model,
    param_distributions = parameter_grid,
    n_iter = 25,
    cv = TimeSeriesSplit(n_splits = 5),
    scoring = scorer
)
param_search.fit(train[trimmed_features], train['target_volatility_rose'])
print(param_search.best_params_) 
print(param_search.best_index_)


rf_model = RandomForestClassifier(n_estimators = 200, min_samples_leaf = 30, max_features = 'sqrt', max_depth = 6)
rf_model.fit(train[trimmed_features], train['target_volatility_rose'])
test_probas = rf_model.predict_proba(test[trimmed_features])[:, 1]
train_probas = rf_model.predict_proba(train[trimmed_features])[:, 1]

train_rocuac = roc_auc_score(train['target_volatility_rose'], train_probas) #
test_rocauc = roc_auc_score(test['target_volatility_rose'], test_probas)

print(f'RF Train ROCAUC: {train_rocauc}, Test ROCAUC: {test_rocauc}')





#XGB TEST


parameter_grid = {
    'max_depth': [2, 3, 4], 
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [50, 100, 200], 
    'subsample': [0.6, 0.8, 1.0]
    }


scorer = make_scorer(roc_auc_score, response_method = 'predict_proba')
model = XGBClassifier()
param_search = RandomizedSearchCV(
    estimator = model,
    param_distributions = parameter_grid,
    n_iter = 25,
    cv = TimeSeriesSplit(n_splits = 5),
    scoring = scorer,
    error_score = 'raise'
)
param_search.fit(train[trimmed_features], train['target_volatility_rose'])
print(param_search.best_params_)  
print(param_search.best_index_)


xgb_model = XGBClassifier(subsample = 0.6, n_estimators = 50, max_depth = 3, learning_rate = 0.01)
xgb_model.fit(train[trimmed_features], train['target_volatility_rose'])
test_probas = xgb_model.predict_proba(test[trimmed_features])[:, 1]
train_probas = xgb_model.predict_proba(train[trimmed_features])[:, 1]

train_rocuac = roc_auc_score(train['target_volatility_rose'], train_probas) #
test_rocauc = roc_auc_score(test['target_volatility_rose'], test_probas)

print(f'XGB Train ROCAUC: {train_rocauc}, Test ROCAUC: {test_rocauc}')


'''{'n_estimators': 300, 'min_samples_leaf': 20, 'max_features': 0.5, 'max_depth': 6}
14
RF Train ROCAUC: 0.559811557625931, Test ROCAUC: 0.7697337906255103
{'subsample': 0.8, 'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.05}
16
XGB Train ROCAUC: 0.559811557625931, Test ROCAUC: 0.7293418259023354'''

In [ ]:
print(all_targets_df[['et_time', 'atr14', 'realized_power']].head(10))

In [ ]:
#W9 D2 WARMUP
'''
**Your turn:** build a dict comprehension mapping each `regime` value to the fraction of its
rows where `et_time`'s hour is exactly 3 or 4 (the London Open window being tested today).'''


m15_df.head()
london_open_regimes = { 
                       regime: ((m15_df.loc[m15_df['regime'] == regime, 'hour'] >= 3) & (m15_df.loc[m15_df['regime'] == regime, 'hour'] <= 4)).mean()
                       for regime in m15_df['regime'].unique()
                     }
print(london_open_regimes)
'''{
'retracement_bull': np.float64(0.11807012286484866), 
'range_recross': np.float64(0.08899254303863503), 
'strong_bear': np.float64(0.09340176757973322), 
'retracement_bear': np.float64(0.11689088191330343), 
'strong_bull': np.float64(0.09317861403318511)}'''


In [ ]:
#W9 D2 T1
'''## Task 1 - Build the two windows cleanly, one trading day at a time (25 min)

- 1a. Add `trade_date` (`et_time.dt.date`). Build `is_london_open = et_time.dt.hour == 3`,
     `is_overlap = et_time.dt.hour.between(8, 11)`.
- 1b. Per `trade_date`, compute `realized_power_lon` (sum of absolute 1-bar returns, ONLY
     `is_london_open` rows) and `atr14_lon_mean` (mean `atr14`, same rows).
- 1c. Per `trade_date`, compute `realized_power_overlap` (same formula, `is_overlap` rows
     only).
- 1d. Merge into one row-per-day table, keeping only days present in BOTH windows.

**In a comment:** print 3 example days' `realized_power_lon` vs `realized_power_overlap`
side by side - sensible, no duplicated/missing days?'''


m15_df['ret_1bar'] = m15_df['ret_1bar'].abs()
m15_df['et_time'] = pd.to_datetime(m15_df['et_time'])
m15_df['trade_date'] = pd.to_datetime(m15_df['et_time']).dt.date
m15_df['is_london_open'] = m15_df['et_time'].dt.hour == 3
m15_df['is_overlap'] = m15_df['et_time'].dt.hour.between(8, 11)


rp_lon_per_day = m15_df.loc[m15_df['is_london_open'], ['trade_date', 'ret_1bar']].groupby('trade_date')['ret_1bar'].sum()
atr_lon_per_day = m15_df.loc[m15_df['is_london_open'], ['trade_date', 'atr14']].groupby('trade_date')['atr14'].mean()
rp_overlap_per_day = m15_df.loc[m15_df['is_overlap'], ['trade_date', 'ret_1bar']].groupby('trade_date')['ret_1bar'].sum()

daily_df = pd.DataFrame({
    'realized_power_london': rp_lon_per_day,
    'atr14_london_mean': atr_lon_per_day,
    'realized_power_overlap': rp_overlap_per_day
})
daily_df = daily_df.dropna()

daily_df.head(20)
'''
realized_power_lon	atr14_lon_mean	realized_power_overlap
trade_date			
2021-01-07	-0.001199	3.631319	-0.001082
2021-01-08	0.001472	5.322136	-0.025071
2021-01-11	-0.000345	4.619172	0.007740'''

In [ ]:
#W9 D2 T2
'''
## Task 2 - Build the log-ratio target and features (20 min)

**Concept - why this avoids a leak, briefly re-stated:** `realized_power_lon` (the
feature) and `realized_power_overlap` (half the target) come from COMPLETELY DIFFERENT,
non-overlapping bars (hour 3 vs hours 8-11) - no shared timestamp this time, which is
exactly what caused Friday/yesterday's leak.

- 2a. Build `target_log_ratio = np.log(realized_power_overlap / realized_power_lon)`.
     Handle any zero/negative-safety edge cases (shouldn't occur with sums of absolute
     values, but check `.describe()` for anything at exactly 0).
- 2b. `.describe()` the target - does it look roughly centered near 0 (meaning overlap and
     London-open volatility are similar on a typical day) or skewed one way?

**In a comment:** convert the target's median back with `np.exp()` - in plain words, what
does that number say about a typical day's overlap-vs-London-open volatility ratio?
'''

daily_df['target_log_ratio'] = np.log(daily_df['realized_power_overlap'] / daily_df['realized_power_london'])
daily_df['target_log_ratio'].describe()

'''count    677.000000
mean       1.039797
std        1.611758
min       -6.435213
25%        0.134109
50%        1.053711
75%        1.984417
max        6.530566'''


print(np.exp(daily_df['target_log_ratio'].median())) #2.868275931022899

#It seems the overlap volatility is roughly 3x the London volatility, which kinda seems to make sense

In [ ]:
#W9 D2 T3
'''## Task 3 - Fit a regressor, check the gap (30 min)

**Concept - regression metrics, briefly (new today):** classification used AUC; regression
uses different metrics. `R²` (0 to 1, roughly "how much of the target's variance does the
model explain") and `MAE` (mean absolute error, in the SAME units as the target - here,
log-ratio units) are the two to look at. A model that just predicts the TRAINING MEAN every
time is your baseline - if your model's MAE isn't meaningfully better than that baseline's
MAE, it isn't adding value.

- 3a. Time-aware split by `trade_date` (80/20, no shuffling).
- 3b. Baseline: predict the train set's mean `target_log_ratio` for every test row - compute
     that baseline's MAE on test.
- 3c. Fit `RandomForestRegressor` (bounded grid, same spirit as before - `max_depth`
     [3,4,5,6], `min_samples_leaf` [5,10,20,30]) via `RandomizedSearchCV` + `TimeSeriesSplit`,
     `scoring='neg_mean_absolute_error'`, using `realized_power_lon` and `atr14_lon_mean` as
     features.
- 3d. Compare: train R²/MAE vs test R²/MAE (healthy gap this time?), and test MAE vs the
     baseline's MAE from 3b.

**In a comment - the actual verdict:** does the model beat the naive "always predict the
mean" baseline on test MAE? Is the train/test gap in the expected direction now (train
performing at least as well as test, not inverted like Friday)?'''

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, make_scorer

train = daily_df[:int(0.8*len(daily_df))]
test = daily_df[int(0.8*len(daily_df)):]

features = ['realized_power_london', 'atr14_london_mean']

parameter_grid = {
     'max_depth': [3, 4, 5, 6],
     'min_samples_leaf': [5, 10, 20, 30]
}

scorer_r2 = make_scorer(r2_score)
searcher = RandomizedSearchCV(
     estimator = RandomForestRegressor(),
     param_distributions = parameter_grid,
     n_iter = 20,
     cv = TimeSeriesSplit(n_splits = 5),
     scoring = scorer_r2,
     random_state = 42,
     error_score = 'raise'
)
searcher.fit(train[features], train['target_log_ratio'])
print(searcher.best_params_) #{'min_samples_leaf': 10, 'max_depth': 4}
print(searcher.best_score_) #0.5823657827805055


#baseline mae
baseline_mae = np.full(len(test), train['target_log_ratio'].mean())
baseline_mae = mean_absolute_error(baseline_mae, test['target_log_ratio'])
print(baseline_mae) #0.4775303415538093

model = RandomForestRegressor(
     min_samples_leaf = 10,
     max_depth = 4
)
model.fit(train[features], train['target_log_ratio'])


train_predictions = model.predict(train[features])
train_mae = mean_absolute_error(train['target_log_ratio'], train_predictions)
train_r2 = r2_score(train['target_log_ratio'], train_predictions)

predictions = model.predict(test[features])
test_mae = mean_absolute_error(test['target_log_ratio'], predictions)
test_r2 = r2_score(test['target_log_ratio'], predictions)

print(f'Train mae: {train_mae}, R2: {train_r2}')
print(f'Test mae: {test_mae}, R2: {test_r2}')

'''Train mae: 0.26795405428041047, R2: 0.6453352440402251
Test mae: 0.36402322441028406, R2: 0.363398775584278'''

In [ ]:
#W9 D3 WARMUP
'''Same mapping pattern as yesterday, but with a numeric escalation: build a dict comprehension
mapping each `regime` to the fraction of its rows falling in the new premarket window
(hour == 7), AND nest a second dict comprehension inside computing the same fraction split
further by `day_of_week`.'''

m15_df['day_of_week'] = pd.to_datetime(m15_df['trade_date']).dt.day_of_week
regimes_premarket = {
    regime: (m15_df.loc[m15_df['regime'] == regime, 'hour'] == 7).mean()
    for regime in m15_df['regime'].unique()
}
print(regimes_premarket)

In [ ]:
#W9 D3 T1
###NEW START - przez te modele po drodze RUN ALL na całym notebooku ładuje się z kilka minut - szkoda czasu
#tu dam od nowa potrzebne importy, load dfa itd.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



m15_df = pd.read_csv('project4_trend_regime/m15_with_regime.csv')
m15_df['deviation_30'] = m15_df['close'] - m15_df['close'].rolling(30).mean()
m15_df['z_score_30'] = m15_df['deviation_30'] / m15_df['deviation_30'].rolling(30).std()

is_extreme_high = m15_df['z_score_30'] > 2.5
is_extreme_low = m15_df['z_score_30'] < -2.5
m15_df['new_high_episode'] = is_extreme_high & ~is_extreme_high.shift(1, fill_value=False)
m15_df['new_low_episode'] = is_extreme_low & ~is_extreme_low.shift(1, fill_value=False)

m15_df['ref_close'] = m15_df.apply(lambda x: x['close'] if x['new_high_episode'] == 1 or x['new_low_episode'] == 1 else np.nan, axis = 1)
m15_df['ref_close'] = m15_df['ref_close'].ffill()

####Added 13.10 
m15_df['atr_trend'] = m15_df['atr14'] - m15_df['atr14'].shift(5)
m15_df['atr_trend'] = m15_df['atr_trend'].apply(lambda x: 1 if x > 0 else 0)
m15_df['regime_id'] = (m15_df['regime'] != m15_df['regime'].shift(1)).cumsum()
m15_df['regime_start'] = m15_df.groupby('regime_id')['et_time'].transform('first')
m15_df['regime_duration_so_far_mins'] = (pd.to_datetime(m15_df['et_time']) - pd.to_datetime(m15_df['regime_start'])).dt.seconds // 60
m15_df['day_of_week'] = pd.to_datetime(m15_df['et_time']).dt.day_of_week

m15_df['volatility_ratio'] = m15_df['close'].pct_change().rolling(10).std() / m15_df['close'].pct_change().rolling(50).std()
m15_df['realized_power'] = m15_df['close'].pct_change().abs().rolling(10).sum() #atr proxy - sum of absolute moves
m15_df['variance_ratio_10']  = m15_df['close'].pct_change(10).rolling(200).var() / (10 * m15_df['close'].pct_change().rolling(200).var())


#7 NEW MATH-HEAVY FEATURES ADDED AFTER RESEARCH, ON 14.10

#F1 - Efficiency Ratio ER
m15_df['net_displacement_10'] = m15_df['close'].pct_change().rolling(10).sum().abs()
m15_df['efficiency_ratio_10'] = m15_df['net_displacement_10'] / m15_df['realized_power']

#F2 - Concentration (CONC)
m15_df['l1_sum_10'] = m15_df['close'].pct_change().abs().rolling(10).sum()
m15_df['l2_sum_10'] = (m15_df['close'].pct_change()**2).rolling(10).sum()
m15_df['concentration_10'] = (m15_df['l1_sum_10'] ** 2) / (10 * m15_df['l2_sum_10'])

#F3 - signed semivariance assymmetry (SJV)
m15_df['ret_1bar'] = m15_df['close'].pct_change()
m15_df['squared_returns_10_positive_bars'] = (m15_df['ret_1bar'].where(m15_df['ret_1bar'] > 0, 0) ** 2).rolling(10).sum()
m15_df['squared_returns_10_negative_bars'] = (m15_df['ret_1bar'].where(m15_df['ret_1bar'] < 0, 0) ** 2).rolling(10).sum()
m15_df['sjv_10'] = (m15_df['squared_returns_10_positive_bars'] -  m15_df['squared_returns_10_negative_bars']) / (m15_df['squared_returns_10_positive_bars'] +  m15_df['squared_returns_10_negative_bars'])

#F4 - relative RP
m15_df['realized_power_50'] = m15_df['close'].pct_change().abs().rolling(50).sum()
m15_df['relative_rp'] = (m15_df['realized_power'] / 10) / (m15_df['realized_power_50'] / 50)

#F5 - Center of Mass (COM) - LLM assisted here, as it was a bit too difficult
def center_of_mass(abs_returns_window):
    n = len(abs_returns_window)
    positions = np.arange(1, n + 1)
    weighted_sum = np.sum(positions * abs_returns_window)
    total_weight = np.sum(abs_returns_window)
    if total_weight == 0:
        return 0.0 
    raw_com = weighted_sum / total_weight
    center = (n + 1) / 2
    radius = (n - 1) / 2
    return (raw_com - center) / radius

m15_df['abs_returns_1bar'] = m15_df['close'].pct_change().abs()
m15_df['center_of_mass'] = m15_df['abs_returns_1bar'].rolling(10).apply(center_of_mass, raw=True)

#F6+F7 Wick to close ratio (WTC) + Wick rejection assymetry (WRA)
m15_df['wtc_10'] = np.log(m15_df['high'] / m15_df['low']).rolling(10).sum() / m15_df['realized_power']
m15_df['upper_wick'] = m15_df['high'] - np.maximum(m15_df['open'], m15_df['close'])
m15_df['lower_wick'] = np.minimum(m15_df['open'], m15_df['close']) - m15_df['low']
m15_df['wra_10'] = (m15_df['upper_wick'].rolling(10).sum() - m15_df['lower_wick'].rolling(10).sum()) / (m15_df['high'] - m15_df['low']).rolling(10).sum()

#######

#Volatility target pivot - 16.10.26
m15_df['rp_at_event'] = np.where((m15_df['new_low_episode'] == True) | (m15_df['new_high_episode'] == True), m15_df['realized_power'], None)
m15_df['rp_at_event'] = m15_df['rp_at_event'].ffill()
m15_df = m15_df.dropna()
m15_df['rp_future_20'] = m15_df['realized_power'].shift(-20)
m15_df['target_volatility_rose'] = m15_df.apply(lambda x: 1 if x['rp_future_20'] > x['rp_at_event'] else 0, axis = 1)


#otf_streaksy --- 

from typing import List

def otf_streak(highs: List, lows: List) -> List:
     otf_streaks = [0] * len(highs)
     try:
          for i in range(1, len(otf_streaks)):
               if (lows[i] > lows[i - 1]) and (highs[i] < highs[i - 1]): #inside bar
                    otf_streaks[i] = otf_streaks[i - 1]
                    
               elif (lows[i] > lows[i - 1]):
                    if otf_streaks[i - 1] < 0:
                         otf_streaks[i] = 1
                    else:
                         otf_streaks[i] = otf_streaks[i - 1] + 1
                         
               elif (highs[i] < highs[i - 1]):
                    if otf_streaks[i - 1] > 0:
                         otf_streaks[i] = -1
                    else:
                         otf_streaks[i] = otf_streaks[i - 1] - 1
     except Exception as e: 
          print(f'Unhandled exception: {str(e)}')
     return otf_streaks 
          
     #low > low-1 #otfup
     #high < high -1 #otfdown
     
m15_df['otf_streak'] = otf_streak(m15_df['high'].tolist(), m15_df['low'].tolist())


#abs z-score + hour

m15_df['abs_z_score'] = m15_df['z_score_30'].abs()
m15_df['trade_date'] = pd.to_datetime(m15_df['et_time']).dt.date
m15_df['et_time'] = pd.to_datetime(m15_df['et_time'])
m15_df['hour'] = pd.to_datetime(m15_df['et_time']).dt.hour


m15_df = m15_df.reset_index(drop=True)




'''## Task 1 - Build the two new windows (25 min)

- 1a. `is_premarket = et_time.dt.hour == 7`, `is_overlap1h = et_time.dt.hour == 8`.
- 1b. Per `trade_date`, build ALL premarket-window features listed above (one groupby per
     feature, or a single `.agg()` call with a dict mapping column -> function - your
     choice, but keep it one clean table at the end).
- 1c. Per `trade_date`, build `realized_power_overlap1h` (same sum-of-abs formula, `is_overlap1h`
     rows only).
- 1d. Merge into one row-per-day table, keep only days present in both windows, `.dropna()`.

**In a comment:** print 3 example days across all columns - anything that looks structurally
wrong (e.g. a streak/duration value that's suspiciously constant, or still negative where it
shouldn't be)?'''

m15_df['is_premarket'] = m15_df['et_time'].dt.hour == 7
m15_df['is_overlap'] = m15_df['et_time'].dt.hour == 8
m15_df['day_of_week'] = pd.to_datetime(m15_df['trade_date']).dt.day_of_week

premarket_daily = m15_df.groupby(['trade_date', 'is_premarket']).agg(
    realized_power_premarket = ('abs_returns_1bar', 'sum'),
    atr14_premarket_mean = ('atr14', 'mean'),
    otf_streak_premarket = ('otf_streak', 'last'),
    sjv_10_premarket_mean = ('sjv_10', 'mean'),
    efficiency_ratio_10_premarket_mean = ('efficiency_ratio_10', 'mean'),
    wtc_10_premarket_mean = ('wtc_10', 'mean'),
    center_of_mass_mean = ('center_of_mass', 'mean'),
    atr_trend_premarket = ('atr_trend', 'last'),
    regime_duration_premarket = ('regime_duration_so_far_mins', 'last'),
    day_of_week = ('day_of_week', 'last')
).reset_index()

premarket_daily.head()
premarket_daily = premarket_daily[premarket_daily['is_premarket'] == True]

overlap_fh_daily = m15_df.groupby(['trade_date', 'is_overlap']).agg(
    realized_power_overlap = ('abs_returns_1bar', 'sum'),
    atr14_premarket_overlap = ('atr14', 'mean'),
    otf_streak_overlap = ('otf_streak', 'last'),
    sjv_10_premarket_overlap = ('sjv_10', 'mean'),
    efficiency_ratio_10_overlap = ('efficiency_ratio_10', 'mean'),
    wtc_10_overlap = ('wtc_10', 'mean'),
    center_of_mass_mean_overlap = ('center_of_mass', 'mean'),
    atr_trend_overlap = ('atr_trend', 'last'),
    regime_duration_overlap = ('regime_duration_so_far_mins', 'last'),
    #day_of_week = ('day_of_week', 'last') - żeby nie dublować
).reset_index()

overlap_fh_daily = overlap_fh_daily[overlap_fh_daily['is_overlap'] == True]
overlap_fh_daily.head()


daily_df = premarket_daily.merge(overlap_fh_daily, on = 'trade_date', how = 'inner')
'''
trade_date	is_premarket	realized_power_premarket	atr14_premarket_mean	otf_streak_premarket	sjv_10_premarket_mean	efficiency_ratio_10_premarket_mean	wtc_10_premarket_mean	center_of_mass_mean	atr_trend_premarket	...	atr14_premarket_overlap	otf_streak_overlap	sjv_10_premarket_overlap	efficiency_ratio_10_overlap	wtc_10_overlap	center_of_mass_mean_overlap	atr_trend_overlap	regime_duration_overlap	day_of_week_y	target_log_ratio
0	2021-01-06	True	0.006679	6.518668	-2	-0.735842	0.413374	1.615206	0.118933	1	...	6.149160	1	-0.697124	0.335071	1.851333	-0.269022	1	120	2	-0.414028
1	2021-01-07	True	0.003003	3.596889	-2	0.034003	0.259330	2.021943	-0.311091	1	...	4.456369	-1	-0.456103	0.259046	2.432421	0.275001	1	1365	3	0.826455
2	2021-01-08	True	0.013695	4.667762	1	-0.496160	0.335645	1.816359	0.387053	1	...	6.819526	-3	-0.097387	0.318072	1.574594	0.292252	1	1365	4	0.011318
3	2021-01-11	True	0.004509	3.868514	-7	-0.851180	0.741223	1.992824	0.053418	1	...	4.664387	2	-0.612114	0.635269	2.718302	-0.008137	1	1365	0	-0.068061'''

#Eveyrthing seems to look fine
daily_df.head()

In [ ]:
#W9 D3 T2
'''## Task 2 - Build the log-ratio target (15 min)

- 2a. `target_log_ratio = np.log(realized_power_overlap1h / realized_power_premarket)`.
- 2b. `.describe()` - compare its spread (std) to yesterday's version. Narrower or wider? Any
     intuition for why a shorter, tighter window pair might behave differently?

**In a comment:** convert the median back with `np.exp()` and state in plain words what it
means for a typical day.'''

daily_df['target_log_ratio'] = np.log(daily_df['realized_power_overlap'] / daily_df['realized_power_premarket'])
daily_df['target_log_ratio'].describe()
'''count    1424.000000
mean        0.001632
std         0.646571
min        -2.166348
25%        -0.427756
50%        -0.019391
75%         0.404998
max         2.465844
Name: target_log_ratio, dtype: float64'''

#Obviously it doesn't differ from yesterday

daily_df.head()

In [ ]:
#W9 D3 T3
'''## Task 3 - Fit a regressor on the full feature set, check the gap (40 min)

- 3a. Time-aware split by `trade_date` (80/20, no shuffling).
- 3b. Baseline MAE: predict train mean on test.
- 3c. Fit `RandomForestRegressor` via `RandomizedSearchCV` + `TimeSeriesSplit`,
     `scoring='neg_mean_absolute_error'`, same bounded grid as yesterday
     (`max_depth` [3,4,5,6], `min_samples_leaf` [5,10,20,30]), using ALL features from Task 1.
- 3d. Compare train/test R²/MAE vs baseline. Pull `feature_importances_` - which 2-3 features
     actually carry weight, and does that match your intuition from the univariate screens
     back in Week 8?

**In a comment - the actual verdict:** does the bigger feature set beat yesterday's 2-feature
result on test MAE? If not, what does that tell you about whether more features was ever
going to be the lever that mattered here?'''


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score

train = daily_df[:int(0.8 * len(daily_df))]
test = daily_df[int(0.8 * len(daily_df)):]

baseline_mae = np.full(len(test), train['target_log_ratio'].mean())
baseline_mae = mean_absolute_error(baseline_mae, test['target_log_ratio'])
print(baseline_mae) #0.481953381955044

scorer_mae = make_scorer(mean_absolute_error)
features = [
    "realized_power_premarket",
    "atr14_premarket_mean",
    "otf_streak_premarket",
    "sjv_10_premarket_mean",
    "efficiency_ratio_10_premarket_mean",
    "wtc_10_premarket_mean",
    "center_of_mass_mean",
    "atr_trend_premarket",
    "regime_duration_premarket",
    "day_of_week"
]

param_grid = {
     'max_depth': [3, 4, 5, 6],
     'min_samples_leaf': [5, 10, 20, 30]
}
searcher = RandomizedSearchCV(
     estimator = RandomForestRegressor(),
     param_distributions = param_grid,
     n_iter = 20,
     cv = TimeSeriesSplit(n_splits = 5),
     scoring = scorer_mae,
     error_score = 'raise',
     random_state = 42
     )


searcher.fit(train[features], train['target_log_ratio'])
print(searcher.best_params_)


model = RandomForestRegressor(min_samples_leaf = 5, max_depth = 6)
model.fit(train[features], train['target_log_ratio'])
train_predictions = model.predict(train[features])
test_predictions = model.predict(test[features])

train_mae = mean_absolute_error(train['target_log_ratio'], train_predictions)
train_r2 = r2_score(train['target_log_ratio'], train_predictions)

test_r2 = r2_score(test['target_log_ratio'], test_predictions)
test_mae = mean_absolute_error(test['target_log_ratio'], test_predictions)

print(f'Train MAE: {train_mae}, Test MAE: {test_mae}')
print(f'Train R2: {train_r2}, Test R2: {test_r2}')
'''Train MAE: 0.3295705219739197, Test MAE: 0.4253363181593707
Train R2: 0.5929026693043737, Test R2: 0.18348532919776828'''


#MAE looks quite alright, yet R2 is a bit scary

print(model.feature_importances_)
imps = [importance for importance in model.feature_importances_]
results = pd.DataFrame(
    {
        'feature': [feature for feature in features],
        'importance': [imp for imp in imps]
    }
).sort_values(by = 'importance', ascending = False)
print(results)
'''                              feature  importance
0            realized_power_premarket    0.655038
1                atr14_premarket_mean    0.128631
5               wtc_10_premarket_mean    0.044523
6                 center_of_mass_mean    0.041060
4  efficiency_ratio_10_premarket_mean    0.038321
8           regime_duration_premarket    0.038182
3               sjv_10_premarket_mean    0.026403
9                         day_of_week    0.017256
2                otf_streak_premarket    0.010444
7                 atr_trend_premarket    0.000141'''


In [ ]:
#W9 D4 NEW START#2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

m5_df = pd.read_csv('xauusd_m5_et.csv', parse_dates=['et_time'])
pd.options.display.max_columns = 50



# realized_power (same def as every prior session)
m5_df['ret_1bar'] = m5_df['close'].pct_change()
m5_df['abs_ret_1bar'] = m5_df['ret_1bar'].abs()

# atr14 (Wilder ATR, 14-period) - M5 file has no atr14 column, built fresh here
m5_df['prev_close'] = m5_df['close'].shift(1)
m5_df['tr'] = m5_df[['high', 'low', 'prev_close']].apply(
    lambda row: max(row['high'] - row['low'],
                    abs(row['high'] - row['prev_close']),
                    abs(row['low'] - row['prev_close'])), axis=1)
m5_df['atr14'] = m5_df['tr'].ewm(alpha=1/14, adjust=False).mean()

# otf_streak - exact function from Week 9 Day 1, unchanged
from typing import List
def otf_streak(highs: List, lows: List) -> List:
    otf_streaks = [0] * len(highs)
    for i in range(1, len(otf_streaks)):
        if (lows[i] > lows[i-1]) and (highs[i] < highs[i-1]):
            otf_streaks[i] = otf_streaks[i-1]
        elif lows[i] > lows[i-1]:
            otf_streaks[i] = 1 if otf_streaks[i-1] < 0 else otf_streaks[i-1] + 1
        elif highs[i] < highs[i-1]:
            otf_streaks[i] = -1 if otf_streaks[i-1] > 0 else otf_streaks[i-1] - 1
    return otf_streaks
m5_df['otf_streak'] = otf_streak(m5_df['high'].tolist(), m5_df['low'].tolist())

# atr_trend (1 if atr rising over last 5 bars, else 0 - same def as Week 8)
m5_df['atr_trend'] = (m5_df['atr14'] - m5_df['atr14'].shift(5) > 0).astype(int)

# efficiency_ratio_10, sjv_10, center_of_mass, wtc_10 - same window-10 defs as Week 8 Day 3
m5_df['net_displacement_10'] = m5_df['ret_1bar'].rolling(10).sum().abs()
m5_df['realized_power_10'] = m5_df['abs_ret_1bar'].rolling(10).sum()
m5_df['efficiency_ratio_10'] = m5_df['net_displacement_10'] / m5_df['realized_power_10']

m5_df['sq_ret_pos_10'] = (m5_df['ret_1bar'].where(m5_df['ret_1bar'] > 0, 0) ** 2).rolling(10).sum()
m5_df['sq_ret_neg_10'] = (m5_df['ret_1bar'].where(m5_df['ret_1bar'] < 0, 0) ** 2).rolling(10).sum()
m5_df['sjv_10'] = (m5_df['sq_ret_pos_10'] - m5_df['sq_ret_neg_10']) / (m5_df['sq_ret_pos_10'] + m5_df['sq_ret_neg_10'])

def center_of_mass(abs_returns_window):
    n = len(abs_returns_window)
    positions = np.arange(1, n + 1)
    total_weight = np.sum(abs_returns_window)
    if total_weight == 0:
        return 0.0
    raw_com = np.sum(positions * abs_returns_window) / total_weight
    center = (n + 1) / 2
    radius = (n - 1) / 2
    return (raw_com - center) / radius
m5_df['center_of_mass'] = m5_df['abs_ret_1bar'].rolling(10).apply(center_of_mass, raw=True)

m5_df['wtc_10'] = np.log(m5_df['high'] / m5_df['low']).rolling(10).sum() / m5_df['realized_power_10']


####added 5.11.2026 - Parkinson volatility which also captures the intrabar swing, not only close-close

m5_df['parkinson_volatility'] = (np.log(m5_df['high'] / m5_df['low']) ** 2) / (4 * np.log(2))

#+ hh/min added
m5_df['hour'] = pd.to_datetime(m5_df['et_time']).dt.hour
m5_df['minute'] = pd.to_datetime(m5_df['et_time']).dt.minute


In [ ]:
#W9 D4 WARMUP
'''on `m5_df`, build a dict comprehension mapping each
`day_of_week` to the fraction of its rows whose `et_time` hour is in `[8, 9)` (the overlap's
first hour) AND minute is `< 15` (first 15 min specifically) - combine both conditions with
`&` on the boolean masks, same pattern as every warm-up this month, just two conditions
`&`-ed together instead of one.'''
m5_df['hour'] = pd.to_datetime(m5_df['et_time']).dt.hour
m5_df['minute'] = pd.to_datetime(m5_df['et_time']).dt.minute
print(m5_df.dtypes)
dow_above_8915 = {
    dow: (
        m5_df.loc[m5_df['day_of_week'] == dow, 'hour'].between(8, 9) 
          & m5_df.loc[m5_df['day_of_week'] == dow, 'minute'] < 15
          ).mean()
    for dow in m5_df['day_of_week'].unique()
}

print(dow_above_8915)

In [ ]:
#W9 D4 T1
'''
- 1a. On `m5_df`: `is_obs = (et_time.dt.hour == 8) & (et_time.dt.minute < 15)`,
     `is_pred = (et_time.dt.hour == 8) & (et_time.dt.minute >= 15) | (et_time.dt.hour == 8... )` -
     actually simpler: build `is_pred` as "hour==8 and minute>=15" OR "hour==8, minute==45..59"
     - think about this as one clean boundary: `is_pred = (et_time.dt.hour == 8) &
     (et_time.dt.minute >= 15)` already covers 8:15 through 8:59, which IS the rest of the
     hour - no second OR needed. Confirm this with `.sum()` on both masks before moving on.
- 1b. Per `trade_date`, aggregate ALL observation-window features (mean for
     efficiency_ratio_10/sjv_10/center_of_mass/wtc_10, sum of `abs_ret_1bar` for
     `realized_power_obs`, mean for `atr14`, last value for `otf_streak`/`atr_trend`) -
     `.agg()` with a dict, same pattern as Day 3.
- 1c. Per `trade_date`, build `realized_power_pred` (sum of `abs_ret_1bar`, `is_pred` rows
     only).
- 1d. Merge, keep only days in both, `.dropna()`.

**In a comment:** print 3 example days - anything look off (e.g. observation window with
only 2-3 rows instead of the expected 3)?'''

m5_df['is_obs'] = (m5_df['et_time'].dt.hour == 8) & (m5_df['et_time'].dt.minute < 15)
m5_df['is_pred'] = (m5_df['et_time'].dt.hour == 8) & (m5_df['et_time'].dt.minute >= 15)

print(m5_df.columns)
observation_window = m5_df.groupby(['is_obs', 'trade_date']).agg(
    realized_power_observation = ('abs_ret_1bar', 'sum'),
    atr14_mean_observation = ('atr14', 'mean'),
    otf_streak_observation = ('otf_streak', 'last'),
    sjv_10_observation = ('sjv_10', 'mean'),
    efficiency_ratio_10_observation = ('efficiency_ratio_10', 'mean'),
    wtc_10_observation = ('wtc_10', 'mean'),
    center_of_mass_mean_observation = ('center_of_mass', 'mean'),
    atr_trend_observation = ('atr_trend', 'last'),
    parkinson_obs = ('parkinson_volatility', 'sum')
).reset_index()

predicted_window = m5_df.groupby(['is_pred', 'trade_date']).agg(
    realized_power_predicted = ('abs_ret_1bar', 'sum'),
    atr14_mean_predicted = ('atr14', 'mean'),
    otf_streak_predicted = ('otf_streak', 'last'),
    sjv_10_predicted = ('sjv_10', 'mean'),
    efficiency_ratio_10_predicted = ('efficiency_ratio_10', 'mean'),
    wtc_10_predicted = ('wtc_10', 'mean'),
    center_of_mass_mean_predicted = ('center_of_mass', 'mean'),
    atr_trend_predicted = ('atr_trend', 'last'),
    parkinson_predicted = ('parkinson_volatility', 'sum')
).reset_index()

observation_window = observation_window[observation_window['is_obs'] == True]
predicted_window = predicted_window[predicted_window['is_pred'] == True]

merged_df = observation_window.merge(predicted_window, how = 'inner', on = 'trade_date')
merged_df = merged_df.dropna()
merged_df.head()

In [ ]:
#W9 D4 T2
'''- 2a. `target_log_ratio = np.log(realized_power_pred / realized_power_obs)`.
- 2b. `.describe()` - compare std to both prior days (1.61, then 0.65). Prediction, before
     looking: given this is the SHORTEST window pair yet, do you expect std to keep
     shrinking, or could the tiny 15-min observation window (only 3 bars) make estimates
     noisier and push std back UP?

**In a comment:** state your prediction from 2b BEFORE running `.describe()`, then say
whether the actual result matched it.'''

merged_df['target_log_ratio'] = np.log(merged_df['realized_power_predicted'] / merged_df['realized_power_observation'])
merged_df['target_log_ratio'].describe()

'''count    1426.000000
mean        0.970094
std         0.646964
min        -0.848754
25%         0.528205
50%         0.917537
75%         1.369972
max         4.463253'''

#the std is 0.64, so almost the same.

In [ ]:
#W9 D4 T3
'''- 3a. Time-aware split by `trade_date` (80/20, no shuffling).
- 3b. Baseline MAE: predict train mean on test.
- 3c. Fit `RandomForestRegressor` via `RandomizedSearchCV` + `TimeSeriesSplit`,
     `scoring='neg_mean_absolute_error'`, same bounded grid as both prior days
     (`max_depth` [3,4,5,6], `min_samples_leaf` [5,10,20,30]), using all 8 observation-window
     features (realized_power_obs, atr14, otf_streak, sjv_10, efficiency_ratio_10, wtc_10,
     center_of_mass, atr_trend).
- 3d. Compare train/test R²/MAE vs baseline. Pull `feature_importances_`.

**In a comment - the actual verdict, tying together all three days:** does the shortest,
same-session window pair beat BOTH prior days on margin-over-baseline (MAE), confirming the
research finding that shorter horizons carry a stronger signal - or does the opposite happen
(a gap this short being mostly noise, not real predictability)? Either answer closes this
3-day arc properly.'''

merged_df['day_of_week'] = pd.to_datetime(merged_df['trade_date']).dt.day_of_week

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score

train = merged_df[:int(0.8 * len(merged_df))]
test = merged_df[int(0.8 * len(merged_df)):]

print(train.columns)

baseline_mae = np.full(len(test), train['target_log_ratio'].mean())
baseline_mae = mean_absolute_error(baseline_mae, test['target_log_ratio'])
print(baseline_mae) #0.481953381955044

scorer_mae = make_scorer(mean_absolute_error)
features = [
    "realized_power_observation",
    "atr14_mean_observation",
    "otf_streak_observation",
    "sjv_10_observation",
    "efficiency_ratio_10_observation",
    "wtc_10_observation",
    "center_of_mass_mean_observation",
    "atr_trend_observation",
    "day_of_week"
]

'''Index(['is_obs', 'trade_date', 'realized_power_observation',
       'atr14_mean_observation', 'otf_streak_observation',
       'sjv_10_observation', 'efficiency_ratio_10_observation',
       'wtc_10_observation', 'center_of_mass_mean_observation',
       'atr_trend_observation', 'is_pred', 'realized_power_predicted',
       'atr14_mean_predicted', 'otf_streak_predicted', 'sjv_10_predicted',
       'efficiency_ratio_10_predicted', 'wtc_10_predicted',
       'center_of_mass_mean_predicted', 'atr_trend_predicted',
       'target_log_ratio'],
      dtype='str')'''

param_grid = {
     'max_depth': [3, 4, 5, 6],
     'min_samples_leaf': [5, 10, 20, 30]
}
searcher = RandomizedSearchCV(
     estimator = RandomForestRegressor(),
     param_distributions = param_grid,
     n_iter = 20,
     cv = TimeSeriesSplit(n_splits = 5),
     scoring = scorer_mae,
     error_score = 'raise',
     random_state = 42
     )


searcher.fit(train[features], train['target_log_ratio'])
print(searcher.best_params_)

best_params = {
    'min_samples_leaf': 5,
    'max_depth': 6
}

model = RandomForestRegressor(min_samples_leaf = 5, max_depth = 6)
model.fit(train[features], train['target_log_ratio'])
train_predictions = model.predict(train[features])
test_predictions = model.predict(test[features])

train_mae = mean_absolute_error(train['target_log_ratio'], train_predictions)
train_r2 = r2_score(train['target_log_ratio'], train_predictions)

test_r2 = r2_score(test['target_log_ratio'], test_predictions)
test_mae = mean_absolute_error(test['target_log_ratio'], test_predictions)

print(f'Train MAE: {train_mae}, Test MAE: {test_mae}')
print(f'Train R2: {train_r2}, Test R2: {test_r2}')


#MAE looks quite alright, yet R2 is a bit scary

print(model.feature_importances_)
imps = [importance for importance in model.feature_importances_]
results = pd.DataFrame(
    {
        'feature': [feature for feature in features],
        'importance': [imp for imp in imps]
    }
).sort_values(by = 'importance', ascending = False)
print(results)




In [ ]:
import mlflow

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('xauusd_volatility_windows')

with mlflow.start_run(run_name='day4_real_model'):
    mlflow.log_params({'min_samples_leaf': 5, 'max_depth': 6})
    mlflow.log_metric('test_mae', test_mae)
    mlflow.log_metric('test_r2', test_r2)
    mlflow.sklearn.log_model(
    model,
    'model',
    skops_trusted_types=['sklearn.tree._tree.Tree']
)

    run_id = mlflow.active_run().info.run_id

print(run_id)


In [ ]:
loaded_model = mlflow.sklearn.load_model(f'runs:/{run_id}/model')
print((loaded_model.predict(test[features]) == model.predict(test[features])).all())


In [ ]:
import predict_volatility_multiplier

observation_features = test[features].iloc[0].to_dict()

model = predict_volatility_multiplier.load_model()
predicted_value = predict_volatility_multiplier.predict_volatility(model, observation_features)

print(predicted_value) #2.193873111824186

In [ ]:
#W10 D1 - TASK 2 - request to our FastAPI endpoint
'''In a notebook cell (not the FastAPI file) - this simulates how any OTHER program (not just
the `/docs` page) would use your API.'''

# import requests

# one_row = test[features].iloc[0].to_dict()
# response = requests.post('http://127.0.0.1:8000/predict', json=one_row)
# print(response.status_code)
# print(response.json())

#200
#{'predicted_volatility_multiplier': 2.193873111824186}

#Everything works properly!
#I've encountered an issue because the oclumns in row were different in my dict vs the FastAPI endpoint schema,
# but I've fixed it quickly

In [ ]:
#W10 D2 T3 - the rest for today in live_data.py :))

import live_data
import pandas as pd
latest_rows = live_data.get_recent_bars(symbol = 'XAU/USD', interval = '5min')
print(latest_rows)


'''#W10 D2

              datetime        open        high         low       close
0  2026-10-27 12:30:00  4021.01759   4029.1793  4021.01759  4023.54418
1  2026-10-27 12:15:00  4013.43137  4022.36435  4013.16318  4021.92106
2  2026-10-27 12:00:00  4012.73716  4017.25417  4010.96801  4013.25788
3  2026-10-27 11:45:00  4009.46367  4014.12974  4009.14084  4012.60037
4  2026-10-27 11:30:00  4011.34993  4013.94654  4008.65623   4009.9457'''


#Desired format
'''et_time,open,high,low,close,volume,day_of_week,trade_date
2021-01-04 00:00:00,1921.648,1922.658,1921.448,1922.305,0.2,1.0,2021-01-04'''


#I've stwitched timezone from UTC to America/New_York to make sure we're aligned from the beginning

latest_rows['day_of_week'] = pd.to_datetime(latest_rows['datetime']).dt.day_of_week
latest_rows['trade_date'] = pd.to_datetime(latest_rows['datetime']).dt.date
latest_rows['et_time'] = pd.to_datetime(latest_rows['datetime'])

print(latest_rows.dtypes)
latest_rows.head()

'''datetime                  str
open                      str
high                      str
low                       str
close                     str
day_of_week             int32
trade_date             object
et_time        datetime64[us]
dtype: object
datetime	open	high	low	close	day_of_week	trade_date	et_time
0	2026-10-27 12:45:00	4023.8693	4024.95472	4018.64652	4024.23346	1	2026-10-27	2026-10-27 12:45:00
1	2026-10-27 12:30:00	4021.01759	4029.1793	4021.01759	4023.41301	1	2026-10-27	2026-10-27 12:30:00
2	2026-10-27 12:15:00	4013.43137	4022.36435	4013.16318	4021.92106	1	2026-10-27	2026-10-27 12:15:00
3	2026-10-27 12:00:00	4012.73716	4017.25417	4010.96801	4013.25788	1	2026-10-27	2026-10-27 12:00:00
4	2026-10-27 11:45:00	4009.46367	4014.12974	4009.14084	4012.60037	1	2026-10-27	2026-10-27 11:45:00'''

In [ ]:
#W10 D3 T1

'''## Task 1 - Filter to only CLOSED bars in the observation window (25 min)

- 1a. Call `get_recent_bars('XAU/USD', '5min', outputsize=20)` - you already have this
     working. Confirm the returned `datetime` column, converted to a proper pandas
     datetime, lets you check `.dt.hour` and `.dt.minute`.
- 1b. Drop the forming candle: the bar matching the LATEST timestamp in the batch is
     always still in progress (yesterday's finding) - filter it out explicitly before
     anything else touches this DataFrame. One clean way: sort by datetime descending,
     drop row 0, keep the rest.
- 1c. From the remaining CLOSED bars, filter to the observation window: hour == 8 AND
     minute in `[0, 5, 10]` (equivalently `minute < 15`). In a live system running right
     now (not necessarily during that window), this might return 0 rows - that's expected
     and correct, not a bug. For TESTING today, you'll want to either (i) run this when
     it's actually near 8am ET, or (ii) temporarily test against whatever hour/minute IS
     live right now by adjusting the filter, then switch back.

**In a comment:** how many rows did you get after 1c, and does that match your expectation
(3, if you're testing exactly in-window; 0 otherwise)?'''


import live_data
import pandas as pd
import datetime as dt
from zoneinfo import ZoneInfo

latest_rows = live_data.get_recent_bars(symbol = 'XAU/USD', interval = '5min').sort_values(by = 'datetime')
latest_rows['hour'] = pd.to_datetime(latest_rows['datetime']).dt.hour
latest_rows['minute'] = pd.to_datetime(latest_rows['datetime']).dt.minute
latest_rows['trade_date'] = pd.to_datetime(latest_rows['datetime']).dt.date

current_time = dt.datetime.now(ZoneInfo('America/New_York')) #adjusted to ET timezone
current_hour = current_time.hour
current_minute = current_time.minute
current_date = current_time.date()
print(f'Current time: {current_time}, hour: {current_hour}, minute: {current_minute}, date: {current_date}')



###FEATURES CALC - COPIED FROM ABOVE CELLS


latest_rows['close'] = latest_rows['close'].astype('float')
latest_rows['open'] = latest_rows['open'].astype('float')
latest_rows['high'] = latest_rows['high'].astype('float')
latest_rows['low'] = latest_rows['low'].astype('float')

# realized_power (same def as every prior session)
latest_rows['ret_1bar'] = latest_rows['close'].pct_change()
latest_rows['abs_ret_1bar'] = latest_rows['ret_1bar'].abs()

# atr14 (Wilder ATR, 14-period) - M5 file has no atr14 column, built fresh here
latest_rows['prev_close'] = latest_rows['close'].shift(1)
latest_rows['tr'] = latest_rows[['high', 'low', 'prev_close']].apply(
    lambda row: max(row['high'] - row['low'],
                    abs(row['high'] - row['prev_close']),
                    abs(row['low'] - row['prev_close'])), axis=1)
latest_rows['atr14'] = latest_rows['tr'].ewm(alpha=1/14, adjust=False).mean()

# otf_streak - exact function from Week 9 Day 1, unchanged
from typing import List
def otf_streak(highs: List, lows: List) -> List:
    otf_streaks = [0] * len(highs)
    for i in range(1, len(otf_streaks)):
        if (lows[i] > lows[i-1]) and (highs[i] < highs[i-1]):
            otf_streaks[i] = otf_streaks[i-1]
        elif lows[i] > lows[i-1]:
            otf_streaks[i] = 1 if otf_streaks[i-1] < 0 else otf_streaks[i-1] + 1
        elif highs[i] < highs[i-1]:
            otf_streaks[i] = -1 if otf_streaks[i-1] > 0 else otf_streaks[i-1] - 1
    return otf_streaks
latest_rows['otf_streak'] = otf_streak(latest_rows['high'].tolist(), latest_rows['low'].tolist())

# atr_trend (1 if atr rising over last 5 bars, else 0 - same def as Week 8)
latest_rows['atr_trend'] = (latest_rows['atr14'] - latest_rows['atr14'].shift(5) > 0).astype(int)

# efficiency_ratio_10, sjv_10, center_of_mass, wtc_10 - same window-10 defs as Week 8 Day 3
latest_rows['net_displacement_10'] = latest_rows['ret_1bar'].rolling(10).sum().abs()
latest_rows['realized_power_10'] = latest_rows['abs_ret_1bar'].rolling(10).sum()
latest_rows['efficiency_ratio_10'] = latest_rows['net_displacement_10'] / latest_rows['realized_power_10']

latest_rows['sq_ret_pos_10'] = (latest_rows['ret_1bar'].where(latest_rows['ret_1bar'] > 0, 0) ** 2).rolling(10).sum()
latest_rows['sq_ret_neg_10'] = (latest_rows['ret_1bar'].where(latest_rows['ret_1bar'] < 0, 0) ** 2).rolling(10).sum()
latest_rows['sjv_10'] = (latest_rows['sq_ret_pos_10'] - latest_rows['sq_ret_neg_10']) / (latest_rows['sq_ret_pos_10'] + latest_rows['sq_ret_neg_10'])

def center_of_mass(abs_returns_window):
    n = len(abs_returns_window)
    positions = np.arange(1, n + 1)
    total_weight = np.sum(abs_returns_window)
    if total_weight == 0:
        return 0.0
    raw_com = np.sum(positions * abs_returns_window) / total_weight
    center = (n + 1) / 2
    radius = (n - 1) / 2
    return (raw_com - center) / radius
latest_rows['center_of_mass'] = latest_rows['abs_ret_1bar'].rolling(10).apply(center_of_mass, raw=True)

latest_rows['wtc_10'] = np.log(latest_rows['high'] / latest_rows['low']).rolling(10).sum() / latest_rows['realized_power_10']


##########

prediction_window = latest_rows.loc[
    (latest_rows['hour'] == current_hour) & 
    (latest_rows['trade_date'] == current_date) &
    (latest_rows['minute'].between(0, 10))
    ]

prediction_window = prediction_window.groupby('trade_date').agg(
    realized_power_observation = ('abs_ret_1bar', 'sum'),
    atr14_mean_observation = ('atr14', 'mean'),
    otf_streak_observation = ('otf_streak', 'last'),
    sjv_10__observation = ('sjv_10', 'mean'),
    efficiency_ratio_10_observation = ('efficiency_ratio_10', 'mean'),
    wtc_10_observation = ('wtc_10', 'mean'),
    center_of_mass_mean_observation = ('center_of_mass', 'mean'),
    atr_trend_observation = ('atr_trend', 'last')
).reset_index()

prediction_window['day_of_week'] = current_date.weekday()
prediction_window.head()




In [ ]:
prediction_window = latest_rows.loc[
    (latest_rows['hour'] == current_hour) & 
    (latest_rows['trade_date'] == current_date) &
    (latest_rows['minute'].between(0, 10))
    ]

prediction_window = prediction_window.groupby('trade_date').agg(
    realized_power_observation = ('abs_ret_1bar', 'sum'),
    atr14_mean_observation = ('atr14', 'mean'),
    otf_streak_observation = ('otf_streak', 'last'),
    sjv_10__observation = ('sjv_10', 'mean'),
    efficiency_ratio_10_observation = ('efficiency_ratio_10', 'mean'),
    wtc_10_observation = ('wtc_10', 'mean'),
    center_of_mass_mean_observation = ('center_of_mass', 'mean'),
    atr_trend_observation = ('atr_trend', 'last')
).reset_index()

prediction_window['day_of_week'] = current_date.weekday()
prediction_window.head()

In [ ]:
range_pred = m5_df[m5_df['is_pred']].groupby('trade_date').agg(
    high_pred=('high', 'max'),
    low_pred=('low', 'min')
).reset_index()
range_pred['range_points_predicted'] = range_pred['high_pred'] - range_pred['low_pred']



merged_df = merged_df.merge(range_pred[['trade_date', 'range_points_predicted']], on='trade_date')
merged_df['range_to_power_ratio'] = merged_df['range_points_predicted'] / merged_df['realized_power_predicted']
merged_df['range_to_power_ratio'].describe()
'''
count    1426.000000
mean     1755.873094
std       854.202469
min       541.120060
25%      1165.142538
50%      1539.910752
75%      2079.532406
max      8433.699471
Name: range_to_power_ratio, dtype: float64'''


k_p10 = merged_df['range_to_power_ratio'].quantile(0.10)
k_p25 = merged_df['range_to_power_ratio'].quantile(0.25)
k_p75 = merged_df['range_to_power_ratio'].quantile(0.75)
k_p90 = merged_df['range_to_power_ratio'].quantile(0.90)
print(k_p10, k_p25, k_p75, k_p90)

##947.3450254351877 1165.1425383657856 2079.532405528847 2961.722885905217

#These are the typical bands of range per unif ot power predicted by the model,
#in 10th, 25th, 75th and 90th percentile.
#The most common band will be the 1165-2079 (the IQR here),
#but in more extreme cases we can encounter lower nad higher values




In [ ]:
import live_data
import predict_volatility_multiplier as pvm
data = live_data.get_recent_bars(symbol = 'XAU/USD', interval = '5min')
prediction_features, latest_close = live_data.get_live_observation_features(data)
'''
{'trade_date': datetime.date(2026, 10, 28), 'realized_power_observation': 0.0024814671549869827, 'atr14_premarket_observation': 3.1675227520529723, 'otf_streak_observation': -4, 'sjv_10_premarket_observation': nan, 'efficiency_ratio_10_observation': nan, 'wtc_10_observation': nan, 'center_of_mass_mean_observation': nan, 'atr_trend_observation': 1, 'day_of_week': 2}

'''

model = pvm.load_model()
live_prediction = pvm.predict_volatility(model, prediction_features)
predicted_power = prediction_features['realized_power_observation'] * live_prediction
typical_low, typical_high, wide_low, wide_high = pvm.calibrate_range_bounds(predicted_power, latest_close)

print(f'Predicted volatility power: {live_prediction}')
print(f'Predicted typical low: {typical_low}, typical high: {typical_high}')
print(f'Predicted wide low: {wide_low}, wide high: {wide_high}')


































In [ ]:
#W11 D3 WARMUP TASK

'''```python
results = [
    {'model': 'baseline', 'test_mae': 0.482},
    {'model': 'linear', 'test_mae': 0.371},
    {'model': 'rf', 'test_mae': 0.351},
]
```

- W1. A list comprehension with only the model names: `['baseline', 'linear', 'rf']`.
- W2. A dict comprehension `{model_name: test_mae}` from the same list.
- W3. (one step up) A dict comprehension with only the models that beat the baseline
  (`test_mae < 0.482`), and the value is the improvement `0.482 - test_mae`, rounded to 3
  places.

W3 is the exact shape you'll need in Task 3 (dropping a feature -> MAE change).'''


results = [
    {'model': 'baseline', 'test_mae': 0.482},
    {'model': 'linear', 'test_mae': 0.371},
    {'model': 'rf', 'test_mae': 0.351},
]

model_names = [result['model'] for result in results]
model_w_names = [{'model_name': result['model']} for result in results]
model_w_names_above_baseline = [{'model_name': result['model']} for result in results if result['test_mae'] < 0.482]

print(model_names)
print(model_w_names)
print(model_w_names_above_baseline)

In [ ]:
#W11 D3 T1
'''
**Concept - why a harness.** If every model is scored with its own copied code, a small
difference (another split, MAE computed on train by mistake) makes the comparison
meaningless without you noticing. One function = every model is measured the same way.

**Concept - skill score.** The raw MAE (mean absolute error - how far a prediction is from
reality on average) of 0.35 means nothing on its own. Compare it with the baseline (always
predict the mean of train):

- `test_mae / baseline_mae` = what share of the baseline's error is left over
  (0.35 / 0.48 = 0.73, so 73% of the error is still there)
- `1 - that` = what share of the error the model removed (27%)

That last number is the "skill". 0 = no better than guessing the mean, negative = worse
than guessing the mean.

- 1a. Write `evaluate(model, features, train, test) -> dict`: fit on `train[features]`,
  predict on both train and test, return `train_mae`, `test_mae`, `train_r2`, `test_r2`,
  `skill`.
  The target column name can be fixed inside the function (`target_log_ratio`).
- 1b. Compute `baseline_mae` once, outside the function (the same way as Day 4: the train
  mean predicted for every test row). It should come out at `0.4820`.
- 1c. Test the harness on the production configuration:
  `RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42)` with the 9
  features. Test MAE should be close to the Day 4 result (~0.351). If it's far off, the
  split or the table differs from Day 4.
'''

def evaluate(model, features: List, target: str, train: pd.DataFrame, test: pd.DataFrame) -> dict:
    '''
    Harness function used to quickly evaluate model's MAE/R2
    
    In model you should put the actual Scikit-learn model with parameters
    eg. RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42)
    
    It returns a dict with:
    - train_mae, test_mae
    - train_r2, test_r2,
    - skill score = 1 - (test_mae / baseline_mae); to check what share of the baseline's error model removed
    '''
    
    from sklearn.metrics import mean_absolute_error, r2_score
    
    model = model
    model.fit(train[features], train[target])
    train_preds = model.predict(train[features])
    test_preds = model.predict(test[features])
    
    baseline_mae = np.full(len(test), train[target].mean())
    baseline_mae = mean_absolute_error(baseline_mae, test[target])
    
    train_mae = mean_absolute_error(train[target], train_preds)
    test_mae = mean_absolute_error(test[target], test_preds)
    
    train_r2 = r2_score(train[target], train_preds)
    test_r2 = r2_score(test[target], test_preds)
    
    skill = 1 - (test_mae / baseline_mae)
    
    return {
      'train_mae': train_mae,
      'test_mae': test_mae,
      'train_r2': train_r2,
      'test_r2': test_r2,
      'skill': skill
    }
    
model = RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42)
features = [
    "realized_power_observation",
    "atr14_mean_observation",
    "otf_streak_observation",
    "sjv_10_observation",
    "efficiency_ratio_10_observation",
    "wtc_10_observation",
    "center_of_mass_mean_observation",
    "atr_trend_observation",
    "day_of_week"
]

results = evaluate(model, features = features, target = 'target_log_ratio', train = train, test = test)
print(results)
    
#{'train_mae': 0.24047620307105522, 
# 'test_mae': 0.3491531829154676, 
# 'train_r2': 0.7551578289860965, 
# 'test_r2': 0.4917812593334536, 
# 'skill': 0.293199545232805}
    
    
    

In [ ]:
#W11 D3 T2
'''## Task 2 - Linear regression vs RF on the same features (25 min)

**Concept - why a linear model is the benchmark that matters.** The standard model in
volatility forecasting (HAR-RV, Corsi 2009) is plain linear regression on the LOGARITHM
of realized volatility. If a straight line gets close to RF, RF isn't buying much, and the
simpler model wins (easier to explain, harder to overfit).

**Concept - why the log.** `realized_power_observation` and `atr14_mean_observation` are
strongly right-skewed (lots of small values, a few huge ones). A linear model fits a single
slope across the whole range, so a few huge days would set that slope. After `np.log()`,
the values spread out evenly. A side effect that helps here: the target is already a log,
so `log(power_obs)` vs `log(ratio)` is a natural linear relationship.

```python
import numpy as np
x = np.array([0.0005, 0.001, 0.002, 0.004, 0.03])
print(np.log(x))   # gaps become even: each doubling = +0.69
```

- 2a. Add two columns to `merged_df`: `log_power_obs` and `log_atr14_obs` (the log of the
  two columns above). Split train/test again after adding them.
- 2b. Run 3 linear configurations through `evaluate()` with `LinearRegression()`:
  - L1: `['log_power_obs']` only
  - L2: `['log_power_obs', 'log_atr14_obs']`
  - L3: all 9 features, with the 2 skewed ones swapped for their logs
- 2c. Put everything in one DataFrame (one row = one configuration): baseline, L1, L2, L3,
  RF-9. Columns: `train_mae`, `test_mae`, `test_r2`, `skill`. This is benchmark table #1.

**In a comment:** how much of RF's skill does L1 (one feature, a straight line) already
get? Is the train/test gap bigger for RF or for the linear models, and what does that say
about overfitting?'''

from sklearn.linear_model import LinearRegression

merged_df['log_power_obs'] = np.log(merged_df['realized_power_observation'])
merged_df['log_atr14_obs'] = np.log(merged_df['atr14_mean_observation'])

merged_df.head()

train = merged_df[:int(0.8 * len(merged_df))]
test = merged_df[int(0.8 * len(merged_df)):]

model = LinearRegression()

#My RF scores for comparison:
#{'train_mae': 0.24047620307105522, 
# 'test_mae': 0.3491531829154676, 
# 'train_r2': 0.7551578289860965, 
# 'test_r2': 0.4917812593334536,  26pp gap vs train
# 'skill': 0.293199545232805}
  

#Eval 1 - LinReg with just the log_power_obs
results = evaluate(model, 
                   features = ['log_power_obs'], 
                   target = 'target_log_ratio', 
                   train = train, 
                   test = test)

print(results)
#{
  # 'train_mae': 0.31161545929341183, 
  # 'test_mae': 0.4301211885607545,  #+8pp MAE vs RF
  # 'train_r2': 0.5989559602098272,  
  # 'test_r2': 0.26720741419989325, #-23pp R2 vs RF, with a 26pp gap between train/test
  # 'skill': 0.12929376973959705} #-17pp vs RF



#Eval 2- LinReg with just the log_power_obs + log_atr14_obs
results = evaluate(model, 
                   features = ['log_power_obs', 'log_atr14_obs'], 
                   target = 'target_log_ratio', 
                   train = train, 
                   test = test)

print(results)
# {
  # 'train_mae': 0.28718532712224676, 
  # 'test_mae': 0.3442837419689125,  #even MAE with RF
  # 'train_r2': 0.6609887611321572, 
  # 'test_r2': 0.5354195757539904, #+4pp R2 vs RF and only a 13pp gap between train/test
  # 'skill': 0.30305688935537145} #+1pp vs RF
  
#It seems like Eval2 already works better than my RF with all the features
  
#Eval 3 - LinReg with 9 features + 2 skewed ones replaced with their log versions


features = [
    "log_power_obs",
    "log_atr14_obs",
    "otf_streak_observation",
    "sjv_10_observation",
    "efficiency_ratio_10_observation",
    "wtc_10_observation",
    "center_of_mass_mean_observation",
    "atr_trend_observation",
    "day_of_week"
]



results = evaluate(model, 
                   features = features, 
                   target = 'target_log_ratio', 
                   train = train, 
                   test = test)

print(results)
#{
  # 'train_mae': 0.28217261897802426, 
  # 'test_mae': 0.32600611046393874, #-2pp vs RF + double log LinReg, only 4pp gap from train mae
  # 'train_r2': 0.6719236461802452, 
  # 'test_r2': 0.5768179947071942, #+4pp vs dual log LinReg and +8pp vs my RF and ONLY 10pp gap between test/train
  # 'skill': 0.3400568048420658} #+4pp vs my RF; 34% baseline error removed

#I'm seeing improvement compared to my RF here, which means we could maybe use LinReg instead of that variation
#What's also clear is that it's probably better than a standard HAR and couldm aybe be trimmed EVEN more if complemented with relevant features



In [ ]:
#W11 D3 T3

'''Task 3 - Honest importance: permutation + ablation (25 min)

**Concept - why `feature_importances_` isn't enough.** RF importance shows how often a
feature was used for splits during TRAINING. Two weaknesses:

- the importances sum to 1, so one dominant feature squeezes the others' share toward zero
- two correlated features split the credit between themselves

**Permutation importance** answers a different question: on the TEST set, shuffle one
column (break its link with the target) and see how much MAE gets worse. If it doesn't get
worse, the model didn't use that feature for predictions that generalise.

```python
from sklearn.inspection import permutation_importance

result = permutation_importance(
    rf_model, test[features], test['target_log_ratio'],
    scoring='neg_mean_absolute_error', n_repeats=20, random_state=42
)
# result.importances_mean[i] = average MAE increase after shuffling feature i
# ~0 or negative = the model gets nothing out of this feature
```

**Ablation** is the bluntest test: remove a feature, retrain, compare test MAE.

- 3a. Permutation importance on the fitted RF-9. Sort and put it next to
  `feature_importances_` in one DataFrame.
- 3b. Ablation loop: for each of the 9 features, run `evaluate()` on RF without that one
  feature. Record `test_mae` and the change from RF-9 (a dict comprehension like W3, or a
  list of dicts -> DataFrame).
- 3c. One last run: RF on the top 3 by permutation importance only.

**In a comment - answer your own question from today:** does removing the 6 weak features
IMPROVE the model, do nothing, or make it worse? "Adds nothing" and "removing it helps"
are two different claims - which one do the numbers support?'''

features = [
    "log_power_obs",
    "log_atr14_obs",
    "otf_streak_observation",
    "sjv_10_observation",
    "efficiency_ratio_10_observation",
    "wtc_10_observation",
    "center_of_mass_mean_observation",
    "atr_trend_observation",
    "day_of_week"
]

from sklearn.inspection import permutation_importance
result = permutation_importance(
    model, test[features], test['target_log_ratio'],
    scoring = 'neg_mean_absolute_error', n_repeats = 20, random_state = 42
)


print(pd.DataFrame({'feature': [feature for feature in features],
    'importance_mean': [result for result in result.importances_mean]}))

'''                           feature  importance_mean
0                    log_power_obs         0.347238
1                    log_atr14_obs         0.106658
2           otf_streak_observation         0.005640
3               sjv_10_observation        -0.000317
4  efficiency_ratio_10_observation         0.000762
5               wtc_10_observation         0.000259
6  center_of_mass_mean_observation         0.007562
7            atr_trend_observation         0.000552
8                      day_of_week         0.001161'''

# Now it's evident that log_power_abs IS NOT AS STRONG as it was suggested,
# log_atr14 is still a considerable feature that improves the results
# and the other features seem to be nearly useless



numer = 0
for feature in features:
    numer += 1
    updated_list = features.copy()
    updated_list.remove(feature)
    print(updated_list)
    model = LinearRegression()
    results = evaluate(model, 
            features = updated_list, 
            target = 'target_log_ratio', 
            train = train, 
            test = test)
    print(f'Ablation {numer} - Results without {feature}: {results}')


'''Ablation 1 - Results without log_power_obs: {'train_mae': 0.37880819172817387, 'test_mae': 0.44117100936044223, 'train_r2': 0.38379652599767855, 'test_r2': 0.20378188642295347, 'skill': 0.10692531157146357}
['log_power_obs', 'otf_streak_observation', 'sjv_10_observation', 'efficiency_ratio_10_observation', 'wtc_10_observation', 'center_of_mass_mean_observation', 'atr_trend_observation', 'day_of_week']
Ablation 2 - Results without log_atr14_obs: {'train_mae': 0.30242965634111885, 'test_mae': 0.39651945005822825, 'train_r2': 0.6269640656514084, 'test_r2': 0.3621788042609476, 'skill': 0.19731469928187162}
['log_power_obs', 'log_atr14_obs', 'sjv_10_observation', 'efficiency_ratio_10_observation', 'wtc_10_observation', 'center_of_mass_mean_observation', 'atr_trend_observation', 'day_of_week']
Ablation 3 - Results without otf_streak_observation: {'train_mae': 0.28385404548548965, 'test_mae': 0.3265564942767245, 'train_r2': 0.6685353726722987, 'test_r2': 0.5750076512752423, 'skill': 0.33894264765201765}
['log_power_obs', 'log_atr14_obs', 'otf_streak_observation', 'efficiency_ratio_10_observation', 'wtc_10_observation', 'center_of_mass_mean_observation', 'atr_trend_observation', 'day_of_week']
Ablation 4 - Results without sjv_10_observation: {'train_mae': 0.28272076620335385, 'test_mae': 0.324982325150291, 'train_r2': 0.6715285917239412, 'test_r2': 0.580755670600712, 'skill': 0.34212928179681656}
['log_power_obs', 'log_atr14_obs', 'otf_streak_observation', 'sjv_10_observation', 'wtc_10_observation', 'center_of_mass_mean_observation', 'atr_trend_observation', 'day_of_week']
Ablation 5 - Results without efficiency_ratio_10_observation: {'train_mae': 0.28237959876154256, 'test_mae': 0.3266455549341125, 'train_r2': 0.6700220675215325, 'test_r2': 0.5744398142473015, 'skill': 0.33876235969755}
['log_power_obs', 'log_atr14_obs', 'otf_streak_observation', 'sjv_10_observation', 'efficiency_ratio_10_observation', 'center_of_mass_mean_observation', 'atr_trend_observation', 'day_of_week']
Ablation 6 - Results without wtc_10_observation: {'train_mae': 0.282569476247194, 'test_mae': 0.3268771196120675, 'train_r2': 0.6716934230899385, 'test_r2': 0.5738506107856883, 'skill': 0.33829359690890815}
['log_power_obs', 'log_atr14_obs', 'otf_streak_observation', 'sjv_10_observation', 'efficiency_ratio_10_observation', 'wtc_10_observation', 'atr_trend_observation', 'day_of_week']
Ablation 7 - Results without center_of_mass_mean_observation: {'train_mae': 0.28423157423937695, 'test_mae': 0.3359536948138652, 'train_r2': 0.6680859161855919, 'test_r2': 0.5496066584142693, 'skill': 0.31991963443550153}
['log_power_obs', 'log_atr14_obs', 'otf_streak_observation', 'sjv_10_observation', 'efficiency_ratio_10_observation', 'wtc_10_observation', 'center_of_mass_mean_observation', 'day_of_week']
Ablation 8 - Results without atr_trend_observation: {'train_mae': 0.2822477998547821, 'test_mae': 0.3270166912660005, 'train_r2': 0.6718384660875434, 'test_r2': 0.5750514291373823, 'skill': 0.3380110581456962}
['log_power_obs', 'log_atr14_obs', 'otf_streak_observation', 'sjv_10_observation', 'efficiency_ratio_10_observation', 'wtc_10_observation', 'center_of_mass_mean_observation', 'atr_trend_observation']
Ablation 9 - Results without day_of_week: {'train_mae': 0.28238630606530984, 'test_mae': 0.32909796936304186, 'train_r2': 0.6706262634386289, 'test_r2': 0.5739514493912388, 'skill': 0.3337978692719692}'''

#Ablation also shows that 2 first features make the difference, others are merely noise that don't affacet the results on their own in a positive sense

In [ ]:
#ML Phase W11 D4 warmup
'''
- W1. `{dropped_feature: test_mae}` as one dict.
- W2. `{dropped_feature: change vs full_mae}` (test_mae - full_mae, rounded to 4 places).
- W3. Same as W2, but only features whose removal HURT the model (change > 0).'''

ablation = [
    {'dropped': 'otf_streak', 'test_mae': 0.3266},
    {'dropped': 'sjv_10', 'test_mae': 0.3250},
    {'dropped': 'center_of_mass', 'test_mae': 0.3360},
]
full_mae = 0.3260


dropped_features_test_mae = [{feature['dropped'] : feature['test_mae']} for feature in ablation]
print(dropped_features_test_mae)

feature_change_vs_full_mae = [{feature['dropped'] : feature['test_mae'] - test_mae} for feature in ablation]
print(feature_change_vs_full_mae)
'''[{'otf_streak': -0.025889835708519326}, {'sjv_10': -0.027489835708519317}, {'center_of_mass': -0.016489835708519307}]'''

feature_change_vs_full_mae_filtered = [{feature['dropped'] : feature['test_mae'] - test_mae} for feature in ablation if feature['test_mae'] - test_mae > 0]
print(feature_change_vs_full_mae_filtered) #[]

In [ ]:
#ML Phase W11 D4 T1
'''
**Concept - `**` unpacking in a dict.** `{'config': 'L1', **some_dict}` builds a new dict:
first the `config` key, then every key:value pair copied from `some_dict`. Handy for turning
`evaluate()` results into table rows:

```python
row = {'config': 'L1', **{'test_mae': 0.43, 'skill': 0.13}}
# {'config': 'L1', 'test_mae': 0.43, 'skill': 0.13}
```

- 0a. Benchmark table #1 as ONE DataFrame: a list of dicts (`{'config': ..., **evaluate(...)}`)
  for L1, L2, L3, RF-9 -> `pd.DataFrame(rows)`. Keep this list - every new configuration
  today gets appended to it.
- 0b. Top-3 run: `['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation']` on
  BOTH `LinearRegression()` and RF (`max_depth=6, min_samples_leaf=5, random_state=42`).
  Call this feature set `TOP3` - it's today's starting point.

**In a comment:** is TOP3-linear within ~0.003 MAE of L3? If yes, 3 features do the work of 9.

---

## Important - keeping the comparison fair today

Task 1 adds lag features, so the first ~22 rows get NaN (no history before them). Dropping
those rows changes the train/test split, and then the new numbers can't be compared with
yesterday's.'''


features = [
    "log_power_obs",
    "log_atr14_obs",
    "otf_streak_observation",
    "sjv_10_observation",
    "efficiency_ratio_10_observation",
    "wtc_10_observation",
    "center_of_mass_mean_observation",
    "atr_trend_observation",
    "day_of_week"
]

top3 = ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation']

train = merged_df[:int(0.8 * len(merged_df))]
test = merged_df[int(0.8 * len(merged_df)):]

rows = []
rows.append({'config': 'L1 linear', **evaluate(LinearRegression(), ['log_power_obs'], 'target_log_ratio', train, test)})
rows.append({'config': 'L2 linear', **evaluate(LinearRegression(), ['log_power_obs', 'log_atr14_obs'], 'target_log_ratio', train, test)})
rows.append({'config': 'L3 linear all', **evaluate(LinearRegression(), features, 'target_log_ratio', train, test)})
rows.append({'config': 'L3 linear top3', **evaluate(LinearRegression(), top3, 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 all', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), features, 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 top3', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), top3, 'target_log_ratio', train, test)})


results = pd.DataFrame(rows)
results.head(6)

'''
config	train_mae	test_mae	train_r2	test_r2	skill
0	L1 linear	0.311615	0.430121	0.598956	0.267207	0.129294
1	L2 linear	0.287185	0.344284	0.660989	0.535420	0.303057
2	L3 linear all	0.282173	0.326006	0.671924	0.576818	0.340057
3	L3 linear top3	0.284973	0.331048	0.665349	0.565849	0.329850
4	RF-9 all	0.240653	0.349238	0.754934	0.491510	0.293028
5	RF-9 top3	0.249792	0.342928	0.736790	0.499740	0.305802'''

#It's evident that LinearRegression is simply better - not THAT MUCH, but definitely better
#For that reason I'm opting towards using it + probably tuning as well, if possible

#It's not within 0.003 MAE, it's +0.5pp better XD, not much, but better, 
# and +1~pp r2, with +1pp skill better. It's not a big deal for 6 features, but it's a bit better. 
# I guess we need to look for features that are stronger (if there are any)

In [ ]:

#W11 D4 T2

'''**Concept - shift BEFORE rolling (leakage trap).** `rolling(5).mean()` on row t includes
row t itself, which is today's target window - a leak. `shift(1)` first moves every value
one row down, so row t only sees days t-1 and earlier:

```python
s = pd.Series([1, 2, 3, 4, 5, 6])
s.rolling(3).mean()           # row 5 = mean(4, 5, 6)  <- includes today = LEAK
s.shift(1).rolling(3).mean()  # row 5 = mean(3, 4, 5)  <- only the past = OK
```

- 1a. Sort `merged_df` by `trade_date`. Build three columns from `realized_power_predicted`:
  - `log_har_d` = log of yesterday's value (`shift(1)`)
  - `log_har_w` = log of the mean of the last 5 days (`shift(1)` then `rolling(5).mean()`)
  - `log_har_m` = log of the mean of the last 22 days (`shift(1)` then `rolling(22).mean()`)
- 1b. Sanity check on 3 rows: `log_har_d` on row t must equal `log(realized_power_predicted)`
  on row t-1. Print it and confirm by eye.
- 1c. (After the dropna + re-split described above.) Evaluate `TOP3 + har_d`,
  `TOP3 + har_d + har_w + har_m` on both linear and RF. Append to the benchmark list.

Small caveat to know about: one row = one trading day in the table, so "shift(1)" means
"previous day in the table" - after holidays or dropped days that's 2-3 calendar days back.
That's the standard approach, just worth knowing.'''

merged_df['log_har_d'] = merged_df['realized_power_predicted'].shift(1)
merged_df['log_har_w'] = merged_df['realized_power_predicted'].shift(1).rolling(5).mean()
merged_df['log_har_m'] = merged_df['realized_power_predicted'].shift(1).rolling(22).mean()

merged_df = merged_df.dropna()
merged_df.head()

train = merged_df[:int(0.8 * len(merged_df))]
test = merged_df[int(0.8 * len(merged_df)):]


rows = []

rows.append({'config': 'L3 linear top3', **evaluate(LinearRegression(), top3, 'target_log_ratio', train, test)})
rows.append({'config': 'L3 linear top3 + har_d', **evaluate(LinearRegression(), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'log_har_d'], 'target_log_ratio', train, test)})
rows.append({'config': 'L3 linear top3 + har_d-w-m', **evaluate(LinearRegression(), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'log_har_d', 'log_har_w', 'log_har_m'], 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 top3', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), top3, 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 top3 + har_d', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'log_har_d'], 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 top3 + har_d-w-m', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'log_har_d', 'log_har_w', 'log_har_m'], 'target_log_ratio', train, test)})

'''old pred'''
'''
config	train_mae	test_mae	train_r2	test_r2	skill
0	L3 linear top3	0.278710	0.315511	0.676170	0.606498	0.366424
1	L3 linear top3 + har_d	0.278508	0.315500	0.676425	0.606748	0.366446
2	L3 linear top3 + har_d-w-m	0.275900	0.316697	0.687834	0.607183	0.364044
3	RF-9 top3	0.240889	0.342112	0.752742	0.515972	0.313007
4	RF-9 top3 + har_d	0.238543	0.346566	0.758571	0.505740	0.304063
5	RF-9 top3 + har_d-w-m	0.233787	0.341580	0.770650	0.517318	0.314076'''

#har-dwm doesn't give much, I don't think it's useful


results = pd.DataFrame(rows)
results.head(6)

#updated har-d-w-m calcs
'''config	train_mae	test_mae	train_r2	test_r2	skill
0	L3 linear top3	0.276987	0.318100	0.680640	0.599456	0.361299
1	L3 linear top3 + har_d	0.275018	0.313203	0.686026	0.613923	0.371131
2	L3 linear top3 + har_d-w-m	0.274323	0.319688	0.688436	0.600041	0.358110
3	RF-9 top3	0.239875	0.345161	0.755230	0.507255	0.306964
4	RF-9 top3 + har_d	0.234452	0.339350	0.767084	0.524039	0.318632
5	RF-9 top3 + har_d-w-m	0.229950	0.335036	0.776319	0.542127	0.327293'''

#Now it seems like HAR-D is also a little, but still improvement, which is probably worht implementing

In [ ]:
'''## Task 2 - Range-based volatility: Parkinson estimator (15 min)

**Concept.** `realized_power` uses only closes (close-to-close returns). A bar can swing
wildly inside and still close near where it opened - close-to-close misses that. The
Parkinson estimator (1980) uses each bar's high and low instead, which captures the
intrabar swing. In theory it's ~5x more efficient (less noise for the same number of bars) -
useful with only 3 bars in our observation window.

Per bar: `(ln(high / low))^2 / (4 * ln(2))`. For the window: sum over the 3 bars.

- 2a. On `m5_df`, rows where `is_obs` is True: compute the per-bar Parkinson value, then
  sum per `trade_date` -> `parkinson_obs`. Merge into `merged_df` on `trade_date`, then take
  the log -> `log_parkinson_obs`.
- 2b. Check correlation with `log_power_obs` (`.corr()`). If it's ~0.95+, it's mostly the
  same information measured differently.
- 2c. Evaluate `TOP3 + log_parkinson_obs` and `log_parkinson_obs` swapped in FOR
  `log_power_obs`, on both models.

**In a comment:** does Parkinson add something on top of power, or just replace it?'''

#Moved up to where I build features 
m5_df['parkinson_volatility'] = (np.log(m5_df['high'] / m5_df['low']) ** 2) / (4 * np.log(2))
m5_df.head()

merged_df['log_parkinson_obs'] = np.log(merged_df['parkinson_obs'])
merged_df.head()

merged_df['log_parkinson_obs'].corr(merged_df['log_power_obs']) #np.float64(0.8074390728679085)

train = merged_df[:int(0.8 * len(merged_df))]
test = merged_df[int(0.8 * len(merged_df)):]

rows = []
rows.append({'config': 'L3 linear top3', **evaluate(LinearRegression(), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation'], 'target_log_ratio', train, test)})
rows.append({'config': 'L3 linear top3 + parkinson_obs', **evaluate(LinearRegression(), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'log_parkinson_obs'], 'target_log_ratio', train, test)})
rows.append({'config': 'L3 top3 log_power_obs replaced with parkinson_obs', **evaluate(LinearRegression(), ['log_parkinson_obs', 'log_atr14_obs', 'center_of_mass_mean_observation'], 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 top3', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), top3, 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 top3 + parkinson_obs', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'log_parkinson_obs'], 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 top3 log_power_obs replaced with parkinson obs', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), ['log_parkinson_obs', 'log_atr14_obs', 'center_of_mass_mean_observation'], 'target_log_ratio', train, test)})


results = pd.DataFrame(rows)
results.head(6)

'''config	train_mae	test_mae	train_r2	test_r2	skill
0	L3 linear top3	0.280567	0.323016	0.670432	0.586541	0.350852
1	L3 linear top3 + parkinson_obs	0.276807	0.307996	0.677563	0.630373	0.381037
2	L3 top3 log_power_obs replaced with parkinson_obs	0.370957	0.393989	0.402302	0.366417	0.208221
3	RF-9 top3	0.245369	0.347285	0.741969	0.498770	0.302081
4	RF-9 top3 + parkinson_obs	0.240626	0.345574	0.751142	0.511239	0.305520
5	RF-9 top3 log_power_obs replaced with parkinso...	0.319197	0.418179	0.549565	0.296077	0.159609
'''

#Interesting an it's plausible that we have another good feature.
#parkinson_obs is worse than standard power_obs,
# and yet when combined, it increases the skill by 3pp, r2 by 4pp and lowers mae by 2pp

#Good call I think

In [ ]:
'''Task 3 - Macro calendar: is there a US data release at 8:30 ET? (15 min)

**Concept.** NFP (jobs report) and CPI (inflation) come out at 8:30 ET - inside our
prediction window (8:15-9:00). On those days gold often moves several times more than usual.
Price data alone can't know a release is coming; the calendar can, and it's known in advance,
so it's not a leak.

`macro_releases.csv` (repo root, official BLS release dates, Jan 2021 - Jul 2026) has columns
`date`, `event` (`NFP` / `CPI`).

**Concept - why `day_of_week` is wrong for a linear model.** It's stored as 0-4, so a linear
model fits ONE slope across the week ("each day later adds X"). That's meaningless - Friday
isn't "4x Monday". A 0/1 flag per meaningful day fixes this. US weekly jobless claims are
also released every Thursday at 8:30 ET, so `is_thursday` is a cheap, meaningful flag.

- 3a. Build `is_nfp`, `is_cpi` (0/1) by matching `trade_date` against the CSV. Make both
  sides the same type first (`pd.to_datetime` on both). Also `is_thursday`.
- 3b. Before any model - look at the data: `merged_df.groupby('is_nfp')['target_log_ratio'].mean()`
  and the same for `is_cpi`. How much higher is the average log ratio on release days?
  Convert the difference to a multiplier with `np.exp()`.
- 3c. Evaluate `TOP3 + is_nfp + is_cpi + is_thursday` on both models.

---

**In a comment - the day's summary:** which feature family improved test MAE vs today's TOP3
reference, by how much, and on which model? Order them by improvement. That ranking is
tomorrow's input for the final feature set.
'''
import pandas as pd

news_releases = pd.read_csv('./macro_releases.csv')
news_releases['date'] = pd.to_datetime(news_releases['date'])

nfp_dates = news_releases.loc[news_releases['event'] == 'NFP', 'date']
cpi_dates = news_releases.loc[news_releases['event'] == 'CPI', 'date']
fomc_dates = news_releases.loc[news_releases['event'] == 'FOMC', 'date']

merged_df['trade_date'] = pd.to_datetime(merged_df['trade_date'])
merged_df['is_nfp'] = merged_df['trade_date'].isin(nfp_dates).astype(int)
merged_df['is_cpi'] = merged_df['trade_date'].isin(cpi_dates).astype(int)
merged_df['is_thursday'] = merged_df.apply(lambda x: 1 if x['day_of_week'] == 4 else 0, axis = 1)

x = merged_df.groupby('is_nfp')['target_log_ratio'].mean()
y = merged_df.groupby('is_cpi')['target_log_ratio'].mean()
#No visible differences


train = merged_df[:int(0.8 * len(merged_df))]
test = merged_df[int(0.8 * len(merged_df)):]

rows = []
rows.append({'config': 'L3 linear top3', **evaluate(LinearRegression(), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation'], 'target_log_ratio', train, test)})
rows.append({'config': 'L3 linear + cpi/nfp/thursday', **evaluate(LinearRegression(), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'is_cpi', 'is_nfp', 'is_thursday'], 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 top3', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation'], 'target_log_ratio', train, test)})
rows.append({'config': 'RF-9 top3 + cpi/nfp/thursday', **evaluate(RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42), ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'is_cpi', 'is_nfp', 'is_thursday'], 'target_log_ratio', train, test)})



results = pd.DataFrame(rows)
results.head(6)


In [ ]:
#W11 D5 WARMUO

## Warm-up (5 min) - dict comprehension

'''```python
features = ['log_power_obs', 'log_atr14_obs', 'log_parkinson_obs']
corrs = [0.71, 0.55, 0.62]
```

- W1 (no-brainer). List comprehension: `features` in upper case.
- W2. ONE dict `{feature: corr}` - use `zip(features, corrs)` to walk both lists together.
  Reminder: `{ key: value for ... }` = one dict. `[ {...} for ... ]` = a list of dicts - not this.
- W3. Same dict, only features with corr > 0.6.as_integer_ratio'''

features = ['log_power_obs', 'log_atr14_obs', 'log_parkinson_obs']
corrs = [0.71, 0.55, 0.62]

uppercase_features = [feature.upper() for feature in features]
print(uppercase_features) #['LOG_POWER_OBS', 'LOG_ATR14_OBS', 'LOG_PARKINSON_OBS']

feature_corrs = {feature : corr for feature, corr in zip(features, corrs)}
print(feature_corrs) #{'log_power_obs': 0.71, 'log_atr14_obs': 0.55, 'log_parkinson_obs': 0.62}


feature_corrs_threshold = {feature : corr for feature, corr in zip(features, corrs) if corr > 0.6}
print(feature_corrs_threshold) #{'log_power_obs': 0.71, 'log_parkinson_obs': 0.62}


In [ ]:
#W11 D5 T1
'''## Task 1 - Fix the clock (20 min)

**Concept - timezones in pandas, in 3 steps.** A timestamp without a timezone is just
numbers. `tz_localize` puts a label on it ("these numbers are UTC") without changing them.
`tz_convert` changes the numbers to another zone and handles daylight saving for you.
`tz_localize(None)` removes the label again, so it compares with plain timestamps.

```python
t = pd.Series(pd.to_datetime(['2024-01-05 07:30', '2024-07-05 06:30']))  # two NFP days, broken clock

utc = t + pd.Timedelta(hours=6)              # 13:30, 12:30   <- broken clock is UTC-6, so +6h = real UTC
ny = utc.dt.tz_localize('UTC')               # 13:30+00:00    <- label only, numbers unchanged
ny = ny.dt.tz_convert('America/New_York')    # 08:30-05:00 (winter), 08:30-04:00 (summer)
ny = ny.dt.tz_localize(None)                 # 08:30, 08:30   <- both releases now at 8:30
```

Note how winter and summer needed different shifts (-5h vs -4h) - `tz_convert` handles that,
a fixed `- Timedelta` would not.

- 1a/1b. DONE - `xauusd_m5_et.csv` was rewritten with exactly these steps (trade_date and
  day_of_week recomputed too, same 1-5 format). Old file kept as `xauusd_m5_utc-6_backup.csv`.
  Nothing to change in the loading cell. The Concept above stays as reference - use it for
  the same check on other instruments (DAX, NQ) if they went through the same conversion.
- 1c. Proof it worked: on NFP/CPI days, which bar has the biggest move? Filter `m5_df` to
  release dates (`trade_date.isin(...)`, as yesterday) and hours 6-10, then per `trade_date`
  find the row with max `abs_ret_1bar` (`groupby('trade_date')['abs_ret_1bar'].idxmax()`
  gives the row labels, `.loc[...]` picks them). Count the bar times with `.value_counts()`.
  08:30 should win by far.
- 1d. Re-run everything after it, in order: the `m5_df` feature cell, the `is_obs`/`is_pred`
  + aggregation cell, then build again `log_power_obs`, `log_atr14_obs`, `log_parkinson_obs`,
  `log_har_d` (this time WITH `np.log()`), `is_nfp`, `is_cpi`.
  Check: every day should have 3 observation bars and 9 prediction bars.

Rolling features (ATR, otf_streak, sjv...) don't change - the rows keep the same order, only
the labels move. What changes is which 3 + 9 bars land in each '''


print(merged_df.groupby('is_cpi')['target_log_ratio'].mean())
#is_cpi
# 0    1.485807
# 1    2.183549
print(merged_df.groupby('is_nfp')['target_log_ratio'].mean())
# is_nfp
# 0    1.488250
# 1    2.164436
# Name: target_log_ratio, dtype: float64
print(merged_df.groupby('is_thursday')['target_log_ratio'].mean())
# is_thursday
# 0    1.489380
# 1    1.635001
# Name: target_log_ratio, dtype: float64

m5_df['trade_date'] = pd.to_datetime(m5_df['trade_date'])
m5_df['is_nfp'] = m5_df['trade_date'].isin(nfp_dates).astype(int)
m5_df['is_cpi'] = m5_df['trade_date'].isin(cpi_dates).astype(int)
m5_df['day_of_week'] = m5_df['trade_date'].dt.day_of_week
m5_df['is_thursday'] = m5_df.apply(lambda x: 1 if x['day_of_week'] == 3 else 0, axis = 1)


cpi_days = m5_df.loc[(m5_df['is_cpi'] == 1) & m5_df['hour'].isin([6, 7, 8, 9, 10])]
cpi_days_max_vals = cpi_days.groupby('trade_date')['abs_ret_1bar'].idxmax().reset_index()
cpi_days_max_candles = m5_df.loc[cpi_days_max_vals['abs_ret_1bar']]
cpi_days_max_candles['hh_mm'] = cpi_days_max_candles['et_time'].dt.strftime('%H:%M')
cpi_days_max_candles['hh_mm'].value_counts()

'''hh_mm
08:30    37
09:35     4
08:35     4
10:15     3
10:30     2
08:50     2
09:30     2
10:00     2
09:40     2
09:25     1
09:00     1
08:55     1
08:10     1
08:20     1
09:15     1
09:45     1
09:55     1
Name: count, dtype: int64'''
#Well, that makes sense XD

#And guess what, not CPI/NFP/Thursday seem to have a positive effect on the results as well..



In [1]:
#NEWSTART 6.11

#all the previous 9 features + log_har_X + parkinson_volatility + cpi/nfp/thursday


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

m5_df = pd.read_csv('xauusd_m5_et.csv', parse_dates=['et_time'])
pd.options.display.max_columns = 50

# realized_power (same def as every prior session)
m5_df['ret_1bar'] = m5_df['close'].pct_change()
m5_df['abs_ret_1bar'] = m5_df['ret_1bar'].abs()

# atr14 (Wilder ATR, 14-period) - M5 file has no atr14 column, built fresh here
m5_df['prev_close'] = m5_df['close'].shift(1)
m5_df['tr'] = m5_df[['high', 'low', 'prev_close']].apply(
    lambda row: max(row['high'] - row['low'],
                    abs(row['high'] - row['prev_close']),
                    abs(row['low'] - row['prev_close'])), axis=1)
m5_df['atr14'] = m5_df['tr'].ewm(alpha=1/14, adjust=False).mean()

# otf_streak - exact function from Week 9 Day 1, unchanged
from typing import List
def otf_streak(highs: List, lows: List) -> List:
    otf_streaks = [0] * len(highs)
    for i in range(1, len(otf_streaks)):
        if (lows[i] > lows[i-1]) and (highs[i] < highs[i-1]):
            otf_streaks[i] = otf_streaks[i-1]
        elif lows[i] > lows[i-1]:
            otf_streaks[i] = 1 if otf_streaks[i-1] < 0 else otf_streaks[i-1] + 1
        elif highs[i] < highs[i-1]:
            otf_streaks[i] = -1 if otf_streaks[i-1] > 0 else otf_streaks[i-1] - 1
    return otf_streaks
m5_df['otf_streak'] = otf_streak(m5_df['high'].tolist(), m5_df['low'].tolist())

# atr_trend (1 if atr rising over last 5 bars, else 0 - same def as Week 8)
m5_df['atr_trend'] = (m5_df['atr14'] - m5_df['atr14'].shift(5) > 0).astype(int)

# efficiency_ratio_10, sjv_10, center_of_mass, wtc_10 - same window-10 defs as Week 8 Day 3
m5_df['net_displacement_10'] = m5_df['ret_1bar'].rolling(10).sum().abs()
m5_df['realized_power_10'] = m5_df['abs_ret_1bar'].rolling(10).sum()
m5_df['efficiency_ratio_10'] = m5_df['net_displacement_10'] / m5_df['realized_power_10']

m5_df['sq_ret_pos_10'] = (m5_df['ret_1bar'].where(m5_df['ret_1bar'] > 0, 0) ** 2).rolling(10).sum()
m5_df['sq_ret_neg_10'] = (m5_df['ret_1bar'].where(m5_df['ret_1bar'] < 0, 0) ** 2).rolling(10).sum()
m5_df['sjv_10'] = (m5_df['sq_ret_pos_10'] - m5_df['sq_ret_neg_10']) / (m5_df['sq_ret_pos_10'] + m5_df['sq_ret_neg_10'])

def center_of_mass(abs_returns_window):
    n = len(abs_returns_window)
    positions = np.arange(1, n + 1)
    total_weight = np.sum(abs_returns_window)
    if total_weight == 0:
        return 0.0
    raw_com = np.sum(positions * abs_returns_window) / total_weight
    center = (n + 1) / 2
    radius = (n - 1) / 2
    return (raw_com - center) / radius
m5_df['center_of_mass'] = m5_df['abs_ret_1bar'].rolling(10).apply(center_of_mass, raw=True)

m5_df['wtc_10'] = np.log(m5_df['high'] / m5_df['low']).rolling(10).sum() / m5_df['realized_power_10']


####added 5.11.2026 - Parkinson volatility which also captures the intrabar swing, not only close-close

m5_df['parkinson_volatility'] = (np.log(m5_df['high'] / m5_df['low']) ** 2) / (4 * np.log(2))

#+ hh/min added
m5_df['hour'] = pd.to_datetime(m5_df['et_time']).dt.hour
m5_df['minute'] = pd.to_datetime(m5_df['et_time']).dt.minute

###9.11.2026 - Garman Klass added, it's similar to Parkinson, yet it works on open-close,
#looking at intrabar swing and where it ended
m5_df['high_low_swing_squared'] = np.log(m5_df['high'] / m5_df['low']) ** 2
m5_df['open_to_close_squared'] = np.log(m5_df['close'] / m5_df['open']) ** 2
m5_df['garman_klass'] = 0.5 * m5_df['high_low_swing_squared'] - (2 * np.log(2) - 1) * m5_df['open_to_close_squared']


m5_df['is_pre'] = (((m5_df['et_time'].dt.hour == 7) & (m5_df['et_time'].dt.minute >= 15)) | ((m5_df['et_time'].dt.hour == 8) & (m5_df['et_time'].dt.minute < 15)))
m5_df['is_obs'] = (m5_df['et_time'].dt.hour == 8) & (m5_df['et_time'].dt.minute < 15)
m5_df['is_pred'] = (m5_df['et_time'].dt.hour == 8) & (m5_df['et_time'].dt.minute >= 15)


print(m5_df.columns)

m5_df['rs_neg_bar'] = m5_df['ret_1bar'].where(m5_df['ret_1bar'] < 0, 0) ** 2
m5_df['rs_pos_bar'] = m5_df['ret_1bar'].where(m5_df['ret_1bar'] > 0, 0) ** 2

pre_window = m5_df[m5_df['is_pre']].groupby('trade_date').agg(
    rs_neg = ('rs_neg_bar', 'sum'),
    rs_pos = ('rs_pos_bar', 'sum')
).reset_index()



#range calculations so that later I can calculate range to power ratio, so that I will be able to get the actual prices range

range_pred = m5_df[m5_df['is_pred']].groupby('trade_date').agg(
    high_pred = ('high', 'max'),
    low_pred = ('low', 'min'),
).reset_index()

#predicted price window for each predicted window
range_pred['range_points_predicted'] = range_pred['high_pred'] - range_pred['low_pred']



observation_window = m5_df.groupby(['is_obs', 'trade_date']).agg(
    realized_power_observation = ('abs_ret_1bar', 'sum'),
    atr14_mean_observation = ('atr14', 'mean'),
    otf_streak_observation = ('otf_streak', 'last'),
    sjv_10_observation = ('sjv_10', 'mean'),
    efficiency_ratio_10_observation = ('efficiency_ratio_10', 'mean'),
    wtc_10_observation = ('wtc_10', 'mean'),
    center_of_mass_mean_observation = ('center_of_mass', 'mean'),
    atr_trend_observation = ('atr_trend', 'last'),
    parkinson_obs = ('parkinson_volatility', 'sum'),
    gk_obs = ('garman_klass', 'sum')
).reset_index()

predicted_window = m5_df.groupby(['is_pred', 'trade_date']).agg(
    realized_power_predicted = ('abs_ret_1bar', 'sum'),
    atr14_mean_predicted = ('atr14', 'mean'),
    otf_streak_predicted = ('otf_streak', 'last'),
    sjv_10_predicted = ('sjv_10', 'mean'),
    efficiency_ratio_10_predicted = ('efficiency_ratio_10', 'mean'),
    wtc_10_predicted = ('wtc_10', 'mean'),
    center_of_mass_mean_predicted = ('center_of_mass', 'mean'),
    atr_trend_predicted = ('atr_trend', 'last'),
    parkinson_predicted = ('parkinson_volatility', 'sum'),
    gk_predicted = ('garman_klass', 'sum')
).reset_index()



observation_window = observation_window[observation_window['is_obs'] == True]
predicted_window = predicted_window[predicted_window['is_pred'] == True]

merged_df = observation_window.merge(predicted_window, how = 'inner', on = 'trade_date')
merged_df = merged_df.merge(pre_window, how = 'inner', on = 'trade_date')
merged_df = merged_df.merge(range_pred, on='trade_date')

###Range to power per day
merged_df['range_to_power_ratio'] = merged_df['range_points_predicted'] / merged_df['realized_power_predicted']

#top 75 and 90 percentiles for the predicitons
k_p75 = merged_df['range_to_power_ratio'].quantile(0.75)
k_p90 = merged_df['range_to_power_ratio'].quantile(0.90)
print(k_p75, k_p90)


#converted parkinson and observed RP to log as they seem to work better
merged_df['log_parkinson_obs'] = np.log(merged_df['parkinson_obs'])
merged_df['log_atr14_obs'] = np.log(merged_df['atr14_mean_observation'])
merged_df['log_power_obs'] = np.log(merged_df['realized_power_observation'])
#9.11.26+also garman-klass + Semivariance splits (squared returns of negative bars only + positive bars only)
merged_df['log_garman_klass_obs'] = np.log(merged_df['gk_obs'])

cols = ['rs_neg', 'rs_pos']
min_pos = merged_df[cols][merged_df[cols] > 0].min()
merged_df[['log_rs_neg', 'log_rs_pos']] = np.log(merged_df[cols].clip(lower=min_pos, axis=1))

print(f'Garman Klass - Parkinson correlation: {merged_df['log_garman_klass_obs'].corr(merged_df['log_parkinson_obs'])}')

#added log_har_d + the rest, as it clearly shows a signal - not super strong, but it's there :))
#also this time it's in the ln format
merged_df['log_har_d'] = np.log(merged_df['realized_power_predicted'].shift(1))
merged_df['log_har_w'] = np.log(merged_df['realized_power_predicted'].shift(1).rolling(5).mean())
merged_df['log_har_m'] = np.log(merged_df['realized_power_predicted'].shift(1).rolling(22).mean())

###added the news releases segment, as it clearly shows a signal as well
news_releases = pd.read_csv('./macro_releases.csv')
news_releases['date'] = pd.to_datetime(news_releases['date'])

nfp_dates = news_releases.loc[news_releases['event'] == 'NFP', 'date']
cpi_dates = news_releases.loc[news_releases['event'] == 'CPI', 'date']
fomc_dates = news_releases.loc[news_releases['event'] == 'FOMC', 'date']

merged_df['trade_date'] = pd.to_datetime(merged_df['trade_date'])
merged_df['day_of_week'] = merged_df['trade_date'].dt.day_of_week
merged_df['is_nfp'] = merged_df['trade_date'].isin(nfp_dates).astype(int)
merged_df['is_cpi'] = merged_df['trade_date'].isin(cpi_dates).astype(int)
merged_df['is_thursday'] = merged_df.apply(lambda x: 1 if x['day_of_week'] == 3 else 0, axis = 1)

merged_df['target_log_ratio'] = np.log(merged_df['realized_power_predicted'] / merged_df['realized_power_observation'])

merged_df = merged_df.dropna()
merged_df.head()






Index(['et_time', 'open', 'high', 'low', 'close', 'volume', 'day_of_week',
       'trade_date', 'ret_1bar', 'abs_ret_1bar', 'prev_close', 'tr', 'atr14',
       'otf_streak', 'atr_trend', 'net_displacement_10', 'realized_power_10',
       'efficiency_ratio_10', 'sq_ret_pos_10', 'sq_ret_neg_10', 'sjv_10',
       'center_of_mass', 'wtc_10', 'parkinson_volatility', 'hour', 'minute',
       'high_low_swing_squared', 'open_to_close_squared', 'garman_klass',
       'is_pre', 'is_obs', 'is_pred'],
      dtype='str')
2138.6352561046688 3063.5539822916144
Garman Klass - Parkinson correlation: 0.9772196295766407


,is_obs,trade_date,realized_power_observation,atr14_mean_observation,otf_streak_observation,sjv_10_observation,efficiency_ratio_10_observation,wtc_10_observation,center_of_mass_mean_observation,atr_trend_observation,parkinson_obs,gk_obs,is_pred,realized_power_predicted,atr14_mean_predicted,otf_streak_predicted,sjv_10_predicted,efficiency_ratio_10_predicted,wtc_10_predicted,center_of_mass_mean_predicted,atr_trend_predicted,parkinson_predicted,gk_predicted,rs_neg,rs_pos,high_pred,low_pred,range_points_predicted,range_to_power_ratio,log_parkinson_obs,log_atr14_obs,log_power_obs,log_garman_klass_obs,log_rs_neg,log_rs_pos,log_har_d,log_har_w,log_har_m,day_of_week,is_nfp,is_cpi,is_thursday,target_log_ratio
22,True,2021-02-03,0.001732,1.310572,0,0.417228,0.347997,2.382314,-0.020366,0,9.526996e-07,7.222580e-07,True,0.005170,1.713521,2,0.090335,0.077608,1.859346,0.079947,1,0.000006,0.000006,6.441080e-07,2.186274e-06,1842.202,1835.707,6.495,1256.318796,-13.863966,0.270464,-6.358191,-14.140883,-14.255399,-13.033312,-4.447797,-4.733799,-4.905177,2,0,0,0,1.093283
23,True,2021-02-04,0.000983,1.561829,2,0.590807,0.418417,1.666993,-0.076072,1,1.211076e-06,1.503110e-06,True,0.008369,1.996975,-6,-0.571002,0.411255,1.866703,0.113726,1,0.000011,0.000010,8.271148e-07,2.938717e-06,1819.485,1806.276,13.209,1578.322086,-13.624002,0.445858,-6.925089,-13.407974,-14.005322,-12.737537,-5.264908,-4.744114,-4.909962,3,0,0,1,2.141870
24,True,2021-02-05,0.002563,1.586192,1,-0.750834,0.445070,1.942095,0.121462,1,2.708463e-06,2.221969e-06,True,0.012397,2.280278,-2,0.035804,0.301401,1.816297,0.217741,1,0.000018,0.000014,4.921180e-06,1.224976e-06,1806.553,1793.325,13.228,1067.029181,-12.819129,0.461336,-5.966539,-13.017117,-12.221962,-13.612589,-4.783219,-4.795576,-4.903710,4,1,0,0,1.576241
25,True,2021-02-08,0.001243,1.700426,1,0.576717,0.246142,1.660124,0.141101,1,2.318687e-06,2.866543e-06,True,0.006365,1.900671,0,0.565262,0.403828,1.810919,-0.014986,0,0.000005,0.000004,1.582456e-06,5.218679e-06,1831.356,1822.345,9.011,1415.782944,-12.974509,0.530879,-6.689830,-12.762404,-13.356532,-12.163266,-4.390298,-4.715317,-4.875995,0,0,0,0,1.632838
26,True,2021-02-09,0.001807,1.283142,-1,-0.359346,0.299653,2.526041,0.191250,1,1.875187e-06,2.055392e-06,True,0.006606,1.757479,-1,0.074869,0.202111,2.182769,0.185794,1,0.000007,0.000007,1.408802e-06,7.603453e-07,1846.972,1838.938,8.034,1216.218582,-13.186802,0.249312,-6.315869,-13.095044,-13.472771,-14.089493,-5.056992,-4.732892,-4.869650,1,0,0,0,1.296050


In [2]:
#W11 D5 T3

'''##Benchmark on correct data (20 min)

- 2a. Before any model: `groupby('is_nfp')['target_log_ratio'].mean()` and the same for
  `is_cpi`. Take the difference between the two means, then `np.exp()` of it = "on release
  days the prediction window is X times more volatile relative to the observation window".
  Compare with yesterday's "no difference".
- 2b. Fix `is_thursday`: `.dt.day_of_week` counts Monday = 0, so Thursday = 3.
- 2c. `.dropna()`, split 80/20, then ONE table, both models (linear + RF
  `max_depth=6, min_samples_leaf=5, random_state=42`), configurations:
  - `TOP3` = `['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation']`
  - `TOP3 + log_parkinson_obs`
  - `TOP3 + log_parkinson_obs + log_har_d`
  - `TOP3 + log_parkinson_obs + log_har_d + is_nfp + is_cpi + is_thursday` (call it `FULL`)

**In a comment:** which families still help on the correct clock? Any that changed verdict
vs yesterday?'''


from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor



print(merged_df.groupby('is_nfp')['target_log_ratio'].mean())
print(merged_df.groupby('is_cpi')['target_log_ratio'].mean())
print(merged_df.groupby('is_thursday')['target_log_ratio'].mean())


'''is_nfp
0    1.488250
1    2.164436
Name: target_log_ratio, dtype: float64
is_cpi
0    1.485807
1    2.183549
Name: target_log_ratio, dtype: float64
is_thursday
0    1.489380
1     1.637220
Name: target_log_ratio, dtype: float64
'''

merged_df = merged_df.dropna()
train = merged_df[:int(0.8 * len(merged_df))]
test = merged_df[int(0.8 * len(merged_df)):]

lin = LinearRegression()
rf = RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42)

top3 = ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation']


top3_parkinson = ['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'log_parkinson_obs']
top3_parkinson_log_har_d = ['log_power_obs', 
                           'log_atr14_obs', 
                           'center_of_mass_mean_observation', 
                           'log_parkinson_obs', 
                           'log_har_d'
                           ]
top3_parkinson_log_hars = ['log_power_obs', 
                           'log_atr14_obs', 
                           'center_of_mass_mean_observation', 
                           'log_parkinson_obs', 
                           'log_har_d', 
                           'log_har_w', 
                           'log_har_m']

top3_parkinson_log_har_d_news_thursdays = ['log_power_obs', 
                                          'log_atr14_obs', 
                                          'center_of_mass_mean_observation', 
                                          'log_parkinson_obs', 
                                          'log_har_d', 
                                          'is_cpi',
                                          'is_nfp',
                                          'is_thursday'
                                          ]



top3_parkinson_log_hars_news_thursdays = ['log_power_obs', 
                                          'log_atr14_obs', 
                                          'center_of_mass_mean_observation', 
                                          'log_parkinson_obs', 
                                          'log_har_d', 
                                          'log_har_w', 
                                          'log_har_m',
                                          'is_cpi',
                                          'is_nfp',
                                          'is_thursday'
                                          ]


top3_parkinson_log_har_d_news_thursdays_log_gk_obs = ['log_power_obs', 
                                          'log_atr14_obs', 
                                          'center_of_mass_mean_observation', 
                                          'log_parkinson_obs', 
                                          'log_har_d',
                                          'log_garman_klass_obs',
                                          'is_cpi',
                                          'is_nfp',
                                          'is_thursday'
                                          ]

top3_parkinson_log_har_d_news_thursdays_log_rs_neg_log_rs_pos = ['log_power_obs', 
                                          'log_atr14_obs', 
                                          'center_of_mass_mean_observation', 
                                          'log_parkinson_obs', 
                                          'log_har_d',
                                          'log_rs_neg',
                                          'log_rs_pos',
                                          'is_cpi',
                                          'is_nfp',
                                          'is_thursday'
                                          ]

top3_parkinson_combined = ['log_power_obs', 
                                          'log_atr14_obs', 
                                          'center_of_mass_mean_observation', 
                                          'log_parkinson_obs', 
                                          'log_har_d',
                                          'log_garman_klass_obs',
                                          'log_rs_neg',
                                          'log_rs_pos',
                                          'is_cpi',
                                          'is_nfp',
                                          'is_thursday'
                                          ]





#evaluate() harness copied from one of the cells above, it will be much smoother to start every run from the cell above
def evaluate(model, features: List, target: str, train: pd.DataFrame, test: pd.DataFrame) -> dict:
    '''
    Harness function used to quickly evaluate model's MAE/R2
    
    In model you should put the actual Scikit-learn model with parameters
    eg. RandomForestRegressor(max_depth=6, min_samples_leaf=5, random_state=42)
    
    It returns a dict with:
    - train_mae, test_mae
    - train_r2, test_r2,
    - skill score = 1 - (test_mae / baseline_mae); to check what share of the baseline's error model removed
    '''
    
    from sklearn.metrics import mean_absolute_error, r2_score
    
    model = model
    model.fit(train[features], train[target])
    train_preds = model.predict(train[features])
    test_preds = model.predict(test[features])
    
    baseline_mae = np.full(len(test), train[target].mean())
    baseline_mae = mean_absolute_error(baseline_mae, test[target])
    
    train_mae = mean_absolute_error(train[target], train_preds)
    test_mae = mean_absolute_error(test[target], test_preds)
    
    train_r2 = r2_score(train[target], train_preds)
    test_r2 = r2_score(test[target], test_preds)
    
    skill = 1 - (test_mae / baseline_mae)
    
    return {
      'train_mae': train_mae,
      'test_mae': test_mae,
      'train_r2': train_r2,
      'test_r2': test_r2,
      'skill': skill
    }



rows = []
rows.append({'config': 'LinReg top3_parkinson_log_har_d_news_thursdays', **evaluate(lin, top3_parkinson_log_har_d_news_thursdays, 'target_log_ratio', train, test)})
rows.append({'config': 'LinReg top3_parkinson_log_har_d_news_thursdays_log_gk_obs', **evaluate(lin, top3_parkinson_log_har_d_news_thursdays_log_gk_obs, 'target_log_ratio', train, test)})
rows.append({'config': 'LinReg top3_parkinson_log_har_d_news_thursdays_log_rs_neg_log_rs_pos', **evaluate(lin, top3_parkinson_log_har_d_news_thursdays_log_rs_neg_log_rs_pos, 'target_log_ratio', train, test)})
rows.append({'config': 'LinReg combined', **evaluate(lin, top3_parkinson_combined, 'target_log_ratio', train, test)})


# rows.append({'config': 'RF top3', **evaluate(rf, top3, 'target_log_ratio', train, test)})
# rows.append({'config': 'RF top3_parkinson', **evaluate(rf, top3_parkinson, 'target_log_ratio', train, test)})
# rows.append({'config': 'RF top3_parkinson_log_hars', **evaluate(rf, top3_parkinson_log_hars, 'target_log_ratio', train, test)})
# rows.append({'config': 'RF top3_parkinson_log_hars_news_thursdays', **evaluate(rf, top3_parkinson_log_hars_news_thursdays, 'target_log_ratio', train, test)})

results = pd.DataFrame(rows)
results.head(15)



#Final results look promising - as expected now, the best eprfoming model is the LinReg with all the recently added features
#and all additions, from parkinson, log_hars and news/thursdays seem to do the trick.
#In the end I get a model with 0.3MAE, 0.62 test R2 and 0.48 skill, which seems pretty solid, and that's not even all
#we can possibly have, as I have more ideas to test, but it's certainly worth implementing

is_nfp
0    1.488250
1    2.164436
Name: target_log_ratio, dtype: float64
is_cpi
0    1.485807
1    2.183549
Name: target_log_ratio, dtype: float64
is_thursday
0    1.488173
1    1.637220
Name: target_log_ratio, dtype: float64


,config,train_mae,test_mae,train_r2,test_r2,skill
0,LinReg top3_parkinson_log_har_d_news_thursdays,0.330890,0.301960,0.638734,0.626880,0.490545
1,LinReg top3_parkinson_log_har_d_news_thursdays...,0.330484,0.303100,0.639117,0.627098,0.488622
2,LinReg top3_parkinson_log_har_d_news_thursdays...,0.328784,0.303707,0.641363,0.626618,0.487597
3,LinReg combined,0.328556,0.304626,0.641849,0.627240,0.486047


In [3]:
#W11 D5 T4
'''**Concept - gradient boosting.** RF builds many trees independently and averages them.
Boosting builds trees one after another - each new tree is trained on the errors left by the
trees before it, so it focuses on what's still wrong. Small steps (`learning_rate`) and small
trees (`num_leaves`) keep it from memorising the training data. LightGBM is a fast library
for it, and `LGBMRegressor` works with `.fit()` / `.predict()` like any sklearn model - so
your `evaluate()` takes it without changes.

```python
# pip install lightgbm   (in the venv, once)
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=300,       # number of trees, built one after another
    learning_rate=0.03,     # each tree corrects only 3% of the remaining error - slow and safe
    num_leaves=15,          # small trees
    min_child_samples=20,   # a leaf needs at least 20 days - against memorising single days
    random_state=42,
    verbose=-1,             # silence training logs
)
```

- 3a. Add LightGBM rows for `TOP3` and `FULL` to the Task 2 table.
- 3b. Ablation on the best model + `FULL`: drop each feature once, record test MAE. Loop
  over `FULL`; in each pass make a copy (`reduced = FULL.copy()`), `reduced.remove(feature)`,
  run `evaluate()` on `reduced`, append `{'dropped': feature, **result}` to a list ->
  DataFrame. Any feature whose removal makes test MAE better or equal goes out.
- 3c. Final benchmark table: 3 models (linear, RF, LightGBM) x 2 feature sets (`TOP3`, final
  set after 3b).

'''
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=300,       # number of trees, built one after another
    learning_rate=0.03,     # each tree corrects only 3% of the remaining error - slow and safe
    num_leaves=15,          # small trees
    min_child_samples=20,   # a leaf needs at least 20 days - against memorising single days
    random_state=42,
    verbose=-1,             # silence training logs
)




rows = []
rows.append({'config': 'LinReg top3', **evaluate(lin, top3, 'target_log_ratio', train, test)})
rows.append({'config': 'LinReg top3_parkinson', **evaluate(lin, top3_parkinson, 'target_log_ratio', train, test)})
rows.append({'config': 'LinReg top3_parkinson_log_hars', **evaluate(lin, top3_parkinson_log_hars, 'target_log_ratio', train, test)})
rows.append({'config': 'LinReg top3_parkinson_log_hars_news_thursdays', **evaluate(lin, top3_parkinson_log_hars_news_thursdays, 'target_log_ratio', train, test)})


rows.append({'config': 'RF top3', **evaluate(rf, top3, 'target_log_ratio', train, test)})
rows.append({'config': 'RF top3_parkinson', **evaluate(rf, top3_parkinson, 'target_log_ratio', train, test)})
rows.append({'config': 'RF top3_parkinson_log_hars', **evaluate(rf, top3_parkinson_log_hars, 'target_log_ratio', train, test)})
rows.append({'config': 'RF top3_parkinson_log_hars_news_thursdays', **evaluate(rf, top3_parkinson_log_hars_news_thursdays, 'target_log_ratio', train, test)})


rows.append({'config': 'LGBM top3', **evaluate(lgbm, top3, 'target_log_ratio', train, test)})
rows.append({'config': 'LGBM top3_parkinson', **evaluate(lgbm, top3_parkinson, 'target_log_ratio', train, test)})
rows.append({'config': 'LGBM top3_parkinson_log_hars', **evaluate(lgbm, top3_parkinson_log_hars, 'target_log_ratio', train, test)})
rows.append({'config': 'LGBM top3_parkinson_log_hars_news_thursdays', **evaluate(lgbm, top3_parkinson_log_hars_news_thursdays, 'target_log_ratio', train, test)})


results = pd.DataFrame(rows)
results.head(16)

'''8	LGBM top3	0.305523	0.394380	0.686611	0.424098	0.334618
9	LGBM top3_parkinson	0.279724	0.366479	0.741313	0.482576	0.381692
10	LGBM top3_parkinson_log_hars	0.246920	0.362336	0.797210	0.495113	0.388680
11	LGBM top3_parkinson_log_hars_news_thursdays	0.219492	0.344038	0.840338	0.563065	0.419553'''
#LGBM doesn't get close to our LinearRegression, but we didn't use any parameters search etc., so I'm wondering if that would do the trick
#All the LBGM istances seem to be overfit, as the train scores are way better and frankly better than any other models we've tested, so I guess
#this is also a matter of bad model prepartion

#Imo we have to test this out properly with proper hyperparams and general best overfitting-avoiding pracitces
#before rejecting LGBM

#That's a good task for next week.

#For now, LinReg with har_d + parkinson + news/thursdays is the way to go for now.


'8\tLGBM top3\t0.305523\t0.394380\t0.686611\t0.424098\t0.334618\n9\tLGBM top3_parkinson\t0.279724\t0.366479\t0.741313\t0.482576\t0.381692\n10\tLGBM top3_parkinson_log_hars\t0.246920\t0.362336\t0.797210\t0.495113\t0.388680\n11\tLGBM top3_parkinson_log_hars_news_thursdays\t0.219492\t0.344038\t0.840338\t0.563065\t0.419553'

In [4]:
#W12 D1 WARMUP

'''results = pd.DataFrame({
    'config': ['lin_final', 'lin_top3', 'rf_final', 'rf_top3'],
    'model': ['lin', 'lin', 'rf', 'rf'],
    'test_mae': [0.302, 0.365, 0.343, 0.371],
})
```

- W1 (no-brainer). One dict `{config: test_mae}` - `zip` the two columns.
- W2. Same dict, only rows with test_mae < 0.35.
- W3. `groupby('model')['test_mae'].min()` - best MAE per model. Then the same result as a
  dict with `.to_dict()`.'''
  
results = pd.DataFrame({
    'config': ['lin_final', 'lin_top3', 'rf_final', 'rf_top3'],
    'model': ['lin', 'lin', 'rf', 'rf'],
    'test_mae': [0.302, 0.365, 0.343, 0.371],
})
  
  
configs_mae = [{config : mae} for config, mae in zip(results['config'], results['test_mae'])]
print(configs_mae)


configs_mae_threshold = [{config : mae} for config, mae in zip(results['config'], results['test_mae'])
                         if mae < 0.35]
print(configs_mae_threshold)

grouped_top_models = results.groupby('model')['test_mae'].min().to_dict()
print(grouped_top_models)



[{'lin_final': 0.302}, {'lin_top3': 0.365}, {'rf_final': 0.343}, {'rf_top3': 0.371}]
[{'lin_final': 0.302}, {'rf_final': 0.343}]
{'lin': 0.302, 'rf': 0.343}


In [5]:
#W12 D1 T1

'''**Concept - Garman-Klass (1980).** Parkinson uses only high and low. Garman-Klass also uses
open and close, so it sees both the swing inside the bar and where the bar ended up. Per bar,
in steps:

```python
hl = np.log(high / low) ** 2              # squared high-low swing (same piece as Parkinson)
co = np.log(close / open) ** 2            # squared open-to-close move
gk = 0.5 * hl - (2 * np.log(2) - 1) * co  # 0.5 * swing, minus 0.386 * open-close move
```

The minus part is a correction: part of the swing is already "explained" by the open-to-close
move, so it's removed to cut noise. Window value = sum over the 3 observation bars, then log.

Watch out: because of the minus, a single bar can come out slightly negative. After summing
3 bars it's almost never a problem, but check `.min() > 0` before `np.log()`.

**Concept - semivariance: "bad" vs "good" volatility.** Realized variance squares every
return. Semivariance splits it in two: squared returns of falling bars only (RS-) and of
rising bars only (RS+). Literature: falling-market volatility predicts future volatility
better. Our `sjv_10` was a ratio of these two and did nothing - here we test the two LEVELS
separately, over a longer window.

```python
r = pd.Series([0.01, -0.02, 0.005, -0.01])
rs_neg = (r.where(r < 0, 0) ** 2).sum()  # where(): keep negatives, others -> 0. = 0.0004 + 0.0001
rs_pos = (r.where(r > 0, 0) ** 2).sum()  # = 0.0001 + 0.000025
```

- 1a. GK: on `m5_df` add `gk_bar` (the 3 steps above, per row), add `gk_obs = ('gk_bar', 'sum')`
  to the observation aggregation, then `log_gk_obs` in `merged_df`. Check `.corr()` with
  `log_parkinson_obs`.
- 1b. Semivariance over the hour before the forecast - bars from 7:15 to 8:10 (12 bars,
  7:15 <= time < 8:15). Build a mask `is_pre` (hour/minute, like `is_obs`), compute per
  bar `ret_1bar.where(...) ** 2` for both signs, sum per `trade_date` on `is_pre` rows,
  merge, log -> `log_rs_neg`, `log_rs_pos`. Check `.min() > 0` before the log.
- 1c. Linear only, one table: `FINAL`, `FINAL + log_gk_obs`, `FINAL + log_rs_neg + log_rs_pos`,
  `FINAL + all three`.

Live check for later: both need only today's bars from 7:15 - one API call. Fine.'''



# hl = np.log(high / low) ** 2              # squared high-low swing (same piece as Parkinson)
# co = np.log(close / open) ** 2            # squared open-to-close move
# gk = 0.5 * hl - (2 * np.log(2) - 1) * co  # 0.5 * swing, minus 0.386 * open-close move


# m5_df['high_low_swing_squared'] = np.log(m5_df['high'] / m5_df['low']) ** 2
# m5_df['open_to_close_squared'] = np.log(m5_df['close'] / m5_df['open']) ** 2
# m5_df['garman-klass'] = 0.5 * m5_df['high_low_swing_squared'] - (2 * np.log(2) - 1) * m5_df['open_to_close_squared']

# m5_df.head()


##FEATURES ADDED IN THE LAUNCHING SECTION ABOVE, AS IT WOULD MAKE NO POINT TO EXECUTE THE CODE HERE :))

'**Concept - Garman-Klass (1980).** Parkinson uses only high and low. Garman-Klass also uses\nopen and close, so it sees both the swing inside the bar and where the bar ended up. Per bar,\nin steps:\n\n```python\nhl = np.log(high / low) ** 2              # squared high-low swing (same piece as Parkinson)\nco = np.log(close / open) ** 2            # squared open-to-close move\ngk = 0.5 * hl - (2 * np.log(2) - 1) * co  # 0.5 * swing, minus 0.386 * open-close move\n```\n\nThe minus part is a correction: part of the swing is already "explained" by the open-to-close\nmove, so it\'s removed to cut noise. Window value = sum over the 3 observation bars, then log.\n\nWatch out: because of the minus, a single bar can come out slightly negative. After summing\n3 bars it\'s almost never a problem, but check `.min() > 0` before `np.log()`.\n\n**Concept - semivariance: "bad" vs "good" volatility.** Realized variance squares every\nreturn. Semivariance splits it in two: squared returns of falling bar

In [6]:
#W12 D1 T2

'''# Task 2 - Lock the feature set with time-series cross-validation (25 min)

**Concept - why one split is not enough.** All week, decisions came from one 80/20 split -
~280 test days. A difference like 0.004 MAE (what `har_d` gave) can be luck: which days
happened to land in the test set. Time-series cross-validation repeats the test on several
consecutive periods, always training on the past and testing on the next block:

```
split 1: train [####      ] test [##]
split 2: train [######    ] test [##]
split 3: train [########  ] test [##]
```

```python
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

tscv = TimeSeriesSplit(n_splits=5)           # 5 consecutive test blocks, never shuffled
scores = cross_val_score(
    LinearRegression(), merged_df[FINAL], merged_df['target_log_ratio'],
    cv=tscv, scoring='neg_mean_absolute_error',
)
# sklearn always "maximises" a score, so it returns MAE with a minus sign: [-0.31, -0.29, ...]
mae_per_split = -scores
mae_per_split.mean(), mae_per_split.std()   # average MAE, and how much it jumps between periods
```

`merged_df` must be sorted by `trade_date` (it is, if it came out of the groupby).

- 2a. Ablation with CV: for every feature in `FINAL` (+ any Task 1 winner), drop it, run the
  CV above, store `{'dropped': feature, 'cv_mae': mean, 'cv_std': std}`. Plus one row for the
  full set. -> DataFrame, sorted by `cv_mae`.
- 2b. Decision rule: a feature stays if dropping it makes `cv_mae` worse. If dropping it
  changes nothing (difference much smaller than `cv_std`), it goes - fewer features = less
  to compute live.
- 2c. Name the final list `FINAL_V2`.

**In a comment:** did `har_d` survive CV? Did the Task 1 features?'''





from sklearn.model_selection import TimeSeriesSplit, cross_val_score

final_features = ['log_power_obs', 
                                          'log_atr14_obs', 
                                          'center_of_mass_mean_observation', 
                                          'log_parkinson_obs', 
                                          'log_har_d', 
                                          'is_cpi',
                                          'is_nfp',
                                          'is_thursday'
                                          ]

tscv = TimeSeriesSplit(n_splits=5)           # 5 consecutive test blocks, never shuffled
scores = cross_val_score(
    LinearRegression(), merged_df[final_features], merged_df['target_log_ratio'],
    cv=tscv, scoring='neg_mean_absolute_error',
)


print(scores)
#[-0.31730564 -0.36518613 -0.35450708 -0.36297316 -0.31037483]

# sklearn always "maximises" a score, so it returns MAE with a minus sign: [-0.31, -0.29, ...]
mae_per_split = -scores
mae_per_split.mean(), mae_per_split.std()   # average MAE, and how much it jumps between periods
#(np.float64(0.3420693697087648), np.float64(0.023425779998606875))

#permutation importance test


model = LinearRegression()
model.fit(train[final_features], train['target_log_ratio'])


from sklearn.inspection import permutation_importance
result = permutation_importance(
    model, test[final_features], test['target_log_ratio'],
    scoring = 'neg_mean_absolute_error', n_repeats = 20, random_state = 42
)


print(pd.DataFrame({'feature': [feature for feature in final_features],
    'importance_mean': [result for result in result.importances_mean]}))

'''                           feature  importance_mean
0                    log_power_obs         0.538727
1                    log_atr14_obs         0.000571
2  center_of_mass_mean_observation         0.003188
3                log_parkinson_obs         0.143568
4                        log_har_d         0.006756
5                           is_cpi         0.026148
6                           is_nfp         0.034553
7                      is_thursday         0.007188'''


###ablation:

numer = 0
rows = []
rows.append(
    {
    'missing_feature': None,
    'train_mae': 0.330890,
    'test_mae': 0.301960,
    'train_r2': 0.638734,
    'test_r2': 0.626880,
    'skill': 0.490545
})
for feature in final_features:
    numer += 1
    updated_list = final_features.copy()
    updated_list.remove(feature)
    print(updated_list)
    results = evaluate(model, 
            features = updated_list, 
            target = 'target_log_ratio', 
            train = train, 
            test = test)
    rows.append({
        'missing_feature': feature,
        'train_mae': results['train_mae'],
        'test_mae': results['test_mae'],
        'train_r2': results['train_r2'],
        'test_r2': results['test_r2'],
        'skill': results['skill']
    })

    # print(f'Ablation {numer} - Results without {feature}: {results}')
    
    
    '''
missing_feature	train_mae	test_mae	train_r2	test_r2	skill
0	log_power_obs	0.425851	0.397011	0.370958	0.394468	0.330178
1	log_atr14_obs	0.330887	0.302171	0.638729	0.626688	0.490190
2	center_of_mass_mean_observation	0.331379	0.303236	0.638074	0.621891	0.488393
3	log_parkinson_obs	0.339320	0.324569	0.613984	0.567171	0.452400
4	log_har_d	0.332492	0.303747	0.634702	0.626590	0.487529
5	is_cpi	0.346721	0.315740	0.593330	0.590884	0.467296
6	is_nfp	0.349205	0.321152	0.594349	0.569463	0.458164
7	is_thursday	0.333929	0.304937	0.631069	0.625092	0.485523'''

#I'd say THERE are some features that seem to have marginal results - log_atr14_obs seems to have the most neglibible effect.
#Yet it isn't the case that the result is BETTER without it, which makes me think MAYBE I should leave it for now.
#I'm not entirely sure about this decision, yet I'll stick with it for now.

    
ablation_results = pd.DataFrame(rows)
ablation_results.head(20)


[-0.31730564 -0.36518613 -0.35450708 -0.36297316 -0.31037483]
                           feature  importance_mean
0                    log_power_obs         0.538727
1                    log_atr14_obs         0.000571
2  center_of_mass_mean_observation         0.003188
3                log_parkinson_obs         0.143568
4                        log_har_d         0.006756
5                           is_cpi         0.026148
6                           is_nfp         0.034553
7                      is_thursday         0.007188
['log_atr14_obs', 'center_of_mass_mean_observation', 'log_parkinson_obs', 'log_har_d', 'is_cpi', 'is_nfp', 'is_thursday']
['log_power_obs', 'center_of_mass_mean_observation', 'log_parkinson_obs', 'log_har_d', 'is_cpi', 'is_nfp', 'is_thursday']
['log_power_obs', 'log_atr14_obs', 'log_parkinson_obs', 'log_har_d', 'is_cpi', 'is_nfp', 'is_thursday']
['log_power_obs', 'log_atr14_obs', 'center_of_mass_mean_observation', 'log_har_d', 'is_cpi', 'is_nfp', 'is_thursday']
['lo

,missing_feature,train_mae,test_mae,train_r2,test_r2,skill
0,NaN,0.330890,0.301960,0.638734,0.626880,0.490545
1,log_power_obs,0.425851,0.397011,0.370958,0.394468,0.330178
2,log_atr14_obs,0.330887,0.302171,0.638729,0.626688,0.490190
3,center_of_mass_mean_observation,0.331379,0.303236,0.638074,0.621891,0.488393
4,log_parkinson_obs,0.339320,0.324569,0.613984,0.567171,0.452400
5,log_har_d,0.332492,0.303747,0.634702,0.626590,0.487529
6,is_cpi,0.346721,0.315740,0.593330,0.590884,0.467296
7,is_nfp,0.349205,0.321152,0.594349,0.569463,0.458164
8,is_thursday,0.333929,0.304937,0.631069,0.625092,0.485523


In [ ]:
'''## Task 3 - Train the final model, save it, recompute the range constants (25 min)

The API turns the model's prediction into a price range with two constants (`k_p75`,
`k_p90`). They were computed on the old, wrong-clock data, so they need recomputing too.

- 3a. Fit `LinearRegression()` on `train[FINAL_V2]`, evaluate with `evaluate()`. Log to MLflow,
  same pattern as before:
  ```python
  import mlflow
  mlflow.set_tracking_uri('sqlite:///mlflow.db')
  mlflow.set_experiment('xauusd_volatility_windows')
  with mlflow.start_run(run_name='w12d1_linreg_final_v2'):
      mlflow.log_param('features', ','.join(FINAL_V2))   # a list must be a string for MLflow
      mlflow.log_metric('test_mae', ...)                 # from your evaluate() dict
      mlflow.log_metric('test_r2', ...)
      mlflow.sklearn.log_model(model, 'model')
      run_id = mlflow.active_run().info.run_id
  print(run_id)
  ```
  Write the `run_id` down in a comment - the API needs it next session.
- 3b. Range constants. Per `trade_date`, on `is_pred` rows: `high.max() - low.min()` =
  `range_points_predicted` (how many points price actually travelled 8:15-9:00). Merge into
  `merged_df`. Then:
  - `range_to_power_ratio = range_points_predicted / realized_power_predicted`
    (how many points of range one unit of power turns into)
  - `k_p75 = ratio.quantile(0.75)`, `k_p90 = ratio.quantile(0.90)`
  Old values: 2079.53 / 2961.72. Write the new ones in a comment.
- 3c. Sanity check on one test day: `predicted_power = realized_power_observation * np.exp(prediction)`,
  `typical range = predicted_power * k_p75`. Compare with that day's real `range_points_predicted`.
  Same order of magnitude?'''



  
  
  
final_model = LinearRegression()
final_features = [
                  'log_power_obs', 
                  'log_atr14_obs', 
                  'center_of_mass_mean_observation', 
                  'log_parkinson_obs', 
                  'log_har_d', 
                  'is_cpi',
                  'is_nfp',
                  'is_thursday'
                                          ]
final_model.fit(train[final_features], train['target_log_ratio'])
metrics = evaluate(final_model, final_features, 'target_log_ratio', train, test) 

# import mlflow
# mlflow.set_tracking_uri('sqlite:///mlflow.db')
# mlflow.set_experiment('xauusd_volatility_windows')
# with mlflow.start_run(run_name='w12d1_linreg_final_v2'):
#     mlflow.log_param('features', ','.join(final_features))   
#     mlflow.log_metric('test_mae', metrics['test_mae'])               
#     mlflow.log_metric('test_r2', metrics['test_r2'])
#     mlflow.sklearn.log_model(final_model, 'model')
    #run_id = mlflow.active_run().info.run_id
#print(run_id) #d068c9f054b04563b8c0e8765ffef091 - latest run_id


k_p75, k_p90 = 2138.6352561046688, 3063.5539822916144
print(k_p75, k_p90)

c:\Users\HARDPC\Desktop\AL\projekty\python_agentic_learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/09 20:52:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


d068c9f054b04563b8c0e8765ffef091
2138.6352561046688 3063.5539822916144


In [8]:
day = test.iloc[[0]]                                   
pred = final_model.predict(day[final_features])[0]                
predicted_power = day['realized_power_observation'].iloc[0] * np.exp(pred)
print(predicted_power * k_p75, day['range_points_predicted'].iloc[0])


10.088882207743469 8.039999999999964


In [5]:
#W12 D3 WARMUP

values = [
    {'datetime': '2026-11-10 08:10:00', 'close': '4075.78'},
    {'datetime': '2026-11-10 08:05:00', 'close': '4074.10'},
    {'datetime': '2026-11-10 08:00:00', 'close': '4076.55'},
]
# ```

# - W1 (no-brainer). List of closes as floats: `[4075.78, 4074.10, 4076.55]`.
# - W2. ONE dict `{datetime: close_as_float}`. Reminder: outer `{}` + `key: value`, no inner `{}`.
# - W3. Same dict, but sorted oldest first - `sorted(values, key=lambda v: v['datetime'])` in the
#   `for` part.


closes = [value['close'] for value in values]
print(closes)
date_close = {value['datetime'] : value['close'] for value in values}
print(date_close)
date_close = {value['datetime'] : value['close'] for value in sorted(values, key = lambda v: v['datetime'])}
print(date_close)

['4075.78', '4074.10', '4076.55']
{'2026-11-10 08:10:00': '4075.78', '2026-11-10 08:05:00': '4074.10', '2026-11-10 08:00:00': '4076.55'}
{'2026-11-10 08:00:00': '4076.55', '2026-11-10 08:05:00': '4074.10', '2026-11-10 08:10:00': '4075.78'}


In [29]:
#W12 D3 T1

'''## Task 1 - Retry with backoff in `get_recent_bars` (20 min)

**Concept - which failures are worth retrying.** An API call can fail in two kinds of ways:

- **Temporary** - the server is busy or the network blinked: HTTP 429 (too many requests),
  500/502/503/504 (server-side trouble), timeouts, connection errors. Trying again a bit later
  often works. This is exactly what hit your Worker this week (429 from in front of Render).
- **Permanent** - your request is wrong: 400 (bad parameters), 401/403 (bad key). Retrying
  just repeats the same error, so fail at once with a clear message.

**Backoff** = wait longer after each failed attempt (2 s, 4 s, 8 s...), so you don't hammer a
server that's already struggling. "Exponential" = the wait doubles each time.

```python
import time
import requests

RETRYABLE = {429, 500, 502, 503, 504}

def get_with_retry(url, params, max_attempts=4):
    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(url, params=params, timeout=10)  # timeout: never hang forever
        except (requests.ConnectionError, requests.Timeout) as e:
            if attempt == max_attempts:
                raise RuntimeError(f'Network error after {max_attempts} attempts: {e}')
        else:
            if response.status_code not in RETRYABLE:
                return response                     # success, or a permanent error - caller decides
            if attempt == max_attempts:
                return response                     # out of attempts, hand back the last answer
        time.sleep(2 ** attempt)                    # 2, 4, 8 s
```

Read it step by step before using it:
- `try / except / else`: `except` runs when `requests.get` raised (no response at all),
  `else` runs when it didn't (we got a response, maybe an error code).
- `2 ** attempt` - 2 to the power of the attempt number: 2, 4, 8 seconds.

- 1a. In this repo's `live_data.py`, make `get_recent_bars` use it (keep your existing checks -
  `status_code != 200` and `status == 'error'` - they now run on the response it returns).
- 1b. Test the permanent path: call it with a wrong API key. It must fail at once (no waiting).
- 1c. **In a comment, one sentence each:** why is 401 not retried, and why is 429 retried?

Twelve Data detail worth knowing: when you hit their limit they often reply with HTTP 200
and `{"status": "error", "code": 429}` in the body - so your `status == 'error'` check is the
one that catches it, not the status code. (Retrying that case is optional.)'''


'''- **Temporary** - the server is busy or the network blinked: HTTP 429 (too many requests),
  500/502/503/504 (server-side trouble), timeouts, connection errors. Trying again a bit later
  often works. This is exactly what hit your Worker this week (429 from in front of Render).
- **Permanent** - your request is wrong: 400 (bad parameters), 401/403 (bad key). Retrying
  just repeats the same error, so fail at once with a clear message.'''



import requests
import os
from dotenv import load_dotenv
import pandas as pd
import time

retried = {429, 500, 502, 503, 504}
API_KEY = os.getenv('TWELVEDATA_API_KEY')

def get_candle_data(url, params, max_attempts = 4):
  '''
  Function used to fetch candle data with HTTP request
  '''
  load_dotenv()
  for attempt in range(1, max_attempts + 1):
    try:
      response = requests.get(
        url = url,
        params = {
          **params,
          'apikey': API_KEY}
      , timeout = 10)
      if attempt == max_attempts:
        print(f'Max attempts reached, data fetch failed')
        break
    except response.status_code not in (retried):
      print(f'Request error {response.status_code}: {response.reason}')
      break
    except response.status_code in retried:
      time.sleep(2 ** attempt)
    else:
      data = response.json()
      data = pd.DataFrame(data['values'])
      return data
      
  
  
  
xd = get_candle_data(url = 'https://api.twelvedata.com/time_series',
                     params = {
                       'symbol': 'AAPL',
                       'interval': '5min'
                     })
print(xd)


               datetime        open       high         low      close   volume
0   2026-11-10 15:55:00   349.42001  349.48001   348.47000  348.63000  1261208
1   2026-11-10 15:50:00   349.60001        350   349.22000  349.39001   573210
2   2026-11-10 15:45:00   349.26999  349.66000   349.10999  349.64499   290793
3   2026-11-10 15:40:00   349.12000  349.45999  349.019989  349.26709   237683
4   2026-11-10 15:35:00      348.75  349.20001   348.66000  349.10001   269586
5   2026-11-10 15:30:00   348.72000  348.79999   348.60001  348.73010   211016
6   2026-11-10 15:25:00   348.73001  348.89999       348.6  348.67990   241129
7   2026-11-10 15:20:00   348.12000     348.75     347.985  348.73001   225185
8   2026-11-10 15:15:00   348.23999  348.26001      347.94  348.11841   189018
9   2026-11-10 15:10:00   348.32001  348.39801   348.20999     348.25   147931
10  2026-11-10 15:05:00   348.19000  348.66000   348.17001  348.32999   147552
11  2026-11-10 15:00:00   348.38000  348.38000      

In [7]:
#W12 D4 WARMUP

bars = [
    {'time': '08:00', 'high': '4081.20', 'low': '4074.90'},
    {'time': '08:05', 'high': '4079.55', 'low': '4075.10'},
    {'time': '08:10', 'high': '4083.00', 'low': '4077.25'},
]
'''```

- W1. One dict `{time: high_as_float}`.
- W2. One dict `{time: bar_range}` where range = high - low, both converted with `float()`,
  rounded to 2 places.
- W3. Same as W2, only bars with range > 5.'''


time_highs = {bar['time']: bar['high'] for bar in bars}
print(time_highs)


time_bar_range = {bar['time']: round(float(bar['high']) - float(bar['low']), 2) for bar in bars}
print(time_bar_range)

time_bar_range_threshold = {bar['time']: round(float(bar['high']) - float(bar['low']), 2) for bar in bars if round(float(bar['high']) - float(bar['low']), 2) > 5}
print(time_bar_range_threshold)


{'08:00': '4081.20', '08:05': '4079.55', '08:10': '4083.00'}
{'08:00': 6.3, '08:05': 4.45, '08:10': 5.75}
{'08:00': 6.3, '08:10': 5.75}


In [ ]:
#W12 D4 T1


import time
import requests

RETRYABLE = {429, 500, 502, 503, 504}

def get_with_retry(url, params, max_attempts=4):
    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(url, params=params, timeout=10)
        except (requests.ConnectionError, requests.Timeout) as e:   # no response at all
            if attempt == max_attempts:
                raise RuntimeError(f'Network error after {max_attempts} attempts: {e}')
        else:                                                        # we got a response
            if response.status_code not in RETRYABLE:
                return response          # 200, or a permanent error like 401 - caller decides
            if attempt == max_attempts:
                return response          # still 429/5xx after all attempts
        time.sleep(2 ** attempt)         # 2, 4, 8 s
        



In [25]:
#W12 D4 T2
'''## Task 2 - Gold's volatility profile by hour of day (15 min)

Data: `xauusd_m5_et.csv` (5-min bars, real New York time since last week's clock fix).

- 2a. Load with `parse_dates=['et_time']`, `set_index('et_time')`.
- 2b. Build hourly candles with `resample('1h').agg({...})`: open = `'first'`, high = `'max'`,
  low = `'min'`, close = `'last'`. Then `.dropna()` - hours when the market is closed come out
  as empty rows.
- 2c. `range = high - low` per hourly candle. Then `groupby(df.index.hour)['range'].mean()` -
  the average hourly range for each hour of the day (0-23).
- 2d. Sort descending and look at the top 3 hours.

**In a comment:** which hours are the most volatile, and does that match what you know
happens then (8:30 US data releases, 9:30 US stock market open, 10:00 more data releases)?
If the top hours were 7 or 6, the clock would still be broken.
'''



import pandas as pd

xau_data = pd.read_csv('xauusd_m5_et.csv', parse_dates = ['et_time'])
xau_data = xau_data.set_index('et_time')
xau_data.head()

hourly_candles = xau_data.resample('1h').agg(
    open = ('open', 'first'),
    low = ('low', 'min'),
    high = ('high', 'max'),
    close = ('close', 'last')
).reset_index()

hourly_candles.head()
hourly_candles = hourly_candles.dropna()
hourly_candles['range'] = hourly_candles['high'] - hourly_candles['low']
hourly_candles['hour'] = hourly_candles['et_time'].dt.hour

hourly_range_means = hourly_candles.groupby('hour')['range'].mean().reset_index()
hourly_range_means = hourly_range_means.sort_values(by = 'range', ascending = False)
hourly_range_means.head(3)

,hour,range
9,9,13.911660
10,10,13.807344
8,8,12.748737
